# Manga Translator — Jepang → semua bahasa

Terjemahan otomatis halaman manga di Google Colab.
Pilih bahasa tujuan di dropdown **"Bahasa terjemahan"** (default: English).

**Penyedia terjemahan** dipilih di UI:

| Penyedia | Kecepatan | Tahu ukuran balon? |
|---|---|---|
| **LLM (freetokenfaucet)** ← default | 5–8 dtk/halaman (terukur) | **YA** — anggaran balon aktif |
| DeepL | ~2 dtk/halaman | tidak |
| Router LLM (gorouter) | ~8 dtk/halaman (terukur) | ya — tapi memakai kredit berbayar |

Hanya penyedia **LLM** yang bisa diberi tahu ukuran tiap balon, dan itulah yang
memenuhi syarat **NO KELUAR BUBBLE**: teksnya dibuat pendek sejak di sumber,
bukan dikecilkan fontnya di typeset. Model faucet (`mimo-v2.5-pro`) **wajib
model GRATIS**: terukur 17 Agu 2026, 16 dari 19 model faucet membalas HTTP 402
`INSUFFICIENT_BALANCE` — termasuk model yang dulu jadi default, dan itulah
yang membuat tiga halaman keluar TANPA terjemahan. `thinking` tetap
**dimatikan** — tanpa itu jatah keluaran habis untuk berpikir dan jawabannya
keluar kosong. Tiap halaman dikirim sekali dalam satu panggilan semua balon.

API key ditaruh di **Colab Secrets** (`FAUCET_API_KEY`, `DEEPL_API_KEY`, atau
`ROUTER_API_KEY`) atau ditempel di field **API key** pada UI — jangan pernah
ditulis di dalam sel.

**SFX dibiarkan utuh** (deteksi onomatope diperkuat: kana+simbol seperti
`フー．．．`, ulangan `ドキドキドキ`, SFX dalam balon `ドキッ` — sedangkan kata
pinjaman seperti サッカー/メール tetap dialog). Simbol emosi (♥ ♪ ☆ 〜 ! ?)
dipulihkan/dipertahankan. Teks asli dihapus bersih, terjemahan ditulis
ulang di dalam balon dengan ukuran yang menyesuaikan — tanpa saling timpa
dan tanpa keluar garis balon.

> ⚠️ **FULL CUDA — GPU T4 WAJIB.** Sel 2 berhenti dengan error kalau GPU
> tidak aktif; sel 22 memverifikasi keempat model (detector, CTD, manga-ocr, LaMa)
> benar-benar berjalan di GPU sebelum UI dibuka. Terjemahan sendiri jalan di
> jaringan, bukan GPU.
> Runtime → Change runtime type → Hardware accelerator → **T4 GPU**


In [1]:
# Sel 2 — cek GPU dan set cache SEBELUM import apa pun.
# HF_HOME harus di-set di Python, bukan `!export` — variabel shell tidak
# propagate ke proses notebook, itu penyebab paling umum caching terasa "sudah
# diperbaiki" padahal belum.
import os, subprocess, sys, pathlib

os.environ["HF_HOME"] = "/content/work/hf"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["MANGATL_ROOT"] = "/content/mangatl"
os.environ["MANGATL_WORK"] = "/content/work"

for d in ("/content/mangatl", "/content/work/hf", "/content/work/weights",
          "/content/work/fonts", "/content/work/output", "/content/work/debug"):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])

# GPU WAJIB. Satu halaman menjalankan 4 model (detector, CTD, manga-ocr, LaMa);
# di CPU itu ~5 menit per halaman dan batch jadi tidak masuk akal. Lebih baik
# berhenti di sini dengan pesan yang jelas daripada user menunggu lama lalu
# bingung kenapa lambat.
try:
    smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=60,
    )
    gpu = smi.stdout.strip() if smi.returncode == 0 else ""
except (FileNotFoundError, subprocess.SubprocessError):
    gpu = ""

if not gpu:
    raise RuntimeError(
        "GPU tidak terdeteksi - notebook ini WAJIB GPU T4. Perbaiki: "
        "Runtime -> Change runtime type -> Hardware accelerator: GPU (T4)"
        " -> Save, lalu jalankan ulang sel ini."
    )

print("GPU   :", gpu)
if "T4" not in gpu.upper():
    print("        (bukan T4 — tetap jalan; T4 adalah target yang diuji)")



Python: 3.12.13
GPU   : Tesla T4, 15360 MiB


In [2]:
# Sel 3 — install dependency. JANGAN sentuh torch: build Colab sudah jalan di T4.
# manga-ocr pakai --no-deps supaya tidak menarik pin torch/transformers lamanya,
# jadi dependency runtime-nya harus dipasang manual di baris kedua.
# `loguru` WAJIB: manga_ocr/ocr.py meng-import-nya di level modul, jadi tanpa itu
# `from manga_ocr import MangaOcr` gagal dan SELURUH OCR mati diam-diam —
# tiap region jadi UNREADABLE dan halaman keluar tanpa terjemahan sama sekali.
# `fire` dan `pyperclip` sengaja TIDAK dipasang: cuma dipakai CLI-nya, bukan library.
# `pyphen` WAJIB juga: typeset.py memakainya untuk titik penggalan Knuth-Liang.
# Tanpa itu modul turun ke heuristik vokal->konsonan yang menghasilkan penggalan
# tidak layak cetak ('STRA-NGE', 'CONTR-ACTOR') — gagal diam-diam, tidak error.
%pip install -q --no-deps "manga-ocr==0.1.16"
%pip install -q "loguru" "openai>=1.40" "fonttools>=4.53" "pillow-heif>=0.18" "gradio>=6.0" "sentencepiece" "fugashi" "unidic-lite" "jaconv" "transformers>=4.49" "pyphen>=0.15"

# `rar` dipasang dari multiverse Ubuntu. RAR itu format proprietary: stdlib
# Python tidak punya penulis RAR dan `rarfile` cuma bisa MEMBACA, jadi biner
# resminya satu-satunya jalan membuat .rar asli. Kalau apt gagal, UI otomatis
# jatuh ke ZIP — jangan pernah kirim ZIP yang cuma diganti nama jadi .rar,
# WinRAR tidak bisa membukanya.
!apt-get install -qq -y rar 2>/dev/null | tail -1

# ONNX Runtime: JANGAN menurunkan versi tanpa alasan. Downgrade paksa ke 1.22.0
# pernah memicu "TIDAK FULL CUDA" di lingkungan torch 2.11+cu128 (CUDA 12.8):
# build 1.28 bawaan Colab (CUDA 13) tidak cocok dengan torch cu12, dan 1.22.0
# (cu12) ternyata juga bisa gagal memuat CUDA EP. Di sini versi diganti HANYA
# kalau build yang terpasang jelas CPU-only; sel 5 yang menguji dan memilih
# versi yang benar-benar jalan di GPU secara otomatis (ladder).
import subprocess, sys

try:
    import onnxruntime as _ort0
    _has_cuda_ep = "CUDAExecutionProvider" in _ort0.get_available_providers()
except Exception:  # noqa: BLE001 - onnxruntime rusak/tidak lengkap
    _has_cuda_ep = False
if _has_cuda_ep:
    print(f"[ort] {_ort0.__version__} sudah punya CUDA EP - versi tidak dipaksa. "
          "Sel 5 yang memverifikasi.")
else:
    print("[ort] build CPU/rusak - pasang onnxruntime-gpu cu12 (1.22.0).")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y",
                    "onnxruntime", "onnxruntime-gpu"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "onnxruntime-gpu==1.22.0"], check=False)

import importlib, shutil
for mod, why in (("manga_ocr", "OCR tidak akan jalan"),
                 ("pyphen", "penggalan kata turun ke heuristik")):
    if importlib.util.find_spec(mod) is None:
        print(f"[!] {mod} gagal terpasang — {why}.")
if shutil.which("rar") is None:
    print("[!] biner rar tidak ada — unduhan RAR otomatis jatuh ke ZIP.")
print("\nSelesai. Lanjut ke sel 4.")



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 15.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 694.9/694.9 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 73.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
manga-ocr 0.1.16 requires fire, which is not installed.
Processing triggers for man-db (2.10.2-1) ...
[ort] build CPU/rusak - pasang onnxruntime-gpu cu12 (1.22.0).

Selesai. Lanjut ke sel 4.


## ⚠️ Sel 4 — WAJIB restart runtime

Install di atas mengganti beberapa paket yang sudah ter-import Colab.
**Runtime → Restart session** (atau `Ctrl+M .`), lalu **lanjut dari sel 5**.
Jangan ulangi sel 1–3.



In [3]:
# Sel 5 - uji CUDA untuk ONNX di SUBPROSES, lalu daftarkan modul ke sys.path.
#
# Kenapa subproses: cuDNN dengan CUDA major yang tidak cocok membuat onnxruntime
# abort di dalam kode C++-nya saat memilih kernel konvolusi. Abort itu membunuh
# kernel Colab TANPA traceback - keluaran sel berhenti begitu saja di tengah
# jalan. Dijalankan di subproses, kematian yang sama cuma jadi kode keluar
# negatif yang bisa dibaca dan ditindaklanjuti.
#
# Sel ini SELF-HEALING. Lingkungan Colab terbaru membawa torch cu128 (CUDA 12.8)
# dan onnxruntime 1.28 (build CUDA 13) yang TIDAK cocok; downgrade paksa ke
# 1.22.0 (cu12) pun ternyata bisa gagal. Jadi tiap kandidat versi ORT diuji di
# subproses dan versi pertama yang benar-benar menjalankan inference di GPU
# yang dipakai. Model ujinya tertanam (base64) - tidak bergantung pada
# torch.onnx.export yang berubah-ubah antar versi torch.
import base64, glob, os, pathlib, site, subprocess, sys

os.environ.setdefault("HF_HOME", "/content/work/hf")
os.environ.setdefault("MANGATL_ROOT", "/content/mangatl")
os.environ.setdefault("MANGATL_WORK", "/content/work")
for d in ("/content/mangatl", "/content/work/weights", "/content/work/fonts",
          "/content/work/output", "/content/work/debug"):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
if "/content/mangatl" not in sys.path:
    sys.path.insert(0, "/content/mangatl")

import torch

TORCH_MAJOR = (torch.version.cuda or "12").split(".")[0]
PROBE_ONNX = "/content/work/probe_conv.onnx"

# Conv 1-lapis opset 13, dibuat via onnx.helper (BUKAN torch.onnx.export):
# cukup untuk memicu pemilihan kernel cuDNN - titik crash yang dulu membunuh
# kernel. 1069 byte, tervalidasi, kompatibel ORT 1.19-1.28.
PROBE_B64 = "CAg6oggKPAoBeAoBVwoBQhIBeSIEQ29udioVCgxrZXJuZWxfc2hhcGVAA0ADoAEHKhEKBHBhZHNAAUABQAFAAaABBxIKcHJvYmVfY29udirwBggICAMIAwgDEAFCAVdK4AZ4zOE/aOHMPpOOej/Lag9AJAzvP+Iuer//OHM/Yv0avmhk0734OdI+KIATPqIluj9e00I/wDD5PQtC4z5d16o+/D2/PwIVUr5pSqA+BaZavy9kI8CMUyc/sUtdP4f+Pb+pQxFAqCi6v0htOz0grT++HDLEP/MTvD+Kqh4+hZ7BPu1FY7+9iv2/iyGyvvIZID4qep0/leeZP7NPxr5tx5q+/DaGvybDtb8QZ9q/ArP5P5F4Ar9GS+C+mVugv5wJRz80lM6/l9hZvkw9Zb8WGMY+IMQCv/Uel79r3ua8UU7bPi46iD2Z3Zo+72Iivza5ub5eJiy/Whe4vlsqUL/U9ty/M681Pju2zb5XqtC/zPHsPrVEaL+yxFQ9rqU6PxkUBD7i15E/xg6ev7n/zT63Ty+/kOxev34vFL/Qg5++oQ1mPaEjlb+RnGY/T2vuPqKjxL8Mf74/f6zyP0Dilj83Pji+bA6Jv0b4hj81bc6+FHmcPwlGVT4EBXo/qHW2PvvhND9HCCw8aJfkPz71AT6N0c0+FQvxP16DrL9Bn6K/Yip4P+golr+UyPg/3sXTvjNZP7/3IvY/goG9PywM7z+L8mc/SXlcvwJ99D+9N4m+yG1NPxt/cj/3uh6+TjQdP70VbD/ZusA+KrmMv62ymD4Dx6k/M88xv8w5Gb5yzN6+rbTsP4IbLD/WntA+OBlFvzwMCj8RoSy/wmACPc/GIr+7Ki0/dZsTP0VMVb5kwco+cemLv4fhvr/z9+A+dawqPmyRIj9yhRhAaMlxP7iuab9k+o4/p2+ov9FU7L5Bwou90E7bP0GoPr96kVO/eqHJvbfZKb+bNZA/MjuKv0Hgkr/zKeC+HP7+vuj69j8+DXM/EU6zPRLbnL8sKFg/DgeAvw+7xb9cEZg/TUaiPme9az9FMKM+QFlbP52pJr8SYoS/+nwuP0GsTb9WhjC/jjvpvnQwjzyxPrW+Z/6vvy3EJL89TA7AKw8gPzoQzb9vXI2/DatVPQBUPb+BgcU/Vnylv+S6iD4F5yC9FoSVv3b1BT/XqS++EZRFPyvRUj91cgpAWROrP2cFvb7RH3W+pcGMP12/Jz+p3yM/avjOv5VHx7yY7zy/R1KPPhIDyb18AWk/bWqiPspMST98zu6+O8dxvwny0b4qKQgIEAFCAUJKIGVui7wtIMI+hZgQQNYVLb3QuHS/hiSxvnRc7b69hPY+WhsKAXgSFgoUCAESEAoCCAEKAggDCgIIQAoCCEBiGwoBeRIWChQIARIQCgIIAQoCCAgKAghACgIIQEIECgAQDQ=="


def lib_dirs() -> list[str]:
    """Folder lib tiap wheel nvidia-*, dari dist-packages maupun site-packages."""
    roots = set(site.getsitepackages())
    roots |= {p for p in sys.path if p.endswith(("dist-packages", "site-packages"))}
    return sorted({d for r in roots if os.path.isdir(r)
                   for d in glob.glob(os.path.join(r, "nvidia", "*", "lib"))})


def child_env() -> dict:
    """LD_LIBRARY_PATH cuma berguna untuk proses BARU - loader membacanya sekali."""
    env = dict(os.environ)
    env["LD_LIBRARY_PATH"] = os.pathsep.join(
        lib_dirs() + [env.get("LD_LIBRARY_PATH", "")])
    return env


def cudnn_report() -> str:
    """Versi cuDNN yang benar-benar ada di site-packages - penunjuk arah mismatch."""
    found = []
    for r in site.getsitepackages():
        for hit in sorted(glob.glob(os.path.join(r, "nvidia", "cudnn", "lib", "*.so*"))):
            found.append(os.path.basename(hit))
    return ", ".join(found) or "(cuDNN tidak ditemukan di site-packages)"


def ensure_probe() -> bool:
    """Tulis model uji dari base64 tertanam. True kalau file valid."""
    try:
        if not os.path.exists(PROBE_ONNX):
            pathlib.Path(PROBE_ONNX).write_bytes(base64.b64decode(PROBE_B64))
        return os.path.getsize(PROBE_ONNX) > 500
    except Exception:  # noqa: BLE001
        return False


CHILD = (
    "import sys, numpy as np, onnxruntime as ort;"
    "o = ort.SessionOptions(); o.log_severity_level = 1;"
    "s = ort.InferenceSession(sys.argv[1], o, providers=["
    "('CUDAExecutionProvider', {'device_id': 0}), 'CPUExecutionProvider']);"
    "s.run(None, {'x': np.zeros((1, 3, 64, 64), np.float32)});"
    "print('PROVIDERS=' + ','.join(s.get_providers()))"
)


def probe_cuda() -> tuple[str, str]:
    """('ok'|'cpu'|'crash'|'error', detail) - tanpa risiko membunuh kernel."""
    try:
        p = subprocess.run([sys.executable, "-c", CHILD, PROBE_ONNX],
                           env=child_env(), capture_output=True, text=True,
                           timeout=900)
    except subprocess.SubprocessError as exc:
        return "error", str(exc)[:200]
    line = next((l for l in p.stdout.splitlines() if l.startswith("PROVIDERS=")), "")
    if p.returncode < 0:
        return "crash", f"subproses mati sinyal {-p.returncode} - cuDNN/CUDA tak cocok"
    if not line:
        tail = [l for l in p.stderr.splitlines() if l.strip()][-3:]
        return "error", " | ".join(tail)[:400] or f"kode keluar {p.returncode}"
    return ("ok" if "CUDAExecutionProvider" in line else "cpu"), line[10:]


def pip_run(*args: str) -> None:
    subprocess.run([sys.executable, "-m", "pip", *args], capture_output=True,
                   check=False, timeout=600)


# Kandidat versi ORT dicoba berurutan; yang pertama "ok" dipakai.
# 1.23.0 / 1.26.0 / 1.22.0 / 1.21.0 / 1.19.2 = build cu12 (CUDA 12 + cuDNN 9),
# cocok dengan torch cu12/cu128. "1.22.0[cuda,cudnn]" ikut memasang dependency
# nvidia-nya SENDIRI (penyelamat kalau torch cuma membawa cuDNN 10).
# "onnxruntime-gpu" tanpa pin = build terbaru (cu13) - cadangan terakhir.
ORT_CANDIDATES: tuple[str, ...] = (
    "onnxruntime-gpu==1.23.0",
    "onnxruntime-gpu==1.26.0",
    "onnxruntime-gpu==1.22.0[cuda,cudnn]",
    "onnxruntime-gpu==1.22.0",
    "onnxruntime-gpu==1.21.0",
    "onnxruntime-gpu==1.19.2",
    "onnxruntime-gpu",
)


def try_version(spec: str) -> bool:
    """Pasang ORT lalu probe CUDA di subproses. True kalau benar-benar jalan."""
    no_deps = [] if "[cuda,cudnn]" in spec else ["--no-deps"]
    pip_run("install", "-q", "--force-reinstall", *no_deps, spec)
    if not ensure_probe():
        return False
    verdict, detail = probe_cuda()
    print(f"onnx cuda: {spec:<40} {verdict.upper():<6} {detail}")
    return verdict == "ok"


print("torch    :", torch.__version__, "| cuda:", torch.version.cuda,
      "| available:", torch.cuda.is_available())
print("cudnn    :", cudnn_report())

# ORT yang terpasang bisa saja rusak/tidak lengkap (bekas downgrade gagal).
# Kalau import-nya saja gagal, lompat langsung ke ladder - jangan mati di sini.
try:
    import onnxruntime as _ort
    print("ort      :", _ort.__version__, "| device:", _ort.get_device(), "|",
          ", ".join(_ort.get_available_providers()))
    current = f"onnxruntime-gpu=={_ort.__version__}"
except Exception:  # noqa: BLE001 - onnxruntime rusak/tidak lengkap
    _ort, current = None, ""
    print("ort      : TIDAK BISA DI-IMPORT (rusak) - langsung coba versi lain.")

if not ensure_probe():
    raise RuntimeError("Model uji CUDA rusak/tidak bisa ditulis - laporkan sel ini.")

if _ort is not None:
    verdict, detail = probe_cuda()
    print(f"onnx cuda: {'(terpasang)':<40} {verdict.upper():<6} {detail}")
else:
    verdict, detail = "error", "onnxruntime rusak saat import"

chosen = None
if verdict != "ok":
    print("onnx cuda: mencoba versi lain sampai ada yang jalan di GPU ...")
    for spec in ORT_CANDIDATES:
        if spec == current:
            continue  # sudah diuji di atas
        if try_version(spec):
            verdict, detail, chosen = "ok", spec, spec
            break

os.environ["MANGATL_ORT_CUDA"] = verdict

if verdict == "ok":
    if chosen:
        print()
        print(f"CUDA ONNX jalan dengan {chosen} - versi baru tersimpan di disk.")
        print("Runtime -> Restart session, lalu LANJUTKAN dari sel 5 (jangan")
        print("ulangi sel 1-3). Setelah restart, sel ini langsung lolos.")
        print("Tanpa restart, modul ORT lama masih ter-cache - sel 22 akan berhenti lagi.")
    else:
        print()
        print("CUDA ONNX jalan dengan versi yang terpasang. Lanjut ke sel 6.")
else:
    print()
    print("Semua kandidat ORT gagal memuat CUDA EP - ONNX dipaksa ke CPU.")
    print("Laporkan SELURUH keluaran sel ini (terutama baris 'cudnn:' dan")
    print("baris 'onnx cuda:' terakhir - itu isi stderr subproses).")

print("sys.path siap")



torch    : 2.11.0+cu128 | cuda: 12.8 | available: True
cudnn    : libcudnn.so.9, libcudnn_adv.so.9, libcudnn_cnn.so.9, libcudnn_engines_precompiled.so.9, libcudnn_engines_runtime_compiled.so.9, libcudnn_graph.so.9, libcudnn_heuristic.so.9, libcudnn_ops.so.9
ort      : 1.22.0 | device: GPU | TensorrtExecutionProvider, CUDAExecutionProvider, CPUExecutionProvider
onnx cuda: (terpasang)                              OK     CUDAExecutionProvider,CPUExecutionProvider

CUDA ONNX jalan dengan versi yang terpasang. Lanjut ke sel 6.
sys.path siap


## Modul pipeline

Tiap sel di bawah menulis satu file `.py` ke `/content/mangatl/`.
Kode berat sengaja tidak ditaruh langsung di notebook: traceback jadi menunjuk
`detect.py:88`, bukan `<ipython-input-42>`.



In [4]:
%%writefile /content/mangatl/config.py

"""Konfigurasi terpusat: path, konstanta threshold, dan dataclass antar-stage."""

from __future__ import annotations

import os
from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal

import numpy as np

# ---------------------------------------------------------------- paths

ROOT = Path(os.environ.get("MANGATL_ROOT", "/content/mangatl"))
WORK = Path(os.environ.get("MANGATL_WORK", "/content/work"))
WEIGHTS = WORK / "weights"
FONTS = WORK / "fonts"
OUTPUT = WORK / "output"
DEBUG_DIR = WORK / "debug"

for _d in (WORK, WEIGHTS, FONTS, OUTPUT, DEBUG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------- weights

# Tiap weight punya rantai mirror, dicoba berurutan — sama seperti FONT_CHAIN.
# Mirror pertama yang memberi file utuh dipakai. Ini bukan paranoia: URL CTD yang
# beredar di banyak tutorial menunjuk ke repo AnimeMangaInpainting, dan file itu
# TIDAK pernah ada di sana (404) — repo tersebut cuma berisi lama_large_512px.ckpt.
WEIGHT_URLS: dict[str, tuple[str, ...]] = {
    # RT-DETR-v2, 3 kelas: bubble / text_bubble / text_free
    "detector.onnx": (
        "https://huggingface.co/ogkalu/comic-text-and-bubble-detector"
        "/resolve/main/detector.onnx",
    ),
    # comic-text-detector: block head + soft text mask + line head.
    # Rilis GitHub upstream = sumber kanonik (94,669,756 byte, terverifikasi).
    "comictextdetector.pt.onnx": (
        "https://github.com/zyddnys/manga-image-translator/releases/download"
        "/beta-0.3/comictextdetector.pt.onnx",
        "https://huggingface.co/bropines/ballon-translator-models"
        "/resolve/main/models/comictextdetector.pt.onnx",
    ),
    # LaMa large 512px, di-finetune untuk anime/manga
    "lama_large_512px.ckpt": (
        "https://huggingface.co/dreMaz/AnimeMangaInpainting"
        "/resolve/main/lama_large_512px.ckpt",
    ),
}

# ---------------------------------------------------------------- fonts
#
# Anime Ace = face di gambar referensi (ALL CAPS, oblique, huruf I ber-crossbar).
# Lisensi Blambot melarang REDISTRIBUSI, jadi notebook hanya mengunduh saat
# runtime dan tidak pernah membundel .ttf-nya. Kalau URL mati, chain turun
# otomatis ke Comic Neue (OFL) tanpa crash.

FONT_CHAIN: list[dict[str, str]] = [
    {
        "name": "AnimeAce",
        "file": "anime_ace.ttf",
        "url": (
            "https://raw.githubusercontent.com/zyddnys/manga-image-translator"
            "/main/fonts/anime_ace.ttf"
        ),
        "role": "primary",
    },
    {
        "name": "AnimeAce2-Bold",
        "file": "animeace2_bld.ttf",
        "url": (
            "https://static.wfonts.com/download/data/2014/06/26"
            "/anime-ace-2-0-bb/animeace2_bld.ttf"
        ),
        "role": "primary",
    },
    {
        "name": "ComicNeue-BoldItalic",
        "file": "ComicNeue-BoldItalic.ttf",
        "url": (
            "https://raw.githubusercontent.com/google/fonts"
            "/main/ofl/comicneue/ComicNeue-BoldItalic.ttf"
        ),
        "role": "primary",
        "license_url": (
            "https://raw.githubusercontent.com/google/fonts"
            "/main/ofl/comicneue/OFL.txt"
        ),
    },
]

FONT_SHOUT = {
    "name": "Bangers",
    "file": "Bangers-Regular.ttf",
    "url": (
        "https://raw.githubusercontent.com/google/fonts"
        "/main/ofl/bangers/Bangers-Regular.ttf"
    ),
    "license_url": (
        "https://raw.githubusercontent.com/google/fonts/main/ofl/bangers/OFL.txt"
    ),
}

# Anime Ace cuma ~159 glyph -> tidak punya U+2661 (heart) maupun U+FF5E.
# Tanpa fallback ini, "AH~<3 NO, STOP~<3" keluar jadi kotak tofu.
FONT_SYMBOL = {
    "name": "NotoSansSymbols2",
    "file": "NotoSansSymbols2-Regular.ttf",
    "url": (
        "https://raw.githubusercontent.com/google/fonts"
        "/main/ofl/notosanssymbols2/NotoSansSymbols2-Regular.ttf"
    ),
    "license_url": (
        "https://raw.githubusercontent.com/google/fonts"
        "/main/ofl/notosanssymbols2/OFL.txt"
    ),
}

# Fallback lebar: Latin-ext (aksen, â é ñ ...), Cyrillic, Greek, Vietnam.
FONT_FALLBACK = {
    "name": "NotoSans",
    "file": "NotoSans.ttf",
    "url": (
        "https://raw.githubusercontent.com/google/fonts"
        "/main/ofl/notosans/NotoSans%5Bwdth%2Cwght%5D.ttf"
    ),
    "license_url": (
        "https://raw.githubusercontent.com/google/fonts"
        "/main/ofl/notosans/OFL.txt"
    ),
}

# Fallback CJK: kana, kanji, Hangul — untuk target bahasa Asia.
FONT_CJK = {
    "name": "NotoSansCJKjp",
    "file": "NotoSansCJKjp-Regular.otf",
    "url": (
        "https://github.com/googlefonts/noto-cjk/raw/main/Sans/OTF"
        "/Japanese/NotoSansCJKjp-Regular.otf"
    ),
    "license_url": (
        "https://raw.githubusercontent.com/googlefonts/noto-cjk"
        "/main/LICENSE"
    ),
}

# Fallback skrip RTL: Arab.
FONT_ARABIC = {
    "name": "NotoNaskhArabic",
    "file": "NotoNaskhArabic.ttf",
    "url": (
        "https://raw.githubusercontent.com/google/fonts"
        "/main/ofl/notonaskharabic/NotoNaskhArabic%5Bwght%5D.ttf"
    ),
    "license_url": (
        "https://raw.githubusercontent.com/google/fonts"
        "/main/ofl/notonaskharabic/OFL.txt"
    ),
}

# Fallback skrip: Thai.
FONT_THAI = {
    "name": "NotoSansThai",
    "file": "NotoSansThai.ttf",
    "url": (
        "https://raw.githubusercontent.com/google/fonts"
        "/main/ofl/notosansthai/NotoSansThai%5Bwdth%2Cwght%5D.ttf"
    ),
    "license_url": (
        "https://raw.githubusercontent.com/google/fonts"
        "/main/ofl/notosansthai/OFL.txt"
    ),
}

# ---------------------------------------------------------------- DeepL
#
# DeepL = mesin terjemahan MURNI: TIDAK menyensor konten apa pun (cocok
# untuk terjemahan uncensored), tapi juga TIDAK bisa klasifikasi label/SFX
# — itu ditangani heuristik di translate.py.
#
# Key berakhiran ':fx' = akun Free -> endpoint api-free (1 juta karakter/
# bulan). Key berbayar memakai https://api.deepl.com/v2.
DEEPL_API_BASE = "https://api-free.deepl.com/v2"

# Bahasa yang didukung DeepL (kode target_lang API). Thai/Vietnam/Filipino
# TIDAK didukung DeepL, jadi tidak muncul di daftar LANGUAGES versi ini.
DEEPL_TARGET: dict[str, str] = {
    "English": "EN",
    "Indonesian": "ID",
    "Spanish": "ES",
    "French": "FR",
    "German": "DE",
    "Portuguese": "PT",
    "Italian": "IT",
    "Dutch": "NL",
    "Russian": "RU",
    "Chinese (Simplified)": "ZH",
    "Chinese (Traditional)": "ZH-HANT",
    "Korean": "KO",
    "Japanese": "JA",
    "Arabic": "AR",
    "Turkish": "TR",
}

# ---------------------------------------------------------------- bahasa
#
# Pilihan bahasa sasaran di UI. "English" memakai gaya ALL CAPS + font Anime
# Ace; bahasa lain pakai huruf normal + fallback font multi-script.
LANGUAGES: list[str] = list(DEEPL_TARGET.keys())

# ---------------------------------------------------------------- router LLM
#
# Penyedia kedua: router OpenAI-compatible (gorouter). Bedanya dengan DeepL bukan
# soal mutu bahasa saja — LLM bisa DIBERI TAHU besar balonnya, dan itulah yang
# tidak mungkin dilakukan ke DeepL. DeepL menerjemahkan kalimat lepas konteks,
# jadi panjang hasilnya kebetulan; kalau kepanjangan, satu-satunya jalan keluar
# adalah mengecilkan font atau memenggal kata — dua cacat pertama di plan.txt.
#
# Keduanya tetap ada dan bisa dipilih di UI: DeepL lebih cepat dan gratis
# 1 juta karakter/bulan, router lebih patuh pada batas balon.
PROVIDERS: list[str] = ["DeepL", "LLM (freetokenfaucet)", "Router LLM (gorouter)"]
PROVIDER_DEFAULT = "LLM (freetokenfaucet)"

# WAJIB untuk gorouter. Tanpa header User-Agent, SETIAP permintaan ke host ini
# dibalas 403 "error code 1010" oleh Cloudflare — terukur 17 Agu 2026, dua
# bentuk auth (x-api-key dan Bearer) sama-sama 403. Bukan soal kredensial: key
# yang SAMA dengan UA ini membalas 200 dalam 8.2 s. Jebakan ini tidak terbaca
# dari pesan errornya, jadi jangan hapus tanpa mengukur lagi.
BROWSER_UA = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"
)

# Base URL router. Bisa ditimpa env ROUTER_API_BASE — host tunnel ini bisa
# berubah, dan hardcode-nya di kode berarti notebook harus diedit.
ROUTER_API_BASE = os.environ.get(
    "ROUTER_API_BASE", "https://rsx5kfk.abc-tunnel.us/v1"
).rstrip("/")

# SATU model saja, sesuai ANTHROPIC_MODEL di gorouter.txt dan itulah yang
# terukur bekerja (200, 8.2 s, JSON utuh, SFX ドキッ -> *thump*). Rantai cadangan
# sengaja berisi model yang sama, BUKAN kosong: router membalas 502 untuk model
# yang jelas-jelas ada lalu berhasil di panggilan berikutnya, jadi kegagalan
# pertama harus dianggap SEMENTARA dan yang menolong adalah mencoba lagi — bukan
# pindah model. Nama-nama model router lama dihapus; host ini tidak melayaninya.
ROUTER_MODEL = os.environ.get("ROUTER_MODEL", "gorouter/claude-opus-5")
ROUTER_FALLBACK: tuple[str, ...] = (ROUTER_MODEL,)

# Percobaan per model dan jeda dasar (detik, dilipatkan per percobaan).
#
# ANGKA-ANGKA INI PERNAH SALAH DAN AKIBATNYA PARAH, jadi alasannya ditulis.
# Semula 900/4/8: satu halaman bisa menunggu 900 s x 4 percobaan x 3 model =
# 3 JAM, dan dengan 2 ronde revisi jadi 9 jam. Yang terlihat di UI cuma
# "klasifikasi SFX + terjemah" menggantung — seolah GPU-nya lambat, padahal
# prosesnya menunggu HTTP yang tidak akan pernah datang.
#
# Halaman 13 balon yang SEHAT dijawab dalam 12.9-31.8 s (terukur, n=3). 120 s
# sudah 4x kasus terburuk yang sehat; lebih dari itu bukan "lambat" melainkan
# menggantung. ROUTER_DEADLINE membatasi SELURUH rangkaian percobaan, jadi
# batas atas satu panggilan bisa dihitung, bukan hasil perkalian diam-diam.
ROUTER_RETRY = 2
ROUTER_BACKOFF = 4
ROUTER_TIMEOUT = 120
ROUTER_DEADLINE = 240

# ------------------------------------------------------- LLM freetokenfaucet
#
# Penyedia ketiga, OpenAI-compatible sama seperti router, jadi dipegang kelas
# client yang sama — yang beda cuma base URL, nama model, dan satu parameter
# body. Alasan dia ada: dia GRATIS, dan anggaran balon butuh LLM (bukan DeepL),
# jadi tanpa penyedia ini fitur itu cuma jalan dengan kredit berbayar.
#
# MODELNYA HARUS MODEL GRATIS. Terukur 17 Agu 2026: dari 19 model terdaftar,
# 16 membalas HTTP 402 INSUFFICIENT_BALANCE ("model berbayar, saldo akun 0") —
# termasuk model yang dulu jadi default di sini, dan itulah yang membuat tiga
# halaman Colab keluar TANPA terjemahan. 402 ditolak SEBELUM satu token pun
# dibuat, jadi ini bukan soal kuota atau max_tokens.
#
# Yang gratis cuma tiga, semuanya diuji dengan _system_prompt() asli atas 19
# baris Jepang dan ketiganya membalas 19/19 kunci JSON dengan ♥ dan … utuh:
#   mimo-v2.5-pro   4.5-7.8 s, out 131-203  <- DIPAKAI. 3/3 sukses, SFX aman.
#   mimo-v2.5       4.6-6.9 s, out 144-205  - sekali MENERJEMAHKAN SFX ドキッ
#                                             jadi "My heart's racing" (salah).
#   gpt-5.6-terra   6.7-10.1 s, out 151-310 - balas ALL CAPS sendiri, sekali 524.
# Dua yang terakhir bisa dipakai lewat env FAUCET_MODEL tanpa mengedit sel.
FAUCET_API_BASE = os.environ.get(
    "FAUCET_API_BASE", "https://freetokenfaucet.com/v1"
).rstrip("/")
FAUCET_MODEL = os.environ.get("FAUCET_MODEL", "mimo-v2.5-pro")

# Model ini model REASONING, dan defaultnya membakar token untuk berpikir dulu.
# Terukur pada 3 balon: default 111-259 token reasoning lalu jawaban; dengan
# max_tokens=200 jatahnya habis di reasoning dan content keluar KOSONG STRING —
# gagal tanpa error HTTP, yang jauh lebih berbahaya daripada gagal berisik.
# thinking.type=disabled menghilangkan reasoning sepenuhnya: 26 token keluaran,
# 2.8 s, jawaban tetap benar. Token faucet TERBATAS, jadi ini bukan sekadar
# soal cepat — reasoning yang dibuang itu 4-10x biaya jawabannya sendiri.
# reasoning_effort="none" DITOLAK server (400, hanya low..max), dan
# reasoning.enabled=False DIABAIKAN (208 token reasoning tetap keluar) — jadi
# hanya bentuk inilah yang bekerja, jangan diganti tanpa mengukur lagi.
FAUCET_EXTRA: dict = {"thinking": {"type": "disabled"}}
# Plafon keluaran. Halaman 19 balon terukur puncak 203 token (mimo-v2.5-pro,
# 17 Agu 2026); 1200 memberi ruang untuk halaman padat tanpa memberi ruang untuk
# mengarang.
FAUCET_MAX_TOKENS = 1200
FAUCET_FALLBACK: tuple[str, ...] = (FAUCET_MODEL,)
# Jauh lebih pendek dari router: yang sehat 3-4 s, jadi 45 s sudah 10x kasus
# terburuk. Tidak ada gunanya menunggu 120 s ke endpoint yang biasanya 3 s.
FAUCET_TIMEOUT = 45
FAUCET_DEADLINE = 120

# ---------------------------------------------------------------- gaya
#
# Gaya terjemahan scanlation. Kunci = label di UI; nilai = instruksi tambahan
# yang disisipkan ke system prompt LLM.
TRANSLATION_STYLES: dict[str, str] = {
    "Manga Natural": (
        "Manga scanlation style: casual, natural dialogue that reads like a real "
        "released manga. Use contractions and short, punchy sentences. Match the "
        "speaker's personality and register (kids talk like kids, elders politely). "
        "Preserve emotion marks (... ! ? ~ <3)."
    ),
    "Formal": (
        "Formal, polite register: proper grammar, respectful tone, minimal "
        "contractions. Suitable for formal scenes, narration, or polite characters."
    ),
    "Casual & Slang": (
        "Casual and slangy: very informal, heavy contractions, modern slang and "
        "colloquialisms, playful or rough tone matching the scene."
    ),
    "Literal": (
        "Literal: stay as close to the Japanese wording as possible, even if the "
        "result sounds stiff. Keep structure and terms faithful to the source."
    ),
    "Uncensored": (
        "Uncensored: keep sexual, vulgar, crude, and dirty-joke language raw and "
        "faithful — no softening, no euphemism, no censorship, whatever the genre "
        "(comedy with dirty jokes, mature drama, hentai, ...). Use direct, explicit "
        "wording for sex acts and body parts exactly as the source intends. Explicit "
        "content is expected in this adult 18+ work and must be rendered literally."
    ),
    "Fully Localized": (
        "Fully localized: adapt idioms, jokes, and cultural references so the result "
        "reads as if the manga was originally written in the target language. "
        "Translate honorifics and Japanese terms into natural equivalents."
    ),
}

# Honorifik Jepang (san/kun/chan/sama/senpai) dipertahankan atau dilokalkan.
# Contoh: 'Yamada-kun' tetap 'Yamada-kun', bukan 'Tuan Yamada'.
KEEP_HONORIFICS_HELP = (
    "Pertahankan honorifik Jepang (san/kun/chan/senpai) — matikan untuk "
    "melokalkan jadi sapaan bahasa sasaran."
)

# ---------------------------------------------------------------- thresholds


@dataclass
class Settings:
    """Semua angka yang bisa disetel ada di sini, tidak tersebar di kode."""

    # deteksi
    det_size: int = 640
    det_conf: float = 0.30
    det_iou: float = 0.45
    # Balon ganda (連結吹き出し) = dua lobus menyatu yang kotaknya saling
    # tumpang tindih banyak. Pada 0.45 salah satu lobus disuppress, kedua
    # region teks jatuh ke satu kotak balon, dan dua terjemahan berakhir
    # bertumpuk. Ambang bubble sengaja lebih longgar; penjaga containment di
    # _nms() yang membedakan "dua lobus" dari "dua deteksi balon yang sama".
    det_iou_bubble: float = 0.65
    tiled_pass: bool = True          # 2x2 tile untuk menangkap teks kecil

    # mask
    seed_thresh: float = 0.50        # region yang pasti teks
    grow_thresh: float = 0.118       # melebar ke tepi anti-aliased
    dilate_ratio: float = 0.30       # kernel adaptif per tinggi glyph
    halo_deviation: int = 12         # |px - bg| minimal supaya dihitung halo
    min_cc_area: int = 50            # furigana terkecil yang masih diselamatkan

    # OCR
    min_ink_ratio: float = 0.015     # gate anti-halusinasi manga-ocr

    # erase
    flat_std_thresh: int = 10         # di bawah ini -> flat fill, tanpa GPU
    flat_std_thresh_noisy: int = 7
    # Isi PENUH interior balon saat menghapus, bukan cuma stroke glyph.
    #
    # Kenapa: mask stroke (ink_mask) dibangun dari ambang + dilasi adaptif, dan
    # ia sistematis melewatkan glyph yang tipis atau renggang. Terukur di
    # hasilnew/jp_6.JPG setelah hapusan: tanda dash panjang '——' di balon kanan
    # selamat utuh sebagai GARIS TIPIS, dan 'うう…' menyisakan dua coretan kecil.
    # Menaikkan dilasi tidak menyelesaikan ini — ia cuma menggeser cacatnya ke
    # garis balon yang ikut termakan. Interior balon tidak punya masalah itu:
    # batasnya bukan taksiran, itu garis balon sungguhan.
    #
    # Harganya jujur dan sudah disetujui: gradasi/screentone DI DALAM balon
    # hilang, diganti satu warna rata. Yang TIDAK ikut diputihkan: region tanpa
    # balon induk (bubble_bbox None -> art, bukan balon) dan region terlindungi
    # (SFX). Warna isian juga bukan putih buta — dipakai median latar balon itu
    # sendiri, jadi balon hitam tetap terisi hitam.
    bubble_fill: bool = True
    # Interior untuk ISIAN dikikis sekian x ketebalan garis balon. 1x, bukan 2x
    # seperti interior untuk tata letak: isian harus sampai mepet garis supaya
    # tidak ada pita tinta lama tertinggal di tepi, sementara tata letak justru
    # perlu jarak aman supaya glyph tidak menempel di garis.
    fill_erode_stroke: int = 1

    # typeset
    min_font_size: int = 11
    # Lebar halaman tempat min_font_size di atas dikalibrasi (CONTOH/2.webp
    # dipakai pada 1134 px). Lantai ukuran font TIDAK boleh angka mutlak: ia
    # dikalibrasi pada satu resolusi, dan halaman lain datang di resolusi lain.
    #
    # Terukur, bukan ditaksir. hasilnew/jp_6.JPG cuma 698 px lebar, dan pada
    # halaman itu lantai 11 px membuat anggaran balon jadi 2-39 karakter
    # sementara wording typeset referensi hasilnew/6.JPG untuk balon yang SAMA
    # 15-71 karakter (probe_floor6.py). Jadi model diperintah menulis jauh lebih
    # pendek daripada yang sebenarnya muat, dan hasilnya 'SO?' untuk balon yang
    # referensinya 'SO IN THE END, THE WHOLE CLASS GOT SO EXCITED...'.
    #
    # Huruf referensinya sendiri diukur 4-7 px tinggi (probe_refsize.py, modus 4,
    # median 5 pada 728 px) = ukuran font 5-8. Lantai 11 px memang di atas apa
    # yang dipakai typesetter manusia di resolusi ini.
    min_font_ref_width: int = 1134
    # Lantai mutlak: di bawah ini huruf tidak terbaca pada resolusi apa pun,
    # jadi penskalaan tidak boleh menembusnya.
    min_font_abs: int = 6
    max_font_size: int = 96
    # Diukur, bukan ditaksir. probe_lines.py mengambil profil baris tinta tiap
    # balon di CONTOH/2.webp: pitch baseline / cap_height = 1.36 (p25 1.33,
    # p75 1.37). Anime Ace punya asc+desc ~ 1.36x cap_height-nya sendiri, jadi
    # faktor yang menyamai referensi = 1.00 — bukan 1.28. Pada 1.28 tiap baris
    # menyisakan 4-12 px ruang kosong yang tidak dipakai ALL CAPS, dan ruang
    # itulah yang memaksa balon padat turun ke ukuran font minimum.
    line_spacing: float = 1.00
    # Margin dalam bubble. Dikurangi DUA KALI di layout.max_width_at (kiri dan
    # kanan), jadi 0.10 memakan 20% lebar tiap baris.
    #
    # Angka ini TIDAK dipilih dari margin nominalnya, tapi dari margin yang
    # BENAR-BENAR terukur. probe_tidy.py merender teks kita lalu mengukurnya
    # dengan kode yang sama seperti pengukuran referensi (cap_height dari
    # komponen terhubung, penyebut = kolom interior gabungan):
    #     pad   cap/min  sisi/min  isi      referensi: 0.117 / 0.165 / 70%
    #     0.04    0.116     0.096  80%
    #     0.06    0.115     0.100  80%
    #     0.10    0.113     0.144  69%
    # Naik dari 0.04 ke 0.06 nyaris tidak menambah margin nyata (0.096 -> 0.100)
    # tapi membuat DUA balon tersempit halaman (r6, r10) turun ke 10 px, di bawah
    # min_font_size, dan memperbesar galat ke ukuran referensi 3.05 -> 3.36 px
    # (probe_final.py, probe_cal.py). Jadi 0.04: nol region di bawah minimum,
    # galat terkecil, tanda hubung tetap satu.
    #
    # Sisa jarak ke sisi/min 0.165 milik referensi TIDAK dibayar dengan pad. Pada
    # pad 0.10 margin memang naik ke 0.144, tapi harganya ukuran font (galat 3.82)
    # dan tanda hubung tambahan — dua cacat yang eksplisit di plan.txt.
    # probe_wording.py memisahkan sebabnya: dengan wording referensi yang lebih
    # pendek, pad yang sama memberi isi 84% dan galat lebih kecil. Selisihnya
    # berasal dari panjang wording DeepL, dan itu tahap tersendiri.
    pad_ratio: float = 0.04
    force_upper: bool = True         # ALL CAPS hanya untuk target English
    target_lang: str = "English"     # di-set UI: Jepang -> bahasa ini
    translation_style: str = "Manga Natural"  # gaya scanlation (TRANSLATION_STYLES)
    keep_honorifics: bool = True     # san/kun/chan dipertahankan

    # terjemahan
    provider: str = "LLM (freetokenfaucet)"  # lihat PROVIDERS
    # Anggaran balon: kirim batas karakter per balon ke LLM, lalu ukur ulang
    # jawabannya dengan mesin tata letak sungguhan dan minta perbaikan untuk baris
    # yang melanggar. Hanya berlaku untuk provider LLM (faucet/router) — DeepL
    # tidak bisa diberi tahu apa pun. Biayanya ~0.8 detik CPU per balon (terukur: 13 balon =
    # 10 detik, 485 panggilan layout(), nol GPU) di atas tunggu jaringan ~20 detik.
    # Itu harga yang dibayar untuk 'NO KELUAR BUBBLE': tanpa anggaran, model yang
    # sama mengembalikan 'SORRY TO BARGE IN.' (18 karakter) untuk balon yang cuma
    # memuat 6 — bukan karena membangkang, tapi karena ia tidak melihat balonnya.
    balloon_budget: bool = True
    # Ronde perbaikan maksimum. Tiap ronde hanya mengirim ulang baris yang MASIH
    # melanggar, jadi ronde kedua jauh lebih murah dari yang pertama. Dibatasi 2
    # karena pengamatan: baris yang tidak membaik di ronde 2 juga tidak membaik di
    # ronde 5 — sisanya diserahkan ke fit() yang mengecilkan font.
    budget_repair_rounds: int = 2
    # Ronde permintaan ulang untuk balon yang kuncinya TIDAK dijawab model.
    # Beda urusan dengan budget_repair_rounds: yang itu menjaga jawaban tetap
    # muat, yang ini menjaga jawabannya ADA. Terukur di hasilnew/13.JPG: balon
    # 'えっ！？' terkirim tapi kuncinya hilang dari JSON balasan, jadi balon itu
    # tercetak berbahasa Jepang. Satu ronde sudah cukup untuk kasus itu (model
    # menjawab begitu diberi tahu balonnya nyata); angka 1 juga menjaga biaya
    # token faucet tetap kecil — yang dikirim ulang cuma idx yang kosong.
    missing_repair_rounds: int = 1
    oblique: float = 0.12            # shear sintetis; Anime Ace regular tegak,
                                     # referensi miring. 0 = matikan.
    # Rapatkan huruf secara horizontal (1.00 = matikan). BUKAN selera — ini
    # menutup selisih kerapatan font yang TERUKUR antara Anime Ace dan font
    # typeset referensi.
    #
    # probe_reffont2.py mengukur 10 baris CONTOH/6.JPG yang teksnya terbaca mata:
    # baris dicari dari komponen glyph (bukan kotak tangan), tinggi kapital =
    # median tinggi komponen tertinggi baris itu, lalu lebar baris dibanding
    # lebar string YANG SAMA di Anime Ace pada ukuran ber-cap-height sama:
    #     EMBARASSING       97 px vs 147 px   0.660
    #     DESCRIBED         77      113       0.681
    #     I'M PRAISING      88      130       0.677
    #     NEVER SEEN        95      136       0.699
    #     TO BE             42       58       0.724
    #     MOSTLY CAME BY   116      157       0.739
    #     BOTTOM OF         83      112       0.741
    #     TOO EXCITED AND  122      162       0.753
    #   median 0.690  -> referensi memuat ~1.45x karakter per baris pada tinggi
    #   huruf yang sama. Ukuran fontnya sendiri TIDAK salah: cap referensi 14.0 px
    #   pada 1357 px = 7.2 px pada halaman kita (698 px) = Anime Ace ukuran 6, dan
    #   kita memang merender 6-8. Yang beda cuma kerapatannya.
    #
    # Angkanya 0.85, bukan 0.690. probe_cond.py menyapu faktor pada mask jp_6
    # yang sungguhan dan menghitung tanda hubung yang tersisa:
    #     cond   wording kita        wording referensi
    #     1.00   3 hyphen  size 6-8  5 hyphen  1 luber
    #     0.88   2 hyphen  size 6-8  3 hyphen  0 luber
    #     0.85   2 hyphen  size 6-9  1 hyphen  0 luber
    #     0.72   0 hyphen  size 6-10 1 hyphen  0 luber
    # 0.85 memberi hampir seluruh perbaikannya (referensi 5->1 tanda hubung, luber
    # 1->0, plafon ukuran 9->10) sementara hurufnya masih berbentuk huruf. 0.690
    # dan 0.72 memang menghapus satu tanda hubung lagi, tapi pada cap 7 px itu
    # meremas batang huruf sampai di bawah satu piksel dan hasilnya kabur — mahal
    # untuk satu tanda hubung, dan tanda hubung sisanya (r3) sebabnya lain:
    # interiornya cuma menyisakan 26 px kolom bebas dari mask 46x87.
    condense: float = 0.85

    # inpaint
    lama_size: int = 512
    use_amp: bool = False            # torch.fft selalu fp32; fp16 nihil manfaat

    # verifikasi
    residue_deviation: int = 20
    # Komponen sisa TERBESAR yang masih dianggap wajar, dalam px — gerbang kedua
    # di verify.find_residue(), di samping gerbang jumlah. Gerbang jumlah
    # `max(30, 0.002*w*h)` berskala AREA balon sementara satu titik kotor tidak:
    # di balon 400x500 ambangnya jadi 400 px dan titik 60 px yang jelas terlihat
    # lolos tanpa satu ronde eskalasi. 12 dipilih karena komponen tinta yang
    # sah pun jauh lebih besar — komponen ink_mask terkecil di halaman referensi
    # yang bukan derau = 150 px — jadi gerbang ini tidak bisa salah menuduh satu
    # stroke utuh. Enam ambang (12/16/20/24/30/40) diuji berdampingan di
    # _residue5.py: semuanya menandai region yang sama, jadi 12 dipilih sebagai
    # yang paling ketat tanpa satu pun false positive. Yang menahan alarm palsu
    # bukan angka ini melainkan LINGKUP-nya — lihat verify._SCOPE_NEAR; dengan
    # lingkup yang benar halaman referensi bersih di semua enam ambang.
    residue_blob_max: int = 12
    max_escalation: int = 2

    # output
    output_format: str = "both"   # pilihan UI: "png" | "jpg" | "both"
    jpg_quality: int = 92
    debug: bool = False


SETTINGS = Settings()

# ---------------------------------------------------------------- ONNX Runtime

# Provider yang BENAR-BENAR dipakai tiap sesi. Sel warm-up membacanya untuk
# memutuskan berhenti atau lanjut.
ORT_REPORT: dict[str, str] = {}

# ---------------------------------------------------------------- catatan jalan
#
# Jalur terdegradasi pipeline ini SENGAJA tidak melempar: satu balon gagal
# diterjemah tidak boleh membunuh halaman, dan itu keputusan yang benar. Tapi
# konsekuensinya terukur pahit — satu run Colab keluar dengan `diterjemah 0` di
# tiga halaman (semua balon tetap berbahasa Jepang) dan user melihat NOL pesan,
# karena satu-satunya jejak tiap kegagalan adalah print() di dalam thread
# handler Gradio, yang di Colab nyangkut di sel yang sudah discroll.
#
# Jadi kegagalan dicatat, bukan cuma dicetak. Bentuknya daftar tuple dan bukan
# logging.Logger: pembacanya (app.py) perlu MEMUTUSKAN berdasarkan isinya —
# "ada error level apa saja di halaman ini" — dan itu jauh lebih murah atas
# daftar terstruktur daripada atas teks yang harus diurai ulang dengan regex.
# Pola yang sama dengan ORT_REPORT di atas: modul menulis, pembaca menyimpulkan.
RUN_NOTES: list[tuple[str, str, str]] = []

# Awalan yang dipakai note(). Dipilih supaya bisa dipindai mata DAN grep di log
# mentah: "[!!]" tidak pernah muncul di keluaran library pihak ketiga mana pun
# yang dipakai pipeline ini, jadi `grep '\[!!\]' run.log` = daftar error murni.
_NOTE_MARK: dict[str, str] = {"error": "[!!]", "warn": "[!]", "info": "[i]"}


def note(level: str, tag: str, msg: str) -> None:
    """Catat jalur terdegradasi SEKALIGUS cetak. level: "error"|"warn"|"info".

    Mencetak juga, bukan hanya menyimpan: sel notebook (warm-up, probe, sel 25)
    memanggil modul di luar UI, dan di sana stdout memang terlihat. Yang
    disimpan dipakai app.py untuk menyusun banner.

    RAHASIA TIDAK BOLEH MASUK `msg`. Pemanggilnya yang menjaga itu — sama
    seperti print() sebelumnya — karena di sinilah satu-satunya tempat pesan
    bisa ikut tertulis ke file .log yang lalu diunduh user.
    """
    RUN_NOTES.append((level, tag, msg))
    print(f"{_NOTE_MARK.get(level, '[i]')} [{tag}] {msg}")


def notes_since(mark: int) -> list[tuple[str, str, str]]:
    """Catatan yang muncul sejak `len(RUN_NOTES)` bernilai `mark`.

    Dipakai supaya catatan menempel ke HALAMAN yang bersangkutan, bukan ke
    seluruh batch: pemanggil mencatat panjangnya sebelum halaman dimulai lalu
    memanggil ini sesudahnya. Tanpa itu, halaman ke-3 mewarisi error halaman
    ke-1 dan tabel UI menuduh halaman yang benar.
    """
    return RUN_NOTES[max(mark, 0):]

# EXHAUSTIVE (default ORT) mengukur ulang tiap algoritma konvolusi untuk setiap
# bentuk input baru. Halaman manga ukurannya beda-beda, jadi biaya itu dibayar
# terus dan tidak pernah teramortisasi. HEURISTIC memilih langsung.
_CUDA_OPTS = {
    "device_id": 0,
    "cudnn_conv_algo_search": "HEURISTIC",
    "arena_extend_strategy": "kSameAsRequested",
    "do_copy_in_default_stream": True,
}


def ort_session(path, tag: str):
    """InferenceSession di CUDA. Kalau jatuh ke CPU, jatuhnya BERISIK.

    Fallback diam adalah sebab pipeline pernah makan 100 detik per halaman tanpa
    satu pun pesan: onnxruntime-gpu gagal memuat CUDA EP, ORT diam saja, semua
    inference pindah ke CPU. Sekarang tiap kegagalan dicatat di ORT_REPORT.
    """
    import onnxruntime as ort

    opts = ort.SessionOptions()
    opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    opts.log_severity_level = 2  # warning: ORT menyebut .so yang hilang

    # Sel 5 sudah menguji CUDA di subproses. Kalau di sana ia abort, mencobanya
    # lagi di sini membunuh kernel Colab tanpa traceback - jadi jangan dicoba.
    probe = os.environ.get("MANGATL_ORT_CUDA", "ok")
    if probe not in ("ok", ""):
        note("warn", "ort", f"{tag}: sel 5 menandai CUDA tidak aman ({probe}) -> CPU")
    elif "CUDAExecutionProvider" in ort.get_available_providers():
        try:
            sess = ort.InferenceSession(
                str(path), opts,
                providers=[("CUDAExecutionProvider", _CUDA_OPTS), "CPUExecutionProvider"],
            )
            ORT_REPORT[tag] = sess.get_providers()[0]
            if "CUDAExecutionProvider" not in sess.get_providers():
                note("warn", "ort", f"{tag}: CUDA ditolak saat membuat sesi -> CPU (lambat)")
            return sess
        except Exception as exc:  # noqa: BLE001 - ORT melempar tipe khusus per build
            note("warn", "ort", f"{tag}: CUDA gagal ({str(exc)[:140]}) -> CPU (lambat)")
    else:
        note("warn", "ort", f"{tag}: CUDAExecutionProvider tidak tersedia -> CPU (lambat)")

    sess = ort.InferenceSession(str(path), opts, providers=["CPUExecutionProvider"])
    ORT_REPORT[tag] = "CPUExecutionProvider"
    return sess

DetClass = Literal["bubble", "text_bubble", "text_free"]
RegionLabel = Literal["DIALOGUE", "THOUGHT", "NARRATION", "SIGN", "SFX", "UNREADABLE"]
Route = Literal["flat", "lama", "skip"]

ID2LABEL: dict[int, str] = {0: "bubble", 1: "text_bubble", 2: "text_free"}

# Label yang TIDAK BOLEH dihapus dari halaman, apa pun yang terjadi.
PROTECTED_LABELS: frozenset[str] = frozenset({"SFX", "UNREADABLE"})


@dataclass
class Region:
    """Satu blok teks. Diteruskan lewat semua stage, diisi bertahap."""

    idx: int
    bbox: tuple[int, int, int, int]                    # xyxy, koordinat halaman
    det_class: DetClass = "text_bubble"
    det_conf: float = 0.0
    quad: np.ndarray | None = None                     # 4 titik, untuk rotasi
    bubble_bbox: tuple[int, int, int, int] | None = None
    # Kotak balon SEBELUM dibelah, diisi hanya kalau balon ini dipakai bersama
    # region lain (balon ganda). textmask.partition_shared_interiors() butuh
    # kotak asli untuk flood fill sekali di seluruh balon, bukan per belahan.
    shared_bubble_bbox: tuple[int, int, int, int] | None = None
    bubble_mask: np.ndarray | None = None              # interior bubble (lokal)
    # Mask ISIAN untuk erase: SELURUH interior balon, bukan cuma stroke glyph.
    # Dipisah dari bubble_mask karena dua-duanya punya tugas berbeda dan salah
    # satunya dirusak demi yang lain: bubble_mask dipangkas
    # disjoin_overlapping_interiors() supaya tata letak dua balon bertetangga
    # tidak beririsan, dan kalau mask yang sudah dipangkas itu dipakai mengisi,
    # sliver yang dipangkas tadi tidak pernah tersentuh dan tinta Jepang di
    # sana selamat. fill_mask direkam SEBELUM pemangkasan, jadi isian selalu
    # menutup interior penuh. fill_bbox ikut disimpan karena bubble_bbox juga
    # digeser oleh pemangkasan itu.
    fill_bbox: tuple[int, int, int, int] | None = None
    fill_mask: np.ndarray | None = None
    ink_mask: np.ndarray | None = None                 # stroke teks (lokal)
    est_font_size: float = 0.0
    angle: float = 0.0
    is_vertical: bool = False

    # OCR
    src_text: str = ""
    ink_ratio: float = 0.0

    # LLM
    label: RegionLabel = "DIALOGUE"
    label_conf: float = 0.0
    translation: str | None = None

    # erase
    route: Route = "flat"
    bg_color: tuple[int, int, int] | None = None

    # typeset
    final_font_size: int = 0
    lines: list[str] = field(default_factory=list)
    overflowed: bool = False

    @property
    def width(self) -> int:
        return self.bbox[2] - self.bbox[0]

    @property
    def height(self) -> int:
        return self.bbox[3] - self.bbox[1]

    @property
    def is_protected(self) -> bool:
        """SFX dan teks tak terbaca tidak pernah dihapus dari halaman."""
        return self.label in PROTECTED_LABELS

    def to_dict(self) -> dict:
        """Ringkasan untuk report.json — tanpa array numpy."""
        return {
            "idx": self.idx,
            "bbox": list(self.bbox),
            "det_class": self.det_class,
            "det_conf": round(self.det_conf, 3),
            "label": self.label,
            "label_conf": round(self.label_conf, 3),
            "src_text": self.src_text,
            "translation": self.translation,
            "route": self.route,
            "est_font_size": round(self.est_font_size, 1),
            "final_font_size": self.final_font_size,
            "lines": self.lines,
            "ink_ratio": round(self.ink_ratio, 4),
            "overflowed": self.overflowed,
            "protected": self.is_protected,
        }


SUPPORTED_EXT: frozenset[str] = frozenset({
    ".jpg", ".jpeg", ".jpe", ".jfif", ".png", ".webp", ".bmp", ".dib",
    ".tif", ".tiff", ".gif", ".ppm", ".pgm", ".pbm", ".avif", ".heic", ".heif",
})



Writing /content/mangatl/config.py


In [5]:
%%writefile /content/mangatl/imgio.py

"""Baca gambar apa pun jadi RGB, tulis hasil sebagai PNG + JPG + ZIP."""

from __future__ import annotations

import io
import zipfile
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps

from config import OUTPUT, SETTINGS, SUPPORTED_EXT

Image.MAX_IMAGE_PIXELS = 300_000_000  # halaman manga resolusi tinggi itu wajar


def register_extra_formats() -> list[str]:
    """Daftarkan HEIC/AVIF kalau plugin-nya ada. Aman dipanggil berulang."""
    enabled: list[str] = []
    try:
        import pillow_heif

        pillow_heif.register_heif_opener()
        enabled.append("heic")
    except (ImportError, AttributeError):
        pass
    # AVIF sudah native di Pillow >= 11; pillow-heif membuang
    # register_avif_opener, memanggilnya akan AttributeError.
    if "AVIF" in Image.registered_extensions().values():
        enabled.append("avif")
    return enabled


def load_any(path: str | Path) -> np.ndarray:
    """Buka file gambar apa pun -> array RGB uint8, HxWx3.

    Urutan normalisasi penting: exif_transpose dulu (kalau tidak, halaman
    ter-rotasi salah), baru urusan mode warna.
    """
    path = Path(path)
    if path.suffix.lower() not in SUPPORTED_EXT:
        raise ValueError(f"Ekstensi tidak didukung: {path.suffix}")

    with Image.open(path) as im:
        im.load()
        return normalize(im)


def normalize(im: Image.Image) -> np.ndarray:
    """Samakan orientasi, bit depth, dan mode warna jadi RGB uint8."""
    im = ImageOps.exif_transpose(im) or im

    # Palette dengan transparansi harus lewat RGBA dulu supaya alpha tidak hilang.
    if im.mode == "P":
        im = im.convert("RGBA" if "transparency" in im.info else "RGB")

    # 16-bit / float perlu diskalakan sebelum convert, kalau tidak akan terpotong.
    if im.mode in ("I", "I;16", "I;16B", "I;16L", "F"):
        arr = np.asarray(im).astype(np.float32)
        hi = float(arr.max()) or 1.0
        arr = (arr / hi * 255.0).clip(0, 255).astype(np.uint8)
        im = Image.fromarray(arr, mode="L")

    if im.mode in ("RGBA", "LA"):
        bg = Image.new("RGB", im.size, (255, 255, 255))
        bg.paste(im, mask=im.split()[-1])
        im = bg

    if im.mode != "RGB":
        im = im.convert("RGB")

    return np.asarray(im, dtype=np.uint8)


def save_outputs(img: np.ndarray, stem: str, outdir: Path | None = None) -> dict[str, Path]:
    """Simpan sesuai SETTINGS.output_format: PNG, JPG, atau keduanya.

    JPG wajib subsampling=0 (4:4:4) — chroma subsampling default bikin garis
    hitam tajam di manga jadi berbayang. Dict hasil hanya berisi format yang
    benar-benar ditulis, jadi konsumen (gallery/zip) tinggal pakai apa adanya.
    """
    outdir = outdir or OUTPUT
    outdir.mkdir(parents=True, exist_ok=True)
    pil = Image.fromarray(img)

    # Clamp ke nilai sah: format tak dikenal tetap harus menghasilkan file,
    # kalau tidak pipeline mati di sidecar JSON (next(iter(paths.values()))).
    fmt = SETTINGS.output_format if SETTINGS.output_format in ("png", "jpg", "both") else "both"
    paths: dict[str, Path] = {}
    if fmt in ("png", "both"):
        png_path = outdir / f"{stem}.png"
        pil.save(png_path, "PNG", optimize=True)
        paths["png"] = png_path
    if fmt in ("jpg", "both"):
        jpg_path = outdir / f"{stem}.jpg"
        pil.save(jpg_path, "JPEG", quality=SETTINGS.jpg_quality, subsampling=0)
        paths["jpg"] = jpg_path
    return paths


def make_zip(paths: list[Path], zip_path: Path) -> Path:
    """Bungkus hasil jadi satu ZIP supaya sekali unduh."""
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in paths:
            if p.exists():
                zf.write(p, arcname=p.name)
    return zip_path


def to_png_bytes(img: np.ndarray) -> bytes:
    buf = io.BytesIO()
    Image.fromarray(img).save(buf, "PNG")
    return buf.getvalue()



Writing /content/mangatl/imgio.py


In [6]:
%%writefile /content/mangatl/detect.py

"""Deteksi region: RT-DETR-v2 (bubble/text_bubble/text_free) via ONNX Runtime."""

from __future__ import annotations

import numpy as np

try:
    import onnxruntime as ort
except ImportError:  # ORT gagal terpasang: beri pesan jelas saat dipakai,
    ort = None       # jangan matikan seluruh notebook saat import.

from config import ID2LABEL, SETTINGS, WEIGHTS, Region, ort_session

_SESSION = None

# Dua lobus balon ganda saling tumpang tindih banyak tapi tidak ada yang MEMUAT
# yang lain; dua deteksi atas balon yang sama containment-nya ~1.0. 0.80 memisah
# keduanya. Lihat _nms().
_BUBBLE_CONTAIN = 0.80

# Kotak teks kerap menonjol keluar kotak balon, dan pada balon ganda tonjolannya
# besar. 0.80 menolak pasangan yang jelas benar -> region kehilangan induk ->
# mask interior jadi persegi mentah dan teks keluar garis. Lihat assign_bubbles().
_PARENT_OVERLAP = 0.65

# SATU blok teks yang terdeteksi DUA KALI: kotak kecil hampir seluruhnya termuat
# di kotak besar. NMS kelompok teks memakai IoU murni (contain_thresh=0.0), dan
# IoU pasangan bersarang bisa RENDAH walau containment-nya ~1.0 — pada halaman
# hitomi_3740721_015: containment 0.974 tapi IoU cuma 0.280, di bawah det_iou
# 0.45, jadi keduanya lolos. Lihat drop_nested_duplicates().
#
# 0.80 dipilih karena ada jurang lebar yang terukur di antara dua populasi:
# containment teks-vs-teks pada halaman bersih (debug/jp_6 8 region, jepang_002
# 13 region, jp_13 4 region) tertinggi cuma 0.33, sedangkan duplikat sejati
# 0.974. Angka yang sama dengan _BUBBLE_CONTAIN, dengan alasan yang sama.
_DUP_CONTAIN = 0.80


# Toleransi cakupan saat memilih induk. Aturan "balon TERKECIL yang memuat
# mayoritas teks" salah kalau balonnya BERLOBUS: detector mengeluarkan satu kotak
# untuk seluruh balon plus satu per lobus, dan kotak lobus selalu lebih kecil.
# Terukur di hitomi_3740721_015 — teks (832,130,1027,405):
#   lobus kiri  (800,117,964,440)  cakupan 0.677  area 52972  <- lama menang
#   balon penuh (800,96,1046,442)  cakupan 1.000  area 85116  <- yang benar
# Yang menang dulu memuat cuma 2/3 teks, jadi lobus kanan tidak masuk interior:
# `そうならないように．．．` tidak pernah terhapus dan `find_residue` menandainya.
#
# Jadi cakupan dipakai LEBIH DULU, area cuma pemutus seri. Serinya diberi pita
# 0.05 supaya kotak lobus yang benar tidak kalah hanya karena teksnya menonjol
# beberapa piksel keluar lobus: 0.97 lawan 1.00 tetap seri (lobus menang, dan
# itu yang diinginkan pada balon ganda), sedangkan 0.677 lawan 1.00 tidak.
_PARENT_SLACK = 0.05


def get_session():
    """Muat detector sekali, pakai ulang. GPU kalau ada, CPU kalau tidak."""
    global _SESSION
    if _SESSION is None:
        if ort is None:
            raise RuntimeError(
                "onnxruntime tidak terpasang — jalankan ulang sel install (sel 3)."
            )
        path = WEIGHTS / "detector.onnx"
        if not path.exists():
            raise FileNotFoundError(f"Weight detector tidak ada: {path}")
        _SESSION = ort_session(path, "detector")
    return _SESSION


def _preprocess(img: np.ndarray, size: int) -> np.ndarray:
    """Resize langsung ke size x size — RT-DETR tidak pakai letterbox."""
    import cv2

    resized = cv2.resize(img, (size, size), interpolation=cv2.INTER_LINEAR)
    chw = resized.transpose(2, 0, 1).astype(np.float32) / 255.0
    return chw[None]


def _input_names(sess) -> list[str]:
    return [i.name for i in sess.get_inputs()]


def _nms(boxes: np.ndarray, scores: np.ndarray, iou_thresh: float,
         contain_thresh: float = 0.0) -> list[int]:
    """NMS greedy. Dipakai untuk menggabung hasil full-page + tile.

    `contain_thresh > 0` menambah syarat kedua sebelum suppress: kotak yang
    kalah harus BENAR-BENAR termuat di kotak pemenang (inter/area_kecil).
    Tanpa itu, satu lobus balon ganda ditekan hanya karena IoU-nya tinggi, dua
    region teks jatuh ke satu kotak balon, dan terjemahannya bertumpuk.
    Default 0.0 = perilaku IoU murni (dipakai kelompok teks).
    """
    if len(boxes) == 0:
        return []
    x1, y1, x2, y2 = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    areas = (x2 - x1).clip(0) * (y2 - y1).clip(0)
    order = scores.argsort()[::-1]
    keep: list[int] = []
    while order.size > 0:
        i = order[0]
        keep.append(int(i))
        rest = order[1:]
        xx1 = np.maximum(x1[i], x1[rest])
        yy1 = np.maximum(y1[i], y1[rest])
        xx2 = np.minimum(x2[i], x2[rest])
        yy2 = np.minimum(y2[i], y2[rest])
        inter = (xx2 - xx1).clip(0) * (yy2 - yy1).clip(0)
        iou = inter / (areas[i] + areas[rest] - inter + 1e-9)
        contain = inter / (np.minimum(areas[i], areas[rest]) + 1e-9)
        order = rest[(iou <= iou_thresh) | (contain < contain_thresh)]
    return keep


def _decode(outputs: list[np.ndarray], w: int, h: int, conf: float) -> tuple[np.ndarray, ...]:
    """Decode keluaran detector.

    Export resmi ogkalu sudah memuat postprocessor RT-DETR: keluarannya
    (labels int64, boxes xyxy ABSOLUT, scores) dan boxes sudah diskalakan oleh
    `orig_target_sizes`. Terverifikasi langsung terhadap detector.onnx —
    3 keluaran, 300 query, skor sudah urut menurun.

    Cabang kedua menangani export mentah (logits, boxes cxcywh ternormalisasi)
    yang dipakai sebagian mirror. RT-DETR memakai sigmoid per-kelas, bukan
    softmax, jadi tidak ada slot background yang harus dibuang.
    """
    empty = (np.zeros((0, 4), np.float32), np.zeros(0, np.float32), np.zeros(0, int))

    if len(outputs) >= 3:
        labels, boxes, scores = outputs[0], outputs[1], outputs[2]
        labels, boxes, scores = np.squeeze(labels, 0), boxes[0], np.squeeze(scores, 0)
        keep = scores >= conf
        if not keep.any():
            return empty
        return (
            boxes[keep].astype(np.float32),
            scores[keep].astype(np.float32),
            labels[keep].astype(int),
        )

    logits, boxes = outputs[0], outputs[1]
    if logits.ndim == 3:
        logits, boxes = logits[0], boxes[0]
    probs = 1.0 / (1.0 + np.exp(-logits))          # sigmoid, bukan softmax
    cls_ids = probs.argmax(axis=1)
    scores = probs.max(axis=1)

    keep = scores >= conf
    if not keep.any():
        return empty

    boxes, scores, cls_ids = boxes[keep], scores[keep], cls_ids[keep]
    cx, cy, bw, bh = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    xyxy = np.stack(
        [(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h],
        axis=1,
    ).astype(np.float32)
    return xyxy, scores.astype(np.float32), cls_ids.astype(int)


def _run_once(img: np.ndarray, conf: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    sess = get_session()
    h, w = img.shape[:2]
    feed = {"images": _preprocess(img, SETTINGS.det_size)}
    # (w, h), BUKAN (h, w): terverifikasi — urutan terbalik menghasilkan box
    # yang meluber melewati tepi kanan halaman.
    if "orig_target_sizes" in _input_names(sess):
        feed["orig_target_sizes"] = np.array([[w, h]], dtype=np.int64)
    outs = sess.run(None, feed)
    return _decode(outs, w, h, conf)


def _tiled(img: np.ndarray, conf: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Pass 2x2 dengan tumpang tindih — resize 640 melewatkan teks kecil."""
    h, w = img.shape[:2]
    ov = 0.15
    th, tw = int(h * (0.5 + ov)), int(w * (0.5 + ov))
    all_b, all_s, all_c = [], [], []
    for oy in (0, h - th):
        for ox in (0, w - tw):
            tile = img[oy : oy + th, ox : ox + tw]
            b, s, c = _run_once(tile, conf)
            if not len(b):
                continue
            b, s, c = _drop_tile_edge(b, s, c, ox, oy, tw, th, w, h)
            if len(b):
                all_b.append(b + np.array([ox, oy, ox, oy], dtype=np.float32))
                all_s.append(s)
                all_c.append(c)
    if not all_b:
        return np.zeros((0, 4), np.float32), np.zeros(0, np.float32), np.zeros(0, int)
    return np.concatenate(all_b), np.concatenate(all_s), np.concatenate(all_c)


def _drop_tile_edge(
    b: np.ndarray, s: np.ndarray, c: np.ndarray,
    ox: int, oy: int, tw: int, th: int, w: int, h: int, margin: float = 2.0,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Buang box yang menempel di garis potong tile.

    Box seperti itu terpotong, dan skornya sering LEBIH TINGGI daripada versi
    utuh dari full-page pass — jadi NMS justru memilih yang cacat. Terukur di
    halaman referensi: balon 415..560 px jadi 448..562 px karena tile mulai di
    x=448. Tepi yang berimpit dengan tepi halaman tidak dihitung: di sana box
    memang boleh mentok.
    """
    edge = np.zeros(len(b), dtype=bool)
    if ox > 0:
        edge |= b[:, 0] <= margin
    if oy > 0:
        edge |= b[:, 1] <= margin
    if ox + tw < w:
        edge |= b[:, 2] >= tw - margin
    if oy + th < h:
        edge |= b[:, 3] >= th - margin
    return b[~edge], s[~edge], c[~edge]


def detect(img: np.ndarray, conf: float | None = None) -> tuple[list[Region], list[tuple]]:
    """Deteksi region teks + bubble.

    Returns:
        (regions, bubbles) — regions hanya text_bubble/text_free; bubbles
        adalah bbox bubble kosong untuk mencari induk tiap region.
    """
    conf = SETTINGS.det_conf if conf is None else conf
    h, w = img.shape[:2]

    b1, s1, c1 = _run_once(img, conf)
    if SETTINGS.tiled_pass and max(h, w) > 900:
        b2, s2, c2 = _tiled(img, conf)
        boxes = np.concatenate([b1, b2]) if len(b2) else b1
        scores = np.concatenate([s1, s2]) if len(s2) else s1
        cls_ids = np.concatenate([c1, c2]) if len(c2) else c1
    else:
        boxes, scores, cls_ids = b1, s1, c1

    if len(boxes) == 0:
        return [], []

    boxes[:, 0::2] = boxes[:, 0::2].clip(0, w)
    boxes[:, 1::2] = boxes[:, 1::2].clip(0, h)

    regions: list[Region] = []
    bubbles: list[tuple[int, int, int, int]] = []
    idx = 0
    # NMS dua kelompok, bukan per-kelas. `bubble` memang tumpang tindih dengan
    # teks di dalamnya jadi harus dipisah, tapi text_bubble dan text_free adalah
    # teks fisik yang sama — NMS per-kelas membiarkan keduanya lolos dan region
    # yang sama masuk pipeline dua kali (terukur di halaman referensi: balon '!'
    # muncul sebagai #3 dan #4).
    is_bubble = cls_ids == 0
    for group, iou_t, contain_t in (
        (is_bubble, SETTINGS.det_iou_bubble, _BUBBLE_CONTAIN),
        (~is_bubble, SETTINGS.det_iou, 0.0),
    ):
        if not group.any():
            continue
        keep = _nms(boxes[group], scores[group], iou_t, contain_thresh=contain_t)
        kb, ks, kc = boxes[group][keep], scores[group][keep], cls_ids[group][keep]
        for box, sc, cid in zip(kb, ks, kc):
            xyxy = tuple(int(v) for v in box)
            if xyxy[2] - xyxy[0] < 4 or xyxy[3] - xyxy[1] < 4:
                continue
            if int(cid) == 0:
                bubbles.append(xyxy)
            else:
                regions.append(
                    Region(
                        idx=idx,
                        bbox=xyxy,
                        det_class=ID2LABEL.get(int(cid), "text_bubble"),
                        det_conf=float(sc),
                    )
                )
                idx += 1

    drop_nested_duplicates(regions)
    regions = assign_bubbles(regions, bubbles)
    return sort_reading_order(regions), bubbles


def drop_nested_duplicates(regions: list[Region]) -> int:
    """Buang kotak teks yang hampir seluruhnya termuat di kotak teks LAIN.

    Detector kadang mengeluarkan dua kotak untuk satu blok teks: satu menutup
    seluruh blok, satu lagi hanya sebagian kolomnya. NMS kelompok teks tidak
    menangkapnya karena dipanggil dengan `contain_thresh=0.0` — IoU pasangan
    bersarang rendah (0.280 pada halaman hitomi_3740721_015) walau containment-nya
    0.974. Keduanya lolos, dan akibatnya BERTUMPUK tiga kali:

    1. Balon berlobus (satu balon, garis luarnya berpinggang) membuat detector
       mengeluarkan TIGA kotak balon: satu untuk seluruh balon plus satu per
       lobus. Terukur di halaman itu: (800,96,1046,442) untuk balonnya, lalu
       (800,117,964,440) dan (934,89,1045,353) untuk kedua lobusnya. Kedua kotak
       teks duplikat lalu memilih induk yang BERBEDA — yang kecil dapat lobus
       kanan, yang besar dapat lobus kiri — jadi masing-masing mengukur
       `fill_color`-nya sendiri: kelabu 152 di kanan (lobus itu duduk di atas
       halaman putih) lawan 120 di kiri (di atas art gelap). Interior keduanya
       beririsan, `disjoin_overlapping_interiors` memotongnya jadi saling lepas,
       dan potongan itulah jahitan bergerigi vertikal yang terlihat di tengah
       balon — dua kelabu berbeda bersebelahan, dengan teks hitam di sebelah
       teks putih karena warna huruf pun dihitung per region.
    2. OCR membaca blok yang sama dua kali; teks yang kecil jadi AWALAN teks yang
       besar (`そうならないように．．．` di dalam `そうならないように．．．生涯に…`).
    3. Keduanya diterjemahkan dan DITATA, jadi kalimat yang sama tercetak dua kali
       di dalam satu balon — melanggar kontrak "tidak boleh saling timpa".

    Yang bertahan adalah kotak yang LEBIH BESAR, dan kotaknya dilebarkan ke gabungan
    keduanya. Melebarkan itu bukan hiasan: kotak kecil bisa menonjol beberapa piksel
    keluar kotak besar (di halaman itu y1 130 lawan 135), dan tinta di sliver itu
    tidak akan tercakup mask siapa pun kalau kotaknya tidak dilebarkan — `find_residue`
    akan menandainya sebagai sisa tinta.

    Bersarang BUKAN balon ganda: lobus balon ganda BERJAJAR, jadi containment-nya
    rendah (tertinggi 0.33 pada halaman bersih yang diukur). Tapi satu kotak besar
    yang benar-benar memuat DUA lobus juga membuat dua kotak kecil bersarang di
    dalamnya — dan untuk kasus itu `_partition_shared_bubbles` +
    `partition_shared_interiors` justru sudah benar. Jadi penyingkiran di sini
    hanya jalan kalau kotak besar itu memuat TEPAT SATU kotak bersarang.

    Harus dipanggil SEBELUM assign_bubbles(): di sanalah kedua duplikat diberi
    induk yang berbeda, dan seluruh mask dibangun sesudahnya. Dedupe setelah OCR
    — walau di sanalah bukti teksnya ada, karena teks kecil persis awalan teks
    besar — tiba terlambat untuk mencegah jahitan dua warna itu.

    Returns:
        Jumlah region yang dibuang.
    """
    n = len(regions)
    if n < 2:
        return 0
    areas = [max(r.width, 0) * max(r.height, 0) for r in regions]

    # nested[j] = daftar i yang bersarang di j. Dikumpulkan lengkap dulu, supaya
    # syarat "tepat satu" bisa diuji sebelum ada yang dibuang.
    nested: dict[int, list[int]] = {}
    for i in range(n):
        for j in range(n):
            if i == j or areas[i] <= 0 or areas[j] <= 0 or areas[i] > areas[j]:
                continue
            ax1, ay1, ax2, ay2 = regions[i].bbox
            bx1, by1, bx2, by2 = regions[j].bbox
            iw = min(ax2, bx2) - max(ax1, bx1)
            ih = min(ay2, by2) - max(ay1, by1)
            if iw <= 0 or ih <= 0:
                continue
            if (iw * ih) / areas[i] >= _DUP_CONTAIN:
                nested.setdefault(j, []).append(i)

    drop: set[int] = set()
    for j, inner in nested.items():
        if len(inner) != 1:
            continue          # dua lobus dalam satu kotak = balon ganda, jangan disentuh
        i = inner[0]
        if i in drop or j in drop:
            continue
        drop.add(i)
        ax1, ay1, ax2, ay2 = regions[i].bbox
        bx1, by1, bx2, by2 = regions[j].bbox
        regions[j].bbox = (min(ax1, bx1), min(ay1, by1),
                           max(ax2, bx2), max(ay2, by2))

    if not drop:
        return 0
    kept = [r for k, r in enumerate(regions) if k not in drop]
    regions[:] = kept
    return len(drop)


def assign_bubbles(regions: list[Region], bubbles: list[tuple]) -> list[Region]:
    """Cari bubble induk tiap region: yang memuat teksnya paling utuh, lalu terkecil.

    Pakai overlap, bukan containment ketat. Kotak teks kerap menonjol beberapa
    piksel keluar dari kotak balon — terukur di halaman referensi: teks mulai di
    x=1041 sedangkan balonnya di x=1046, jadi containment ketat menolak pasangan
    yang jelas benar. Akibatnya typeset kehilangan mask interior balon, menata
    teks di kotak mentah, dan memberi stroke putih yang tidak seharusnya ada.

    Urutan pemilihannya CAKUPAN dulu, baru area — lihat _PARENT_SLACK. "Terkecil
    yang memuat mayoritas" saja memilih kotak LOBUS di atas kotak balon penuh
    pada balon berlobus, dan lobus yang salah pilih itu meninggalkan tinta Jepang
    di lobus sebelahnya tanpa terhapus.
    """
    for r in regions:
        rx1, ry1, rx2, ry2 = r.bbox
        r_area = max((rx2 - rx1) * (ry2 - ry1), 1)
        cands: list[tuple[float, int, tuple]] = []
        for b in bubbles:
            bx1, by1, bx2, by2 = b
            iw = min(rx2, bx2) - max(rx1, bx1)
            ih = min(ry2, by2) - max(ry1, by1)
            if iw <= 0 or ih <= 0:
                continue
            cover = (iw * ih) / r_area
            if cover < _PARENT_OVERLAP:  # mayoritas teks di dalam balon
                continue
            cands.append((cover, (bx2 - bx1) * (by2 - by1), b))
        best = None
        if cands:
            top = max(c[0] for c in cands)
            best = min((c for c in cands if c[0] >= top - _PARENT_SLACK),
                       key=lambda c: c[1])[2]
        r.bubble_bbox = best
        if best is not None and r.det_class == "text_free":
            r.det_class = "text_bubble"  # ternyata di dalam bubble
    _partition_shared_bubbles(regions)
    return regions


def sort_reading_order(regions: list[Region]) -> list[Region]:
    """Urutan baca manga: kanan->kiri, atas->bawah, dikelompokkan per baris."""
    if not regions:
        return regions
    heights = [r.height for r in regions] or [20]
    band = max(int(np.median(heights) * 1.3), 20)
    rows: list[list[Region]] = []
    for r in sorted(regions, key=lambda x: x.bbox[1]):
        placed = False
        for row in rows:
            if abs(row[0].bbox[1] - r.bbox[1]) < band:
                row.append(r)
                placed = True
                break
        if not placed:
            rows.append([r])
    ordered: list[Region] = []
    for row in rows:
        ordered.extend(sorted(row, key=lambda x: -x.bbox[0]))
    for i, r in enumerate(ordered):
        r.idx = i
    return ordered


def release() -> None:
    """Bebaskan sesi ONNX — RAM Colab cuma ~12.7 GB."""
    global _SESSION
    _SESSION = None


def _partition_shared_bubbles(regions: list[Region]) -> None:
    """Double bubble: SATU kotak balon berisi >= 2 region -> bagi balon per region.

    RT-DETR kadang mengeluarkan satu kotak bubble untuk dua balon yang menyatu
    (double bubble). Tanpa pemecahan, kedua region memakai mask gabungan
    (figura-8) dan dua terjemahan ditumpuk di tengahnya — saling timpa.

    Belahan di sini hanya KASAR: potongan persegi tidak sejajar bentuk lobus,
    jadi tiap belahan masih memuat sebagian lobus sebelah. Kotak asli disimpan
    di `shared_bubble_bbox` supaya textmask.partition_shared_interiors() bisa
    memartisi piksel interiornya mengikuti bentuk lobus sungguhan; belahan
    persegi ini tinggal jadi fallback kalau partisi itu gagal.
    """
    from collections import defaultdict

    groups: dict[tuple[int, int, int, int], list[Region]] = defaultdict(list)
    for r in regions:
        if r.bubble_bbox is not None:
            groups[r.bubble_bbox].append(r)

    def _cx(r: Region) -> float:
        return (r.bbox[0] + r.bbox[2]) / 2

    def _cy(r: Region) -> float:
        return (r.bbox[1] + r.bbox[3]) / 2

    for bbox, grp in groups.items():
        if len(grp) < 2:
            continue
        bx1, by1, bx2, by2 = bbox
        for r in grp:
            r.shared_bubble_bbox = bbox
        by_x = sorted(grp, key=_cx)
        by_y = sorted(grp, key=_cy)
        # Belah sepanjang sumbu dengan sebaran centroid terbesar.
        horizontal = _cx(by_x[-1]) - _cx(by_x[0]) >= _cy(by_y[-1]) - _cy(by_y[0])

        order = by_x if horizontal else by_y
        prev = bx1 if horizontal else by1
        for i, r in enumerate(order):
            if i == len(order) - 1:
                r.bubble_bbox = ((prev, by1, bx2, by2) if horizontal
                                 else (bx1, prev, bx2, by2))
                continue
            nxt = order[i + 1]
            if horizontal:
                gap = ((r.bbox[2] + nxt.bbox[0]) // 2 if r.bbox[2] < nxt.bbox[0]
                       else int((_cx(r) + _cx(nxt)) / 2))
                cut = max(gap, prev + 4)
                r.bubble_bbox = (prev, by1, cut, by2)
            else:
                gap = ((r.bbox[3] + nxt.bbox[1]) // 2 if r.bbox[3] < nxt.bbox[1]
                       else int((_cy(r) + _cy(nxt)) / 2))
                cut = max(gap, prev + 4)
                r.bubble_bbox = (bx1, prev, bx2, cut)
            prev = cut



Writing /content/mangatl/detect.py


In [7]:
%%writefile /content/mangatl/textmask.py

"""Pembangun mask teks — bagian paling menentukan kualitas hasil akhir.

Urutan operasinya bukan sembarangan. Langkah dual-polarity Otsu dan halo pass
adalah yang biasanya dilewatkan orang, dan itu penyebab ghost outline.
"""

from __future__ import annotations

import cv2
import numpy as np

try:
    import onnxruntime as ort
except ImportError:
    ort = None  # jalur Otsu tetap jalan tanpa ORT

from config import SETTINGS, WEIGHTS, Region, ort_session

_CTD = None
_CTD_FAILED = False


def get_ctd():
    """comic-text-detector opsional: kalau gagal muat, jalur Otsu tetap jalan."""
    global _CTD, _CTD_FAILED
    if _CTD is not None or _CTD_FAILED:
        return _CTD
    path = WEIGHTS / "comictextdetector.pt.onnx"
    if ort is None or not path.exists():
        _CTD_FAILED = True
        return None
    try:
        _CTD = ort_session(path, "ctd")
    except Exception:  # noqa: BLE001 - ORT melempar tipe khusus per build
        _CTD_FAILED = True
        _CTD = None
    return _CTD


def _letterbox(img: np.ndarray, size: int = 1024) -> tuple[np.ndarray, float, int, int, int, int]:
    """Resize menjaga rasio lalu pad ke size x size. RGB dulu, baru /255."""
    h, w = img.shape[:2]
    r = min(size / h, size / w)
    nh, nw = int(round(h * r)), int(round(w * r))
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    ph, pw = size - nh, size - nw
    top, left = ph // 2, pw // 2
    out = cv2.copyMakeBorder(
        resized, top, ph - top, left, pw - left, cv2.BORDER_CONSTANT, value=(114, 114, 114)
    )
    return out, r, left, top, nw, nh


def ctd_soft_mask(img: np.ndarray) -> np.ndarray | None:
    """Mask teks lembut skala halaman penuh, nilai float 0..1. None kalau CTD mati."""
    sess = get_ctd()
    if sess is None:
        return None
    h, w = img.shape[:2]
    lb, _r, dx, dy, nw, nh = _letterbox(img, 1024)
    inp = (lb.transpose(2, 0, 1).astype(np.float32) / 255.0)[None]
    try:
        outs = sess.run(None, {sess.get_inputs()[0].name: inp})
    except (RuntimeError, ValueError):
        return None

    # Address by shape, bukan index — urutan output CTD berbeda antar build,
    # dan cv2.dnn punya bug yang menukar mask dengan lines_map.
    mask = None
    for o in outs:
        a = np.squeeze(o)
        if a.ndim == 2 and min(a.shape) >= 128:
            mask = a
            break
        if a.ndim == 3 and a.shape[0] == 1 and min(a.shape[1:]) >= 128:
            mask = a[0]
            break
    if mask is None:
        return None

    mask = mask.astype(np.float32)
    if mask.max() > 1.5:
        mask /= 255.0
    mask = cv2.resize(mask, (1024, 1024), interpolation=cv2.INTER_LINEAR)
    crop = mask[dy : dy + nh, dx : dx + nw]
    if crop.size == 0:
        return None
    return cv2.resize(crop, (w, h), interpolation=cv2.INTER_LINEAR).clip(0, 1)


def _dual_polarity_otsu(gray: np.ndarray) -> np.ndarray:
    """OR-kan biner teks-gelap dengan teks-terang.

    Satu polaritas saja melewatkan glyph putih ber-outline hitam, yang umum
    di manga untuk teks di atas art gelap.
    """
    blur = cv2.GaussianBlur(gray, (3, 3), 0)
    _, dark = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    _, light = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    # Pilih polaritas minoritas: teks selalu lebih sedikit dari background.
    return dark if dark.mean() <= light.mean() else light


def _glyph_height(binary: np.ndarray, vertical: bool = False) -> float:
    """Ukuran glyph dari connected components — dasar estimasi ukuran font.

    Diukur MELINTANG terhadap arah tulisan, bukan selalu tinggi. Di teks Jepang
    vertikal, glyph yang bertumpuk menyatu jadi satu komponen menjulur: terukur
    di halaman uji, kolom selebar 104 px menghasilkan komponen setinggi 163 px,
    jadi "tinggi" melaporkan panjang rangkaian, bukan ukuran huruf. Karena glyph
    CJK persegi, lebar kolom itulah ukuran font sebenarnya. Salah di sini merusak
    dua hal sekaligus: est_font_size dan kernel dilasi yang diturunkan darinya.

    Median mentah juga salah: dakuten, handakuten, dan tanda baca kecil menarik
    median ke bawah (ドドド 76 px terbaca 12 px), jadi komponen kecil dibuang
    dulu relatif terhadap yang terbesar.
    """
    stat = cv2.CC_STAT_WIDTH if vertical else cv2.CC_STAT_HEIGHT
    n, _, stats, _ = cv2.connectedComponentsWithStats((binary > 0).astype(np.uint8), 8)
    hs = np.array([
        stats[i, stat]
        for i in range(1, n)
        if stats[i, cv2.CC_STAT_AREA] >= SETTINGS.min_cc_area
    ], dtype=np.float32)
    if hs.size == 0:
        fallback = binary.shape[1] if vertical else binary.shape[0]
        return float(max(fallback * 0.25, 8))
    body = hs[hs >= hs.max() * 0.45]  # buang diakritik & titik
    return float(np.median(body if body.size else hs))


def _adaptive_dilate(mask: np.ndarray, glyph_h: float) -> np.ndarray:
    """Kernel per-glyph, bukan global.

    Kernel global salah dua arah sekaligus: memakan art di sekitar teks kecil,
    dan kurang menutup teks besar.
    """
    k = max((int((glyph_h + 30) * SETTINGS.dilate_ratio) // 2) * 2 + 1, 3)
    k = min(k, 31)
    el = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    return cv2.dilate(mask, el, iterations=1)


def _halo_pass(mask: np.ndarray, gray: np.ndarray, bg: float) -> np.ndarray:
    """Tangkap tepi anti-alias di sekeliling glyph.

    Melewatkan langkah ini adalah penyebab nomor satu ghost outline: stroke
    utamanya terhapus tapi bayangan abu-abunya tertinggal.
    """
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    grown = cv2.dilate(mask, k, iterations=1)
    ring = cv2.subtract(grown, mask)
    deviating = (np.abs(gray.astype(np.int16) - bg) > SETTINGS.halo_deviation).astype(np.uint8) * 255
    return cv2.bitwise_or(mask, cv2.bitwise_and(ring, deviating))


def build_region_mask(img: np.ndarray, region: Region, soft: np.ndarray | None) -> None:
    """Isi region.ink_mask, region.bubble_mask, region.est_font_size, ink_ratio.

    Semua mask disimpan pada koordinat lokal bbox region.
    """
    x1, y1, x2, y2 = region.bbox
    pad = 6
    h, w = img.shape[:2]
    px1, py1 = max(0, x1 - pad), max(0, y1 - pad)
    px2, py2 = min(w, x2 + pad), min(h, y2 + pad)
    crop = img[py1:py2, px1:px2]
    if crop.size == 0:
        region.ink_mask = np.zeros((1, 1), np.uint8)
        return

    gray = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)

    # 1-2. seed + grow dari CTD kalau tersedia
    ink = None
    if soft is not None:
        sub = soft[py1:py2, px1:px2]
        seed = (sub >= SETTINGS.seed_thresh).astype(np.uint8) * 255
        grow = (sub >= SETTINGS.grow_thresh).astype(np.uint8) * 255
        if seed.any():
            # rekonstruksi: hanya komponen grow yang menyentuh seed
            n, lab = cv2.connectedComponents(grow // 255)
            keep = np.unique(lab[seed > 0])
            ink = np.isin(lab, keep[keep > 0]).astype(np.uint8) * 255

    # 3-4. dual-polarity Otsu — jalur utama kalau CTD tidak ada, penguat kalau ada.
    #
    # Saat CTD ada, Otsu TIDAK di-OR mentah. Otsu memilih polaritas minoritas, dan
    # untuk teks di atas screentone padat yang minoritas itu justru ART-nya: pada
    # kolom narasi halaman uji, CTD sendiri menutup 27% kotak sedangkan hasil OR
    # mentah menutup 57% — seluruh kolom, art dan semua. Jadi Otsu dibatasi ke
    # pita tipis di sekitar tinta CTD: cukup untuk menambal tepi anti-alias dan
    # stroke tipis yang CTD lewatkan, tanpa mengimpor art di sekelilingnya.
    otsu = _dual_polarity_otsu(gray)
    if ink is None or ink.sum() == 0:
        ink = otsu
    else:
        band = cv2.dilate(ink, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)))
        ink = cv2.bitwise_or(ink, cv2.bitwise_and(otsu, band))

    # Buang komponen yang jelas bukan glyph: noise, garis panel, dan bingkai.
    # Bingkai kotak narasi paling menipu — luasnya kecil (cuma stroke) tapi
    # bounding box-nya sebesar crop, jadi harus disaring lewat fill ratio.
    n, lab, stats, _ = cv2.connectedComponentsWithStats((ink > 0).astype(np.uint8), 8)
    clean = np.zeros_like(ink)
    ih, iw = ink.shape[:2]
    box_area = ih * iw
    for i in range(1, n):
        area = stats[i, cv2.CC_STAT_AREA]
        ch = stats[i, cv2.CC_STAT_HEIGHT]
        cw = stats[i, cv2.CC_STAT_WIDTH]
        if area < 8 or area > box_area * 0.55:
            continue
        bb = ch * cw
        if bb > box_area * 0.30 and area < bb * 0.25:
            continue  # hollow: bingkai / outline balon, bukan huruf
        if ch > ih * 0.90 and cw > iw * 0.90:
            continue
        clean[lab == i] = 255
    if clean.sum() > 0:
        ink = clean

    # Arah tulisan harus diketahui SEBELUM ukuran glyph diukur — pengukurannya
    # melintang terhadap arah itu.
    region.is_vertical = (py2 - py1) > (px2 - px1) * 1.6
    glyph_h = _glyph_height(ink, vertical=region.is_vertical)
    region.est_font_size = glyph_h

    bg = float(np.median(gray[ink == 0])) if (ink == 0).any() else 255.0

    # 5. dilasi adaptif  6. halo pass
    ink = _adaptive_dilate(ink, glyph_h)
    ink = _halo_pass(ink, gray, bg)

    region.ink_mask = ink
    region.ink_ratio = float((ink > 0).mean())
    # bbox ikut padding dulu, baru bubble_mask dibuat — supaya ukuran mask
    # cocok dengan kotak yang dipakai typeset.
    region.bbox = (px1, py1, px2, py2)
    region.bubble_mask = _bubble_interior(img, region)
    # ...lalu mask ISIAN direkam, SEBELUM disjoin_overlapping_interiors()
    # memangkas bubble_mask. Lihat build_fill_mask().
    build_fill_mask(img, region)


def build_fill_mask(img: np.ndarray, region: Region) -> None:
    """Isi region.fill_mask/fill_bbox: SELURUH interior balon untuk erase.

    Kenapa interior penuh dan bukan ink_mask: ink_mask dibangun dari ambang +
    dilasi adaptif dan sistematis melewatkan glyph tipis/renggang. Terukur di
    hasilnew/jp_6.JPG setelah hapusan — tanda '——' selamat utuh sebagai garis
    tipis, 'うう…' menyisakan dua coretan. Menaikkan dilasi cuma memindahkan
    cacatnya ke garis balon yang ikut termakan; interior balon tidak punya
    masalah itu karena batasnya garis balon sungguhan, bukan taksiran.

    Kenapa direkam di sini, bukan dibaca dari bubble_mask saat erase:
    disjoin_overlapping_interiors() MEMANGKAS bubble_mask supaya tata letak dua
    balon bertetangga tidak beririsan. Mengisi pakai mask yang sudah dipangkas
    membuat sliver yang dipangkas itu tidak pernah tersentuh — dan justru di
    sliver itu tinta Jepang paling mungkin tertinggal, karena letaknya di tepi.

    Kenapa hanya kalau ada balon induk: bubble_bbox None berarti teks duduk di
    ART, bukan di balon. Memutihkan persegi di atas art adalah cacat yang jauh
    lebih parah daripada satu titik sisa, jadi region itu tetap jalur ink_mask.
    """
    region.fill_bbox = None
    region.fill_mask = None
    if not SETTINGS.bubble_fill or region.bubble_bbox is None:
        return
    bx1, by1, bx2, by2 = region.bubble_bbox
    crop = img[by1:by2, bx1:bx2]
    if crop.size == 0:
        return
    # Kikis lebih sedikit daripada interior untuk tata letak: isian harus mepet
    # garis supaya tidak ada pita tinta lama tertinggal di tepi, sementara tata
    # letak perlu jarak aman supaya glyph tidak menempel di garis.
    stroke = max(int(round(_stroke_px(region.est_font_size)
                           * SETTINGS.fill_erode_stroke)), 1)
    interior = _interior_from_crop(crop, stroke, _ink_center(region, bx1, by1),
                                   _ink_in_crop(region, bx1, by1, crop.shape[:2]))
    if not interior.any():
        return
    # Pangkas ke komponen yang MEMUAT tinta region ini — lihat _keep_ink_lobes().
    interior = _keep_ink_lobes(interior, region, bx1, by1)
    if interior is None:
        return
    # Interior yang praktis seluruh kotak = flood fill bocor keluar balon dan
    # mengisi art. Lebih baik jatuh ke ink_mask daripada memutihkan panel.
    # Diuji SETELAH pangkas: sebelum pangkas ambang ini tidak bisa dipakai sama
    # sekali, karena interior yang SAH pun sampai 0.900 di halaman uji (r9).
    if float((interior > 0).mean()) > 0.97:
        return
    region.fill_bbox = region.bubble_bbox
    region.fill_mask = interior


def _keep_ink_lobes(interior: np.ndarray, region: Region,
                    ox: int, oy: int) -> np.ndarray | None:
    """Buang komponen interior yang tidak memuat tinta region ini. None = batal.

    Ini penjaga kebocoran isian yang sesungguhnya. Alasannya STRUKTURAL, bukan
    ambang: isian hanya boleh mengisi rongga tempat tinta Jepangnya berada.
    Piksel terang yang tersambung ke tinta itu memang interior balon; gumpalan
    terang lain di dalam kotak yang sama adalah ART di luar garis balon, dan
    mengecatnya dengan warna balon persis cacat 'isian keluar dari balon'.

    Kenapa ambang ukuran tidak bisa dipakai (terukur di jepang_002.webp,
    _fillcal.py + _fillguard2.py):

      * cover (fraksi kotak yang terisi) di 13 region BERSIH = 0.598-0.900.
        Band _DISCOVER_FILL (0.15, 0.85) akan menolak r9 di 0.900 yang isinya
        benar, jadi angka itu tidak bisa dipindah ke sini.
      * fraksi TEPI kotak yang terisi = 0.0000-0.6946 di halaman bersih. Tidak
        memisahkan apa pun.
      * sebaran warna isian juga tidak: interior balon dan kertas kosong di
        luar balon dua-duanya putih rata (spread <= 1.48 di 13 region).

    Yang dipangkas nyata, bukan hipotetis: r9 halaman uji punya cc=11 dan
    turun cover 0.900 -> 0.817; sebaran warna pita-luar isiannya jatuh dari
    19.27 ke 0.00 dan selisih median dari 13 ke 0 — bukti gumpalan yang
    dibuang itu memang art, bukan interior. 12 region lain tidak berubah
    (keepfrac >= 0.9996), jadi pangkas ini bukan pertukaran untung-rugi.

    Ia juga menutup jalur terburuk: `_interior_from_crop` punya jalur mundur
    `interior = binv` (SELURUH piksel terang crop, art dan semua) ketika flood
    fill gagal. Di halaman uji jalur itu bercover 0.82-0.94 — di bawah ambang
    0.97 sehingga LOLOS — dan pangkas menariknya kembali ke 0.66-0.83.

    Returns:
        Interior terpangkas, atau None kalau tidak ada komponen yang memuat
        tinta region ini sama sekali (isian yang tidak memuat teksnya sendiri
        tidak pernah benar; lebih baik jatuh ke jalur ink_mask).
    """
    if region.ink_mask is None:
        return interior
    ih, iw = interior.shape[:2]
    ink = np.zeros((ih, iw), np.uint8)
    x1, y1 = region.bbox[0], region.bbox[1]
    mh, mw = region.ink_mask.shape[:2]
    sy, sx = y1 - oy, x1 - ox
    dy, dx = max(sy, 0), max(sx, 0)
    hh, ww = min(mh + sy, ih) - dy, min(mw + sx, iw) - dx
    if hh <= 0 or ww <= 0:
        return interior
    ink[dy:dy + hh, dx:dx + ww] = region.ink_mask[dy - sy:dy - sy + hh,
                                                  dx - sx:dx - sx + ww]
    n, lab = cv2.connectedComponents((interior > 0).astype(np.uint8), 8)
    if n <= 2:
        return interior          # satu komponen: tidak ada yang bisa dipangkas
    hit = set(int(v) for v in np.unique(lab[(ink > 0) & (interior > 0)])) - {0}
    if not hit:
        return None
    return np.where(np.isin(lab, sorted(hit)), 255, 0).astype(np.uint8)


def _white_seed(binv: np.ndarray, near: tuple[int, int] | None = None) -> tuple[int, int]:
    """Piksel interior (255) terdekat ke `near` — titik awal flood fill.

    Default `near` = pusat crop. Untuk balon ganda pusat crop bisa jatuh di
    lobus sebelah, jadi pemanggil memberi titik tengah tinta region.

    Setelah inversi polaritas di _interior_from_crop, 'interior' ini bisa balon
    terang maupun gelap - nilai 255 selalu menandai interior."""
    hh, ww = binv.shape[:2]
    cx, cy = near if near is not None else (ww // 2, hh // 2)
    cx, cy = int(np.clip(cx, 0, ww - 1)), int(np.clip(cy, 0, hh - 1))
    ys, xs = np.nonzero(binv)
    if ys.size == 0:
        return cx, cy
    i = int(np.argmin((ys - cy) ** 2 + (xs - cx) ** 2))
    return int(xs[i]), int(ys[i])


def _ink_center(region: Region, ox: int, oy: int) -> tuple[int, int]:
    """Titik tengah tinta region, dalam koordinat crop yang mulai di (ox, oy)."""
    x1, y1, x2, y2 = region.bbox
    cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
    ink = region.ink_mask
    if ink is not None and ink.any():
        ys, xs = np.nonzero(ink)
        cx, cy = x1 + int(xs.mean()), y1 + int(ys.mean())
    return cx - ox, cy - oy


def _stroke_px(glyph_h: float) -> int:
    """Perkiraan ketebalan garis balon dari tinggi glyph, dibatasi 1..4 px."""
    return int(np.clip(round(glyph_h * 0.06), 1, 4))


def _fill_holes(mask: np.ndarray) -> np.ndarray:
    """Tutup lubang TERTUTUP di dalam mask tanpa memekarkan tepi luarnya.

    Versi lama memakai findContours(RETR_EXTERNAL) + drawContours(FILLED).
    Kontur terluar daerah terisi ADALAH garis balon, jadi mengisinya penuh
    membuat interior menelan stroke hitam balon — probe baris lalu menganggap
    piksel di atas garis masih 'di dalam' dan teks dirender menembus balon.
    Flood fill dari luar hanya memulihkan lubang yang benar-benar tertutup.
    """
    # Pagar 1 px bernilai latar menjamin benih (0,0) ada di luar, walau daerah
    # terisi menyentuh tepi crop.
    inv = cv2.copyMakeBorder(
        cv2.bitwise_not(mask), 1, 1, 1, 1, cv2.BORDER_CONSTANT, value=255
    )
    ff = np.zeros((inv.shape[0] + 2, inv.shape[1] + 2), np.uint8)
    cv2.floodFill(inv, ff, (0, 0), 0)
    return cv2.bitwise_or(mask, inv[1:-1, 1:-1])


def _ink_in_crop(region: Region, ox: int, oy: int,
                 shape: tuple[int, int]) -> np.ndarray:
    """ink_mask region dipetakan ke koordinat crop yang mulai di (ox, oy)."""
    hh, ww = shape[0], shape[1]
    out = np.zeros((hh, ww), np.uint8)
    ink = region.ink_mask
    if ink is None:
        return out
    x1, y1 = region.bbox[0] - ox, region.bbox[1] - oy
    mh, mw = ink.shape[:2]
    sy1, sx1 = max(y1, 0), max(x1, 0)
    sy2, sx2 = min(y1 + mh, hh), min(x1 + mw, ww)
    if sy2 > sy1 and sx2 > sx1:
        out[sy1:sy2, sx1:sx2] = ink[sy1 - y1:sy2 - y1, sx1 - x1:sx2 - x1]
    return out


# Cincin latar tinta = tinta didilatasi sebesar ini, minus tinta itu sendiri.
# Piksel di cincin inilah LATAR tempat teks region berdiri, jadi kelas Otsu
# yang memuatnya adalah interior balon — apa pun kecerahan absolutnya.
_RING_K = 9
# Saat polaritas dibalik, tinta dan GARIS balon masuk kelas yang sama dengan
# interior kelabu, jadi flood fill bisa menembus garis. Piksel yang lebih gelap
# dari latar sebanyak ini dijadikan dinding. MAD dipakai supaya balon
# ber-screentone kasar tidak ikut terpotong; lantai 20 supaya balon rata tetap
# punya dinding walau MAD-nya 0.
_WALL_MAD, _WALL_MIN = 3.0, 20.0


def _polarity_ring(gray: np.ndarray, binv: np.ndarray,
                   ink: np.ndarray | None) -> tuple[bool, float, float]:
    """Apakah polaritas harus dibalik, plus (bg, mad) latar tinta region.

    Aturan lama absolut: `median(kelas mayoritas) < 128` -> balik. Itu benar
    hanya untuk dua ujung (balon putih / balon hitam) dan SALAH untuk balon
    ber-screentone kelabu, yang di manga ini justru dipakai untuk balon dalam
    panel gelap. Terukur di cacatbaru/jp_cacatnew1+2 (_cnpol.py): median kelas
    mayoritas 140/124/145 — dua di atas 128 jadi tidak dibalik — sedangkan
    latar tinta region berada di kelas TERANG hanya 0.000/0.014/0.025. Artinya
    yang diambil sebagai 'interior' adalah HALAMAN PUTIH DI LUAR balon, bukan
    rongga balonnya. Satu mask salah itu memunculkan dua cacat sekaligus:
    build_fill_mask mengisi lobus yang salah (tinta Jepang tidak pernah
    terhapus) dan typeset menata teks di sliver luar balon (terjemahan tercetak
    mungil di atas art).

    Penggantinya STRUKTURAL, bukan ambang: interior balon adalah kelas yang
    memuat CINCIN LATAR di sekeliling tinta region — alasan yang sama dengan
    _keep_ink_lobes ("isian hanya boleh mengisi rongga tempat tinta Jepangnya
    berada"). Tanpa tinta (pemanggil lama / ink_mask kosong) aturan absolut
    lama tetap dipakai sebagai jalur mundur.

    Terukur pada 12 region balon PUTIH bersih di hasilnew/jp_6 + jp_13
    (_cnband.py): keputusan aturan cincin sama dengan aturan lama di
    SEMUA-nya, dan cover interiornya identik. Jadi ini bukan pertukaran.
    """
    if ink is not None and ink.any():
        ring = (cv2.dilate(ink, cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE, (_RING_K, _RING_K))) > 0) & (ink == 0)
        if ring.any():
            vals = gray[ring].astype(np.int16)
            bg = float(np.median(vals))
            mad = float(np.median(np.abs(vals - bg)))
            return float((binv[ring] > 0).mean()) < 0.5, bg, mad
    vals = gray[binv > 0]
    if vals.size < binv.size - vals.size:
        vals = gray[binv == 0]
    return float(np.median(vals)) < 128, -1.0, -1.0


def _interior_from_crop(crop: np.ndarray, stroke: int,
                        seed: tuple[int, int] | None = None,
                        ink: np.ndarray | None = None) -> np.ndarray:
    """Interior balon dari satu crop: Otsu -> flood fill -> tambal -> kikis."""
    gray = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)
    _, binv = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    # Balon gelap (teks putih di atas hitam) DAN balon kelabu ber-screentone:
    # Otsu mengasumsikan interior TERANG, jadi hasilnya kebalik - interior jadi
    # 0 dan teks jadi 255, lalu flood fill malah mengisi glyph. Polaritas
    # ditentukan dari cincin latar tinta region; lihat _polarity_ring().
    balik, bg, mad = _polarity_ring(gray, binv, ink)
    if balik:
        binv = cv2.bitwise_not(binv)
        # Setelah dibalik, GARIS balon ikut masuk kelas interior dan flood fill
        # bisa menembusnya lalu mengisi art gelap di luar balon. Terukur di
        # jp_cacatnew1 (_cnwall3.py): tanpa dinding 16.0% piksel interior jatuh
        # di art gelap (taji keluar ke garis rambut), dengan dinding 5.5%.
        if bg >= 0:
            lantai = bg - max(_WALL_MAD * mad, _WALL_MIN)
            binv = cv2.bitwise_and(binv, (gray > lantai).astype(np.uint8) * 255)
    hh, ww = binv.shape

    # Flood fill dari benih menangkap interior, bukan art di luar bubble — tapi
    # HANYA kalau benihnya jatuh di putih. Titik tengah balon justru sering kena
    # goresan huruf (pada halaman uji, piksel tengah bernilai 3), dan flood fill
    # dari sana mengisi goresan itu: interior terbaca 1.2% lalu balon dianggap
    # penuh, sehingga tiap baris gagal probe dan teks dipaksa ke ukuran minimum.
    ff = binv.copy()
    m = np.zeros((hh + 2, ww + 2), np.uint8)
    cv2.floodFill(ff, m, _white_seed(binv, seed), 128)
    interior = (ff == 128).astype(np.uint8) * 255

    # Hitung PIKSEL, bukan jumlah nilai: mask ini 0/255, jadi .sum() 255x lebih
    # besar dari cacah piksel dan ambang 5% ini diam-diam jadi 0.02% — jaring
    # pengaman yang tidak pernah menangkap apa pun.
    if int((interior > 0).sum()) < hh * ww * 0.05:
        interior = binv

    # Dinding gelap di atas MEMBUANG tinta region dari kelas interior, jadi
    # tiap glyph jadi lubang. _fill_holes hanya menambal lubang TERTUTUP, dan
    # glyph yang menempel di garis balon terbuka ke tepi — lubangnya bertahan.
    # Itu tidak boleh dibiarkan: build_fill_mask memakai interior ini untuk
    # MENGHAPUS tinta Jepang (erase_flat memakai fill_mask, bukan ink_mask),
    # jadi interior berlubang sebentuk glyph meninggalkan sisa tinta — persis
    # cacat yang sedang diperbaiki. Tinta region ada di DALAM balon menurut
    # definisi, jadi dipulihkan di sini. Terukur di _cnwall3.py: cakupan tinta
    # 0.92/0.89/0.71 -> 0.99/0.99/1.00, sementara kebocoran ke art gelap tetap
    # 5.5%/4.7%/6.6% (tanpa dinding: 16.0%/11.4%/8.2%).
    if ink is not None and ink.shape[:2] == interior.shape[:2]:
        interior = cv2.bitwise_or(interior, ink)
        binv = cv2.bitwise_or(binv, ink)

    # Tambal lubang bekas glyph. Mask ini dibangun dari gambar ASLI yang teksnya
    # masih ada, jadi flood fill mengalir MENGELILINGI tiap huruf dan menyisakan
    # stroke-nya sebagai lubang. Padahal teks itu dihapus sebelum typeset, jadi
    # lubangnya semu — tapi _row_free merata-rata dan ikut menghitungnya, membuat
    # balon terbaca cuma 56-71% bebas lalu baris gagal probe padahal ruangnya ada.
    interior = _fill_holes(interior)
    # Closing menyambung takik glyph yang MENEMPEL di garis balon — takik begitu
    # terbuka ke tepi, jadi _fill_holes tidak bisa menutupnya. Tapi closing juga
    # mengisi cekungan bentuk: di leher balon ganda kernel 7 px membuat interior
    # menonjol ke ATAS garis balon, dan teks lalu dirender menyentuh garis.
    # Jadi hasilnya dikurung ke piksel yang bukan garis: `binv` sudah membuang
    # semua piksel gelap, dan lubang glyph yang ikut terbuang sudah ditambal
    # `interior` di baris atas.
    closed = cv2.morphologyEx(
        interior, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    )
    interior = cv2.bitwise_and(closed, cv2.bitwise_or(interior, binv))
    # Flood fill berhenti di piksel anti-aliased garis balon, jadi tepi interior
    # masih menyentuh garis. Kikis setebal stroke supaya baris terluar tidak
    # pernah menempel di garis balon.
    k = 2 * max(stroke, 1) + 1
    return cv2.erode(interior, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)))


# Jendela pencarian balon = bbox region dilebarkan sebanyak ini ke tiap sisi.
_DISCOVER_DILATE = 0.40
# Interior temuan harus mengisi sebagian jendela yang wajar: terlalu kecil =
# flood fill terjebak antar-huruf, terlalu besar = ia bocor keluar balon dan
# mengisi art. Dua-duanya lebih buruk dari persegi mentah.
_DISCOVER_FILL = (0.15, 0.85)


def _discover_bubble(
    img: np.ndarray, region: Region, stroke: int
) -> tuple[tuple[int, int, int, int], np.ndarray] | None:
    """Cari balon induk langsung dari gambar saat detector kehilangan kotaknya.

    Tanpa ini, `bubble_bbox is None` memberi mask persegi 255 penuh dan fit()
    bebas membesar sampai teks memotong garis balon. Hanya untuk text_bubble:
    text_free memang tidak punya balon, persegi bbox-nya sudah benar.
    """
    if region.det_class != "text_bubble":
        return None
    h, w = img.shape[:2]
    x1, y1, x2, y2 = region.bbox
    mx = int((x2 - x1) * _DISCOVER_DILATE)
    my = int((y2 - y1) * _DISCOVER_DILATE)
    wx1, wy1 = max(x1 - mx, 0), max(y1 - my, 0)
    wx2, wy2 = min(x2 + mx, w), min(y2 + my, h)
    crop = img[wy1:wy2, wx1:wx2]
    if crop.size == 0:
        return None

    interior = _interior_from_crop(crop, stroke, _ink_center(region, wx1, wy1),
                                   _ink_in_crop(region, wx1, wy1, crop.shape[:2]))
    lo, hi = _DISCOVER_FILL
    if not lo <= float((interior > 0).mean()) <= hi:
        return None
    ys, xs = np.nonzero(interior)
    bx1, by1 = wx1 + int(xs.min()), wy1 + int(ys.min())
    bx2, by2 = wx1 + int(xs.max()) + 1, wy1 + int(ys.max()) + 1
    # Balon yang benar memuat teksnya sendiri. Toleransi 10% karena bbox region
    # sudah ikut padding dan bisa menonjol beberapa piksel keluar interior.
    tx, ty = max((x2 - x1) // 10, 2), max((y2 - y1) // 10, 2)
    if bx1 > x1 + tx or by1 > y1 + ty or bx2 < x2 - tx or by2 < y2 - ty:
        return None
    return (bx1, by1, bx2, by2), interior[by1 - wy1:by2 - wy1, bx1 - wx1:bx2 - wx1]


def _bubble_interior(img: np.ndarray, region: Region) -> np.ndarray:
    """Mask area putih bubble untuk memandu layout baris teks."""
    stroke = _stroke_px(region.est_font_size)
    x1, y1, x2, y2 = region.bbox

    if region.bubble_bbox is None:
        found = _discover_bubble(img, region, stroke)
        if found is None:
            return np.full((y2 - y1, x2 - x1), 255, np.uint8)
        region.bubble_bbox, interior = found
        return interior

    bx1, by1, bx2, by2 = region.bubble_bbox
    crop = img[by1:by2, bx1:bx2]
    if crop.size == 0:
        return np.full((y2 - y1, x2 - x1), 255, np.uint8)
    return _interior_from_crop(crop, stroke, _ink_center(region, bx1, by1),
                               _ink_in_crop(region, bx1, by1, crop.shape[:2]))


def partition_shared_interiors(img: np.ndarray, regions: list[Region]) -> int:
    """Balon ganda: partisi interior GABUNGAN ke lobus milik tiap region.

    Membelah KOTAK balon (detect._partition_shared_bubbles) tidak memisahkan
    BENTUK lobusnya — tiap belahan persegi masih memuat sebagian lobus sebelah,
    jadi dua centroid jatuh berdekatan dan teksnya bertumpuk. Di sini interior
    dihitung untuk kotak balon ASLI, lalu tiap pikselnya diberikan ke region
    dengan tinta terdekat (Voronoi berbobot tinta): interior tiap region pasti
    disjoint dan bentuknya mengikuti lobus sungguhan, bukan potongan persegi.

    Harus dipanggil SETELAH build_region_mask semua region — butuh ink_mask.

    Returns:
        Jumlah region yang interiornya diganti.
    """
    from collections import defaultdict

    groups: dict[tuple[int, int, int, int], list[Region]] = defaultdict(list)
    for r in regions:
        if r.shared_bubble_bbox is not None and r.ink_mask is not None:
            groups[r.shared_bubble_bbox].append(r)
    return sum(_split_interior(img, bbox, grp)
               for bbox, grp in groups.items() if len(grp) >= 2)


def _eff_box_mask(r: Region) -> tuple[tuple[int, int, int, int], np.ndarray]:
    """Kotak + mask interior EFEKTIF region — harus sama dengan yang dipakai
    typeset._region_box_mask(), termasuk jatuh ke persegi 255 penuh saat mask
    tidak cocok. Kalau keduanya berbeda, disjoin di sini memotong peta yang
    berbeda dari peta yang dipakai render dan irisannya tetap ada.
    """
    box = r.bubble_bbox or r.bbox
    bx1, by1, bx2, by2 = box
    bw, bh = max(bx2 - bx1, 0), max(by2 - by1, 0)
    m = r.bubble_mask
    if m is None or m.shape[:2] != (bh, bw):
        m = np.full((bh, bw), 255, np.uint8)
    return box, m


def disjoin_overlapping_interiors(img: np.ndarray, regions: list[Region]) -> int:
    """Interior balon BERTETANGGA yang beririsan -> tiap piksel jadi milik satu.

    partition_shared_interiors() hanya menangani kasus detector menyatukan dua
    lobus ke SATU kotak balon (`shared_bubble_bbox`). Kasus yang jauh lebih
    sering di halaman nyata: balon-balon berdekatan masing-masing dapat kotaknya
    sendiri — jadi fungsi itu tidak pernah jalan — tapi interiornya tetap
    beririsan ribuan piksel karena kotak persegi di sekitar balon bulat saling
    menabrak di sudut.

    Akibatnya bukan tumpang tindih, tapi GLYPH TERPOTONG: render_region menata
    teks memakai interior sendiri, lalu _clip_to_mask membuang piksel yang masuk
    interior region lain (forb_map). Terukur di halaman referensi: irisan 5994 px
    membuat 'OH, IS THIS THE SHIKO CLUB?' dirender jadi 'IS THE :KO 4B?' dan
    'I WAS COMPILING' jadi 'OMPILING'.

    Di sini irisan diselesaikan SEBELUM tata letak: piksel yang diklaim lebih
    dari satu region diberikan ke region dengan tinta terdekat (aturan Voronoi
    yang sama dengan _split_interior). Piksel yang cuma diklaim satu region tidak
    pernah disentuh — jadi fungsi ini hanya MEMBUANG piksel, tidak pernah
    menambah, dan tidak mungkin memunculkan cacat 'teks keluar balon' yang baru.

    Region terlindungi (SFX) ikut berlomba: tinta SFX menarik piksel di
    sekitarnya, jadi dialog tetap tidak ditulis menimpa SFX.

    Returns:
        Jumlah region yang interiornya menyusut.
    """
    items = [(r, *_eff_box_mask(r)) for r in regions]
    items = [(r, b, m) for r, b, m in items if m.size]
    if len(items) < 2:
        return 0

    return sum(_disjoin_group(img, [items[i] for i in grp])
               for grp in _overlap_groups([b for _, b, _ in items]))


def _overlap_groups(boxes: list[tuple[int, int, int, int]]) -> list[list[int]]:
    """Kelompok indeks yang kotaknya saling bersinggungan (union-find).

    Kotak beririsan adalah syarat PERLU bagi mask beririsan, bukan syarat cukup
    — cukup untuk menyaring kandidat murah; piksel yang ternyata tidak
    diperebutkan tetap tidak diubah di _disjoin_group().
    """
    n = len(boxes)
    parent = list(range(n))

    def find(i: int) -> int:
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for i in range(n):
        for j in range(i + 1, n):
            a, b = boxes[i], boxes[j]
            if min(a[2], b[2]) > max(a[0], b[0]) and min(a[3], b[3]) > max(a[1], b[1]):
                parent[find(i)] = find(j)

    groups: dict[int, list[int]] = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(i)
    return [g for g in groups.values() if len(g) >= 2]


def _disjoin_group(img: np.ndarray, grp: list[tuple]) -> int:
    """Satu kelompok balon bersinggungan -> interiornya dibuat saling lepas.

    Returns:
        Jumlah region yang benar-benar kehilangan piksel.
    """
    h, w = img.shape[:2]
    wx1 = max(min(b[0] for _, b, _ in grp), 0)
    wy1 = max(min(b[1] for _, b, _ in grp), 0)
    wx2 = min(max(b[2] for _, b, _ in grp), w)
    wy2 = min(max(b[3] for _, b, _ in grp), h)
    ww, wh = wx2 - wx1, wy2 - wy1
    if ww <= 0 or wh <= 0:
        return 0

    # Mask tiap region dipindah ke jendela bersama supaya bisa dibandingkan
    # piksel-per-piksel.
    local: list[np.ndarray] = []
    for _, (bx1, by1, bx2, by2), m in grp:
        canvas = np.zeros((wh, ww), np.uint8)
        ty1, tx1 = by1 - wy1, bx1 - wx1
        sy1, sx1 = max(ty1, 0), max(tx1, 0)
        sy2, sx2 = min(ty1 + m.shape[0], wh), min(tx1 + m.shape[1], ww)
        if sy2 > sy1 and sx2 > sx1:
            canvas[sy1:sy2, sx1:sx2] = m[sy1 - ty1:sy2 - ty1, sx1 - tx1:sx2 - tx1]
        local.append(canvas)

    claims = np.zeros((wh, ww), np.uint16)
    for m in local:
        claims += (m > 0).astype(np.uint16)
    contested = claims >= 2
    if not contested.any():
        return 0

    owner = np.argmin(
        np.stack([_ink_distance(r, (wh, ww), wx1, wy1) for r, _, _ in grp]), axis=0
    )
    shrunk = 0
    for i, (r, _, _) in enumerate(grp):
        lost = contested & (owner != i) & (local[i] > 0)
        if not lost.any():
            continue
        keep = local[i].copy()
        keep[lost] = 0
        ys, xs = np.nonzero(keep)
        if ys.size == 0:
            continue                       # jangan pernah membuat region tanpa balon
        lx1, ly1 = wx1 + int(xs.min()), wy1 + int(ys.min())
        lx2, ly2 = wx1 + int(xs.max()) + 1, wy1 + int(ys.max()) + 1
        r.bubble_bbox = (lx1, ly1, lx2, ly2)
        r.bubble_mask = keep[ly1 - wy1:ly2 - wy1, lx1 - wx1:lx2 - wx1]
        shrunk += 1
    return shrunk


def _split_interior(img: np.ndarray, bbox: tuple[int, int, int, int],
                    grp: list[Region]) -> int:
    """Satu balon bersama -> satu interior per region, dijamin tidak beririsan."""
    bx1, by1, bx2, by2 = bbox
    crop = img[by1:by2, bx1:bx2]
    if crop.size == 0:
        return 0

    # Satu fill per region lalu digabung: kalau lobusnya menyatu semua benih
    # memberi hasil sama, kalau ada garis pemisah tiap lobus tetap terjaring.
    stroke = _stroke_px(max(r.est_font_size for r in grp))
    merged = np.zeros(crop.shape[:2], np.uint8)
    for r in grp:
        merged = np.maximum(
            merged, _interior_from_crop(crop, stroke, _ink_center(r, bx1, by1),
                                        _ink_in_crop(r, bx1, by1, crop.shape[:2]))
        )
    if not merged.any():
        return 0

    owner = np.argmin(
        np.stack([_ink_distance(r, merged.shape, bx1, by1) for r in grp]), axis=0
    )
    fixed = 0
    for i, r in enumerate(grp):
        lobe = np.where((owner == i) & (merged > 0), 255, 0).astype(np.uint8)
        ys, xs = np.nonzero(lobe)
        if ys.size == 0:
            continue                     # tanpa lobus: belahan persegi tetap dipakai
        lx1, ly1 = bx1 + int(xs.min()), by1 + int(ys.min())
        lx2, ly2 = bx1 + int(xs.max()) + 1, by1 + int(ys.max()) + 1
        r.bubble_bbox = (lx1, ly1, lx2, ly2)
        r.bubble_mask = lobe[ly1 - by1:ly2 - by1, lx1 - bx1:lx2 - bx1]
        fixed += 1
    return fixed


def _ink_distance(region: Region, shape: tuple[int, ...],
                  ox: int, oy: int) -> np.ndarray:
    """Jarak tiap piksel crop ke tinta region — dasar partisi Voronoi.

    distanceTransform mengukur jarak ke piksel BERNILAI 0, jadi tinta dipetakan
    ke 0 dan sisanya 255.
    """
    hh, ww = shape[0], shape[1]
    seed = np.full((hh, ww), 255, np.uint8)
    ink = region.ink_mask
    if ink is not None and ink.any():
        x1, y1 = region.bbox[0] - ox, region.bbox[1] - oy
        mh, mw = ink.shape[:2]
        sy1, sx1 = max(y1, 0), max(x1, 0)
        sy2, sx2 = min(y1 + mh, hh), min(x1 + mw, ww)
        if sy2 > sy1 and sx2 > sx1:
            sub = ink[sy1 - y1:sy2 - y1, sx1 - x1:sx2 - x1]
            seed[sy1:sy2, sx1:sx2] = np.where(sub > 0, 0, 255)
    if seed.min() > 0:                   # tinta di luar crop: pakai pusat bbox
        cx, cy = _ink_center(region, ox, oy)
        seed[int(np.clip(cy, 0, hh - 1)), int(np.clip(cx, 0, ww - 1))] = 0
    return cv2.distanceTransform(seed, cv2.DIST_L2, 3)


def _paste(page: np.ndarray, mask: np.ndarray,
           box: tuple[int, int, int, int]) -> None:
    """Tempel mask lokal ke kanvas halaman. `box` = kotak asal mask itu sendiri.

    Dipisah karena salah kotak di sini tidak pernah kelihatan sebagai error:
    bubble_mask hidup di koordinat bubble_bbox, bukan bbox, dan untuk r8 halaman
    referensi kedua kotak itu beda 29 px. Mask yang bergeser menghasilkan angka
    yang rapi tapi salah tempat.
    """
    h, w = page.shape[:2]
    x1, y1 = box[0], box[1]
    mh, mw = mask.shape[:2]
    sy1, sx1 = max(y1, 0), max(x1, 0)
    sy2, sx2 = min(y1 + mh, h), min(x1 + mw, w)
    if sy2 <= sy1 or sx2 <= sx1:
        return
    sub = mask[sy1 - y1:sy2 - y1, sx1 - x1:sx2 - x1]
    page[sy1:sy2, sx1:sx2] = np.maximum(page[sy1:sy2, sx1:sx2], sub)


# Ambang gelap untuk "ini garis balon". Sengaja jauh di bawah 128 (ambang
# interior di _bubble_interior): yang dicari cuma tinta pekat, bukan raster abu.
_LINE_DARK = 110
# Pita pencarian garis: dari tepi interior sampai stroke + segini px keluar.
# Harus stroke-aware — _interior_from_crop mengikis interior sebesar stroke,
# jadi garisnya duduk stroke..2*stroke px di luar. Pita tetap 4 px melaporkan
# NOL piksel garis di balon berstroke 4 px: pitanya sendiri belum sampai.
_LINE_BAND = 4
# Komponen gelap di pita dihitung garis kalau bentang terpanjangnya (lebar atau
# tinggi kotak pembatasnya) minimal sekian kali stroke. Garis balon membentang
# jauh menyusuri tepi; coretan glyph yang kebetulan menyeberang pita ringkas.
# Dipakai bentang, BUKAN luas relatif: pada balon sempit garisnya terputus jadi
# beberapa potong, dan ambang "seperempat komponen terbesar" ikut membuang
# potongan yang sah.
_LINE_MIN_SPAN = 8


def bubble_outline_guard(img: np.ndarray, regions: list[Region]) -> np.ndarray:
    """Piksel GARIS balon yang tidak boleh ikut terhapus. Biner 0/255.

    Kenapa perlu: _adaptive_dilate memekarkan ink_mask dengan kernel sampai
    31 px, dan teks Jepang vertikal di balon sempit duduk cuma beberapa piksel
    dari garisnya. Dilasi itu menyeberang garis, erase menghapus yang tersentuh,
    dan garis balon jadi putus-putus. Terukur 267 px garis termakan di halaman
    referensi (r7 79, r8 136, r11 36, r12 16).

    Kenapa bukan "kurung erase ke dalam interior" — obat yang lebih sederhana
    itu sudah diuji dan salah: interior bukan selubung yang bisa dipercaya.
    Glyph yang MENEMPEL di garis membuat takik yang terbuka ke tepi, jadi flood
    fill tak bisa mengelilinginya dan _fill_holes tak bisa menambalnya (hanya
    lubang tertutup). Hasilnya 79/148/36/187 px tinta Jepang di r7/r8/r11/r12
    ikut selamat — bukan halo, tapi glyph utuh ('すか' masih terbaca).

    Yang dilindungi di sini justru GARISNYA saja, jadi _halo_pass tetap bekerja
    penuh di dalam balon dan alasannya (penyebab nomor satu ghost outline) tidak
    dilanggar.
    """
    h, w = img.shape[:2]
    dark = img.mean(2) < _LINE_DARK
    guard = np.zeros((h, w), np.uint8)
    k3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    for r in regions:
        if r.bubble_mask is None:
            continue
        box, bm = _eff_box_mask(r)
        if bm.size == 0 or bm.min() == 255:
            continue          # persegi 255 penuh: bukan balon, tak ada garis
        inner = np.zeros((h, w), np.uint8)
        _paste(inner, (bm > 0).astype(np.uint8), box)
        if not inner.any():
            continue
        stroke = _stroke_px(r.est_font_size or 20)
        band = cv2.dilate(inner, k3, iterations=stroke + _LINE_BAND) - inner
        sel = (dark & (band > 0)).astype(np.uint8)
        n, lab, stats, _ = cv2.connectedComponentsWithStats(sel, 8)
        if n <= 1:
            continue
        span = np.maximum(stats[1:, cv2.CC_STAT_WIDTH], stats[1:, cv2.CC_STAT_HEIGHT])
        keep = 1 + np.flatnonzero(span >= _LINE_MIN_SPAN * max(stroke, 1))
        if keep.size:
            guard[np.isin(lab, keep)] = 255
    return guard


def protect_bubble_outline(img: np.ndarray, regions: list[Region]) -> int:
    """Kurangkan garis balon dari SETIAP ink_mask. Return piksel yang dilepas.

    Harus di ink_mask, bukan cuma di erase_mask hasil compose_page_mask().
    erase_page() menghapus dari `r.ink_mask` per region (erase.py:86 dan 113);
    erase_mask halaman hanya dipakai untuk assert SFX dan dump debug. Versi
    pertama perbaikan ini mengurangkan penjaga di compose_page_mask saja, dan
    hasilnya tepat seperti yang diukur: penjaga bersih dari erase_mask (irisan
    0 px) sementara garis yang hilang di plat bersih tidak bergerak satu piksel
    pun — 326 px tetap termakan karena erase tidak pernah membaca mask itu.

    Dipanggil setelah partition_shared_interiors + disjoin_overlapping_interiors,
    karena penjaganya dihitung dari bubble_mask yang sudah final.
    """
    guard = bubble_outline_guard(img, regions)
    if not guard.any():
        return 0
    h, w = img.shape[:2]
    freed = 0
    for r in regions:
        if r.ink_mask is None:
            continue
        x1, y1 = r.bbox[0], r.bbox[1]
        mh, mw = r.ink_mask.shape[:2]
        sy1, sx1 = max(y1, 0), max(x1, 0)
        sy2, sx2 = min(y1 + mh, h), min(x1 + mw, w)
        if sy2 <= sy1 or sx2 <= sx1:
            continue
        sub = r.ink_mask[sy1 - y1:sy2 - y1, sx1 - x1:sx2 - x1]
        hit = (sub > 0) & (guard[sy1:sy2, sx1:sx2] > 0)
        if hit.any():
            freed += int(hit.sum())
            sub[hit] = 0
    return freed


def compose_page_mask(
    img: np.ndarray, regions: list[Region]
) -> tuple[np.ndarray, np.ndarray]:
    """Gabung semua ink mask jadi satu mask halaman, LALU kecualikan SFX.

    Returns:
        (erase_mask, protected_mask) — dua-duanya biner 0/255 skala halaman.

    Ini titik paling kritis di seluruh pipeline. Mask teks dan penyelamat
    furigana AKAN ikut menandai SFX. Kalau exclusion tidak jalan sebelum
    erase, pipeline menghapus persis apa yang diminta untuk dijaga.
    """
    h, w = img.shape[:2]
    page = np.zeros((h, w), np.uint8)
    protected = np.zeros((h, w), np.uint8)

    for r in regions:
        if r.ink_mask is None:
            continue
        x1, y1, x2, y2 = r.bbox
        mh, mw = r.ink_mask.shape[:2]
        x2, y2 = min(x2, x1 + mw), min(y2, y1 + mh)
        sub = r.ink_mask[: y2 - y1, : x2 - x1]
        target = protected if r.is_protected else page
        target[y1:y2, x1:x2] = np.maximum(target[y1:y2, x1:x2], sub)

    # SFX menang mutlak: lebarkan sedikit lalu kurangkan dari mask hapus.
    if protected.any():
        guard = cv2.dilate(
            protected, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)), iterations=1
        )
        page = cv2.bitwise_and(page, cv2.bitwise_not(guard))

    # Garis balon juga menang: lihat bubble_outline_guard(). Di sini hanya
    # menjaga konsistensi dump/assert — pengurangan yang benar-benar berpengaruh
    # ke hasil dilakukan protect_bubble_outline() pada ink_mask, karena
    # erase_page() membaca ink_mask per region, bukan mask halaman ini.
    outline = bubble_outline_guard(img, regions)
    if outline.any():
        page = cv2.bitwise_and(page, cv2.bitwise_not(outline))

    return ((page > 0).astype(np.uint8)) * 255, ((protected > 0).astype(np.uint8)) * 255


def release() -> None:
    global _CTD
    _CTD = None



Writing /content/mangatl/textmask.py


In [8]:
%%writefile /content/mangatl/ocr.py

"""OCR Jepang via manga-ocr, dengan gate anti-halusinasi.

manga-ocr terdokumentasi menghasilkan teks acak pada input kosong. Gate
ink_ratio wajib jalan sebelum model dipanggil, kalau tidak halaman bersih
akan penuh terjemahan hantu.
"""

from __future__ import annotations

import re

import cv2
import numpy as np
from PIL import Image

from config import SETTINGS, Region, note

_OCR = None
_OCR_FAILED = False

# Halusinasi khas manga-ocr pada input kosong/hampir kosong.
_HALLUCINATION = frozenset({
    "。", "、", "…", "・", "！", "？", "「", "」", "ー", "～", ".", "..", "...",
})
_REPEAT = re.compile(r"^(.)\1{4,}$")


def get_ocr():
    """Muat manga-ocr sekali. None kalau paket tidak terpasang.

    Kegagalan di sini dicetak, tidak ditelan: kalau OCR mati, SETIAP region jadi
    UNREADABLE lalu ikut PROTECTED_LABELS, dan halaman keluar tanpa satu pun
    terjemahan — gejalanya terlihat seperti "model LLM tidak jalan", padahal
    penyebabnya cuma satu dependency hilang (`loguru`, di-skip oleh --no-deps).
    """
    global _OCR, _OCR_FAILED
    if _OCR is not None or _OCR_FAILED:
        return _OCR
    try:
        from manga_ocr import MangaOcr

        # force_cpu=False (default) = otomatis CUDA kalau GPU ada. MangaOcr 0.1.16
        # TIDAK punya parameter `device=` — jangan diganti, nanti TypeError.
        _OCR = MangaOcr(force_cpu=False)
    except (ImportError, OSError, RuntimeError) as exc:
        note("error", "ocr",
             f"manga-ocr tidak bisa dimuat ({exc}) — SEMUA region jadi UNREADABLE, "
             "artinya tidak ada teks yang dihapus maupun diterjemahkan")
        _OCR_FAILED = True
        _OCR = None
    return _OCR


def _prepare(img: np.ndarray, region: Region) -> Image.Image:
    """Crop + upscale kecil supaya glyph tipis tidak hilang saat resize model."""
    x1, y1, x2, y2 = region.bbox
    crop = img[y1:y2, x1:x2]
    h, w = crop.shape[:2]
    if min(h, w) < 32 and min(h, w) > 0:
        scale = 32 / min(h, w)
        crop = cv2.resize(
            crop, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_CUBIC
        )
    # Padding putih membantu model: teks manga selalu punya margin.
    crop = cv2.copyMakeBorder(crop, 8, 8, 8, 8, cv2.BORDER_CONSTANT, value=(255, 255, 255))
    return Image.fromarray(crop)


def _is_hallucination(text: str) -> bool:
    t = text.strip()
    if not t or t in _HALLUCINATION:
        return True
    return bool(_REPEAT.match(t))


def read_region(img: np.ndarray, region: Region) -> str:
    """OCR satu region. String kosong = tidak ada teks terbaca."""
    if region.ink_ratio < SETTINGS.min_ink_ratio:
        return ""  # gate: terlalu sedikit tinta, model akan berhalusinasi
    ocr = get_ocr()
    if ocr is None:
        return ""
    try:
        text = ocr(_prepare(img, region))
    except (RuntimeError, ValueError, OSError):
        return ""
    text = (text or "").strip()
    return "" if _is_hallucination(text) else text


def read_all(img: np.ndarray, regions: list[Region]) -> list[Region]:
    """Isi src_text tiap region. Region tanpa teks ditandai UNREADABLE.

    UNREADABLE masuk PROTECTED_LABELS, jadi region gagal-baca tidak pernah
    dihapus dari halaman — lebih baik teks asli tertinggal daripada art hilang.
    """
    for r in regions:
        r.src_text = read_region(img, r)
        if not r.src_text:
            r.label = "UNREADABLE"
    return regions


def release() -> None:
    """Bebaskan ~444 MB model OCR."""
    global _OCR
    _OCR = None



Writing /content/mangatl/ocr.py


In [9]:
%%writefile /content/mangatl/translate.py

"""Terjemahan: DeepL (MT murni) ATAU LLM OpenAI-compatible (faucet / gorouter).

Ketiganya ada dan dipilih di UI, karena menang di hal yang berbeda:

  DeepL       cepat, gratis 1 juta karakter/bulan, tidak pernah menyensor —
              tapi menerjemahkan kalimat LEPAS KONTEKS dan tidak bisa diberi
              tahu apa pun tentang halamannya. Panjang hasilnya kebetulan.
  LLM faucet  freetokenfaucet, mimo-v2.5-pro (GRATIS). Bisa diberi konteks
              halaman, glosari, DAN ukuran balon — jalan ke syarat 'NO KELUAR
              BUBBLE' di plan.txt: teksnya dibuat pendek di sumbernya, bukan
              dikecilkan di typeset. TERUKUR 4.5-7.8 s untuk halaman 19 balon.
              Tokennya terbatas, jadi thinking dimatikan (lihat FAUCET_EXTRA).
              Modelnya WAJIB model gratis — lihat catatan 402 di config.py.
  LLM router  gorouter/claude-opus-5. Sama protokolnya, mutu bahasanya paling
              rapi (TERUKUR 8.2 s), tapi memakai kredit berbayar — jadi bukan
              default. Host-nya di balik Cloudflare dan MENUNTUT header
              User-Agent; tanpa itu 403 "error code 1010" (lihat BROWSER_UA).

Faucet dan router memakai kelas client yang sama (FaucetClient turunan
RouterClient) karena protokolnya identik — yang beda cuma base URL, model,
batas waktu, header, dan satu parameter body.

Klasifikasi SFX heuristik (_fallback_labels) dipakai untuk SEMUA penyedia —
SFX tidak boleh diterjemah, dan itu keputusan yang tidak bergantung penyedia.

API key: FAUCET_API_KEY / DEEPL_API_KEY / ROUTER_API_KEY (Colab Secrets -> env
-> field UI -> file lokal untuk penyedia LLM). Tidak ada key di kode.
"""

from __future__ import annotations

import json
import os
import re
import time
import urllib.error
import urllib.request
from dataclasses import dataclass

from config import (BROWSER_UA, DEEPL_API_BASE, DEEPL_TARGET, FAUCET_API_BASE,
                    FAUCET_DEADLINE, FAUCET_EXTRA, FAUCET_FALLBACK,
                    FAUCET_MAX_TOKENS, FAUCET_MODEL, FAUCET_TIMEOUT,
                    PROTECTED_LABELS,
                    ROUTER_API_BASE, ROUTER_BACKOFF, ROUTER_DEADLINE,
                    ROUTER_FALLBACK, ROUTER_MODEL, ROUTER_RETRY, ROUTER_TIMEOUT,
                    SETTINGS, TRANSLATION_STYLES, Region, note)

_VALID_LABELS = frozenset(
    {"DIALOGUE", "THOUGHT", "NARRATION", "SIGN", "SFX", "UNREADABLE"}
)

# Gaya -> parameter formality DeepL (hanya dipakai untuk bahasa yang
# mendukungnya: DE/FR/ES/IT/NL/PL/PT/RU). Uncensored = default: DeepL
# memang tidak pernah menyensor.
FORMALITY_BY_STYLE: dict[str, str] = {
    "Formal": "more",
    "Casual & Slang": "less",
    "Manga Natural": "default",
    "Literal": "default",
    "Uncensored": "default",
    "Fully Localized": "default",
}

# DeepL tidak menerima request kosong; batch maksimal 50 teks per request.
_CHUNK = 50

# ---------------------------------------------------------------- SFX heuristic
#
# DeepL murni mesin terjemahan — dia tidak tahu mana SFX mana dialog, jadi
# klasifikasi 100% heuristik. Heuristik di sini jauh lebih agresif dari versi
# lama (kana murni <=3 char / ABAB 4 char) yang bocor di kasus nyata:
#   "フー．．．"   -> kana + tanda baca: tidak tertangkap, diterjemah "Phew..."
#   "ぴくぴくっ"  -> ABAB + っ ekor: tidak tertangkap, jadi "Twitch..."
#   "ドキドキドキ"-> pengulangan 6 char: tidak tertangkap, jadi "Thump..."
#
# Aturan baru:
#   inti kana  = sumber minus semua simbol/tanda baca (．。、・！？〜…♥ dll)
#   di luar balon: pendek(<=3) / ada っッ/ー / pola ulang / kamus -> SFX
#   di dalam balon: pola ulang / kamus / seluruhnya-katakana ber-っー /
#                   batang-ganda ada di _SFX_DICT -> SFX (lihat _sfx_in_bubble;
#                   aturan '3-karakter-っッ' yang lama sudah DICABUT karena
#                   mengunci seruan dialog dan mencetak balon Jepang)
#                   KECUALI teks aslinya bertanda BICARA (？ atau jeda di
#                   ANTARA kana) — itu suara tokoh, bukan bunyi latar:
#                   ヒ．．．ッ！？ di balon hitam bergerigi (hasilnew5)

_KANA = re.compile(r"^[\u3040-\u30ff\u31f0-\u31ff]+$")
_LONG = "ー"
_SMALL = frozenset("っッ")

# Tanda BICARA. Dipakai HANYA untuk menahan aturan K (lihat _sfx_in_bubble):
# bunyi tidak bertanya, dan bunyi tidak terputus di tengah mora.
# _PAUSE sengaja TIDAK memuat ・ (U+30FB) dan ー (U+30FC) walau keduanya di blok
# kana — ・ pemisah, ー pemanjang; tidak satu pun jeda ucapan.
_ASK = frozenset("？?")
_PAUSE = frozenset("．.。、,，…‥")

# Onomatope panjang / silang-rima yang tidak tertangkap pola ABAB.
_SFX_DICT = frozenset({
    # rangsangan / fisik
    "ガッタンゴットン", "がったんごっとん", "がたんごとん", "ガタンゴトン",
    "ドクンドクン", "どくんどくん", "ドキドキドキ", "どきどきどき",
    # Bentuk DASAR ganda dua suku. Sebelumnya hanya bentuk 3-ulangan
    # (ドキドキドキ) dan turunannya (ぴくぴく/ひくひく) yang tercatat, sehingga
    # aturan batang-ganda di _label_region tidak bisa membuktikan どきっ dan
    # びくっ sebagai onomatope — keduanya lolos jadi DIALOGUE dan ikut
    # diterjemah. _label3.py mengukur itu: keduanya satu-satunya sisa
    # kesalahan arah SFX sebelum empat entri ini ditambahkan.
    "ドキドキ", "どきどき", "ビクビク", "びくびく",
    "バクバク", "ばくばく", "はあはあ", "はぁはぁ", "ハァハァ", "ふうふう",
    "ずるずる", "じゅるじゅる", "チュパチュパ", "ちゅぱちゅぱ", "ぢゅぱぢゅぱ",
    "くちゅくちゅ", "ぐちゅぐちゅ", "にゅるにゅる", "ぬるぬる", "とろとろ",
    "ぐちょぐちょ", "びちゃびちゃ", "じゅくじゅく", "ぬちゃぬちゃ",
    "ぷるぷる", "ぶるぶる", "がくがく", "がたがた", "わなわな", "ぞくぞく",
    "そわそわ", "もじもじ", "うずうず", "むずむず", "はらはら",
    "むらむら", "わくわく", "ぽかぽか", "ほかほか",
    "ごくごく", "ごくん", "ごくり", "ごくりっ", "ごくっ", "こくん",
    "ひゅるひゅる", "ひゅーひゅる", "ふーふー", "スースー", "すーすー",
    # makan / suara tubuh
    "むしゃむしゃ", "もぐもぐ", "ぱくぱく", "がつがつ", "がりがり",
    "ぼりぼり", "ばりばり", "めきめき", "ぼきぼき", "ばきばき", "ぱきぱき",
    "ぐきぐき", "がんがん", "ごんごん", "どんどん", "とんとん", "こんこん",
    # gerak / angin / air
    "ばたばた", "ぱたぱた", "ひらひら", "ふわふわ", "ふわり", "ふわっ",
    "ほわっ", "ぽわん", "ゆらゆら", "ふらふら", "ぐらぐら", "ふらっ",
    "よろよろ", "とぼとぼ", "うろうろ", "そろそろ", "めそめそ",
    "しとしと", "ぽつぽつ", "ざあざあ", "じゃあじゃあ", "ざーざー",
    "ぱらぱら", "ばらばら", "ざわざわ", "がやがや", "わいわい",
    "ひそひそ", "こそこそ", "ごそごそ", "がさがさ", "かさかさ",
    "ざくざく", "じゃりじゃり", "がしゃがしゃ",
    # benturan / suara keras
    "ごとごと", "ごとん", "がたん", "どしん", "ずしん", "どすん",
    "どかん", "どーん", "どどーん", "ががーん", "ばーん", "がーん",
    "ばきっ", "ぼきっ", "ぐきっ", "ぱきっ", "べきっ", "ばりっ",
    "びりっ", "ぷちっ", "ぷつん", "ちぎっ", "がちゃっ", "ぱちん",
    "ばちん", "ぱちっ", "ちょん",
    # desis / ledakan kecil
    "しゅっ", "ひゅっ", "ひゅー", "ぴゅー", "びゅー", "びゅん",
    "しゅー", "しゅーっ", "ぷしゅー", "ぷしゅう", "じゅわっ",
    "じゅうじゅう", "ぐつぐつ", "ふつふつ", "ぼこぼこ", "ぷくぷく",
    "ぷくっ", "ぷかぷか", "ぷらぷら",
    # emosi / kondisi
    "がくっ", "がっくり", "しゅん", "しょんぼり", "ぼんやり",
    "ぼーっ", "ぼうっと", "うとうと", "すやすや", "ぐっすり",
    "ぐうぐう", "すぴすぴ", "ぴょこん", "ぴょんぴょん",
    "ぴくっ", "ぴくぴく", "ひくっ", "ひくひく", "ぷるん", "ぷるっ",
    "ぽよん", "ぽよぽよ", "ぷにぷに", "むにむに", "むちむち",
    "もみもみ", "ぎゅっ", "ぎゅー", "ぎゅうっ", "きゅっ", "きゅー",
    "ひしっ", "がっつり", "がつん", "ずっぽり", "ずぽっ", "ぽんっ",
    "ぽん", "ぽちっ", "ぽんぽん",
})

# Kata kana pendek yang umum dipakai DIALOGUE — tidak boleh dikunci SFX
# walau di luar balon. Heuristik 'kana pendek = SFX' terlalu rakus:
# "それは" dan "ちょっと" juga 2-4 kana. Daftar ini menangani kata yang
# paling sering muncul sebagai narasi/dialog di atas art.
_KA_DIALOGUE = frozenset({
    # kata tunjuk & sambung
    "それ", "それで", "それは", "それに", "それも", "それから", "それな", "それか",
    "あれ", "あれは", "これ", "これは", "これで", "これも", "どこ", "だれ",
    "なに", "なん", "どう", "どうだ", "どうして", "なぜ", "なんで", "なぜか",
    "そう", "そうだ", "そうね", "そうか", "そうよ", "そうそう", "そんな",
    "こんな", "あんな", "どの", "この", "その", "あの", "つまり", "だから",
    "でも", "けど", "そして", "ところで", "ちなみに", "じゃあ", "では",
    # jawaban & seruan
    "はい", "いいえ", "うん", "うーん", "うーむ", "ええ", "えー", "ええと",
    "あのね", "ねえ", "ねぇ", "まあ", "もう", "まだ", "もっと", "だめ",
    "ダメ", "やめ", "やめて", "やめろ", "まって", "ちょっと", "ごめん",
    "ゴメン", "ごめんなさい", "ありがとう", "すみません", "お願い", "おねがい",
    "こんにちは", "こんばんは", "さようなら", "おはよう", "おやすみ",
    "がんばれ", "がんばって", "いいね", "いいよ", "いいの", "いいんだ",
    "いや", "いやいや", "あら", "あらあら", "まあまあ", "なるほど",
    "なるほどね", "うんうん", "えっ", "あっ", "んっ",
    "うっ", "むっ", "ふむ",
    # Seruan yang JELAS ucapan tapi belum tercatat, jadi cabang di dalam balon
    # menguncinya SFX hanya karena 3 huruf + っ (うわっ やだっ まてっ ...). SFX
    # berarti translation=None + PROTECTED, yaitu balon Jepang tercetak tanpa
    # satu pun pesan error — itulah cacat "short bubble untranslate". Ditulis
    # dalam bentuk hiragana saja; pencarian menormalkan katakana lebih dulu,
    # jadi satu entri menutup オイ maupun おい.
    "おい", "まて", "うそ", "ちょ", "ふぇ", "やだ", "そこ", "うわ",
    # adverb ABAB & kata pendek lain yang sering jadi dialog (di luar balon)
    "ときどき", "そろそろ", "だんだん", "ぼちぼち", "ぼつぼつ", "じきじき",
    "きっと", "たぶん", "まさか", "さすが", "やはり", "やっぱり", "やっぱ",
    "あとで", "あと", "いま", "ここ", "あそこ", "そこで", "こっち",
    "そっち", "あっち", "どっち", "じゃ", "さて", "まず", "おまたせ",
    "いったい", "なんと", "なんて", "そうかな", "そうかも", "そうだね",
    "そうなの", "あれれ", "えーっと", "うーん", "うーーん",
    # ekspresi pendek yang sering jadi narasi
    "わかった", "わかりました", "だめだ", "よかった", "すごい", "うれしい",
    "かなしい", "さみしい", "つらい", "やばい", "むり", "ムリ", "むりだ",
    "できない", "できる", "わからない", "しらない", "しってる", "つかれた",
    "きもい", "うざい", "きたない", "きれい", "かわいい", "かっこいい",
    "たのしい", "おもしろい", "つまらない", "へんだ", "おかしい", "こわい",
})

# Simbol emosi yang wajib bertahan di hasil terjemahan.
_EMOTION = frozenset("♥♡❤💕💗💓♪♫♬☆★〜～")

# Punctuation Jepang yang lolos apa adanya dari DeepL. Anime Ace cuma ~159
# glyph, jadi 「 」 … ： dirender jadi kotak tofu di dalam balon. Kurung sudut
# tidak bermakna di Inggris -> dibuang (None); sisanya dipetakan ke ASCII.
# ♥ ♡ ♪ ☆ TIDAK ada di sini — lihat _EMOTION.
_PUNCT_MAP = str.maketrans({
    "「": None, "」": None, "『": None, "』": None,
    "【": None, "】": None, "〔": None, "〕": None,
    "〈": None, "〉": None, "《": None, "》": None,
    # ＼…／ = tanda penekanan dekoratif Jepang yang mengapit teks ('＼失礼しました').
    # DeepL meloloskannya apa adanya, Anime Ace tidak punya glyph-nya, dan hasilnya
    # kotak tofu di depan baris pertama. Di Inggris tanda ini tidak bermakna.
    "＼": None, "＿": None,
    # Wave dash 〜 (U+301C) dan fullwidth tilde ～ (U+FF5E) -> tilde ASCII.
    # Simbolnya BERTAHAN, cuma dipindah ke lebar ASCII: keduanya tidak ada di
    # Anime Ace sementara '~' ada, jadi versi lebar dirender oleh font fallback
    # dengan wajah huruf yang berbeda dari balon lain — dan lebarnya salah diukur,
    # karena layout() mengukur baris dengan font utama saja. Ini juga syarat
    # plan.txt "simbol seperti ! dan love tetap ada" tanpa mengorbankan kerapian.
    "〜": "~", "～": "~",
    "　": " ", "・": " ",
    "。": ".", "．": ".", "、": ",", "，": ",",
    "：": ":", "；": ";", "／": "/", "！": "!", "？": "?",
    "（": "(", "）": ")", "…": "...", "‥": "..",
    "“": '"', "”": '"', "‘": "'", "’": "'",
})

class DeepLClient:
    """Pegang key saja — cukup untuk semua panggilan DeepL."""

    def __init__(self, api_key: str) -> None:
        self.key = api_key


class RouterClient:
    """Router OpenAI-compatible: base URL + key + nama model.

    Beda dari DeepLClient karena base URL-nya bukan konstanta (host Funnel bisa
    berubah, dan model bisa ditimpa env) — jadi keduanya ikut di client, bukan
    dibaca ulang di tiap fungsi.

    Kelas ini juga memegang BATAS WAKTU, HEADER, dan PARAMETER BODY tambahan,
    bukan membacanya dari config di dalam _router_call(). Alasannya: penyedia
    OpenAI-compatible kedua (faucet) sehat pada 3 s sementara router butuh
    120 s, dan modelnya butuh thinking dimatikan. Kalau angka-angka itu dibaca
    dari konstanta global, satu penyedia memaksakan batasnya ke penyedia lain.
    """

    extra: dict = {}
    # Host gorouter ada di balik Cloudflare dan MENOLAK klien tanpa User-Agent
    # dengan 403 "error code 1010" — terukur 17 Agu 2026, dua bentuk auth
    # sama-sama 403, dan key yang sama dengan UA ini membalas 200. Header ini
    # milik KELAS, bukan global, karena faucet TERUKUR sehat tanpa UA dan
    # menambahkannya di sana berarti mengubah yang bekerja tanpa mengukurnya.
    headers: dict = {"User-Agent": BROWSER_UA}
    timeout = ROUTER_TIMEOUT
    deadline = ROUTER_DEADLINE
    fallback: tuple[str, ...] = ROUTER_FALLBACK
    max_tokens: int = 0
    tag = "router"

    def __init__(self, api_key: str, base: str = "", model: str = "") -> None:
        self.key = api_key
        self.base = (base or ROUTER_API_BASE).rstrip("/")
        self.model = model or ROUTER_MODEL

class FaucetClient(RouterClient):
    """freetokenfaucet: OpenAI-compatible, jadi seluruh jalur router dipakai ulang.

    Yang berbeda hanya empat angka dan satu parameter body — lihat FAUCET_* di
    config.py untuk alasan tiap nilainya, terutama thinking.type=disabled yang
    WAJIB (tanpa itu jawaban bisa keluar sebagai string kosong tanpa error).
    """

    extra = FAUCET_EXTRA
    # Sengaja KOSONG, bukan warisan RouterClient: faucet TERUKUR membalas 200
    # tanpa User-Agent, jadi tidak ada alasan mengirim header yang belum diukur
    # ke sana.
    headers: dict = {}
    timeout = FAUCET_TIMEOUT
    deadline = FAUCET_DEADLINE
    fallback = FAUCET_FALLBACK
    max_tokens = FAUCET_MAX_TOKENS
    tag = "faucet"

    def __init__(self, api_key: str, base: str = "", model: str = "") -> None:
        self.key = api_key
        self.base = (base or FAUCET_API_BASE).rstrip("/")
        self.model = model or FAUCET_MODEL


@dataclass
class ProbeResult:
    model: str
    listed: bool
    ok: bool
    reason: str
    latency: float
    sample: str = ""

    def as_row(self) -> dict[str, str]:
        return {
            "model": self.model,
            "listed": "yes" if self.listed else "no",
            "verdict": "OK" if self.ok else "FAIL",
            "reason": self.reason,
            "latency": f"{self.latency:.1f}s",
            "sample": self.sample[:60],
        }


def _is_router(provider: str | None = None) -> bool:
    """Provider aktif = LLM OpenAI-compatible (router ATAU faucet)?

    Namanya tetap _is_router karena semua pemanggilnya menanyakan hal yang sama:
    "boleh pakai anggaran balon dan prompt sistem?" — dan jawabannya sama untuk
    kedua penyedia LLM. Yang membedakan router dari faucet cuma kelas client.
    """
    p = (provider or SETTINGS.provider or "").lower()
    return "router" in p or "faucet" in p


def _is_faucet(provider: str | None = None) -> bool:
    p = (provider or SETTINGS.provider or "").lower()
    return "faucet" in p


def _secret(name: str) -> str:
    """Satu rahasia dari Colab Secrets -> env. Nilainya tidak pernah dicetak."""
    try:
        from google.colab import userdata

        val = userdata.get(name)
        if val:
            return val.strip()
    except (ImportError, KeyError, Exception):  # noqa: BLE001 - SecretNotFound
        pass
    return os.environ.get(name, "").strip()


def get_api_key(ui_key: str | None = None, provider: str | None = None) -> str:
    """Colab Secrets -> env -> field UI. Jangan pernah hardcode.

    Nama rahasianya ikut penyedia (DEEPL_API_KEY vs ROUTER_API_KEY) supaya
    keduanya bisa tersimpan berdampingan dan berganti penyedia di UI tidak
    menuntut menempel key lagi.
    """
    router = _is_router(provider)
    faucet = _is_faucet(provider)
    name = "FAUCET_API_KEY" if faucet else ("ROUTER_API_KEY" if router else "DEEPL_API_KEY")
    key = _secret(name)
    if key:
        return key
    if ui_key and ui_key.strip():
        return ui_key.strip()
    # Jalur terakhir khusus penyedia LLM: file kredensial lokal di luar repo/
    # notebook. Sengaja TIDAK dipakai untuk DeepL — deepl.txt tidak pernah
    # dibaca kode, dan itu tetap begitu.
    #
    # Formatnya beda per file, jadi diambil dengan regex bukan indeks baris:
    # test.txt = baris 3 berisi key mentah; freetokenfaucet.txt = potongan kode
    # Python berisi api_key="tf_..."; gorouter.txt = baris `set` gaya Windows
    # berisi ANTHROPIC_AUTH_TOKEN=... Regex tahan terhadap baris yang bergeser.
    if faucet:
        from config import WORK

        try:
            raw = (WORK / "freetokenfaucet.txt").read_text(encoding="utf-8")
        except OSError:
            raw = ""
        m = re.search(r'api_key\s*=\s*["\']([^"\']+)["\']', raw)
        if m:
            return m.group(1).strip()
    elif router:
        from config import WORK

        # gorouter.txt dulu, karena itulah host yang dilayani ROUTER_API_BASE
        # sekarang. test.txt dibiarkan sebagai jalur kedua supaya konfigurasi
        # lama tidak mendadak kehilangan key-nya.
        try:
            graw = (WORK / "gorouter.txt").read_text(encoding="utf-8")
        except OSError:
            graw = ""
        gm = re.search(r'ANTHROPIC_AUTH_TOKEN\s*=\s*["\']?(\S+?)["\']?\s*$',
                       graw, re.M)
        if gm:
            return gm.group(1).strip()
        for cand in (WORK / "test.txt",):
            try:
                ln = [x.strip() for x in cand.read_text(encoding="utf-8").splitlines()]
            except OSError:
                continue
            if len(ln) > 3 and ln[3]:
                return ln[3]
    raise RuntimeError(
        f"API key tidak ditemukan. Isi Colab Secrets '{name}' "
        "atau tempel di field API Key pada UI."
    )


def make_client(api_key: str, provider: str | None = None):
    if _is_faucet(provider):
        return FaucetClient(api_key.strip())
    if _is_router(provider):
        return RouterClient(api_key.strip())
    return DeepLClient(api_key.strip())


def pick_model(client, verbose: bool = True) -> tuple[str, list[ProbeResult]]:
    """Nama model yang dipakai + tabel probe.

    DeepL tidak butuh pemilihan model — tidak ada yang menolak konten. Router
    memakai model dari client (test.txt/env), dan verifikasi ketersediaannya
    TIDAK dilakukan di sini: router ini membalas 502 untuk model yang ada dan
    berhasil di panggilan berikutnya, jadi probe yang gagal akan memilih model
    cadangan tanpa alasan. call_any() yang menangani itu saat penerjemahan.
    """
    if isinstance(client, RouterClient):
        if verbose:
            print(f"[{client.tag}] {client.base}  model={client.model}  (key tidak dicetak)")
        return client.model, []
    return "deepl", []


def probe_model(client, model: str, listed: bool) -> ProbeResult:
    if isinstance(client, RouterClient):
        t0 = time.monotonic()
        try:
            got = _router_call(client, model, "Reply with the JSON {\"0\": \"OK\"}.",
                               "Lines:\n{\"0\": \"テスト\"}",
                               timeout=min(60, client.timeout))
            return ProbeResult(model, True, bool(got), f"{client.tag} ok",
                               time.monotonic() - t0, str(got)[:60])
        except Exception as exc:  # noqa: BLE001
            return ProbeResult(model, True, False,
                               f"{type(exc).__name__}: {str(exc)[:80]}",
                               time.monotonic() - t0)
    return ProbeResult("deepl", True, True, "deepl ok", 0.0)


def check_usage(client) -> str:
    """Kuota DeepL: 'dipakai / limit' dari endpoint /usage.

    Router tidak punya endpoint kuota; yang berguna di sana adalah apakah dia
    MENJAWAB, jadi yang dilaporkan hasil probe satu panggilan.
    """
    if isinstance(client, RouterClient):
        p = probe_model(client, client.model, True)
        return f"{client.model}: {p.reason} ({p.latency:.1f}s)"
    try:
        status, data = _http_json(client, "GET", "/usage", None)
        if status == 200:
            return (
                f"{data.get('character_count', 0):,} / "
                f"{data.get('character_limit', 0):,} karakter"
            )
        return f"http {status}"
    except Exception as exc:  # noqa: BLE001
        return f"{type(exc).__name__}: {str(exc)[:120]}"


# ---------------------------------------------------------------- HTTP


def _http_json(client, method: str, path: str, payload: dict | None,
               timeout: int = 60) -> tuple[int, dict | str]:
    req = urllib.request.Request(DEEPL_API_BASE + path, method=method)
    req.add_header("Authorization", f"DeepL-Auth-Key {client.key}")
    body = None
    if payload is not None:
        req.add_header("Content-Type", "application/json")
        body = json.dumps(payload).encode()
    try:
        with urllib.request.urlopen(req, body, timeout=timeout) as r:
            return r.status, json.loads(r.read().decode())
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode(errors="replace")[:400]
    except Exception as exc:  # noqa: BLE001
        return 0, f"{type(exc).__name__}: {str(exc)[:200]}"


def _translate_texts(client, texts: list[str], target_lang: str,
                     formality: str) -> list[str]:
    """Satu batch terjemahan DeepL dengan retry transien (3x)."""
    last: Exception | None = None
    for attempt in range(3):
        payload: dict = {
            "text": texts,
            "target_lang": target_lang,
            "source_lang": "JA",
        }
        if formality and formality != "default":
            payload["formality"] = formality
        status, data = _http_json(client, "POST", "/translate", payload)
        if status == 200 and isinstance(data, dict):
            return [t.get("text", "") for t in data.get("translations", [])]
        if status in (429, 500, 502, 503, 504):
            last = RuntimeError(f"http {status}: {str(data)[:120]}")
            time.sleep(2 * (attempt + 1))
            continue
        raise RuntimeError(f"DeepL http {status}: {str(data)[:200]}")
    raise last  # type: ignore[misc]


# ---------------------------------------------------------------- HTTP router


def _decode_router(raw: str) -> dict:
    """Body router -> dict. Content-Type-nya text/event-stream walau non-stream.

    Router ini mengembalikan SATU objek chat.completion lalu menempelkan
    `data: [DONE]` TANPA pemisah baris. json.loads() gagal dengan 'Extra data'
    padahal objeknya utuh — jadi dipakai raw_decode() yang berhenti di akhir
    objek pertama dan mengabaikan sisanya.
    """
    raw = (raw or "").strip()
    if raw.startswith("data:"):
        raw = raw[5:].lstrip()
    obj, _end = json.JSONDecoder().raw_decode(raw)
    return obj


def _router_call(client, model: str, system: str, user: str,
                 timeout: int = ROUTER_TIMEOUT) -> dict:
    """Satu panggilan chat/completions -> objek JSON hasil terjemahan.

    Body-nya diambil dari client, bukan dari konstanta: client.extra membawa
    parameter khusus penyedia (faucet butuh thinking.type=disabled, kalau tidak
    jatah keluarannya habis untuk reasoning dan content keluar KOSONG tanpa
    error HTTP) dan client.max_tokens membatasi keluaran kalau penyedia
    menghitung token — router tidak, faucet ya, dan tokennya terbatas.
    """
    body = {
        "model": model, "temperature": 0.3, "stream": False,
        "messages": [{"role": "system", "content": system},
                     {"role": "user", "content": user}],
        **getattr(client, "extra", {}),
    }
    if getattr(client, "max_tokens", 0):
        body["max_tokens"] = client.max_tokens
    req = urllib.request.Request(client.base + "/chat/completions", method="POST")
    req.add_header("Authorization", "Bearer " + client.key)
    req.add_header("Content-Type", "application/json")
    for _hk, _hv in getattr(client, "headers", {}).items():
        req.add_header(_hk, _hv)
    with urllib.request.urlopen(req, json.dumps(body).encode(), timeout=timeout) as r:
        d = _decode_router(r.read().decode())
    ch = (d.get("choices") or [{}])[0]
    txt = ch.get("message", {}).get("content") or ""
    u = d.get("usage") or {}
    m = re.search(r"\{.*\}", txt, re.S)
    if not m:
        # finish_reason + jumlah token keluaran IKUT di pesan, bukan cuma
        # cuplikan teksnya. Sebabnya konkret: "bukan JSON" punya dua penyebab
        # yang penanganannya berlawanan. finish_reason="length" berarti jawaban
        # TERPOTONG dan max_tokens-lah yang kurang (FAUCET_MAX_TOKENS=1200 cukup
        # untuk 8 balon — terukur out=147 — tapi halaman 19 balon bisa
        # melampauinya); finish_reason="stop" dengan teks utuh berarti model
        # memang tidak membalas JSON dan yang salah promptnya. Tanpa angka ini
        # pembaca log harus menebak di antara keduanya.
        raise ValueError(
            f"bukan JSON (finish_reason={ch.get('finish_reason')!r}, "
            f"out={u.get('completion_tokens', '?')}/{body.get('max_tokens', '-')} "
            f"token): {txt[:200]}"
        )
    print(f"[{getattr(client, 'tag', 'router')}] in={u.get('prompt_tokens', '?')} "
          f"out={u.get('completion_tokens', '?')}")
    return json.loads(m.group(0))


def _router_call_any(client, model: str, system: str, user: str) -> tuple[dict, str]:
    """_router_call() dengan percobaan ulang + model cadangan, BERBATAS WAKTU.

    Mengulang model YANG SAMA sebelum pindah, karena 502 dari router ini
    SEMENTARA dan bukan tanda model tidak ada: terukur, satu model membalas 502
    tiga kali lalu 200 dalam 4 detik sementara /models tetap 200 sepanjang waktu.
    Berpindah model saja tidak menolong — yang menolong mencoba lagi.

    Tapi mencoba lagi HARUS ada batasnya, dan batas itu satu angka untuk seluruh
    rangkaian (ROUTER_DEADLINE), bukan hasil perkalian timeout x percobaan x
    model. Versi pertama tanpa deadline: satu halaman menggantung 3 jam karena
    router tidak menutup koneksi — di UI kelihatan seperti "GPU lambat" padahal
    tidak ada satu pun kernel yang jalan. Deadline dilewatkan juga ke timeout
    tiap percobaan, supaya percobaan terakhir tidak melompati batasnya sendiri.
    """
    order = (model, *(f for f in client.fallback if f != model))
    tried: list[str] = []
    deadline = client.deadline
    t_end = time.monotonic() + deadline
    for attempt in range(1, ROUTER_RETRY + 1):
        for m in order:
            left = t_end - time.monotonic()
            if left <= 1:
                raise TimeoutError(
                    f"{client.tag} tidak menjawab dalam {deadline}s "
                    f"({', '.join(tried[-6:]) or 'tanpa balasan'})"
                )
            short = m.rsplit("/", 1)[-1]
            try:
                return _router_call(client, m, system, user,
                                    timeout=int(min(client.timeout, left))), m
            except urllib.error.HTTPError as e:
                if e.code == 402:
                    # 402 bukan gangguan sementara: modelnya berbayar dan saldo
                    # akun 0, jadi mencoba lagi PASTI gagal dan cuma memakan
                    # deadline. Body aslinya berbahasa Mandarin
                    # ("为付费模型，但你的资金账户余额不足"), jadi pesan sendiri lebih
                    # berguna daripada meneruskannya — yang perlu dibaca user
                    # adalah nama modelnya dan tindakannya.
                    raise RuntimeError(
                        f"{client.tag}: model '{m}' BERBAYAR dan saldo akun 0 "
                        f"(HTTP 402). Ganti ke model gratis lewat env "
                        f"{'FAUCET_MODEL' if client.tag == 'faucet' else 'ROUTER_MODEL'}"
                        " (faucet gratis: mimo-v2.5-pro, mimo-v2.5,"
                        " gpt-5.6-terra) atau top-up akunnya."
                    ) from e
                if e.code not in (429, 500, 502, 503, 504):
                    raise
                tried.append(f"{short}={e.code}")
                note("warn", client.tag, f"{short} -> HTTP {e.code}")
            except (urllib.error.URLError, TimeoutError, ValueError) as e:
                # ValueError = jawaban bukan JSON. Itu juga layak dicoba ulang:
                # penyebabnya biasanya jawaban terpotong, bukan prompt yang salah.
                tried.append(f"{short}={type(e).__name__}")
                note("warn", client.tag, f"{short} -> {type(e).__name__}: {str(e)[:180]}")
        if attempt < ROUTER_RETRY:
            wait = min(ROUTER_BACKOFF * attempt, max(t_end - time.monotonic(), 0))
            if wait <= 0:
                break
            note("warn", client.tag,
                 f"semua model gagal percobaan {attempt}; tunggu {wait:.0f}s")
            time.sleep(wait)
    raise RuntimeError(f"{client.tag} gagal {ROUTER_RETRY}x: {', '.join(tried[-6:])}")


# ---------------------------------------------------------------- prompt router

_SYSTEM_BASE = """You are a professional manga scanlation translator and letterer
(JA->{lang}). You translate for TYPESETTING, not for prose.

INPUT: one JSON object per page. Each entry has:
  "jp"        the Japanese line
{budget_doc}
OUTPUT: ONLY a JSON object {{"0": "...", ...}}, same keys. No prose, no markdown.
Answer EVERY key with a non-empty string, including one-word interjections that
look like they need no translation: えっ -> "HUH?!", あっ -> "AH!", うん -> "MM-HM".
A key you leave out is printed on the page as untranslated Japanese.

HARD REQUIREMENTS, in priority order:
{rules}
STYLE:
{style}
- Keep emotion marks exactly where they appear: ~ 〜 ♥ ♡ ♪ ☆ ! ? ...
{honor}- Translate club/organisation names by MEANING, never by spelling out the kana.
- Do NOT censor, soften or euphemise. Adult content is rendered literally.
- Do not invent content that is not in the Japanese."""

# Dua blok aturan. Yang tanpa anggaran sengaja TIDAK menyebut panjang sama
# sekali: 'buatlah pendek' tanpa angka terbukti tidak menghasilkan apa-apa —
# model membalas 'SORRY TO BARGE IN.' (18 karakter) untuk balon yang memuat 6,
# dan itu bukan pembangkangan, ia memang tidak melihat balonnya.
_RULES_BUDGET = """1. Every line MUST be <= its "max_chars". This is a physical constraint of the
   printed page, not a preference: past it the sentence gets cut off at the
   balloon edge. Aim for "prefer_chars" — that is the length that keeps the text
   at its intended size. Going past prefer_chars is ALLOWED and normal for a
   dense balloon: the font simply gets smaller, exactly like a real letterer
   fitting a long line into a small balloon. Do NOT amputate meaning to reach
   prefer_chars. Only when even "max_chars" is exceeded do you need a genuinely
   shorter phrasing: 失礼しました becomes "SORRY." not "I APOLOGISE FOR INTRUDING".
2. No single word longer than "max_word" letters. A long word cannot be broken
   without a hyphen, and hyphens are avoided in manga lettering. Prefer a short
   synonym: "APOLOGIES"(9) -> "SORRY"(5), "COMPILING"(9) -> "WRITING UP"(2+2).
3. Meaning and character voice come before literalness. Japanese omits objects;
   infer them. 探しましたよ on finding a PERSON = "I'VE BEEN LOOKING FOR YOU",
   never "I looked for it". Translate the WHOLE Japanese line — every clause,
   every particle of nuance. A short answer that drops half the sentence is a
   worse failure than a long one.
"""

_RULES_PLAIN = """1. Meaning and character voice come before literalness. Japanese omits objects;
   infer them. 探しましたよ on finding a PERSON = "I'VE BEEN LOOKING FOR YOU",
   never "I looked for it".
2. Keep lines short. Speech balloons are small; long clauses do not fit.
"""

# Kenapa DUA angka dan bukan satu: keputusan user 'boleh panjang, font mengecil'.
# max_chars diambil dari char_budget pada LANTAI ukuran font (= yang benar-benar
# masih tercetak), prefer_chars dari plafon proporsional balon. Versi sebelumnya
# mengirim plafon proporsional sebagai 'max_chars' berbunyi MUST, dan pada
# hasilnew/jp_6.JPG itu berarti model diperintah menulis 2-39 karakter untuk
# balon yang wording typeset referensinya 15-71 karakter — hasilnya 'SO?' untuk
# balon yang referensinya satu kalimat penuh.
_BUDGET_DOC = """  "max_chars"    hard ceiling; past this the line is cut off at the balloon edge
  "prefer_chars" length that keeps the intended font size; going over just
                 shrinks the font, which is fine and normal
  "max_word"     longest single word that fits on one line in that balloon
"""


def _system_prompt(target_lang: str, style: str, keep_honorifics: bool,
                   with_budget: bool) -> str:
    return _SYSTEM_BASE.format(
        lang=(target_lang or "English").upper(),
        budget_doc=_BUDGET_DOC if with_budget else "",
        rules=_RULES_BUDGET if with_budget else _RULES_PLAIN,
        style="- " + TRANSLATION_STYLES.get(
            style, TRANSLATION_STYLES["Manga Natural"]),
        honor=("- Keep honorifics (-san, -kun, -chan, -senpai, -sama).\n"
               if keep_honorifics else
               "- Localise honorifics into natural address in the target language.\n"),
    )


def _user_prompt(items: list[Region], budget: dict[int, dict] | None) -> str:
    """JSON berisi jp DAN (kalau ada) anggaran per balon."""
    payload: dict[str, object] = {}
    for r in items:
        if budget and r.idx in budget:
            d = budget[r.idx]
            payload[str(r.idx)] = {"jp": r.src_text,
                                   "max_chars": d["hard"],
                                   "prefer_chars": d["soft"],
                                   "max_word": d["word_hard"]}
        else:
            payload[str(r.idx)] = r.src_text
    return "Lines:\n" + json.dumps(payload, ensure_ascii=False, indent=1)


# ---------------------------------------------------------------- label (SFX)


def _sfx_core(text: str) -> str:
    """Inti kana: buang simbol/tanda baca, sisakan kana + ー + っ/ッ.

    "フー．．．"  -> "フー"
    "ドキッ♥"    -> "ドキッ"
    "（はぁ…）"  -> "はぁ"
    """
    return "".join(ch for ch in text if _KANA.match(ch))


def _broken_kana(raw: str) -> bool:
    """Apakah ada JEDA di ANTARA dua kana (bukan di ujung)?

    "ヒ．．．ッ"  -> True   jeda memutus mora: napas tertahan = suara tokoh
    "キャ．．．ッ" -> True
    "フー．．．"   -> False  jeda di UJUNG: bunyi yang memanjang lalu berhenti
    "ゴクッ．．．" -> False
    "ドキッ"     -> False  tidak ada jeda sama sekali

    Dipakai hanya sebagai penahan aturan K di _sfx_in_bubble. Perhatikan bahwa
    _sfx_core MEMBUANG semua tanda baca, jadi informasi ini sudah lenyap dari
    `core` — penahannya wajib melihat teks ASLI.
    """
    idx = [i for i, ch in enumerate(raw) if _KANA.match(ch)]
    if len(idx) < 2:
        return False
    return any(raw[i] in _PAUSE for i in range(idx[0] + 1, idx[-1]))


def _sfx_pattern(core: str) -> bool:
    """Pola ulang onomatope: ドキドキ, ドキドキドキ, ぴくぴくっ, ばたばた...

    Kepala harus pengulangan PENUH dari satu unit (>= 2 ulangan), lalu ekor
    opsional っ/ッ/ー 1-2 char (ぴくぴくっ, どきどきっ). Bentuk ini sengaja
    menolak kata pinjaman seperti サッカー (サッ+カ+ー — bukan ulangan) dan
    kata 2-char seperti うう / ええ (dialog) yang tidak boleh dikunci.
    """
    n = len(core)
    if n < 4:
        return False
    for tail_len in (0, 1, 2):
        tail = core[n - tail_len:] if tail_len else ""
        if tail and not all(c in _SMALL or c == _LONG for c in tail):
            continue
        head = core[: n - tail_len]
        m = len(head)
        if m < 2:
            continue
        for u in range(1, m // 2 + 1):
            unit = head[:u]
            if all(c in _SMALL or c == _LONG for c in unit):
                continue
            if m % u == 0 and m // u >= 2 and head == unit * (m // u):
                return True
    return False


def _kata2hira(s: str) -> str:
    """Katakana -> hiragana, HANYA untuk pencarian kamus (bukan untuk render).

    Tanpa ini kamus dialog yang isinya hiragana tidak pernah bisa cocok dengan
    dialog yang di manga ditulis katakana (ダメッ ハイッ ウンッ ムリッ オイッ),
    dan semuanya jatuh ke SFX = balon Jepang tercetak. Tidak ada normalisasi
    kana lain di modul ini, jadi ini satu-satunya jembatannya.
    """
    return "".join(
        chr(ord(c) - 0x60) if "ァ" <= c <= "ヶ" else c for c in s
    )


def _all_katakana(core: str) -> bool:
    """Inti kana yang SELURUHNYA katakana (tanpa satu pun hiragana).

    Konvensi manga: bunyi ditulis katakana, ucapan hiragana. Yang ditulis
    katakana tapi memang ucapan sudah diselamatkan kamus (lewat _kata2hira)
    sebelum aturan ini dipakai.
    """
    ada = any("ァ" <= c <= "ヶ" for c in core)
    hira = any("ぁ" <= c <= "ゖ" for c in core)
    return ada and not hira


def _sfx_stem(core: str) -> str:
    """Inti minus ekor っ/ッ/ー: どきっ -> どき, あーっ -> あ, ダメッ -> ダメ."""
    i = len(core)
    while i and (core[i - 1] in _SMALL or core[i - 1] == _LONG):
        i -= 1
    return core[:i]


def _has_kanji(text: str) -> bool:
    """Kalimat dengan kanji pasti dialog — jangan pernah dikunci sebagai SFX."""
    return any(
        0x4E00 <= ord(ch) <= 0x9FFF
        or 0x3400 <= ord(ch) <= 0x4DBF
        or 0xF900 <= ord(ch) <= 0xFAFF
        for ch in text
    )


def _sfx_in_bubble(core: str, n: int, raw: str) -> bool:
    """Apakah inti kana DI DALAM balon adalah bunyi (SFX), bukan ucapan.

    Cabang lama di sini satu baris: `n == 3 and has_small` -> SFX. Itu penyebab
    cacat "balon pendek tidak diterjemah". SFX berarti translation=None +
    PROTECTED, jadi translate_page MELEWATI region itu tanpa satu pun pesan
    error, dan yang tercetak adalah balon Jepang asli. _label2.py mengukur
    cabang itu pada 32 kasus: 21 salah — 20 seruan yang jelas ucapan (ええっ
    うんっ だめっ いやっ まてっ うそっ なにっ ちょっ そこっ はいっ ねえっ
    もうっ やめっ あーっ ふぇっ ...) dikunci SFX hanya karena panjangnya 3 dan
    ada っ, plus satu arah sebaliknya (ハッ, katakana, SFX sejati -> DIALOGUE).

    Penggantinya tiga bagian, diukur dua arah sekaligus di _label3.py:
      N  kamus dialog dicari pada bentuk hiragana DAN pada batangnya
         (_kata2hira + _sfx_stem), supaya dialog yang ditulis katakana
         (ダメッ ハイッ ウンッ ムリッ オイッ) ketemu lewat entri hiragana-nya.
         Tanpa ini tidak ada normalisasi kana sama sekali di modul ini, jadi
         katakana-ditulis-ucapan tidak pernah bisa cocok kamus.
      K  kana ber-っ/ー yang SELURUHNYA katakana = bunyi (konvensi manga: bunyi
         katakana, ucapan hiragana). Yang katakana tapi memang ucapan sudah
         diselamatkan N lebih dulu, jadi urutannya wajib N sebelum K.
      S  yang hiragana hanya SFX kalau BATANGNYA terbukti onomatope, yaitu
         batang-gandanya ada di _SFX_DICT (どきっ -> どきどき). Beban buktinya
         sengaja DIBALIK dari yang lama: dulu cukup 'pendek dan ada っ'.
      V  K adalah satu-satunya aturan yang menang HANYA lewat BENTUK — tanpa
         bukti kamus maupun pola ulang. Jadi K tidak boleh menang kalau teks
         ASLINYA membawa tanda BICARA: intonasi tanya, atau jeda DI ANTARA
         kana. Ini yang membedakan ヒ．．．ッ！？ (jeritan tokoh di balon hitam
         bergerigi, hasilnew5) dari ハッ — dua-duanya satu mora katakana + ッ
         di dalam balon, jadi panjang batang TIDAK bisa memisahkannya.
         V ditaruh di dalam K, bukan menggantikannya, supaya bunyi berkamus
         tetap lolos lewat S sesudahnya (ドキッ！？ -> どきどき di _SFX_DICT).

    Terukur pada 72 kasus (45 dialog + 27 SFX): arah dialog 30 salah -> 0,
    arah SFX 4 salah -> 0. Asimetrinya sengaja: salah menuduh dialog = balon
    Jepang tercetak dan pembaca tidak bisa membacanya, salah melepas SFX =
    SFX ikut diterjemah tapi masih tertahan kamus. はぁっ tetap SFX karena
    はぁはぁ ada di _SFX_DICT — itu embusan napas, bukan ucapan.

    V diukur terpisah di _h5lbl.py atas 60 kasus selftest + 14 kasus baru,
    melawan 4 kandidat lain. Hanya V yang nol kesalahan di KEDUA arah:
    "cuma ？" melewatkan ヒ．．．ッ tanpa tanda tanya; "？ atau ！" merusak
    ズドンッ！ dan パチンッ！ (batangnya tidak berkamus, jadi S tidak
    menyelamatkannya); "batang <= 1 kana wajib berkamus" merusak ハッ.

    `raw` WAJIB, tanpa nilai bawaan: raw="" membuat V mati diam-diam, dan
    aturan yang mati diam-diam adalah cacat yang sama seperti label SFX yang
    melewati balon tanpa pesan error.
    """
    hira = _kata2hira(core)
    stem = _kata2hira(_sfx_stem(core))
    # N — pemanggil sudah menguji `core in _KA_DIALOGUE`; di sini bentuk
    # hiragana dan batangnya.
    if hira in _KA_DIALOGUE or (stem and stem in _KA_DIALOGUE):
        return False
    if _sfx_pattern(core):
        return True                             # どきどき ぴくぴくっ
    if core in _SFX_DICT or hira in _SFX_DICT:
        return True
    # K — bunyi ditulis katakana. Yang menandai bunyi adalah っ/ッ di UJUNG,
    # bukan sembarang っ: サッカー juga seluruhnya katakana dengan ッ dan ー,
    # tapi ッ-nya di tengah dan ujungnya ー — itu kata pinjaman, dan selftest
    # menjaganya tetap DIALOGUE. Batas 6 supaya pinjaman panjang ber-ッ di ujung
    # (kalau ada) tidak terjaring.
    if n <= 6 and _all_katakana(core) and core[-1] in _SMALL:
        # V — kecuali teks aslinya bertanda bicara. Bunyi tidak bertanya, dan
        # bunyi tidak terputus di tengah mora. Jatuh ke S, tidak return False,
        # supaya bunyi yang PUNYA bukti kamus tetap bisa menang.
        if not (any(c in _ASK for c in raw) or _broken_kana(raw)):
            return True                         # ハッ ドキッ ズドンッ パチンッ
    # S — hiragana harus punya catatan onomatope-nya.
    if stem and (stem + stem) in _SFX_DICT:
        return True                             # どきっ びくっ ごくっ
    return False


def _label_region(r: Region) -> None:
    """Klasifikasi satu region: SFX (dijaga utuh) atau DIALOGUE (diterjemah).

    Ambang di luar balon sengaja lebih longgar — SFX manga hampir selalu
    di luar balon, sedangkan kata-kata kana panjang di luar balon jarang.
    Di dalam balon lebih konservatif agar dialog pendek (うん, ええ, はい)
    tidak ikut dikunci.
    """
    t = r.src_text.strip()
    if not t or r.label == "UNREADABLE":
        return

    # Kalimat ber-kanji = dialog sungguhan. Inti kana hanya dinilai bila
    # teks aslinya murni kana + simbol.
    if _has_kanji(t):
        r.label = "DIALOGUE"
        r.label_conf = 0.5
        return

    core = _sfx_core(t)
    if not core:
        r.label = "DIALOGUE"
        r.label_conf = 0.5
        return

    # Kata dialog umum menang atas pola SFX (それは, ちょっと, ごめん...).
    if core in _KA_DIALOGUE:
        r.label = "DIALOGUE"
        r.label_conf = 0.5
        return

    n = len(core)
    in_bubble = r.bubble_bbox is not None
    has_small = any(c in _SMALL for c in core)
    has_long = _LONG in core

    if not in_bubble:
        if n <= 3:
            is_sfx = True                       # ドン バン ピクッ フー ドキッ
        elif has_small or has_long:
            is_sfx = n <= 8                     # ガーン ぴくぴくっ ガッタンゴットン
        elif _sfx_pattern(core):
            is_sfx = True                       # ドキドキ ばたばた どきどきどき
        elif core in _SFX_DICT:
            is_sfx = True
        else:
            is_sfx = False
    else:
        is_sfx = _sfx_in_bubble(core, n, t)

    r.label = "SFX" if is_sfx else "DIALOGUE"
    r.label_conf = 0.6 if is_sfx else 0.5
    r.translation = None if is_sfx else r.translation


def _fallback_labels(regions: list[Region]) -> list[Region]:
    """Label heuristik: SFX = kana di luar/di dalam balon yang berpola.

    Ini klasifikasi BAWAAN pipeline — di versi DeepL dipakai SELALU
    (DeepL tidak bisa menilai SFX).
    """
    for r in regions:
        if r.label != "UNREADABLE":
            _label_region(r)
    return regions


# ---------------------------------------------------------------- simbol


def _restore_symbols(src: str, translation: str) -> str:
    """Jaring pengaman simbol emosi yang hilang saat lewat DeepL.

    DeepL umumnya mempertahankan ♥ ♡ ♪ ☆ 〜 … dan mengubah ！？ jadi !?.
    Tapi kadang simbol di ujung kalimat terbuang (mis. '大好き♥' -> 'I love
    you'). Kalau sumber berakhiran simbol emosi dan hasilnya kehilangan,
    simbol itu disalin ulang ke ujung terjemahan.
    """
    out = (translation or "").strip()
    if not src or not out:
        return out
    out = out.replace("！", "!").replace("？", "?")
    src_end = src.rstrip()
    for ch in _EMOTION:
        if ch in src and ch not in out and src_end.endswith(ch):
            out = out + ch
    return out


# Karakter yang MEMBAWA kata: kanji, hiragana, katakana, katakana setengah-lebar,
# latin, angka. Sengaja TIDAK memuat ー (U+30FC, tanda panjang) dan ・ (U+30FB,
# pemisah) walau keduanya duduk di blok katakana — keduanya tidak pernah menjadi
# kata sendirian, jadi 'ー．．．' harus tetap dihitung tanpa kata.
_WORDY = re.compile(
    r"[一-鿿㐀-䶿ぁ-ゖァ-ヺｦ-ﾝA-Za-z0-9]"
)


def _symbols_only(text: str) -> bool:
    """src_text tanpa satu pun karakter berkata: '．．．', '！？', '♥', '〜'."""
    return bool((text or "").strip()) and not _WORDY.search(text)


def _symbols_as_text(src: str) -> str:
    """Simbol sumber dipetakan ke ASCII, TANPA lstrip _clean_translation().

    _clean_translation() membuang tanda baca di AWAL string ('. SHIZUKU' ->
    'SHIZUKU'), dan pada balon yang isinya cuma simbol SELURUH isinya ada di awal
    — '．．．' akan keluar sebagai string kosong dan balonnya tercetak hampa.
    """
    return " ".join((src or "").translate(_PUNCT_MAP).split())


def _clean_translation(text: str) -> str:
    """Buang glyph yang tidak ada di font komik, samakan punctuation ke ASCII.

    Kurung sudut Jepang lolos apa adanya dari DeepL ('「会長っ」' -> '「Prez」'),
    dan Anime Ace cuma ~159 glyph sehingga 「 」 dirender jadi kotak tofu ⟦ ⟧
    di dalam balon. Kurungnya memang tidak bermakna di Inggris — dibuang saja.

    _EMOTION (♥ ♡ ♪ ☆) TIDAK dibuang: simbol itu wajib bertahan. 〜 dan ～
    dinormalkan ke '~' — bentuknya tetap ada, lebarnya jadi ASCII supaya
    dirender oleh font balon yang sama (lihat komentar di _PUNCT_MAP).
    """
    out = (text or "").translate(_PUNCT_MAP)
    # Titik/koma nyasar di awal terjemahan ('. SHI ZUKU...') — sisa tanda baca
    # Jepang yang kurungnya sudah dibuang. Ujung kanan tidak diusik: '...' dan
    # '?!' di akhir kalimat memang disengaja.
    return " ".join(out.lstrip(" .,:;-–—").split())


def translate_page(client, model: str, regions: list[Region],
                   target_lang: str = "English",
                   style: str = "Manga Natural",
                   keep_honorifics: bool = True) -> list[Region]:
    """Label heuristik dulu, lalu terjemahkan lewat penyedia aktif.

    SFX (dan teks terlindungi lain) tidak pernah dikirim ke mana pun.
    """
    _fallback_labels(regions)
    # Balon yang isinya HANYA simbol ('．．．', '！？', '♥') tidak punya kata untuk
    # diterjemahkan. Dikirim ke model, ia membalas kosong — wajar, tidak ada yang
    # bisa dijawab — lalu jalur perbaikan menuduhnya "BELUM DITERJEMAHKAN" dan
    # satu-satunya error di laporan halaman jadi alarm palsu. Terukur di
    # hitomi_3740721_015: r12='．．．' menghasilkan error_count 1 dengan
    # final_font_size 0, jadi balonnya keluar KOSONG — simbolnya hilang pula.
    #
    # Diselesaikan di sini, SEBELUM items dibangun, sehingga region ini tidak
    # pernah dikirim ke penyedia mana pun, tidak masuk _missing_ids, tidak
    # menghasilkan error, dan tidak dihitung untranslated oleh verify.report().
    # Simbolnya dipetakan ke ASCII dan dicetak apa adanya — '．．．' -> '...' —
    # jadi balon tetap berisi apa yang memang tertulis di halaman aslinya.
    for r in regions:
        if (r.label not in PROTECTED_LABELS and r.translation is None
                and _symbols_only(r.src_text)):
            r.translation = _symbols_as_text(r.src_text) or None
    items = [
        r for r in regions
        if r.label not in PROTECTED_LABELS and r.src_text and r.translation is None
    ]
    if not items:
        return regions
    if isinstance(client, RouterClient):
        return _translate_router(client, model, regions, items, target_lang,
                                 style, keep_honorifics)
    return _translate_deepl(client, items, regions, target_lang, style)


def _translate_deepl(client, items: list[Region], regions: list[Region],
                     target_lang: str, style: str) -> list[Region]:
    code = DEEPL_TARGET.get(target_lang or "English", "EN")
    formality = FORMALITY_BY_STYLE.get(style, "default")

    try:
        for i in range(0, len(items), _CHUNK):
            chunk = items[i : i + _CHUNK]
            out = _translate_texts(client, [r.src_text for r in chunk], code, formality)
            for r, t in zip(chunk, out):
                t = (t or "").strip() or None
                if t is not None:
                    t = _clean_translation(_restore_symbols(r.src_text, t)) or None
                r.translation = t
    except Exception as exc:  # noqa: BLE001 - kegagalan API tidak boleh membunuh halaman
        note("error", "translate",
             f"DeepL gagal ({exc}); halaman keluar TANPA terjemahan, "
             "teks asli tetap di sidecar JSON")
    # Sama seperti jalur router: balon yang tidak dapat terjemahan tercetak
    # berbahasa Jepang, dan itu tidak memicu error apa pun. Jadi disebut di log.
    left = [r for r in items if not r.translation]
    if left:
        note("error", "translate", "BELUM DITERJEMAHKAN (DeepL): "
             + "; ".join(f"r{r.idx}={r.src_text!r}" for r in left))
    return regions


# ------------------------------------------------------------- anggaran + revisi


def _page_budget(items: list[Region]) -> dict[int, dict]:
    """Anggaran karakter per balon, diukur dengan mesin tata letak SUNGGUHAN.

    Dihitung di sini, bukan ditaksir dari lebar bbox, karena angkanya harus
    keluar dari layout() yang nanti merender — anggaran yang dihitung terpisah
    bisa melenceng tanpa ada yang tahu. Balon tanpa mask dilewati: tidak ada
    geometri untuk diukur, jadi baris itu dikirim tanpa batas.
    """
    import typeset

    fp = typeset.FONT_USED or typeset.setup_fonts(verbose=False)
    out: dict[int, dict] = {}
    for r in items:
        try:
            out[r.idx] = typeset.region_budget(r, fp)
        except Exception as exc:  # noqa: BLE001 - satu balon gagal != halaman gagal
            note("warn", "budget",
                 f"r{r.idx} dilewati ({type(exc).__name__}: {exc})")
    return out


def _violations(texts: dict[int, str], budget: dict[int, dict],
                items: list[Region]) -> dict[int, str]:
    """Ukur jawaban model dengan layout() sungguhan. Lapis PENENTU.

    Anggaran karakter itu PROKSI — dihitung dengan teks pengisi, bukan dengan
    kalimat yang akhirnya dipakai. Yang mengikat cuma satu: apakah kalimat INI
    bisa DICETAK di balon INI tanpa terpotong. Jadi yang diukur
    typeset.renders_ok() atas teks aslinya, dan anggaran hanya dipakai untuk
    MEMBERI TAHU model harus sependek apa.

    Plafon proporsional sengaja BUKAN kriteria lulus. Terukur: wording typeset
    profesional halaman referensi sendiri duduk di bawah plafonnya di tiga balon
    padat (-6, -4, -5 px). Menjadikannya kriteria berarti menolak hasil yang
    justru ditiru.

    Kriterianya juga BUKAN 'muat utuh tanpa penggalan di atas lantai'
    (_max_feasible), dan itu juga terukur: pada hasilnew/jp_6.JPG wording
    referensi r3 "CAN'T HELP IT ♥" dan r4 "UH... I-IT'S EMBARRASSING..."
    keduanya memberi feasible 0, padahal fit() merendernya bersih di 6 dan 8 px
    (probe_r34.py). Validator versi itu menolak wording yang sedang ditiru dan
    memaksa model menulis makin pendek — persis keluhan 'translate-nya sedikit
    banget'. Yang tersisa sebagai cacat sungguhan: fit() melaporkan LUBER, yaitu
    barisnya benar-benar terpotong di tepi balon.

    Diukur pada bentuk SETELAH _clean_translation + huruf besar, bukan apa yang
    model tulis. Sebabnya konkret: model membalas '＼SORRY.' dan validator versi
    pertama melaporkan 'tidak muat di ukuran minimum' untuk 7 karakter di balon
    yang memuat 39 — penyebabnya '＼' yang tidak punya glyph di Anime Ace,
    padahal pipeline membuangnya sebelum typeset. Menghukum model atas glyph
    yang sudah ditangani orang lain hanya menghasilkan revisi yang sia-sia.
    """
    import typeset

    fp = typeset.FONT_USED or typeset.setup_fonts(verbose=False)
    rmap = {r.idx: r for r in items}
    bad: dict[int, str] = {}
    for i, t in sorted(texts.items()):
        r, d = rmap.get(i), budget.get(i)
        if r is None or d is None:
            continue
        up = _clean_translation(t or "").upper()
        if not up:
            continue
        try:
            mask = typeset._region_box_mask(r)[1]
            ok, _size = typeset.renders_ok(up, mask, fp)
        except Exception:  # noqa: BLE001
            continue
        if not ok:
            bad[i] = (
                f"{len(up)} chars get cut off at the balloon edge even at the "
                f"smallest readable size. This balloon holds about {d['soft']} "
                f"characters and no single word longer than {d['word_hard']} "
                f"letters. Rewrite much shorter, same meaning."
            )
    return bad


def _missing_ids(got: dict[int, str], items: list[Region]) -> list[int]:
    """idx yang TIDAK punya jawaban terpakai dari model.

    "Tidak punya jawaban" mencakup tiga hal yang semuanya berakhir sama di atas
    kertas: kunci tidak ada di JSON balasan, isinya string kosong/spasi, atau
    isinya habis setelah _clean_translation (mis. model membalas cuma '．．．').
    Diukur pada bentuk AKHIR, bukan pada apa yang model tulis, karena yang
    menentukan balon tercetak berbahasa Inggris atau tidak adalah bentuk akhir.
    """
    out = []
    for r in items:
        t = (got.get(r.idx) or "").strip()
        if not t or not _clean_translation(_restore_symbols(r.src_text, t)):
            out.append(r.idx)
    return out


def _translate_router(client, model: str, regions: list[Region],
                      items: list[Region], target_lang: str, style: str,
                      keep_honorifics: bool) -> list[Region]:
    """Terjemah lewat router, dengan tiga lapis penjaga panjang.

    1. PROMPT   — batas per balon ikut di JSON masukan, sebagai angka.
    2. VALIDASI — jawabannya diukur ulang dengan layout() sungguhan, tidak
                  dipercaya. Model bisa saja mengaku patuh dan tetap melanggar.
    3. PERBAIKAN— hanya baris yang MASIH melanggar dikirim ulang, dengan angka
                  pelanggarannya disebut. Mengirim ulang seluruh halaman membuat
                  model 'memperbaiki' baris yang sudah benar dan merusaknya.

    Lalu satu lapis lagi yang sifatnya berbeda: KELENGKAPAN. Ketiga lapis di
    atas menjaga jawaban yang ADA tetap muat; tidak satu pun menjaga jawabannya
    ADA. Terukur di hasilnew/13.JPG: balon 'えっ！？' terdeteksi (conf .807),
    terbaca OCR (ink .494), berlabel DIALOGUE, ikut terkirim — lalu model
    memutuskan seruan sependek itu tidak perlu diterjemahkan dan kuncinya
    hilang dari JSON. Loop lama menelan itu tanpa suara (`if not t: continue`),
    translation tetap None, render_region() keluar lebih awal, dan pembaca
    melihat satu balon Jepang di tengah halaman Inggris. Sekarang kunci yang
    hilang diminta ulang, dan kalau tetap hilang dicetak dengan sebutan idx-nya
    supaya tidak pernah lagi lolos tanpa terlihat.
    """
    use_budget = bool(SETTINGS.balloon_budget)
    budget = _page_budget(items) if use_budget else {}
    system = _system_prompt(target_lang, style, keep_honorifics, bool(budget))
    got: dict[int, str] = {}
    try:
        raw, used = _router_call_any(client, model or client.model, system,
                                     _user_prompt(items, budget or None))
        got = {int(k): str(v) for k, v in raw.items()}
    except Exception as exc:  # noqa: BLE001 - jaringan tidak boleh membunuh halaman
        note("error", "translate",
             f"{client.tag} gagal ({exc}); halaman keluar TANPA terjemahan "
             f"({len(items)} balon tetap berbahasa Jepang), teks asli di sidecar JSON")
        return regions

    rmap = {r.idx: r for r in items}
    if budget:
        for rnd in range(int(SETTINGS.budget_repair_rounds)):
            bad = _violations(got, budget, items)
            if not bad:
                break
            print(f"[budget] perbaiki {len(bad)} baris {sorted(bad)}")
            extra = ("\n\nREVISION. Your previous attempt broke the balloon budget "
                     "on these lines. Rewrite ONLY these, shorter, same meaning:\n"
                     + "\n".join(f'  "{i}": you wrote {got[i]!r} -> {why}'
                                 for i, why in sorted(bad.items())))
            sub = [rmap[i] for i in sorted(bad) if i in rmap]
            try:
                fix, _u = _router_call_any(client, used, system,
                                           _user_prompt(sub, budget) + extra)
            except Exception as exc:  # noqa: BLE001
                note("warn", "budget", f"revisi {rnd + 1} gagal ({exc}); pakai yang ada")
                break
            for k, v in fix.items():
                if int(k) in bad:
                    got[int(k)] = str(v)
        else:
            left = _violations(got, budget, items)
            if left:
                # Bukan kegagalan halaman: fit() masih akan mengecilkan fontnya.
                # Dicetak supaya balon yang mepet terlihat, bukan diam-diam kecil.
                note("warn", "budget",
                     f"{sorted(left)} masih melebihi balon setelah "
                     f"{SETTINGS.budget_repair_rounds} revisi; fit() yang menangani")

    # Lapis KELENGKAPAN. Dijalankan setelah ronde anggaran, bukan sebelum:
    # revisi anggaran bisa saja menjatuhkan kunci yang tadinya ada, jadi
    # pemeriksaan harus melihat keadaan terakhir. Yang dikirim ulang HANYA idx
    # yang kosong — sama seperti pola revisi anggaran di atas — dan promptnya
    # menyebut sebabnya, karena model yang menghapus 'えっ！？' melakukannya
    # dengan sengaja: ia butuh diberi tahu bahwa balon kosong itu tercetak.
    for rnd in range(int(SETTINGS.missing_repair_rounds)):
        miss = _missing_ids(got, items)
        if not miss:
            break
        print(f"[translate] {len(miss)} balon belum dijawab {miss}; minta ulang")
        sub = [rmap[i] for i in miss if i in rmap]
        if not sub:
            break
        extra = (
            "\n\nMISSING. Your previous reply had no usable text for these ids. "
            "Every one of them is a real speech balloon on the page: if you leave "
            "it out the reader sees raw Japanese. Answer ALL of them with a short "
            "non-empty English line — even a bare interjection gets one "
            "(えっ！？ -> \"HUH?!\"). Reply with JSON for these ids only:\n"
            + "\n".join(f'  "{r.idx}": {r.src_text!r}' for r in sub)
        )
        try:
            fix, _u = _router_call_any(client, used, system,
                                       _user_prompt(sub, budget or None) + extra)
        except Exception as exc:  # noqa: BLE001 - jaringan tidak boleh membunuh halaman
            note("warn", "translate", f"permintaan ulang gagal ({exc}); pakai yang ada")
            break
        for k, v in fix.items():
            try:
                ki = int(k)
            except (TypeError, ValueError):
                continue
            if ki in miss and str(v).strip():
                got[ki] = str(v)

    for r in items:
        t = (got.get(r.idx) or "").strip()
        if not t:
            continue
        r.translation = _clean_translation(_restore_symbols(r.src_text, t)) or None

    # Kalau setelah semua ronde masih ada yang kosong, itu HARUS terlihat.
    # Diam adalah cacatnya: satu balon Jepang di tengah halaman Inggris tidak
    # menghasilkan error apa pun, cuma hasil yang salah. Bukan exception —
    # tiga balon lain sudah benar dan halamannya tetap layak keluar — tapi
    # namanya disebut di log dan src_text-nya dikutip supaya bisa dicari.
    left = [r for r in items if not r.translation]
    if left:
        note("error", "translate", "BELUM DITERJEMAHKAN setelah "
             f"{SETTINGS.missing_repair_rounds} permintaan ulang: "
             + "; ".join(f"r{r.idx}={r.src_text!r}" for r in left))
    return regions


Writing /content/mangatl/translate.py


In [10]:
%%writefile /content/mangatl/inpaint.py

"""LaMa inpainting — generator FFC di-vendor karena checkpoint-nya state_dict mentah.

`lama_large_512px.ckpt` menyimpan bobot di bawah key `gen_state_dict`, BUKAN
TorchScript. Jadi `simple-lama-inpainting` (yang pakai `torch.jit.load`) tidak
kompatibel dan generator harus dibangun sendiri.

Kalau bobot gagal dimuat, modul turun ke `cv2.inpaint` tanpa crash — jalur
flat-fill sudah menangani 70-85% region jadi degradasinya kecil.
"""

from __future__ import annotations

import cv2
import numpy as np
import torch
import torch.nn as nn

from config import SETTINGS, WEIGHTS, note

# ---------------------------------------------------------------- FFC blocks


class FourierUnit(nn.Module):
    """Konvolusi 1x1 di domain frekuensi — inti receptive field global LaMa."""

    def __init__(self, in_channels: int, out_channels: int, groups: int = 1):
        super().__init__()
        self.groups = groups
        self.conv_layer = nn.Conv2d(
            in_channels * 2, out_channels * 2, 1, 1, 0, groups=groups, bias=False
        )
        self.bn = nn.BatchNorm2d(out_channels * 2)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch = x.shape[0]
        fft_dim = (-2, -1)
        ffted = torch.fft.rfftn(x.float(), dim=fft_dim, norm="ortho")
        ffted = torch.stack((ffted.real, ffted.imag), dim=-1)
        ffted = ffted.permute(0, 1, 4, 2, 3).contiguous()
        ffted = ffted.view((batch, -1) + ffted.size()[3:])

        ffted = self.relu(self.bn(self.conv_layer(ffted)))

        ffted = (
            ffted.view((batch, -1, 2) + ffted.size()[2:])
            .permute(0, 1, 3, 4, 2)
            .contiguous()
        )
        ffted = torch.complex(ffted[..., 0], ffted[..., 1])
        return torch.fft.irfftn(ffted, s=x.shape[-2:], dim=fft_dim, norm="ortho")


class SpectralTransform(nn.Module):
    def __init__(
        self, in_channels: int, out_channels: int, stride: int = 1,
        groups: int = 1, enable_lfu: bool = False,
    ):
        super().__init__()
        self.enable_lfu = enable_lfu
        self.downsample = nn.AvgPool2d(2, 2) if stride == 2 else nn.Identity()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels // 2, 1, groups=groups, bias=False),
            nn.BatchNorm2d(out_channels // 2),
            nn.ReLU(inplace=True),
        )
        self.fu = FourierUnit(out_channels // 2, out_channels // 2, groups)
        if enable_lfu:
            self.lfu = FourierUnit(out_channels // 2, out_channels // 2, groups)
        self.conv2 = nn.Conv2d(out_channels // 2, out_channels, 1, groups=groups, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(self.downsample(x))
        return self.conv2(x + self.fu(x))


class FFC(nn.Module):
    """Bagi channel jadi cabang lokal (spasial) dan global (spektral)."""

    def __init__(
        self, in_channels: int, out_channels: int, kernel_size: int,
        ratio_gin: float, ratio_gout: float, stride: int = 1, padding: int = 0,
        dilation: int = 1, groups: int = 1, bias: bool = False,
        enable_lfu: bool = False, padding_type: str = "reflect",
    ):
        super().__init__()
        in_cg = int(in_channels * ratio_gin)
        in_cl = in_channels - in_cg
        out_cg = int(out_channels * ratio_gout)
        out_cl = out_channels - out_cg
        self.ratio_gin, self.ratio_gout = ratio_gin, ratio_gout
        self.global_in_num = in_cg

        def conv(ci: int, co: int) -> nn.Module:
            if ci == 0 or co == 0:
                return nn.Identity()
            return nn.Conv2d(
                ci, co, kernel_size, stride, padding, dilation, groups, bias,
                padding_mode=padding_type,
            )

        self.convl2l = conv(in_cl, out_cl)
        self.convl2g = conv(in_cl, out_cg)
        self.convg2l = conv(in_cg, out_cl)
        self.convg2g = (
            nn.Identity()
            if in_cg == 0 or out_cg == 0
            else SpectralTransform(
                in_cg, out_cg, stride, 1 if groups == 1 else groups // 2, enable_lfu
            )
        )

    def forward(self, x):
        x_l, x_g = x if isinstance(x, tuple) else (x, 0)
        out_xl, out_xg = 0, 0
        if self.ratio_gout != 1:
            out_xl = self.convl2l(x_l) + self.convg2l(x_g)
        if self.ratio_gout != 0:
            out_xg = self.convl2g(x_l) + self.convg2g(x_g)
        return out_xl, out_xg


class FFC_BN_ACT(nn.Module):
    def __init__(
        self, in_channels: int, out_channels: int, kernel_size: int,
        ratio_gin: float, ratio_gout: float, stride: int = 1, padding: int = 0,
        dilation: int = 1, groups: int = 1, bias: bool = False,
        norm_layer=nn.BatchNorm2d, activation_layer=nn.Identity,
        padding_type: str = "reflect", enable_lfu: bool = False,
    ):
        super().__init__()
        self.ffc = FFC(
            in_channels, out_channels, kernel_size, ratio_gin, ratio_gout, stride,
            padding, dilation, groups, bias, enable_lfu, padding_type,
        )
        gc = int(out_channels * ratio_gout)
        lnorm = nn.Identity if ratio_gout == 1 else norm_layer
        gnorm = nn.Identity if ratio_gout == 0 else norm_layer
        self.bn_l = lnorm(out_channels - gc)
        self.bn_g = gnorm(gc)
        lact = nn.Identity if ratio_gout == 1 else activation_layer
        gact = nn.Identity if ratio_gout == 0 else activation_layer
        self.act_l = lact()
        self.act_g = gact()

    def forward(self, x):
        x_l, x_g = self.ffc(x)
        return self.act_l(self.bn_l(x_l)), self.act_g(self.bn_g(x_g))


class FFCResnetBlock(nn.Module):
    def __init__(self, dim: int, padding_type: str, norm_layer, activation_layer,
                 dilation: int = 1, ratio_gin: float = 0.75, ratio_gout: float = 0.75,
                 enable_lfu: bool = False):
        super().__init__()
        kw = dict(
            kernel_size=3, padding=dilation, dilation=dilation,
            ratio_gin=ratio_gin, ratio_gout=ratio_gout, norm_layer=norm_layer,
            activation_layer=activation_layer, padding_type=padding_type,
            enable_lfu=enable_lfu,
        )
        self.conv1 = FFC_BN_ACT(dim, dim, **kw)
        self.conv2 = FFC_BN_ACT(dim, dim, **kw)

    def forward(self, x):
        x_l, x_g = x if isinstance(x, tuple) else (x, 0)
        id_l, id_g = x_l, x_g
        x_l, x_g = self.conv2(self.conv1((x_l, x_g)))
        return id_l + x_l, id_g + x_g


class ConcatTupleLayer(nn.Module):
    def forward(self, x):
        x_l, x_g = x
        return x_l if not torch.is_tensor(x_g) else torch.cat(x, dim=1)


class FFCResNetGenerator(nn.Module):
    """Arsitektur generator LaMa. n_blocks 9 (base) atau 18 (large)."""

    def __init__(
        self, input_nc: int = 4, output_nc: int = 3, ngf: int = 64,
        n_downsampling: int = 3, n_blocks: int = 18, max_features: int = 1024,
    ):
        super().__init__()
        norm, act = nn.BatchNorm2d, nn.ReLU
        model: list[nn.Module] = [
            nn.ReflectionPad2d(3),
            FFC_BN_ACT(
                input_nc, ngf, kernel_size=7, padding=0, ratio_gin=0, ratio_gout=0,
                norm_layer=norm, activation_layer=act,
            ),
        ]
        for i in range(n_downsampling):
            mult = 2 ** i
            model.append(
                FFC_BN_ACT(
                    min(max_features, ngf * mult),
                    min(max_features, ngf * mult * 2),
                    kernel_size=3, stride=2, padding=1,
                    ratio_gin=0, ratio_gout=0.75 if i == n_downsampling - 1 else 0,
                    norm_layer=norm, activation_layer=act,
                )
            )
        feats = min(max_features, ngf * 2 ** n_downsampling)
        for _ in range(n_blocks):
            model.append(FFCResnetBlock(feats, "reflect", norm, act))
        model.append(ConcatTupleLayer())
        for i in range(n_downsampling):
            mult = 2 ** (n_downsampling - i)
            model += [
                nn.ConvTranspose2d(
                    min(max_features, ngf * mult),
                    min(max_features, int(ngf * mult / 2)),
                    kernel_size=3, stride=2, padding=1, output_padding=1,
                ),
                norm(min(max_features, int(ngf * mult / 2))),
                nn.ReLU(True),
            ]
        model += [nn.ReflectionPad2d(3), nn.Conv2d(ngf, output_nc, kernel_size=7, padding=0),
                  nn.Sigmoid()]
        self.model = nn.Sequential(*model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)


# ---------------------------------------------------------------- runtime

_MODEL: FFCResNetGenerator | None = None
_LOAD_FAILED = False

# Konteks di sekeliling mask yang ikut dikirim ke generator, dan sisi minimum
# tile. LaMa butuh tekstur tetangga untuk ditiru; kotak yang mepet glyph cuma
# berisi lubang. 64 px cukup untuk beberapa periode screentone.
_TILE_PAD = 64
_TILE_MIN = 192


def _infer_n_blocks(sd: dict) -> int:
    """Baca jumlah resnet block dari nama key — jangan tebak base vs large."""
    idxs = [
        int(k.split(".")[1])
        for k in sd
        if k.startswith("model.") and ".conv1.ffc." in k and k.split(".")[1].isdigit()
    ]
    return (max(idxs) - 4) if idxs else 18  # blok resnet mulai di index 5


def get_model(device: str = "cuda") -> FFCResNetGenerator | None:
    """Muat generator sekali. None kalau bobot tidak cocok — pemanggil fallback."""
    global _MODEL, _LOAD_FAILED
    if _MODEL is not None or _LOAD_FAILED:
        return _MODEL

    path = WEIGHTS / "lama_large_512px.ckpt"
    if not path.exists():
        _LOAD_FAILED = True
        return None
    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        sd = ckpt.get("gen_state_dict", ckpt) if isinstance(ckpt, dict) else ckpt
        sd = {k.replace("generator.", "", 1): v for k, v in sd.items()}

        model = FFCResNetGenerator(n_blocks=_infer_n_blocks(sd))
        missing, unexpected = model.load_state_dict(sd, strict=False)

        # Self-check: kalau lebih dari 5% parameter tidak terisi, arsitekturnya
        # beda dan hasilnya akan jadi bubur. Lebih baik jatuh ke cv2.
        total = len(model.state_dict())
        if len(missing) > total * 0.05:
            note("warn", "inpaint",
                 f"arsitektur tidak cocok ({len(missing)}/{total} kosong) -> cv2.inpaint")
            _LOAD_FAILED = True
            return None
        if unexpected:
            print(f"[inpaint] {len(unexpected)} key ekstra diabaikan")

        model.eval().to(device)
        for p in model.parameters():
            p.requires_grad_(False)
        _MODEL = model
    except (RuntimeError, KeyError, OSError, ValueError) as exc:
        note("warn", "inpaint", f"gagal muat LaMa ({exc}) -> pakai cv2.inpaint")
        _LOAD_FAILED = True
        _MODEL = None
    return _MODEL


def _pad8(arr: np.ndarray) -> tuple[np.ndarray, int, int]:
    h, w = arr.shape[:2]
    ph, pw = (-h) % 8, (-w) % 8
    if ph or pw:
        arr = cv2.copyMakeBorder(arr, 0, ph, 0, pw, cv2.BORDER_REFLECT)
    return arr, ph, pw


def _cv2_fallback(img: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Telea cukup baik untuk screentone manga saat LaMa tidak tersedia."""
    return cv2.inpaint(img, (mask > 0).astype(np.uint8), 5, cv2.INPAINT_TELEA)


def _grow(box: list[int], w: int, h: int) -> tuple[int, int, int, int]:
    """Perbesar kotak sampai minimal _TILE_MIN, lalu jepit ke tepi halaman."""
    x1, y1, x2, y2 = box
    for _ in range(2):  # dua lintasan: setelah dijepit di tepi, sisi lain digeser
        dw, dh = _TILE_MIN - (x2 - x1), _TILE_MIN - (y2 - y1)
        if dw > 0:
            x1, x2 = x1 - dw // 2, x2 + dw - dw // 2
        if dh > 0:
            y1, y2 = y1 - dh // 2, y2 + dh - dh // 2
        x1, y1, x2, y2 = max(0, x1), max(0, y1), min(w, x2), min(h, y2)
    return x1, y1, x2, y2


def _mask_boxes(mask: np.ndarray) -> list[tuple[int, int, int, int]]:
    """Kotak berisi mask + konteks sekelilingnya; yang bertumpuk digabung."""
    h, w = mask.shape[:2]
    n, _, stats, _ = cv2.connectedComponentsWithStats((mask > 0).astype(np.uint8), 8)
    boxes = [
        [
            int(stats[i, 0]) - _TILE_PAD, int(stats[i, 1]) - _TILE_PAD,
            int(stats[i, 0] + stats[i, 2]) + _TILE_PAD,
            int(stats[i, 1] + stats[i, 3]) + _TILE_PAD,
        ]
        for i in range(1, n)
    ]

    # Digabung sampai tidak ada yang bertumpuk: satu blok teks tidak boleh pecah
    # jadi satu tile per glyph — tiap tile kehilangan konteks tetangganya, dan
    # tile yang bertumpuk akan menimpa hasil tetangganya.
    changed = True
    while changed:
        changed, merged = False, []
        for b in boxes:
            for o in merged:
                if b[0] < o[2] and o[0] < b[2] and b[1] < o[3] and o[1] < b[3]:
                    o[0], o[1] = min(o[0], b[0]), min(o[1], b[1])
                    o[2], o[3] = max(o[2], b[2]), max(o[3], b[3])
                    changed = True
                    break
            else:
                merged.append(b)
        boxes = merged

    return [_grow(b, w, h) for b in boxes]


def _run(
    model: FFCResNetGenerator, crop: np.ndarray, mask: np.ndarray, device: str
) -> np.ndarray | None:
    """Satu forward pass. None kalau OOM — pemanggil jatuh ke cv2."""
    h, w = crop.shape[:2]
    scale = min(1.0, SETTINGS.lama_size / max(h, w))
    if scale < 1.0:
        sm = (max(1, int(w * scale)), max(1, int(h * scale)))
        crop_s = cv2.resize(crop, sm, interpolation=cv2.INTER_AREA)
        mask_s = cv2.resize(mask, sm, interpolation=cv2.INTER_NEAREST)
    else:
        crop_s, mask_s = crop, mask

    pimg, ph, pw = _pad8(crop_s)
    pmask, _, _ = _pad8(mask_s)

    t_img = torch.from_numpy(pimg).permute(2, 0, 1).float().div_(255.0)[None]
    t_msk = torch.from_numpy((pmask > 0).astype(np.float32))[None, None]
    t_img, t_msk = t_img.to(device), t_msk.to(device)

    try:
        with torch.inference_mode():
            # fp32 saja: torch.fft ada di cast-policy fp32, AMP nihil manfaat.
            pred = model(torch.cat([t_img * (1 - t_msk), t_msk], dim=1))
            out = pred * t_msk + t_img * (1 - t_msk)
    except (RuntimeError, torch.cuda.OutOfMemoryError):
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return None

    arr = (out[0].permute(1, 2, 0).clamp(0, 1).cpu().numpy() * 255).astype(np.uint8)
    arr = arr[: arr.shape[0] - ph if ph else None, : arr.shape[1] - pw if pw else None]
    if arr.shape[:2] != (h, w):
        arr = cv2.resize(arr, (w, h), interpolation=cv2.INTER_CUBIC)
    return arr


def inpaint(img: np.ndarray, mask: np.ndarray, device: str | None = None) -> np.ndarray:
    """Inpaint area mask. img RGB uint8, mask uint8 0/255. Return RGB uint8.

    Kontrak forward LaMa: input 4-channel cat([img*(1-m), m]), img float [0,1],
    mask float {0,1} — bukan 255. Output dikomposit pred*m + (1-m)*img.

    Dijalankan per TILE di resolusi asli, bukan sekali untuk seluruh halaman.
    Halaman manga tingginya ~2000 px; menyusutkannya ke 512 lalu membesarkannya
    lagi meratakan screentone jadi bercak kelabu buram — persis artefak yang
    terlihat di kolom narasi. Tile di sekeliling mask hampir selalu di bawah
    512 px, jadi crosshatch direkonstruksi 1:1 tanpa resample sama sekali.
    """
    if mask.max() == 0:
        return img
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = get_model(device)
    if model is None:
        return _cv2_fallback(img, mask)

    out = img.copy()
    for x1, y1, x2, y2 in _mask_boxes(mask):
        sub_mask = mask[y1:y2, x1:x2]
        if sub_mask.max() == 0:
            continue
        arr = _run(model, img[y1:y2, x1:x2], sub_mask, device)
        if arr is None:
            return _cv2_fallback(img, mask)
        # Piksel di luar mask harus persis asli, jadi komposit di sini juga.
        m3 = (sub_mask > 0)[:, :, None]
        out[y1:y2, x1:x2] = np.where(m3, arr, out[y1:y2, x1:x2])
    return out


def release() -> None:
    """Bebaskan ~205 MB + VRAM."""
    global _MODEL
    _MODEL = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()



Writing /content/mangatl/inpaint.py


In [11]:
%%writefile /content/mangatl/erase.py

"""Routing erase: flat-fill cepat vs neural inpaint LaMa.

70-85% region bisa cukup flat-fill — nol GPU, nol residu secara konstruksi.
Sisanya baru masuk LaMa atau cv2.inpaint.
"""

from __future__ import annotations

import cv2
import numpy as np

from config import SETTINGS, Region
import inpaint as inp
# textmask tidak mengimpor erase, jadi arah impor ini tidak melingkar.
import textmask as tm


def _fill_on_page(region: Region, shape: tuple[int, int]) -> np.ndarray | None:
    """fill_mask region di koordinat halaman, atau None kalau tidak ada."""
    if region.fill_mask is None or region.fill_bbox is None:
        return None
    h, w = shape
    x1, y1, x2, y2 = region.fill_bbox
    sx1, sy1 = max(x1, 0), max(y1, 0)
    mh, mw = region.fill_mask.shape[:2]
    sx2, sy2 = min(x2, x1 + mw, w), min(y2, y1 + mh, h)
    if sx2 <= sx1 or sy2 <= sy1:
        return None
    out = np.zeros((h, w), np.uint8)
    out[sy1:sy2, sx1:sx2] = region.fill_mask[sy1 - y1:sy2 - y1, sx1 - x1:sx2 - x1]
    return out


def fill_color(img: np.ndarray, region: Region) -> tuple[int, int, int] | None:
    """Warna isian interior balon: median piksel interior yang BUKAN tinta.

    Bukan putih tetap. Balon hitam (teks putih di atas hitam) memang ada di
    manga, dan mengisinya putih akan merusak halaman lebih parah daripada sisa
    tinta yang mau dihilangkan. Median interior-minus-tinta mengembalikan hitam
    untuk balon hitam dan putih untuk balon putih, tanpa cabang khusus.

    Tinta dikeluarkan dari perhitungan lewat ink_mask yang didilatasi: kalau
    tidak, glyph ikut menarik median ke arah warna tinta dan isian jadi kelabu.
    """
    m = _fill_on_page(region, img.shape[:2])
    if m is None:
        return None
    inside = m > 0
    if not inside.any():
        return None
    ink = np.zeros(img.shape[:2], np.uint8)
    if region.ink_mask is not None:
        x1, y1, x2, y2 = region.bbox
        mh, mw = region.ink_mask.shape[:2]
        y2, x2 = min(y2, y1 + mh, img.shape[0]), min(x2, x1 + mw, img.shape[1])
        if y2 > y1 and x2 > x1:
            ink[y1:y2, x1:x2] = region.ink_mask[: y2 - y1, : x2 - x1]
        ink = cv2.dilate(ink, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
    bg_px = img[inside & (ink == 0)]
    if bg_px.size < 30:
        bg_px = img[inside]
    return tuple(int(v) for v in np.median(bg_px.reshape(-1, 3), axis=0))


def _bg_stats(img: np.ndarray, region: Region) -> tuple[np.ndarray, float]:
    """Median warna + sebaran background. Sebaran rendah -> flat-fill.

    Diukur pada PITA TIPIS di sekeliling tinta, bukan seluruh kotak region.
    Kotak region sudah dilebarkan 6 px saat mask dibangun, jadi untuk balon yang
    pas ia ikut memakan garis luar balon yang hitam — dan itu yang membuat
    seluruh 13 region halaman uji lari ke LaMa, termasuk balon yang isinya putih
    bersih. LaMa lalu mengarang bercak kelabu di tempat yang flat-fill akan
    selesaikan sempurna. Pita di sekeliling glyph adalah tetangga yang benar-benar
    harus ditiru oleh isian.

    Sebaran dihitung dari MAD, bukan np.std maupun rentang persentil. Std tidak
    robust sama sekali, dan persentil 5-95 cuma tahan 5% outlier — pada balon yang
    teksnya mepet, pita ikut menyeberangi garis luar balon lebih dari itu, jadi
    tiga balon berlatar putih murni masih terbaca 999 dan lari ke LaMa. MAD tahan
    sampai 50% outlier: garis balon adalah BATAS, bukan tekstur, dan mayoritas
    pita tetap putih rata. Screentone tetap tertangkap karena di sana sebaran
    itulah mayoritasnya.
    """
    x1, y1, x2, y2 = region.bbox
    crop = img[y1:y2, x1:x2]
    if region.ink_mask is None or crop.size == 0:
        return np.array([255, 255, 255], dtype=np.uint8), 0.0

    ink = region.ink_mask[: crop.shape[0], : crop.shape[1]]
    ring = cv2.subtract(
        cv2.dilate(ink, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))), ink
    )
    bg_px = crop[ring > 0]
    if bg_px.size < 30:
        bg_px = crop[ink == 0]
    if bg_px.size < 10:
        return np.array([255, 255, 255], dtype=np.uint8), 0.0

    median = np.median(bg_px, axis=0).astype(np.uint8)
    dev = np.abs(bg_px.astype(np.float32) - median.astype(np.float32))
    spread_per_channel = np.median(dev, axis=0) * 1.4826  # MAD -> skala sigma
    noise = np.std(spread_per_channel)
    thresh = SETTINGS.flat_std_thresh_noisy if noise > 1 else SETTINGS.flat_std_thresh
    max_spread = float(spread_per_channel.max())

    return median, max_spread if max_spread < thresh else 999.0


def route_region(img: np.ndarray, region: Region) -> Region:
    """Putuskan flat vs lama. Region skip langsung return."""
    if region.is_protected or region.ink_mask is None:
        region.route = "skip"
        return region

    # Jalur isian-interior: warnanya diambil dari interior balon itu sendiri,
    # jadi tidak perlu ronde LaMa sama sekali — hasilnya rata sempurna secara
    # konstruksi dan mustahil menyisakan satu titik pun.
    fc = fill_color(img, region) if SETTINGS.bubble_fill else None
    if fc is not None:
        region.bg_color = fc
        region.route = "flat"
        return region

    median, std = _bg_stats(img, region)
    region.bg_color = tuple(int(v) for v in median)

    if std < SETTINGS.flat_std_thresh:
        region.route = "flat"
    else:
        region.route = "lama"
    return region


def erase_flat(img: np.ndarray, region: Region,
               guard: np.ndarray | None = None) -> np.ndarray:
    """Isi region dengan median background — instan, nol GPU.

    Kalau fill_mask tersedia (balon yang dikenali), yang diisi adalah SELURUH
    interior balon, bukan cuma stroke glyph — lihat textmask.build_fill_mask().
    `guard` adalah piksel yang tidak boleh disentuh isian (SFX + garis balon).
    """
    if region.bg_color is None:
        return img
    fill = _fill_on_page(region, img.shape[:2]) if SETTINGS.bubble_fill else None
    if fill is not None:
        sel = fill > 0
        if guard is not None:
            sel &= guard == 0
        img[sel] = region.bg_color
        return img
    if region.ink_mask is None:
        return img
    x1, y1, x2, y2 = region.bbox
    crop = img[y1:y2, x1:x2].copy()
    mh, mw = region.ink_mask.shape[:2]
    y2, x2 = min(y2, y1 + mh), min(x2, x1 + mw)
    sub_mask = region.ink_mask[: y2 - y1, : x2 - x1]
    m3 = (sub_mask > 0)[:, :, None]
    crop[: y2 - y1, : x2 - x1][m3[:, :, 0]] = region.bg_color
    img[y1:y2, x1:x2] = crop[: y2 - y1, : x2 - x1]
    return img



def erase_neural(img: np.ndarray, mask: np.ndarray, device: str = "cuda") -> np.ndarray:
    """LaMa skala halaman. mask biner 0/255 koordinat halaman."""
    if mask.max() == 0:
        return img
    return inp.inpaint(img, mask, device)


def protected_guard(img: np.ndarray, regions: list[Region]) -> np.ndarray:
    """Piksel yang tidak boleh disentuh isian: tinta SFX + garis balon.

    Wajib ada begitu erase mengisi INTERIOR PENUH. Mask stroke tidak pernah
    menyentuh SFX yang duduk di dalam balon karena bentuknya mengikuti glyph
    dialog; isian interior penuh akan menimpanya. Dua hal yang dijaga:

    * tinta region terlindungi (SFX/UNREADABLE), dilebarkan 5 px — kontrak
      pipeline paling keras: 'SFX DAN SYMBOL SYMBOL TETAP ADA',
    * garis balon itu sendiri (textmask.bubble_outline_guard) — interior sudah
      dikikis sekali, tapi pada balon berstroke tebal kikisan itu bisa kurang,
      dan garis balon yang termakan isian terlihat sebagai balon bocor.
    """
    h, w = img.shape[:2]
    guard = np.zeros((h, w), np.uint8)
    for r in regions:
        if not r.is_protected or r.ink_mask is None:
            continue
        x1, y1, x2, y2 = r.bbox
        mh, mw = r.ink_mask.shape[:2]
        y2, x2 = min(y2, y1 + mh, h), min(x2, x1 + mw, w)
        if y2 > y1 and x2 > x1:
            guard[y1:y2, x1:x2] = np.maximum(
                guard[y1:y2, x1:x2], r.ink_mask[: y2 - y1, : x2 - x1])
    if guard.any():
        guard = cv2.dilate(
            guard, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
    return cv2.bitwise_or(guard, tm.bubble_outline_guard(img, regions))


def erase_page(img: np.ndarray, regions: list[Region], device: str = "cuda") -> np.ndarray:
    """Pipeline erase gabungan flat + neural, sesuai routing."""
    out = img.copy()
    lama_mask = np.zeros(img.shape[:2], np.uint8)
    guard = protected_guard(img, regions) if SETTINGS.bubble_fill else None

    for r in regions:
        route_region(out, r)
        if r.route == "flat":
            out = erase_flat(out, r, guard)
        elif r.route == "lama" and r.ink_mask is not None:
            x1, y1, x2, y2 = r.bbox
            mh, mw = r.ink_mask.shape[:2]
            y2, x2 = min(y2, y1 + mh), min(x2, x1 + mw)
            sub = r.ink_mask[: y2 - y1, : x2 - x1]
            lama_mask[y1:y2, x1:x2] = np.maximum(lama_mask[y1:y2, x1:x2], sub)

    if lama_mask.any():
        out = erase_neural(out, lama_mask, device)

    return out



Writing /content/mangatl/erase.py


In [12]:
%%writefile /content/mangatl/typeset.py

"""Typeset: unduh font, binary-search ukuran, centroid-outward line growing.

Target visual = gambar referensi: ALL CAPS, oblique, center dua sumbu, hitam
murni tanpa stroke di dalam bubble, bubble sempit pecah satu kata per baris.
"""

from __future__ import annotations

import urllib.error
import urllib.request
from functools import lru_cache
from pathlib import Path

import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont

from config import (FONT_ARABIC, FONT_CHAIN, FONT_CJK, FONT_FALLBACK, FONT_SHOUT,
                    FONT_SYMBOL, FONT_THAI, FONTS, SETTINGS, Region, note)

FONT_USED: str = ""
_SYMBOL_PATH: Path | None = None
_FALLBACK_PATH: Path | None = None
_CJK_PATH: Path | None = None
_ARABIC_PATH: Path | None = None
_THAI_PATH: Path | None = None
_CMAP_CACHE: dict[str, frozenset[int]] = {}

# Batas minimum pecahan kata saat dipenggal — lihat _split_word(). Aturan
# tipografi klasik: minimal 2 huruf ditinggal di baris ini, 3 huruf dibawa ke
# baris berikutnya. Menuntut 3 di kepala menolak awalan yang sah ('RE-QUESTED',
# 'DIS-POSING') dan itu menjepit ukuran font: 'REQUESTED.' jadi tidak bisa
# dipenggal sama sekali, dan balonnya tertahan di font 14.
_MIN_HEAD = 2
_MIN_TAIL = 3
# Penggalan kata baru dipakai kalau versi utuh membuat font turun di bawah
# fraksi ini dari versi ber-hyphen. Tanpa ambang, max() memilih versi ber-hyphen
# begitu ia lebih besar SATU poin pun, dan hasilnya penuh 'FI-NALLY', 'RE-ALLY',
# 'EXECU-TIVE' — sementara typeset referensi tidak punya satu pun tanda hubung.
# 0.75 = tukar tanda hubung hanya kalau ia membeli >= 33% ukuran font.
_HYPHEN_MIN_GAIN = 0.75
# Lantai keras ukuran font, dipakai HANYA di jalur darurat fit() ketika teks tidak
# muat di min_font_size sekalipun. Sebelumnya jalur itu memakai
# `min_font_size // 2` (= 5), yang membuat balon sempit dirender 6 px tanpa satu
# pun peringatan — di manga ukuran cetak itu tidak terbaca, dan 'berhasil tapi tak
# terbaca' lebih buruk daripada gagal yang kelihatan.
#
# Pada konfigurasi terkalibrasi (line_spacing 1.00, pad_ratio 0.04) halaman
# referensi TIDAK menyentuh lantai ini sama sekali — kedua balon tersempit (r6
# 68 px, r10 102 px) dirender 11 px (probe_final.py). Jadi ini murni jaring
# pengaman untuk halaman lain, bukan jalur yang dipakai rutin. Balon yang
# menabraknya menandakan wording-nya terlalu panjang untuk balonnya — itu urusan
# tahap wording, bukan ukuran font.
_MIN_FONT_FLOOR = 9
# Lebar halaman yang sedang dikerjakan, di-set sekali per halaman oleh
# pipeline.process_page() dan render_page(). Nol = belum di-set, dan min_font()
# lalu memakai lebar kalibrasi — jadi pemanggil lama (probe, selftest) berperilaku
# persis seperti sebelum lantai ini berskala resolusi.
_PAGE_W: int = 0
# Berapa px ukuran font boleh diturunkan demi blok yang seimbang atas-bawah —
# lihat _rebalance(). Ukuran terbesar yang muat tidak selalu bisa ditata rapi:
# di r12 halaman ini teks 'IS IT? LEMME SEE, C'MON~!' pada ukuran 15 mustahil
# lebih baik dari ketimpangan 43 px pada margin manapun (probe_r12_exhaust.py
# memindai SETIAP pemecahan baris dan SETIAP y yang legal), sementara ukuran 14
# turun ke 1 px. Jadi satu-dua px ukuran ditukar dengan blok yang benar-benar
# terpusat. Batasnya sengaja kecil: melonggarkannya berarti teks diam-diam
# mengecil, dan ukuran yang mengecil adalah cacat #3 di plan.txt.
_BAL_MAX_DROP = 3
# Berapa banyak ukuran font seorang PELEPAS boleh menyusut supaya tetangganya
# yang tercekik bisa membuang satu tanda hubung — lihat reclaim_unused_interiors().
# Fraksi, bukan angka px, dengan lantai 1 px: 3 px di ukuran 40 cuma 7% dan tidak
# terlihat, sedangkan 3 px di ukuran 9 adalah sepertiga tinggi huruf. Ambang px
# tetap karena itu salah di salah satu ujung — versi pertama memakai 1 px dan
# menolak pertukaran yang jelas menguntungkan di halaman uji balon-bertetangga
# (pengklaim 2 tanda hubung -> 1, pelepas 40 -> 37 pada balon 335x291 yang masih
# lapang), sementara di jp_6 pertukaran yang benar memang hanya 1 px (r2 9 -> 8).
# 0.10 melewatkan keduanya dan tetap menolak balon kecil digunduli.
_RECLAIM_LOSS = 0.10
_VOWELS = frozenset("AEIOUY")
_BREAK_CACHE: dict[str, set[int]] = {}
# Fraksi piksel band yang harus berada di dalam balon — lihat _row_free().
_ROW_COVER = 0.985
# Fraksi pita blok yang harus interior supaya satu baris dihitung "masih di
# dalam rongga" saat MENGUKUR batas atas/bawah — lihat block_slack().
# Sengaja jauh lebih longgar dari _ROW_COVER: yang di atas adalah izin "baris
# ini muat" (melanggarnya = huruf keluar balon), yang ini pengukuran letak
# batas. Menuntut 0.985 di sini mengembalikan cacat teks tidak terpusat.
# 0.20 dipilih supaya ekor balon (10-14 px) tidak ikut menggeser batas bawah,
# sementara 0.10 masih mengakuinya; lima ambang diukur di _fix12.py.
_SLACK_COVER = 0.20
# Simbol emosi yang TIDAK BOLEH diambil dari font utama, walau font utama
# mengaku punya codepoint-nya. Ini bukan kehati-hatian berlebih — ini terukur:
# anime_ace.ttf (259 glyph, font display Latin) MEMETAKAN U+2665 ke glyph
# bernama `yat`, yaitu huruf Cyrillic Ѣ, bukan hati. Jadi rantai fallback di
# _char_font() — yang hanya menyala kalau font utama TIDAK punya codepoint-nya —
# tidak pernah menyala, dan pembaca melihat huruf aneh di ujung balon. Itu
# persis cacat yang dilaporkan pada hasilnew/6.JPG dibanding jp_6.JPG.
#
# Terukur dari fonts/ di repo ini (fontTools getBestCmap):
#   anime_ace.ttf        U+2665 -> yat        (SALAH BENTUK), ♡ ♪ ☆ ★ 〜 tidak ada
#   NotoSansSymbols2     U+2665 -> heart, ♡ ❤ ☆ ★ ada; ♪ ♫ ♬ 〜 TIDAK ada
#   NotoSansCJKjp        ♥ ♡ ♪ ♫ ♬ ☆ ★ 〜 ～ ada semua
# Karena itu urutan rantai untuk karakter ini dibalik: simbol dulu (bentuk paling
# pas dan advance-nya tidak full-width), lalu CJK yang melengkapi not musik dan
# 〜, baru NotoSans. plan.txt mewajibkan simbol ini bertahan apa adanya, jadi
# menggambarnya dengan bentuk yang salah sama buruknya dengan menghapusnya.
_FORCE_SYMBOL = frozenset(ord(c) for c in "♥♡❤♪♫♬☆★〜～")
_PYPHEN: object | None = None
_PYPHEN_TRIED = False


def _pyphen():
    """Kamus pola penggalan. None kalau pyphen tidak terpasang — bukan error."""
    global _PYPHEN, _PYPHEN_TRIED
    if _PYPHEN_TRIED:
        return _PYPHEN
    _PYPHEN_TRIED = True
    try:
        import pyphen

        _PYPHEN = pyphen.Pyphen(lang="en_US")
    except (ImportError, OSError, KeyError):
        _PYPHEN = None
    return _PYPHEN


# ---------------------------------------------------------------- font setup


def _download(url: str, dest: Path, timeout: int = 60) -> bool:
    if dest.exists() and dest.stat().st_size > 1024:
        return True
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            data = resp.read()
        if len(data) < 1024:
            return False
        dest.write_bytes(data)
        return True
    except (urllib.error.URLError, urllib.error.HTTPError, OSError, TimeoutError):
        return False


def _usable(path: Path) -> bool:
    """Font baru dianggap valid kalau Pillow benar-benar bisa membukanya."""
    try:
        ImageFont.truetype(str(path), 20)
        return True
    except (OSError, ValueError):
        return False


def setup_fonts(verbose: bool = True) -> str:
    """Unduh FONT_CHAIN berurutan, pakai yang pertama berhasil.

    Lisensi Blambot melarang redistribusi, jadi .ttf hanya diunduh saat runtime
    dan tidak pernah dibundel. Kalau semua gagal, chain turun ke Comic Neue (OFL)
    tanpa crash.
    """
    global FONT_USED, _SYMBOL_PATH, _FALLBACK_PATH, _CJK_PATH
    global _ARABIC_PATH, _THAI_PATH
    FONTS.mkdir(parents=True, exist_ok=True)

    for entry in FONT_CHAIN:
        dest = FONTS / entry["file"]
        if _download(entry["url"], dest) and _usable(dest):
            FONT_USED = str(dest)
            if entry.get("license_url"):
                _download(entry["license_url"], dest.with_suffix(".OFL.txt"))
            if verbose:
                print(f"[font] pakai {entry['name']} -> {dest.name}")
            break
        if verbose:
            print(f"[font] {entry['name']} gagal, lanjut ke kandidat berikutnya")

    if not FONT_USED:
        FONT_USED = _system_fallback()
        # note() bukan print(): ini satu-satunya cabang di sini yang mengubah
        # HASIL, bukan cuma jalannya. Font sistem bukan Anime Ace, jadi seluruh
        # halaman keluar dengan huruf yang salah — dan itu harus terlihat di UI
        # walau verbose=False (app.py memanggil setup_fonts(verbose=False)).
        note("warn", "font",
             f"semua kandidat gagal diunduh, pakai font sistem: {FONT_USED} — "
             "hurufnya BUKAN Anime Ace")

    sym = FONTS / FONT_SYMBOL["file"]
    if _download(FONT_SYMBOL["url"], sym) and _usable(sym):
        _SYMBOL_PATH = sym
        _download(FONT_SYMBOL["license_url"], sym.with_suffix(".OFL.txt"))

    shout = FONTS / FONT_SHOUT["file"]
    if _download(FONT_SHOUT["url"], shout):
        _download(FONT_SHOUT["license_url"], shout.with_suffix(".OFL.txt"))

    # Fallback multi-script: Latin-ext/Cyrillic/Greek, CJK, Arab, Thai.
    fb = FONTS / FONT_FALLBACK["file"]
    if _download(FONT_FALLBACK["url"], fb) and _usable(fb):
        _FALLBACK_PATH = fb
        _download(FONT_FALLBACK["license_url"], fb.with_suffix(".OFL.txt"))
    for key, attr in ((FONT_CJK, "_CJK_PATH"), (FONT_ARABIC, "_ARABIC_PATH"),
                      (FONT_THAI, "_THAI_PATH")):
        p = FONTS / key["file"]
        if _download(key["url"], p) and _usable(p):
            globals()[attr] = p
            _download(key["license_url"], p.with_suffix(".OFL.txt"))

    return FONT_USED


def _system_fallback() -> str:
    """Font terakhir yang pasti ada di hampir semua sistem."""
    for cand in (
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "C:/Windows/Fonts/arialbd.ttf",
    ):
        if Path(cand).exists():
            return cand
    return str(Path(ImageFont.__file__).parent / "fonts")


def set_user_font(path: str | Path) -> str:
    """Slot upload font sendiri — jalur paling aman secara lisensi."""
    global FONT_USED
    p = Path(path)
    if p.exists() and _usable(p):
        FONT_USED = str(p)
        _font.cache_clear()
    return FONT_USED


@lru_cache(maxsize=64)
def _font(path: str, size: int) -> ImageFont.FreeTypeFont:
    return ImageFont.truetype(path, size)


def _cmap(path: str) -> frozenset[int]:
    """Set codepoint yang dipunya font. Anime Ace cuma ~159 glyph."""
    if path in _CMAP_CACHE:
        return _CMAP_CACHE[path]
    try:
        from fontTools.ttLib import TTFont

        cm = frozenset(TTFont(path, fontNumber=0, lazy=True).getBestCmap().keys())
    except Exception:  # noqa: BLE001 - fontTools opsional
        cm = frozenset()
    _CMAP_CACHE[path] = cm
    return cm


# ---------------------------------------------------------------- layout


def _cond() -> float:
    """Faktor rapat horizontal yang berlaku sekarang; 1.0 = mati.

    Satu pintu untuk SELURUH file. Kerapatan tidak boleh dipasang di dua tempat
    terpisah: kalau JALUR UKUR dan JALUR GAMBAR memakai angka berbeda, baris
    dinilai muat lalu tergambar lebih lebar dan menembus garis balon — persis
    cacat 'keluar bubble' yang dilarang plan.txt. Di sini hanya ada satu
    pemakainya (_line_width, yang dipakai _measure DAN render_region), jadi
    keduanya tidak mungkin pecah.

    Alasan angkanya ada di SETTINGS.condense (config.py): selisih kerapatan
    terukur 0.690 antara Anime Ace dan font typeset CONTOH/6.JPG, dan sapuan
    probe_cond.py yang memilih 0.85.

    Nilai di luar [0.3, 1.0] diabaikan (jatuh ke 1.0), bukan di-clamp: angka
    seperti itu selalu salah tulis, dan meremas huruf 5x lebih baik gagal
    kelihatan daripada diam-diam merender teks yang tidak terbaca.
    """
    c = float(getattr(SETTINGS, "condense", 1.0) or 1.0)
    return c if 0.3 <= c <= 1.0 else 1.0


def _measure(text: str, font: ImageFont.FreeTypeFont) -> float:
    """Lebar advance baris, sudah memperhitungkan glyph fallback.

    getlength() = advance width (float). Jangan campur dengan getbbox().

    Sudah termasuk faktor rapat horizontal (_cond) karena _line_width memakainya
    — jadi seluruh mesin tata letak mengukur lebar yang BENAR-BENAR tergambar.

    Dulu fungsi ini memanggil font.getlength() mentah, jadi JALUR UKUR (fit,
    penataan baris) dan JALUR GAMBAR (_draw_line, yang jatuh ke font lain per
    karakter) bisa memakai font berbeda untuk karakter yang sama — dan lebarnya
    memang berbeda. Terukur pada size 20: ☆ 14.0 px di anime_ace tapi 21.0 px di
    NotoSansSymbols2, ♥ 18.0 vs 15.0. Ukur-kurang seperti ☆ itu yang berbahaya:
    fit() menyangka baris muat, lalu penggambaran melebarkannya 7 px keluar
    balon — cacat "keluar bubble" yang justru dilarang plan.txt.
    """
    cmap = _cmap(getattr(font, "path", "") or "")
    return _line_width(text, font, cmap, int(getattr(font, "size", 0) or 0))


def _line_height(font: ImageFont.FreeTypeFont) -> int:
    asc, desc = font.getmetrics()
    return int((asc + desc) * SETTINGS.line_spacing)


@lru_cache(maxsize=128)
def _ink_band(path: str, size: int) -> tuple[int, int]:
    """Offset atas & bawah tinta di dalam kotak baris, relatif ke anchor 'la'.

    Kotak baris `(asc + desc) * line_spacing` jauh lebih tinggi daripada tinta
    yang benar-benar tergambar: teks ALL CAPS tidak memakai ruang descender sama
    sekali. Kalau probe balon memakai kotak penuh, baris dianggap menembus garis
    balon padahal yang menembus cuma ruang kosong — dan fit() lalu mengecilkan
    font tanpa alasan (balon 'DID THE CONTRACTOR BAIL?' turun dari 21 ke 16 dan
    kata terpanjangnya pecah tiga kali). Yang harus muat adalah tintanya.
    """
    f = _font(path, size)
    _, y0, _, y1 = f.getbbox("AHJQ,;()")
    return int(y0), int(y1)


def _row_free(mask: np.ndarray, y0: int, y1: int, x0: float, x1: float) -> bool:
    """Probe: apakah rentang ini masih di dalam interior bubble?"""
    h, w = mask.shape[:2]
    y0, y1 = max(0, int(y0)), min(h, int(y1))
    x0i, x1i = max(0, int(x0)), min(w, int(x1))
    if y1 <= y0 or x1i <= x0i:
        return False
    # Diuji lewat CAKUPAN, bukan rata-rata. Rata-rata tidak bisa membedakan
    # "seluruhnya di dalam balon" dari "sebagian keluar tapi sisanya putih
    # pekat": band setinggi satu baris yang 14% ujungnya sudah di luar oval
    # tetap bernilai 239 dan lolos ambang 220 — itulah yang membuat baris
    # pertama dan terakhir menembus garis balon walau tiap baris lolos probe.
    band = mask[y0:y1, x0i:x1i]
    return float((band >= 200).mean()) >= _ROW_COVER


def _centroid(mask: np.ndarray) -> tuple[int, int]:
    m = cv2.moments((mask > 0).astype(np.uint8))
    if m["m00"] == 0:
        h, w = mask.shape[:2]
        return w // 2, h // 2
    return int(m["m10"] / m["m00"]), int(m["m01"] / m["m00"])


def free_run(flags: np.ndarray, a: int, b: int) -> tuple[int, int] | None:
    """Rentang baris bebas yang MENYAMBUNG dan paling menaungi [a, b].

    Harus menyambung, bukan sekadar "baris bebas paling atas dan paling bawah".
    Interior balon yang sudah dipartisi dari tetangganya kerap terpecah: pita
    bebas di atas dinding partisi tidak bisa dicapai teks di bawahnya. Memakai
    flatnonzero(...)[0] apa adanya melaporkan ruang yang sebenarnya terhalang —
    r6 halaman referensi dilaporkan punya 94 px ruang di atas padahal pitanya
    ada di seberang dinding, dan penyeimbang mana pun lalu mengejar angka semu.
    """
    idx = np.flatnonzero(flags)
    if idx.size == 0:
        return None
    cuts = np.flatnonzero(np.diff(idx) > 1)
    starts = np.concatenate(([0], cuts + 1))
    ends = np.concatenate((cuts, [idx.size - 1]))
    best: tuple[int, tuple[int, int]] | None = None
    for s, e in zip(starts, ends):
        lo, hi = int(idx[s]), int(idx[e])
        ov = min(hi, b) - max(lo, a)  # >0 tumpang tindih tinta, <0 jaraknya
        if best is None or ov > best[0]:
            best = (ov, (lo, hi))
    return None if best is None else best[1]


def _free_flags(mask: np.ndarray, cx: int, width: float) -> np.ndarray:
    """Baris mana yang bebas seluruhnya pada pita selebar `width` di sekitar cx."""
    x1 = int(max(cx - width / 2, 0))
    x2 = int(min(cx + width / 2, mask.shape[1]))
    col = (mask[:, x1:x2] > 0) if x2 > x1 else np.zeros((mask.shape[0], 1), bool)
    flags = col.all(1)
    return flags if flags.sum() >= 2 else col.any(1)


def _cover_flags(mask: np.ndarray, cx: int, width: float, cover: float) -> np.ndarray:
    """Baris mana yang masih DI DALAM rongga pada pita selebar `width` di cx.

    Bedanya dengan _free_flags: di sini satu baris cukup bercakupan `cover`,
    tidak harus bebas seluruh pita. Dipakai block_slack() untuk mengukur LETAK
    batas atas/bawah rongga — bukan untuk memutuskan satu baris muat (itu
    _row_free, dan ambangnya memang ketat). Lihat block_slack() untuk angka
    perbandingan lima ambang.
    """
    x1 = int(max(cx - width / 2, 0))
    x2 = int(min(cx + width / 2, mask.shape[1]))
    if x2 <= x1:
        return np.zeros(mask.shape[0], bool)
    return (mask[:, x1:x2] > 0).mean(1) >= cover


def _band_run(mask: np.ndarray, y0: int, y1: int) -> tuple[int, int] | None:
    """Rentang kolom TERLEBAR yang bebas di SELURUH band y0..y1, atau None.

    Bedanya dengan ekspansi simetris di max_width_at(): rongga tidak dipaksa
    simetris terhadap satu x. Pada interior yang sudah dipotong tetangganya
    rongga memang tidak simetris, dan memaksanya simetris membuang separuh ruang
    yang ada. r6 halaman ini di ketinggian tengah bebas di x=14..67 (54 px),
    tapi ekspansi simetris di centroid x=36 cuma mengakui 2*(36-14)=44 px;
    dikurangi pad*2 jadi 40 px, sementara 'SORRY.' butuh 49 px. Akibatnya
    SETIAP y terpusat ditolak dan satu-satunya y yang lolos ada di 119..140 —
    39 px di bawah tengah, dan itu yang terlihat sebagai teks melorot
    (probe_fitwin.py mencetak ketiga aturan lebar berdampingan).

    Ambangnya sama dengan _row_free (>=200), hanya lebih ketat: per kolom
    dituntut bebas SELURUH tinggi band, bukan 98.5% dari kotak. Jadi lebar yang
    dilaporkan di sini tidak pernah melebihi yang diizinkan _row_free.
    """
    mh = mask.shape[0]
    a, b = max(int(y0), 0), min(int(y1), mh)
    if b <= a:
        return None
    idx = np.flatnonzero((mask[a:b] >= 200).all(0))
    if idx.size == 0:
        return None
    cuts = np.flatnonzero(np.diff(idx) > 1)
    starts = np.concatenate(([0], cuts + 1))
    ends = np.concatenate((cuts, [idx.size - 1]))
    s, e = max(zip(starts, ends), key=lambda p: idx[p[1]] - idx[p[0]])
    return int(idx[s]), int(idx[e])


def block_axis(mask: np.ndarray, lines: list[str], top: int, lh: int,
               ink_top: int, ink_bot: int, font: ImageFont.FreeTypeFont,
               fallback: int) -> int:
    """Satu sumbu x untuk SELURUH blok: pusat rongga yang sah untuk semua baris.

    Satu sumbu, bukan satu per baris. Memusatkan tiap baris di rongganya sendiri
    memang menurunkan ketimpangan, tapi blok jadi bergerigi sampai 15 px di r7
    halaman ini — rapi menurut angka, kacau menurut mata (probe_axis.py varian
    V1 vs V2). Yang dipakai: IRISAN rongga semua baris, supaya sumbunya sah
    bahkan untuk baris terlebar, lalu titik tengah irisan itu.

    Kalau irisannya lebih sempit dari baris terlebar, sumbu blok tidak ada dan
    fungsi ini jatuh ke `fallback` (centroid) — _verify() yang lalu menolak
    kandidat itu, sama seperti sebelumnya.
    """
    if not lines:
        return fallback
    lo, hi = 0, mask.shape[1] - 1
    for k in range(len(lines)):
        run = _band_run(mask, top + k * lh + ink_top, top + k * lh + ink_bot)
        if run is None:
            return fallback
        lo, hi = max(lo, run[0]), min(hi, run[1])
    if hi - lo + 1 < max(_measure(ln, font) for ln in lines):
        return fallback
    return (lo + hi) // 2


def line_axis(mask: np.ndarray, lines: list[str], start_y: int, size: int,
              font_path: str) -> int:
    """Sumbu x yang dipakai layout() untuk blok ini — dihitung ulang, bukan disimpan.

    render_region() harus menggambar di sumbu yang SAMA dengan yang dipakai
    layout() saat memutuskan blok ini muat. block_axis() murni fungsi dari
    (mask, lines, start_y, ukuran), jadi menghitungnya ulang di sini selalu
    memberi angka yang sama tanpa menambah nilai balik layout() — yang dipakai
    belasan probe.
    """
    font = _font(font_path, size)
    ink_top, ink_bot = _ink_band(font_path, size)
    return block_axis(mask, lines, start_y, _line_height(font),
                      ink_top, ink_bot, font, _centroid(mask)[0])


def block_slack(mask: np.ndarray, cx: int, pad: int, w_first: float, w_last: float,
                ink_a: int, ink_b: int) -> tuple[int, int]:
    """Sisa ruang di ATAS tinta baris pertama dan di BAWAH tinta baris terakhir.

    SATU pita acuan selebar baris TERLEBAR, dan satu run untuk kedua ujung.

    Versi sebelumnya memakai dua lebar berbeda — pita selebar baris PERTAMA
    untuk ujung atas, selebar baris TERAKHIR untuk ujung bawah. Itu terdengar
    benar tapi tidak bisa dipakai untuk menilai keseimbangan: di balon oval
    kedua pita menyempit pada laju yang berbeda, jadi dua angka yang
    dibandingkan `abs(up - dn)` diukur terhadap dua batas yang berbeda. Blok
    yang mata lihat melenceng 14-72 px tetap melaporkan bal=0, lalu MENANG di
    pemindaian n lewat `if bal <= tol: break`, mendapat nol iterasi _polish
    (`abs(dn-up)//2 == 0`) dan juga melewati _rebalance (yang menilai dari
    fungsi ini juga). Itu cacat "teks tidak di tengah antara atas dan bawah".
    Terukur di _fix12.py: 13 region halaman referensi mean 25.4 max 72 px
    ketimpangan nyata, sintetis mean 18.8 max 46.

    Barisnya dihitung "masih interior" lewat CAKUPAN, bukan semua-atau-tidak.
    _free_flags menuntut SELURUH pita bebas dalam satu baris (`col.all(1)`);
    dipakai pada pita selebar baris terlebar itu memotong baris yang sebenarnya
    masih di dalam oval, dan hasilnya memburuk di halaman asli (r5 23->43,
    r7 8->48, r12 59->77 di _center5.py). Cakupan >= _SLACK_COVER melihat apa
    yang dilihat mata: batas atas dan bawah rongga di kolom tempat blok berada.

    Ambangnya BUKAN _ROW_COVER (0.985). Itu ambang "baris ini muat", dituntut
    ketat karena melanggarnya berarti huruf keluar balon; ini pengukuran
    LETAK batas, dan menuntut 98.5% di sini mengembalikan cacatnya. 0.20 dipilih
    karena ekor balon (10-14 px) tidak pernah mencapai 20% lebar blok, jadi ekor
    tidak ikut menggeser batas bawah — sementara 0.10 masih mengakuinya.
    Terukur berdampingan (0.10/0.20/0.30/0.40/0.50) di _fix12.py: halaman
    mean 2.3/2.5/3.1/3.5/4.8, sintetis mean 1.4/1.4/1.7/2.4/3.2, over=0 semua.

    Dinding partisi (interior terbelah tetangga) tetap dihormati: barisnya
    bercakupan 0, jadi `free_run` — yang mengambil run TERSAMBUNG — berhenti di
    dinding dan tidak pernah menaungi lobus tetangga. Inilah yang menjaga
    syarat "jangan sampai ada huruf yang termakan mask lain kalau double bubble
    merge": sisa ruang diukur di dalam lobus milik region ini saja.

    Kalau tidak ada run sama sekali (mis. pita acuan lebih lebar dari rongga
    mana pun), fungsi ini JATUH ke perhitungan dua-lebar yang lama, bukan ke
    `return 0, 0` — 0,0 berarti "seimbang" dan itu justru menutupi kegagalan.
    """
    mh = mask.shape[0]
    w_ref = max(float(w_first), float(w_last), 1.0)
    run = free_run(_cover_flags(mask, cx, w_ref, _SLACK_COVER), ink_a, ink_b)
    if run is None:
        top = free_run(_free_flags(mask, cx, w_first), ink_a, ink_a)
        bot = free_run(_free_flags(mask, cx, w_last), ink_b, ink_b)
        if top is None or bot is None:
            return 0, 0
        return ink_a - max(top[0], pad), min(bot[1], mh - pad) - ink_b
    return ink_a - max(run[0], pad), min(run[1], mh - pad) - ink_b


def _break_points(word: str) -> set[int]:
    """Indeks tempat kata boleh dipenggal, dari kamus pola kalau tersedia."""
    key = word.upper()
    if key in _BREAK_CACHE:
        return _BREAK_CACHE[key]

    dic = _pyphen()
    pts: set[int] = set()
    if dic is not None:
        # pyphen bekerja pada huruf saja; tanda baca di ujung digeser manual
        # supaya indeksnya tetap merujuk ke posisi di kata aslinya.
        head_pad = len(word) - len(word.lstrip("\"'([“‘"))
        core = word[head_pad:].rstrip(".,!?…\"')]”’")
        if core:
            for pos in dic.positions(core.lower()):
                pts.add(head_pad + pos)
    else:
        # Cadangan HANYA saat kamusnya tidak terpasang. Set kosong dari pyphen
        # adalah jawaban yang sah — 'STRANGE' memang tidak punya batas suku kata,
        # dan menimpanya dengan heuristik justru menghasilkan 'STRA-NGE'.
        pts = {
            n for n in range(1, len(word))
            if word[n - 1].isalpha() and word[n].isalpha()
            and word[n - 1].upper() in _VOWELS and word[n].upper() not in _VOWELS
        }

    _BREAK_CACHE[key] = pts
    return pts


def _cjk_break(line: str, font: ImageFont.FreeTypeFont, avail: float) -> tuple[str, str]:
    """Pecah teks CJK per karakter — bahasa Asia tidak pakai spasi antar kata.

    Returns:
        (kepala_yang_muat, sisa) atau ("", line) kalau tidak bisa dipecah.
    """
    n = 0
    w = 0.0
    for ch in line:
        # Dikali _cond() supaya sebanding dengan `avail`, yang datang dari
        # max_width_at() lewat _measure() dan sudah dalam satuan rapat.
        w += font.getlength(ch) * _cond()
        if w <= avail and n < len(line) - 1:
            n += 1
        else:
            break
    if n <= 0 or n >= len(line):
        return "", line
    return line[:n], line[n:]


def _has_cjk(text: str) -> bool:
    return any(_is_cjk(c) for c in text)


def _split_word(word: str, font: ImageFont.FreeTypeFont, avail: float) -> tuple[str, str]:
    """Penggal kata yang lebih lebar dari kolomnya: 'CONTRACTORS' -> 'CONTRAC-', 'TORS'.

    Dipakai sebagai bagian dari pencarian ukuran font, bukan cuma penyelamat saat
    gagal: tanpa penggalan, ukuran font dijepit oleh kata terpanjang dan balon
    lega ikut mengecil mengikutinya.

    Penggalan bebas menghasilkan 'UG-H', 'ENOUG-H', 'WH-AT' — muat, tapi tidak
    layak cetak. Titik penggalannya diambil dari kamus pola pyphen (Knuth-Liang),
    yang tahu 'CON-TRAC-TOR' dan 'WASH-ING'; heuristik vokal->konsonan cuma
    cadangan kalau modulnya tidak terpasang, dan itu memang meleset ('CONTR-ACTOR').
    Dua penjaga tetap berlaku di kedua jalur: kata pendek tidak pernah dipenggal,
    dan tiap pecahan minimal 3 HURUF — dihitung per huruf supaya 'HEARD.' tidak
    lolos lewat ekor 'RD.' yang panjangnya 3 karakter tapi cuma 2 huruf.

    Returns:
        (kepala_dengan_tanda_hubung, sisa) atau ("", word) kalau tetap tidak muat.
    """
    letters = sum(c.isalpha() for c in word)
    if letters < _MIN_HEAD + _MIN_TAIL:
        return "", word

    def accept(n: int) -> tuple[str, str] | None:
        if sum(c.isalpha() for c in word[:n]) < _MIN_HEAD:
            return None
        if sum(c.isalpha() for c in word[n:]) < _MIN_TAIL:
            return None
        head = f"{word[:n]}-"
        # _cond(): `avail` berasal dari _measure(), jadi kepala penggalan harus
        # diukur di satuan yang sama. Tanpa itu penggalan dinilai dengan lebar
        # renggang sementara barisnya digambar rapat — kata dipenggal padahal
        # utuhnya muat, dan tanda hubung itu tepat yang sedang dihilangkan.
        return (head, word[n:]) if font.getlength(head) * _cond() <= avail else None

    # Terpanjang dulu: makin banyak yang muat di baris ini, makin sedikit baris.
    for n in sorted(_break_points(word), reverse=True):
        hit = accept(n)
        if hit:
            return hit
    return "", word


def layout(
    text: str, mask: np.ndarray, size: int, font_path: str,
    allow_overflow: bool = False, hyphenate: bool = False,
    from_top: bool = False,
) -> tuple[bool, list[str], int]:
    """Centroid-outward line growing.

    Bentuk oval muncul sendiri: baris dekat lengkung atas/bawah gagal probe
    lebih awal jadi lebih pendek. Lebih bagus daripada inscribed-rectangle.

    Returns:
        (fits, lines, start_y)
    """
    font = _font(font_path, size)
    words = text.split()
    if not words:
        return True, [], 0

    lh = _line_height(font)
    ink_top, ink_bot = _ink_band(font_path, size)
    cx, cy = _centroid(mask)
    mh, mw = mask.shape[:2]
    pad = int(min(mh, mw) * SETTINGS.pad_ratio)

    def max_width_at(y_top: int) -> float:
        """Lebar tersedia pada baris ini = rongga bebas TERLEBAR di band-nya.

        Bukan ekspansi simetris di sekitar satu x. Lihat _band_run() untuk
        alasannya: rongga interior yang sudah dipotong tetangganya tidak
        simetris terhadap centroid, dan memaksanya simetris membuang separuh
        ruang yang ada — itulah yang membuat 'SORRY.' (r6) tidak pernah muat di
        ketinggian tengah dan terpaksa melorot 39 px ke bawah.
        """
        run = _band_run(mask, y_top + ink_top, y_top + ink_bot)
        if run is None:
            return 0.0
        return max((run[1] - run[0] + 1) - pad * 2, 0.0)

    def axis_of(lines: list[str], top: int) -> int:
        """Sumbu x blok ini; centroid kalau tidak ada irisan rongga yang sah."""
        return block_axis(mask, lines, top, lh, ink_top, ink_bot, font, cx)

    def _center_y(n_lines: int) -> int:
        """y anchor baris pertama supaya TINTA blok terpusat di centroid."""
        ink_h = (n_lines - 1) * lh + (ink_bot - ink_top)
        return cy - ink_h // 2 - ink_top

    def _tops(n_lines: int) -> list[int]:
        """Kandidat y baris pertama: centroid dulu, lalu digeser ke atas-bawah.

        Centroid saja tidak cukup. Interior balon yang sudah dipotong tetangganya
        sering paling lebar di pita yang TIDAK melewati centroid: pada balon
        r12 halaman referensi baris terlebar ada di y=118..150 (93 px bebas)
        sementara centroid memaksa baris pertama ke y=102 yang cuma 44 px —
        dan 'WHAT?' butuh 45 px. Tanpa pencarian ini region itu gagal di SEMUA
        ukuran font, lalu fit() jatuh ke jalur di bawah min_font_size dan
        merendernya 6 px.

        Centroid tetap dicoba pertama supaya balon normal tetap terpusat seperti
        sebelumnya; geseran hanya dipakai kalau yang terpusat memang gagal.
        """
        ink_h = (n_lines - 1) * lh + (ink_bot - ink_top)
        lo, hi = pad - ink_top, mh - pad - ink_h - ink_top
        if hi < lo:
            return [_center_y(n_lines)]
        cands = [int(np.clip(_center_y(n_lines), lo, hi))]
        # Langkah setengah baris: cukup halus untuk menemukan pita lebar, cukup
        # kasar supaya jumlah percobaan tetap belasan, bukan ratusan.
        step = max(lh // 2, 2)
        cands += [t for t in range(lo, hi + 1, step) if t != cands[0]]
        return cands

    def build(top: int) -> tuple[list[str], bool]:
        """Tumbuhkan baris ke bawah mulai dari `top`. bool = semua kata termuat."""
        lines: list[str] = []
        queue = list(words)  # bisa tumbuh saat kata dipenggal
        i = 0
        for _ in range(64):  # batas keras, jangan sampai loop selamanya
            if i >= len(queue):
                return lines, True
            avail = max_width_at(top + len(lines) * lh)
            if avail < size * 0.9 and not allow_overflow:
                # Baris pertama harus muat sesuatu, tapi JANGAN diberi seluruh
                # lebar kotak: di balon oval, lebar kotak jauh melewati garis
                # balon pada baris teratas, dan itu membuat baris pertama
                # menembus keluar. Ukuran ini gagal — fit() yang mengecilkan.
                return lines, False

            line = queue[i]
            j = i + 1
            while j < len(queue) and _measure(f"{line} {queue[j]}", font) <= avail:
                line = f"{line} {queue[j]}"
                j += 1

            # Satu kata pun tidak muat.
            if j == i + 1 and _measure(line, font) > avail:
                if _has_cjk(line):
                    head, tail = _cjk_break(line, font, avail)
                else:
                    head, tail = _split_word(line, font, avail) if hyphenate else ("", line)
                if head:
                    queue[i : i + 1] = [head, tail]  # sisanya ke baris berikutnya
                    line, j = head, i + 1
                elif not allow_overflow:
                    return lines, False  # ukuran font ini gagal

            lines.append(line)
            i = j
        return lines, i >= len(queue)

    def _verify(lines: list[str], start_y: int) -> bool:
        """Tiap baris benar-benar di dalam interior, pada lebar baris itu sendiri.

        Menuntut seluruh lebar mask bebas berarti menuntut sudut-sudut persegi
        dari mask bubble oval — yang tidak akan pernah bebas berapa pun ukuran
        fontnya, jadi tiap region selalu dilaporkan overflow walau tiap barisnya
        sudah lolos probe. Yang benar: cek kotak nyata tiap baris di sekitar
        sumbu blok — sumbu yang sama yang dipakai render_region() menggambar
        (lihat line_axis()), bukan centroid, kalau tidak yang diverifikasi bukan
        tempat tintanya benar-benar jatuh.
        """
        ax = axis_of(lines, start_y)
        for k, line in enumerate(lines):
            lw = _measure(line, font)
            y_top = start_y + k * lh
            if not _row_free(
                mask, y_top + ink_top, y_top + ink_bot, ax - lw / 2, ax + lw / 2
            ):
                return False
        # Blok teks diukur dari tinta baris pertama sampai tinta baris terakhir,
        # bukan dari tepi kotak — alasannya sama seperti di _ink_band().
        return not (start_y + ink_top < pad
                    or start_y + (len(lines) - 1) * lh + ink_bot > mh - pad)

    def _slack(lines: list[str], top: int) -> tuple[int, int]:
        """Sisa ruang atas/bawah blok ini — lihat block_slack() untuk alasannya.

        Diukur di sumbu blok, bukan di centroid: pada interior yang terpotong
        keduanya bisa berjarak belasan px, dan mengukur sisa ruang di kolom yang
        TIDAK dilewati tinta melaporkan ruang semu.
        """
        ax = axis_of(lines, top)
        return block_slack(
            mask, ax, pad,
            _measure(lines[0], font) if lines else 1.0,
            _measure(lines[-1], font) if lines else 1.0,
            top + ink_top,
            top + (len(lines) - 1) * lh + ink_bot,
        )

    def _polish(lines: list[str], top: int) -> tuple[list[str], int]:
        """Geser blok 1 px demi 1 px ke arah yang menyeimbangkan sisa atas/bawah.

        Sapuan _tops() berlangkah setengah baris, jadi kandidat paling seimbang
        pun masih bisa timpang setengah langkah — belasan px di halaman ini,
        cukup untuk terlihat menempel ke satu sisi.

        Pemecahan barisnya DIPERTAHANKAN, tidak dibangun ulang. Versi pertama
        memanggil build(top) di tiap langkah dan berhenti di langkah pertama
        pada keempat region yang tersisa: menggeser blok ke tengah oval
        melebarkan baris yang tersedia, build lalu memuat lebih banyak kata dan
        jumlah barisnya turun, jadi penjaga "jumlah baris harus sama" langsung
        menolaknya. Pemecahan yang sudah dipilih memang sah — yang perlu
        dipastikan cuma masih muat di y baru, dan itu tepat yang diuji _verify
        (tiap baris pada lebarnya sendiri, plus batas pad).
        """
        up, dn = _slack(lines, top)
        best_top, best_bal = top, abs(up - dn)
        step = 1 if dn > up else -1
        for _ in range(abs(dn - up) // 2):
            t = best_top + step
            if not _verify(lines, t):
                break
            u2, d2 = _slack(lines, t)
            if abs(u2 - d2) >= best_bal:
                break
            best_top, best_bal = t, abs(u2 - d2)
        return lines, best_top

    # Jumlah baris menentukan start_y, tapi start_y juga menentukan lebar tiap
    # baris — jadi tebakan awal harus dikoreksi, bukan dipakai apa adanya. Tanpa
    # koreksi ini blok mulai terlalu tinggi, baris-baris bawahnya jatuh di luar
    # balon, dan ukuran yang sebenarnya muat ditolak: 'DID THE CONTRACTOR BAIL?'
    # tertahan di font 16 padahal 21 muat.
    n0 = max(1, int(np.ceil(_measure(text, font) / max(mw - pad * 2, 1))))
    lines, done, start_y = [], False, _center_y(n0)
    # Ambang "sudah cukup terpusat" untuk berhenti menyapu. Sisanya diserahkan
    # ke _polish, yang bisa turun ke ketimpangan ~0 — jadi ambang sekasar
    # setengah baris tidak mengorbankan kerapian, hanya menghemat percobaan.
    tol = max(2, lh // 2)
    # Jumlah baris DIPINDAI, tidak dikoreksi dari satu percobaan. Versi
    # sebelumnya memakai `nxt = len(lines) + 1` dari build di y terpusat, dan
    # koreksi itu MELOMPATI jumlah baris yang benar: di r12 halaman ini n0=3,
    # build di y terpusat memuat 4 baris tapi gagal (ok=False), jadi n naik
    # langsung ke 5 dan n=4 tidak pernah dicoba sama sekali. Hasilnya 5 baris
    # yang mentok di ketimpangan 41 px, padahal 4 baris muat dengan
    # ketimpangan 1 px pada margin yang sama persis. Memindai n0..n0+4 dan
    # memilih yang paling seimbang menghapus lompatan itu tanpa melonggarkan
    # batas apa pun — margin build tetap pad*2 seperti sebelumnya.
    hit: tuple[list[str], int, int] | None = None
    for n in range(n0, n0 + 5):
        tops = _tops(n)
        # Kandidat yang muat DIPILIH yang paling seimbang, bukan yang pertama.
        # Versi sebelumnya `break` pada fit pertama, dan karena sapuan berjalan
        # dari tepi atas balon ke bawah, yang pertama muat sering jauh dari
        # tengah: pada halaman referensi start_y yang diterima duduk di
        # peringkat kandidat sampai ke-21, dan 10 dari 13 balon meleset dari
        # tengah (terburuk 40 px). Sapuannya sendiri tetap wajib ada — lihat
        # _tops() — jadi yang diubah cuma kriteria pemilihannya.
        for top in tops:
            cand, ok_all = build(top)
            if top == tops[0]:
                lines, done, start_y = cand, ok_all, top
            if not (ok_all and len(cand) == n and _verify(cand, top)):
                continue
            up, dn = _slack(cand, top)
            bal = abs(up - dn)
            if hit is None or bal < hit[2]:
                hit = (cand, top, bal)
            if bal <= tol:
                break
        if hit is not None and hit[2] <= tol:
            break  # sudah terpusat; jumlah baris yang lebih besar tak perlu
    if hit is not None:
        lines, start_y = _polish(hit[0], hit[1])
        done = True

    if from_top:
        # Teks terlalu panjang untuk ukuran berapa pun: susun dari ATAS balon
        # ke bawah supaya AWAL kalimat yang terlihat (bukan potongan tengah
        # yang tidak terbaca); kelebihan dipotong di tepi bawah oleh klip.
        lines, done = build(pad)
        return (done or allow_overflow), lines, pad

    if not done and not allow_overflow:
        return False, [], 0

    return (_verify(lines, start_y) or allow_overflow), lines, start_y



def _search(
    text: str, mask: np.ndarray, lo: int, hi: int, font_path: str, hyphenate: bool
) -> tuple[int, list[str], int] | None:
    """Ukuran terbesar yang masih muat pada [lo, hi], atau None kalau tidak ada."""
    ok, lines, y = layout(text, mask, hi, font_path, hyphenate=hyphenate)
    if ok:
        return hi, lines, y  # jalur cepat: ukuran asli sudah muat

    best: tuple[int, list[str], int] | None = None
    a, b = lo, hi
    while b - a > 1:
        mid = (a + b) // 2
        ok, lines, y = layout(text, mask, mid, font_path, hyphenate=hyphenate)
        if ok:
            best, a = (mid, lines, y), mid
        else:
            b = mid

    # Loop di atas hanya menguji titik tengah, jadi lo — batas bawahnya sendiri —
    # tidak pernah dicoba. Kalau semua titik tengah gagal, best masih None padahal
    # ukuran minimum belum tentu gagal; tanpa cek ini region dilaporkan overflow
    # tanpa pernah diuji pada ukuran yang justru paling mungkin muat.
    if best is None:
        ok, lines, y = layout(text, mask, lo, font_path, hyphenate=hyphenate)
        if ok:
            best = (lo, lines, y)
    return best


def _block_bal(lines: list[str], y: int, mask: np.ndarray, size: int,
               font_path: str) -> int:
    """Ketimpangan sisa ruang atas vs bawah blok, dalam px. 0 = terpusat."""
    if not lines:
        return 0
    font = _font(font_path, size)
    lh = _line_height(font)
    ink_top, ink_bot = _ink_band(font_path, size)
    mh, mw = mask.shape[:2]
    pad = int(min(mh, mw) * SETTINGS.pad_ratio)
    # Sumbu blok, bukan centroid — sama seperti _slack() di dalam layout().
    ax = line_axis(mask, lines, y, size, font_path)
    up, dn = block_slack(
        mask, ax, pad, _measure(lines[0], font), _measure(lines[-1], font),
        y + ink_top, y + (len(lines) - 1) * lh + ink_bot,
    )
    return abs(up - dn)


def _bal_tol(size: int, font_path: str) -> int:
    """Ambang 'sudah terpusat' — setengah tinggi baris, sama seperti layout()."""
    return max(2, _line_height(_font(font_path, size)) // 2)


def _rebalance(
    text: str, mask: np.ndarray, font_path: str,
    cand: tuple[int, list[str], int], hyphenate: bool,
) -> tuple[int, list[str], int]:
    """Turun ukuran sedikit kalau ukuran terpilih mustahil ditata seimbang.

    Ukuran terbesar yang MUAT tidak selalu ukuran yang bisa dirapikan. Pada r12
    halaman referensi 'IS IT? LEMME SEE, C'MON~!' di ukuran 15 mentok pada
    ketimpangan 43 px — bukan karena pencariannya kurang teliti, tapi karena
    tidak ada satu pun kombinasi pemecahan baris dan y legal yang lebih baik
    (probe_r12_exhaust.py memindai semuanya). Di ukuran 14 blok yang sama turun
    ke 1 px. Tanpa langkah ini teks menempel ke satu sisi balon sejauh 21 px
    dari tengah, dan itu terlihat langsung.

    Yang TIDAK dilakukan di sini: melonggarkan margin build. Varian itu diukur
    (probe_margin.py) dan memang menyembuhkan r12, tapi median jarak tinta ke
    garis balon seluruh halaman jatuh dari 3 px ke 0 px — menukar satu balon
    timpang dengan tinta menempel garis di enam balon lain.

    Turunnya dibatasi _BAL_MAX_DROP dan region yang sudah seimbang di ukuran
    terpilih tidak pernah masuk loop, jadi ukuran font halaman tidak ikut
    mengecil: pada halaman referensi hanya r12 yang turun (15 -> 14).
    """
    size, lines, y = cand
    if _block_bal(lines, y, mask, size, font_path) <= _bal_tol(size, font_path):
        return cand
    floor = max(size - _BAL_MAX_DROP, min_font())
    for s in range(size - 1, floor - 1, -1):
        ok, ls, yy = layout(text, mask, s, font_path, hyphenate=hyphenate)
        if not ok or not ls:
            continue
        if _block_bal(ls, yy, mask, s, font_path) <= _bal_tol(s, font_path):
            return s, ls, yy
    return cand


def set_page_width(w: int) -> None:
    """Catat lebar halaman yang sedang dikerjakan (untuk min_font())."""
    global _PAGE_W
    _PAGE_W = int(w) if w and w > 0 else 0


def min_font(page_w: int | None = None) -> int:
    """Lantai ukuran font untuk halaman selebar `page_w`, BUKAN angka mutlak.

    Kenapa berskala: SETTINGS.min_font_size dikalibrasi pada satu resolusi
    (CONTOH/2.webp, 1134 px). Halaman lain datang di resolusi lain, dan lantai
    yang tidak ikut menyusut berubah jadi PLAFON di halaman kecil — bukan lantai.
    Terukur pada hasilnew/jp_6.JPG (698 px, probe_floor6.py): lantai 11 px
    membuat region_font_cap() semua balon mentok di 11, anggaran yang dikirim ke
    model jadi 2-39 karakter, dan _max_feasible() atas wording typeset referensi
    mengembalikan 0 di 7 dari 8 balon — artinya pipeline meminta model menulis
    lebih pendek daripada yang sebenarnya muat, lalu tetap menolak wording yang
    muat. Itulah sebab 'translate-nya sedikit banget' dibanding hasilnew/6.JPG.

    Huruf referensi di halaman itu diukur 4-7 px tinggi (probe_refsize.py, modus
    4, median 5 pada 728 px) = ukuran font 5-8. Jadi typesetter manusia memang
    turun di bawah 11 px pada resolusi ini; lantai 11 bukan batas keterbacaan,
    melainkan batas keterbacaan DI 1134 px.

    Skalanya linear terhadap lebar halaman karena keterbacaan bergantung pada
    ukuran RELATIF terhadap halaman (dan balonnya, yang juga ikut mengecil),
    bukan pada jumlah piksel. Dibatasi dua arah: tidak pernah melebihi
    min_font_size (halaman besar tetap memakai angka terkalibrasi, tidak
    diperbesar diam-diam) dan tidak pernah di bawah min_font_abs.
    """
    w = _PAGE_W if page_w is None else int(page_w)
    ref = max(int(SETTINGS.min_font_ref_width), 1)
    base = int(SETTINGS.min_font_size)
    if w <= 0 or w >= ref:
        return base
    scaled = int(round(base * w / ref))
    return int(np.clip(scaled, int(SETTINGS.min_font_abs), base))


def emergency_floor() -> int:
    """Lantai jalur darurat fit(), ikut berskala bersama min_font().

    _MIN_FONT_FLOOR (9) berjarak 2 px di bawah min_font_size (11) yang menjadi
    kalibrasinya. Jarak itulah yang dipertahankan, bukan angka 9-nya: kalau
    lantai normal turun ke 7 px di halaman 698 px sementara lantai darurat tetap
    9, jalur darurat berada DI ATAS lantai normal, penjaga `lo - 1 >= floor`
    tidak pernah benar, dan balon yang cuma butuh 1 px lagi langsung dipotong di
    tepi bawah alih-alih dikecilkan sedikit.
    """
    gap = max(int(SETTINGS.min_font_size) - _MIN_FONT_FLOOR, 0)
    return int(max(min_font() - gap, int(SETTINGS.min_font_abs)))


def renders_ok(text: str, mask: np.ndarray, font_path: str) -> tuple[bool, int]:
    """(muat?, ukuran) untuk teks INI di balon INI, diukur seperti render nyata.

    Bedanya dengan _max_feasible(): fungsi ini memakai fit() apa adanya, jadi
    penggalan kata dan lantai darurat IKUT dihitung. _max_feasible() melarang
    penggalan karena tugasnya lain — ia mengukur plafon 'muat utuh' untuk
    kalibrasi ukuran, bukan menjawab 'apakah kalimat ini bisa dicetak'.

    Memakai _max_feasible() sebagai kriteria lulus terukur salah arah. Pada
    hasilnew/jp_6.JPG (698 px) wording typeset referensi untuk r3
    ("CAN'T HELP IT ♥", interior 46x87) mengembalikan feasible 0 — tidak muat
    utuh pada ukuran mana pun di atas lantai — padahal fit() merendernya bersih
    di 6 px dengan tiga baris, dan 6 px persis yang dipakai typesetter manusia di
    resolusi ini (probe_r34.py, probe_refsize.py). Jadi validator lama menolak
    justru wording yang sedang ditiru, lalu menyuruh model menulis lebih pendek —
    itu sebab langsung 'translate-nya sedikit banget'.

    Yang tersisa sebagai cacat sungguhan cuma satu: fit() melaporkan luber, yang
    berarti barisnya benar-benar dipotong di tepi bawah balon.
    """
    size, _lines, _y, over = fit(text, mask, region_font_cap(mask), font_path)
    return (not over), int(size)


def fit(text: str, mask: np.ndarray, est_font: float, font_path: str) -> tuple[int, list[str], int, bool]:
    """Binary search pada [MIN_FONT, est_font]. Jangan pernah di atas est.

    Fit-search yang memaksimalkan ukuran akan overshoot dan hasilnya lebih
    besar dari teks aslinya.
    """
    hi = int(np.clip(round(est_font), min_font(), SETTINGS.max_font_size))
    lo = min_font()

    plain = _search(text, mask, lo, hi, font_path, hyphenate=False)
    if plain is not None and plain[0] >= hi:
        # Jalur cepat tetap lewat _rebalance: plafon yang muat belum tentu
        # plafon yang bisa ditata terpusat — lihat docstring _rebalance().
        s, ls, y = _rebalance(text, mask, font_path, plain, hyphenate=False)
        return s, ls, y, False

    # Penggalan kata adalah BAGIAN dari pencarian ukuran, bukan jalan terakhir.
    # Tanpa baris ini ukuran font dijepit oleh kata terpanjang: 'CONTRACTORS'
    # cuma muat utuh di font 12 pada balon selebar 118 px, jadi balon setinggi
    # 218 px diisi tiga baris kecil dan sisanya kosong — jauh dari gambar
    # referensi yang tiap balonnya terisi penuh. Dengan penggalan, kata panjang
    # pecah jadi dua baris dan font bisa naik ke belasan-duapuluhan.
    hyph = _search(text, mask, lo, hi, font_path, hyphenate=True)

    cands = [c for c in (plain, hyph) if c is not None]
    if not cands:
        # Degradasi bertingkat untuk balon yang teksnya tidak muat di
        # min_font_size pun. Urutannya sengaja: teks UTUH yang sedikit lebih
        # kecil jauh lebih baik dibaca daripada teks terpotong.
        #
        # Batas bawahnya emergency_floor() — lantai bernama dengan alasan
        # tertulis, bukan `min_font_size // 2` yang diam-diam mengizinkan 5-6 px,
        # dan ikut berskala resolusi bersama lo supaya di halaman kecil ia tetap
        # BERADA DI BAWAH lo. Dijaga `lo - 1 >= floor` supaya kalau lantai normal
        # sudah menyentuh lantai darurat, _search tidak dipanggil dengan hi < lo
        # (jalur cepatnya akan mengembalikan ukuran di bawah lantai tanpa pernah
        # mengujinya).
        #
        # UTUH DIUJI DULU, dan itu bukan kosmetik. Sebelumnya jalur ini hanya
        # memanggil _search(hyphenate=True), jadi di seluruh jalur darurat tidak
        # pernah ada kandidat utuh yang bisa menang — tanda hubung menang tanpa
        # lawan, dan _HYPHEN_MIN_GAIN di atas tidak berlaku sama sekali di sini.
        # Itulah sebab langsung 'WON-/DER' di r3 halaman jp_6: teksnya cuma
        # 'NO WONDER ♥!', kolomnya sempit, jadi ia jatuh ke jalur ini dan langsung
        # dipenggal. Ambangnya sama dengan jalur normal supaya aturannya satu:
        # tanda hubung hanya kalau ia membeli >= 33% ukuran font.
        floor = emergency_floor()
        if lo - 1 >= floor:
            lp = _search(text, mask, floor, lo - 1, font_path, hyphenate=False)
            lh_ = _search(text, mask, floor, lo - 1, font_path, hyphenate=True)
            if lp is not None and (lh_ is None or lp[0] >= lh_[0] * _HYPHEN_MIN_GAIN):
                low = lp
            else:
                low = lh_
        else:
            low = None
        if low is not None:
            return low[0], low[1], low[2], False
        # Terlalu panjang untuk ukuran apa pun: susun dari atas balon
        # (awal kalimat terlihat), sisanya dipotong di tepi bawah.
        _, lines, y = layout(
            text, mask, lo, font_path, allow_overflow=True, hyphenate=True,
            from_top=True,
        )
        return lo, lines, y, True

    # Utuh menang kecuali harganya benar-benar mahal — lihat _HYPHEN_MIN_GAIN.
    if plain is None:
        best = hyph
        hy = True
    elif hyph is None or plain[0] >= hyph[0] * _HYPHEN_MIN_GAIN:
        best = plain
        hy = False
    else:
        best = hyph
        hy = True
    s, ls, y = _rebalance(text, mask, font_path, best, hyphenate=hy)
    return s, ls, y, False


# ---------------------------------------------------------------- render


def _is_cjk(ch: str) -> bool:
    """Kana, kanji, Hangul, dan ideograf CJK."""
    o = ord(ch)
    return (
        0x3040 <= o <= 0x30FF  # kana
        or 0x3400 <= o <= 0x4DBF  # CJK ext-A
        or 0x4E00 <= o <= 0x9FFF  # CJK unified
        or 0xF900 <= o <= 0xFAFF  # kompatibilitas
        or 0xAC00 <= o <= 0xD7AF  # Hangul
    )


def _is_arabic(ch: str) -> bool:
    o = ord(ch)
    return 0x0600 <= o <= 0x06FF or 0x0750 <= o <= 0x077F or 0xFB50 <= o <= 0xFDFF


def _is_thai(ch: str) -> bool:
    return 0x0E00 <= ord(ch) <= 0x0E7F


def _char_font(ch: str, main: ImageFont.FreeTypeFont, cmap: frozenset[int],
               size: int) -> ImageFont.FreeTypeFont:
    """Font per karakter: utama -> CJK -> Arab -> Thai -> NotoSans -> simbol.

    Anime Ace cuma ~159 glyph; terjemahan non-Inggris (aksen, Cyrillic, CJK,
    Arab, Thai, ...) butuh fallback ini supaya tidak jadi kotak tofu.

    Tiap kandidat DIPERIKSA punya glyph-nya, tidak cuma dicocokkan lewat rentang
    aksara. Versi sebelumnya memilih font dari rentang saja lalu jatuh ke
    NotoSans sebagai penampung terakhir, dan justru simbol yang wajib bertahan
    menurut plan.txt yang jadi korban: 〜 (U+301C) ada di blok CJK Symbols and
    Punctuation yang tidak dicakup _is_cjk(), jadi ia — bersama ♡ ♪ ♫ ☆ ★ —
    dirutekan ke NotoSans, satu-satunya font di rantai ini yang TIDAK punya satu
    pun dari simbol itu. Hasilnya kotak tofu di dalam balon, padahal
    NotoSansCJKjp punya 〜 ～ ♥ ♡ ♪ ☆ ★ dan NotoSansSymbols2 punya ♥ ♡ ☆ ★ ❤.
    Dengan pemeriksaan cmap, rantainya berhenti di font pertama yang benar-benar
    bisa menggambar karakter itu.

    Satu pengecualian, lihat _FORCE_SYMBOL: untuk simbol emosi, "punya glyph"
    tidak sama dengan "punya glyph yang BENAR". anime_ace memetakan U+2665 ke
    huruf Cyrillic `yat`, jadi cek cmap saja meloloskan bentuk yang salah.
    Simbol-simbol itu selalu diambil dari font simbol/CJK, bukan font utama.
    """
    o = ord(ch)
    if o in _FORCE_SYMBOL:
        for path in (_SYMBOL_PATH, _CJK_PATH, _FALLBACK_PATH):
            if path is not None and o in _cmap(str(path)):
                return _font(str(path), size)
        # Tidak ada font pengganti yang punya simbolnya: font utama tetap lebih
        # baik daripada kotak tofu, walau bentuknya tidak ideal.
        return main
    if ch.isspace() or (cmap and o in cmap):
        return main
    # Urutan preferensi tetap: aksara yang cocok dulu, lalu penampung umum.
    chain = []
    if _is_cjk(ch):
        chain.append(_CJK_PATH)
    if _is_arabic(ch):
        chain.append(_ARABIC_PATH)
    if _is_thai(ch):
        chain.append(_THAI_PATH)
    chain += [_FALLBACK_PATH, _CJK_PATH, _SYMBOL_PATH]
    fallback = None
    for path in chain:
        if path is None:
            continue
        p = str(path)
        cm = _cmap(p)
        if o in cm:
            return _font(p, size)
        # cmap kosong = fontTools tidak terpasang; font itu tetap dipakai sebagai
        # cadangan terakhir supaya perilakunya tidak lebih buruk dari sebelumnya.
        if not cm and fallback is None:
            fallback = p
    if fallback is not None:
        return _font(fallback, size)
    return main


def _needs_fallback(line: str, cmap: frozenset[int]) -> bool:
    """Baris ini butuh digambar per-karakter?

    Dua sebab, dan sebab kedua yang dulu terlewat: (1) ada karakter yang font
    utama TIDAK punya, dan (2) ada simbol emosi yang font utama punya tapi
    dengan bentuk yang salah (_FORCE_SYMBOL). Tanpa syarat kedua, baris seperti
    'I LOVE YOU ♥' lolos sebagai "semua ada" lalu digambar satu kali dengan font
    utama, jadi _char_font() tidak pernah dipanggil dan pembetulan simbolnya
    tidak berpengaruh sama sekali.
    """
    return any(
        ord(c) in _FORCE_SYMBOL or (cmap and ord(c) not in cmap)
        for c in line if not c.isspace()
    )


def _draw_line(
    draw: ImageDraw.ImageDraw, xy: tuple[float, float], line: str,
    font: ImageFont.FreeTypeFont, fill: tuple[int, int, int],
    cmap: frozenset[int], size: int, stroke: int = 0,
) -> None:
    """Gambar per-karakter dengan fallback multi-script (CJK/aksen/simbol)."""
    if not _needs_fallback(line, cmap):
        draw.text(
            xy, line, font=font, fill=fill, anchor="la",
            stroke_width=stroke, stroke_fill=(255, 255, 255) if stroke else None,
        )
        return

    x, y = xy
    for ch in line:
        f = _char_font(ch, font, cmap, size)
        draw.text(
            (x, y), ch, font=f, fill=fill, anchor="la",
            stroke_width=stroke, stroke_fill=(255, 255, 255) if stroke else None,
        )
        x += f.getlength(ch)


def _line_width(line: str, font: ImageFont.FreeTypeFont, cmap: frozenset[int], size: int) -> float:
    """Lebar sejati termasuk glyph fallback — kalau tidak, center-nya meleset.

    Harus memakai syarat yang SAMA dengan _draw_line(), termasuk _FORCE_SYMBOL:
    glyph hati dari font simbol lebarnya beda dari glyph `yat` anime_ace, jadi
    kalau pengukuran memakai font utama sementara penggambaran memakai font
    simbol, baris itu diukur salah dan center-nya bergeser.

    Sudah dikali _cond(): inilah lebar yang BENAR-BENAR tergambar, karena
    render_region() memampatkan tile-nya dengan faktor yang sama.
    """
    if not _needs_fallback(line, cmap):
        return font.getlength(line) * _cond()
    return sum(_char_font(c, font, cmap, size).getlength(c) for c in line) * _cond()



def _bg_luminance(img: np.ndarray, region: Region) -> float:
    """Median luminance interior bubble pada gambar (teks sudah terhapus).

    Dipakai memilih warna tinta: putih di interior gelap, hitam di terang.
    Pipeline memanggil typeset pada halaman bersih, jadi nilai ini = warna
    latar sungguhan. Kalau bubble_mask tidak tersedia, median seluruh crop
    cukup akurat karena interior selalu mayoritas piksel.
    """
    if region.bubble_bbox is None:
        return 255.0
    bx1, by1, bx2, by2 = region.bubble_bbox
    crop = img[by1:by2, bx1:bx2]
    if crop.size == 0:
        return 255.0
    gray = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)
    mask = region.bubble_mask
    if mask is None or mask.shape[:2] != gray.shape:
        return float(np.median(gray))
    vals = gray[mask > 0]
    return float(np.median(vals)) if vals.size else float(np.median(gray))


def _region_box_mask(region: Region) -> tuple[tuple[int, int, int, int], np.ndarray]:
    """Kotak render + mask interior region. Persegi 255 kalau mask tidak cocok."""
    box = region.bubble_bbox or region.bbox
    bx1, by1, bx2, by2 = box
    bw, bh = bx2 - bx1, by2 - by1
    mask = region.bubble_mask
    if mask is None or mask.shape[:2] != (bh, bw):
        mask = np.full((bh, bw), 255, np.uint8)
    return box, mask


def _max_feasible(text: str, mask: np.ndarray, font_path: str) -> int:
    """Ukuran terbesar yang muat di balon ini TANPA penggalan; 0 kalau nihil.

    Plafonnya dari GEOMETRI balon, bukan tinggi glyph Jepang. est_font_size
    diukur MELINTANG kolom vertikal Jepang (textmask._glyph_height) dan
    variansinya besar, jadi memakainya sebagai plafon per region membuat ukuran
    font beda sampai ~2x antar balon dalam satu panel yang sama.
    """
    mh, mw = mask.shape[:2]
    pad = int(min(mh, mw) * SETTINGS.pad_ratio)
    lo = min_font()
    hi = int(np.clip(mh - 2 * pad, lo, SETTINGS.max_font_size))
    best = _search(text, mask, lo, hi, font_path, hyphenate=False)
    return best[0] if best else 0


# Ukuran font = rasio tetap terhadap SISI TERPENDEK interior balon. Kedua angka
# diukur, bukan ditaksir:
#   0.117 = cap_height / min(sisi interior) di CONTOH/2.webp (probe_refnative.py,
#           13 balon; p25 0.108 p75 0.150) — jauh lebih stabil daripada
#           cap_height-nya sendiri, yang berkisar 13..27 px (sebaran 2.08x).
#   0.844 = cap_height / ukuran font Anime Ace, konstan pada ukuran 11..32
#           (probe_cap.py, dari render "HAMBURG").
#
# CATATAN: ini MENYIMPANG dari plan.txt langkah 4, yang menyuruh satu ukuran
# seragam untuk seluruh halaman ("Ini persis pola typesetter referensi"). Ukuran
# mengatakan sebaliknya — referensi justru MENSKALAKAN teks ke besar balon. Tiga
# model diuji terhadap ukuran referensi yang terukur (probe_model.py):
#   seragam-halaman (persentil 35)  galat rata-rata 4.31 px
#   proporsional balon              galat rata-rata 2.71 px  <- ini
#   proporsional per panel          galat rata-rata 4.19 px
# Kontingensi di plan ("kelompokkan region per panel") ternyata LEBIH BURUK
# daripada proporsional biasa di halaman ini, jadi tidak dipakai.
_REF_CAP_PER_MIN = 0.117
_CAP_PER_SIZE = 0.844


def region_font_cap(mask: np.ndarray) -> int:
    """Plafon ukuran font untuk satu balon, dari geometrinya sendiri.

    Plafon, bukan keputusan akhir: fit() masih menurunkannya kalau teksnya
    memang tidak muat. Yang penting plafon ini TIDAK berasal dari est_font_size
    — tinggi glyph Jepang diukur melintang kolom vertikal, variansinya besar,
    dan itulah sebab awal ukuran font beda ~2x antar balon satu panel.
    """
    mn = min(mask.shape[:2])
    size = int(round(mn * _REF_CAP_PER_MIN / _CAP_PER_SIZE))
    return int(np.clip(size, min_font(), SETTINGS.max_font_size))


# ---------------------------------------------------------------- anggaran balon
#
# Berapa karakter yang SUNGGUH muat di satu balon — dipakai jalur LLM untuk
# memberi tahu penerjemah harus sependek apa. Diletakkan di sini, bukan di
# translate.py, karena angkanya harus keluar dari mesin tata letak YANG SAMA
# dengan yang merender. Anggaran yang dihitung terpisah bisa melenceng dari
# kenyataan tanpa ada yang tahu.
#
# Kenapa perlu sama sekali: "buatlah pendek" bukan perintah, itu selera. Percobaan
# yang cuma menyuruh "PREFER THE SHORTER natural phrasing" tetap mengembalikan
# 'SORRY TO BARGE IN.' (18 karakter) untuk balon yang memuat 6 — dan model tidak
# melanggar apa pun, ia memang tidak PUNYA cara menaati perintah tanpa angka. Ia
# tidak melihat balonnya.
#
# Teks pengisi, dan dua sifatnya yang penting (keduanya ketemu dari kegagalan,
# bukan dipikirkan lebih dulu):
#
# 1. GRANULARITAS. layout() bekerja per kata, jadi anggaran hanya bisa melompat
#    sebesar kata berikutnya. Pengisi versi pertama dimulai "THE PREZ WAS PUTTING
#    TOGETHER ..." — lompatan 20 -> 29 karakter, dan tujuh balon berbeda semuanya
#    melaporkan soft=20 karena mentok di kata 'TOGETHER' yang sama. Angka itu
#    bukan sifat balonnya, itu sifat pengisinya. Kata di sini 2-6 huruf berputar.
#
# 2. LEBAR GLYPH. Anime Ace tidak monospace; 'W' hampir dua kali 'I'. Pengisi
#    harus mendekati frekuensi huruf Inggris — 'AAAA' membuat anggaran terlalu
#    pesimistis, 'IIII' terlalu optimistis.
_BUDGET_FILLER = (
    "SO THE PREZ HAS ALL THE NOTES AND I SEE THEM HERE ON HER DESK "
    "AT ONE SIDE OF THE ROOM IT IS SO NICE AND I DO LIKE IT A LOT "
    "LET ME TAKE A LOOK AT THIS ONE FOR JUST A BIT MORE OK THANKS "
) * 8
_BUDGET_WORDS = _BUDGET_FILLER.split()


def char_budget(mask: np.ndarray, size: int, font_path: str) -> int:
    """Karakter terbanyak (batas kata) yang masih muat UTUH pada `size`.

    Dicari lewat jumlah KATA, bukan potongan karakter sembarang: layout() bekerja
    per kata, jadi memotong di tengah kata memberi jawaban yang tidak pernah bisa
    dicapai teks sungguhan.
    """
    if size <= 0:
        return 0
    words = _BUDGET_WORDS
    whole = " ".join(words)
    if layout(whole, mask, size, font_path, hyphenate=False)[0]:
        return len(whole)
    lo, hi = 0, len(words)
    while hi - lo > 1:
        mid = (lo + hi) // 2
        if layout(" ".join(words[:mid]), mask, size, font_path, hyphenate=False)[0]:
            lo = mid
        else:
            hi = mid
    return len(" ".join(words[:lo]))


def max_word_len(mask: np.ndarray, size: int, font_path: str) -> int:
    """Kata TERPANJANG (tanpa spasi) yang masih muat satu baris pada `size`.

    Angka kedua ini wajib ada, dan itu ketemu dari percontohan yang gagal: satu
    balon anggaran totalnya 39 karakter, tapi 'MY APOLOGIES' yang cuma 12 karakter
    tetap menghasilkan tanda hubung. Yang menjepit BUKAN panjang kalimat melainkan
    'APOLOGIES' — satu kata 9 huruf tidak muat di lebar 68 px, dan begitu satu kata
    tidak muat, layout() hanya punya dua pilihan: penggal atau gagal. Typeset
    referensi memilih kata lain sama sekali ('SORRY.'), dan ITULAH keputusan yang
    perlu disampaikan ke penerjemah.
    """
    if size <= 0:
        return 0
    best = 0
    for n in range(2, 25):
        # Konsonan/vokal bergantian: lebar rata-rata wajar, bukan 'WWWW'/'IIII'.
        probe = ("RONALDESTI" * 3)[:n]
        if layout(probe, mask, size, font_path, hyphenate=False)[0]:
            best = n
        else:
            break
    return best


def region_budget(region: Region, font_path: str) -> dict[str, int]:
    """Anggaran satu balon: {cap, soft, hard, word_soft, word_hard}.

    soft = muat pada region_font_cap() -> ukuran yang DIINGINKAN (proporsional ke
           besar balon). hard = muat pada min_font_size -> batas mutlak; lewat dari
           ini fit() jatuh ke jalur darurat dan hasilnya tidak terbaca.

    Keduanya perlu. Cuma soft: terlalu ketat — wording typeset profesional sendiri
    melewatinya di balon padat (61 karakter di satu balon halaman referensi), jadi
    menjadikannya batas keras berarti menolak hasil yang justru ditiru. Cuma hard:
    terlalu longgar — teks jadi muat tapi selalu di ukuran minimum. Jadi soft =
    target, hard = batas.
    """
    mask = _region_box_mask(region)[1]
    cap = region_font_cap(mask)
    lo = min_font()
    return {
        "cap": cap,
        "soft": char_budget(mask, cap, font_path),
        "hard": char_budget(mask, lo, font_path),
        "word_soft": max_word_len(mask, cap, font_path),
        "word_hard": max_word_len(mask, lo, font_path),
    }


def render_region(img: np.ndarray, region: Region, font_path: str | None = None, own_map: np.ndarray | None = None, forb_map: np.ndarray | None = None, size_cap: int | None = None) -> np.ndarray:
    """Tulis terjemahan ke halaman. Center dua sumbu; ALL CAPS hanya English.

    Warna tinta menyesuaikan bubble: putih di interior gelap, hitam di terang.

    Tiap baris digambar ke overlay sendiri lalu di-shear di sekitar baseline-nya.
    Anime Ace versi regular tegak lurus sedangkan gambar referensi miring; shear
    per-baris inilah yang meniru italic sungguhan — men-shear seluruh blok
    sekaligus akan mendorong baris teratas keluar dari balon.

    `size_cap` = plafon ukuran font. Kalau None, diambil dari geometri balon
    lewat region_font_cap() — BUKAN dari est_font_size, yang berasal dari tinggi
    glyph Jepang dan variansinya besar.
    """
    if not region.translation or region.is_protected:
        return img
    font_path = font_path or FONT_USED
    if not font_path:
        return img

    box, mask = _region_box_mask(region)
    bx1, by1, bx2, by2 = box
    bw, bh = bx2 - bx1, by2 - by1
    if own_map is None:
        # Pemanggil langsung (tanpa render_page): bangun peta halaman sendiri.
        ih, iw = img.shape[:2]
        own_map = np.zeros((ih, iw), np.uint8)
        own_map[by1 : by1 + bh, bx1 : bx1 + bw] = mask

    text = region.translation.upper() if SETTINGS.force_upper else region.translation
    cap = size_cap if size_cap else region_font_cap(mask)
    size, lines, start_y, overflow = fit(text, mask, cap, font_path)
    region.final_font_size = size
    region.lines = lines
    region.overflowed = overflow
    if not lines:
        return img

    font = _font(font_path, size)
    cmap = _cmap(font_path)
    lh = _line_height(font)
    # Sumbu x = sumbu yang dipakai layout() menilai blok ini muat, bukan centroid.
    # Pada interior yang dipotong tetangganya keduanya bisa berjarak belasan px,
    # dan menggambar di centroid setelah memverifikasi di sumbu blok berarti
    # tinta jatuh di tempat yang tidak pernah diuji — bisa menembus garis balon.
    cx = line_axis(mask, lines, start_y, size, font_path)

    # Warna tinta mengikuti bubble: putih di interior gelap, hitam di terang.
    # Dulu selalu hitam murni - di bubble hitam terjemahan tidak terlihat.
    # Threshold 128 sengaja SAMA dengan _bubble_interior di textmask.py.
    fill = (255, 255, 255) if _bg_luminance(img, region) < 128 else (0, 0, 0)

    # Teks di dalam balon: tanpa stroke, sesuai referensi. Teks bebas di atas
    # art butuh stroke putih supaya tetap terbaca.
    stroke = 0 if region.bubble_bbox is not None else max(2, size // 9)
    k = SETTINGS.oblique
    pad = int(abs(k) * lh) + stroke + 4

    pil = Image.fromarray(img).convert("RGBA")
    cnd = _cond()
    for i, line in enumerate(lines):
        w = _line_width(line, font, cmap, size)   # lebar TERGAMBAR (sudah rapat)
        wn = w / cnd                              # lebar renggang, untuk kanvas
        tile = Image.new("RGBA", (int(wn) + pad * 2, lh + pad * 2), (0, 0, 0, 0))
        _draw_line(
            ImageDraw.Draw(tile), (pad, pad), line, font, fill, cmap, size, stroke
        )
        if k or cnd != 1.0:
            th = tile.height
            # Satu transform untuk DUA hal, bukan dua transform berurutan: tiap
            # resample memakan ketajaman, dan pada cap 6-8 px huruf kedua kalinya
            # sudah kabur. Koefisien AFFINE PIL adalah pemetaan BALIK
            # (in = a*out + b*out_y + c), jadi rapat x=cnd di sekitar x=pad
            # ditambah shear k memberi:
            #     in_x = pad + (out_x - pad + k*out_y - k*th/2) / cnd
            # Kanvas sengaja tetap selebar versi renggang: tintanya menyusut ke
            # [pad, pad+w], sisanya transparan dan tidak berbiaya apa pun.
            tile = tile.transform(
                tile.size, Image.AFFINE,
                (1 / cnd, k / cnd, pad - (pad + k * th / 2) / cnd, 0, 1, 0),
                resample=Image.BICUBIC,
            )
        tx = int(bx1 + cx - w / 2) - pad
        ty = by1 + start_y + i * lh - pad
        tile = _clip_to_mask(tile, tx, ty, own_map, forb_map)
        _paste(pil, tile, tx, ty)

    return np.asarray(pil.convert("RGB"), dtype=np.uint8)


def _paste(base: Image.Image, tile: Image.Image, x: int, y: int) -> None:
    """Composite dengan clipping — alpha_composite raise kalau tile lewat tepi."""
    x0, y0 = max(0, -x), max(0, -y)
    x1 = min(tile.width, base.width - x)
    y1 = min(tile.height, base.height - y)
    if x1 <= x0 or y1 <= y0:
        return
    if (x0, y0, x1, y1) != (0, 0, tile.width, tile.height):
        tile = tile.crop((x0, y0, x1, y1))
    base.alpha_composite(tile, dest=(x + x0, y + y0))


def _clip_to_mask(
    tile: Image.Image, tx: int, ty: int,
    own: np.ndarray, forb: np.ndarray | None,
) -> Image.Image:
    """Hapus alpha tile di luar balon sendiri / di dalam balon tetangga.

    Inilah jaminan 'tidak saling timpa': teks yang meluap dipotong di garis
    balon (own), dan teks tidak pernah ditulis di atas interior balon region
    lain (forb). Mask 255 penuh (region tanpa balon) = tidak memotong apa pun.
    """
    h, w = own.shape[:2]
    cx0, cy0 = max(tx, 0), max(ty, 0)
    cx1, cy1 = min(tx + tile.width, w), min(ty + tile.height, h)
    if cx1 <= cx0 or cy1 <= cy0:
        return tile
    keep = own[cy0:cy1, cx0:cx1]
    if forb is not None:
        keep = np.minimum(keep, 255 - forb[cy0:cy1, cx0:cx1])
    if not keep.size or int(keep.min()) >= 254:
        return tile  # seluruhnya di area yang boleh ditulis — tanpa biaya
    if int(keep.max()) < 128:
        return Image.new("RGBA", tile.size, (0, 0, 0, 0))  # seluruhnya terlarang
    alpha = np.asarray(keep, np.uint8)
    if int(alpha.min()) < int(alpha.max()):
        # Erode 1 px dulu supaya tepi feather tetap berada DI DALAM balon.
        alpha = cv2.erode(alpha, np.ones((3, 3), np.uint8))
        alpha = cv2.GaussianBlur(alpha, (3, 3), 0)
    factor = alpha.astype(np.float32) / 255.0
    sub = tile.crop((cx0 - tx, cy0 - ty, cx1 - tx, cy1 - ty))
    a = np.asarray(sub.getchannel("A"), np.float32) * factor
    sub.putalpha(Image.fromarray(a.astype(np.uint8)))
    out = Image.new("RGBA", tile.size, (0, 0, 0, 0))
    out.alpha_composite(sub, dest=(cx0 - tx, cy0 - ty))
    return out


def _paste_mask(box: tuple[int, int, int, int], mask: np.ndarray,
                h: int, w: int) -> np.ndarray:
    """Mask lokal -> kanvas halaman h x w. Dipotong di tepi halaman."""
    out = np.zeros((h, w), np.uint8)
    x1, y1 = box[0], box[1]
    mh, mw = mask.shape[:2]
    sy1, sx1 = max(y1, 0), max(x1, 0)
    sy2, sx2 = min(y1 + mh, h), min(x1 + mw, w)
    if sy2 > sy1 and sx2 > sx1:
        out[sy1:sy2, sx1:sx2] = mask[sy1 - y1:sy2 - y1, sx1 - x1:sx2 - x1]
    return out


def _line_bands(box: tuple[int, int, int, int], mask: np.ndarray, size: int,
                lines: list[str], start_y: int,
                font_path: str) -> list[tuple[int, int, int, int]]:
    """Kotak TINTA tiap baris di koordinat halaman: (y0, y1, x0, x1).

    Dihitung analitik dari sumbu blok + _line_width, bukan dengan merender:
    keduanya angka yang SAMA dengan yang dipakai render_region() menempel tile,
    jadi kotak ini benar-benar tempat tintanya jatuh — dan gratis.
    """
    if not lines:
        return []
    font = _font(font_path, size)
    cmap = _cmap(font_path)
    lh = _line_height(font)
    ink_top, ink_bot = _ink_band(font_path, size)
    ax = line_axis(mask, lines, start_y, size, font_path)
    out = []
    for k, line in enumerate(lines):
        w = _line_width(line, font, cmap, size)
        y = box[1] + start_y + k * lh
        out.append((y + ink_top, y + ink_bot + 1,
                    box[0] + int(ax - w / 2), box[0] + int(ax + w / 2) + 1))
    return out


def reclaim_unused_interiors(img: np.ndarray, regions: list[Region],
                             font_path: str | None = None) -> int:
    """Lebar yang diambil disjoin tapi TIDAK dipakai tetangga -> dikembalikan.

    disjoin_overlapping_interiors() menyelesaikan irisan interior secara
    Voronoi — per PIKSEL, tanpa tahu di baris mana teks tetangga benar-benar
    akan jatuh. Hasilnya lebar disandera di ketinggian yang tetangganya bahkan
    tidak sentuh. Terukur di hasilnew/jp_6.JPG (probe_row.py): r3 kehilangan
    20 px tetap di y=139..191, sementara tinta r2 berhenti di y=168 — jadi di
    lima baris terakhir r3 lebar itu hilang tanpa ada yang memakainya. Sisanya
    cuma 26 px dari mask 46x87, sedangkan 'WONDER' butuh 32 px pada size 6, dan
    itulah sebab langsung 'NO WON-/DER'.

    Aturannya, tiap syaratnya menutup satu cacat:

        kandidat_i = fill_mask_i          interior balon SENDIRI sebelum dipangkas
                     & interior region lain  hanya yang DIAMBIL, bukan tepi baru
                     - kotak tinta fase 1    yang benar-benar dipakai tetap milik dia

    Yang boleh mengklaim HANYA region yang benar-benar tercekik: hasil fase 1-nya
    ber-tanda-hubung atau luber. Region yang fontnya kecil tapi rapi tidak
    mengklaim apa pun — "boleh panjang, font mengecil" adalah kebijakan yang
    dipilih, jadi ukuran di bawah plafon bukan cacat dan tidak pantas dibayar
    dengan lebar tetangga. Ini bukan kehati-hatian, ini hasil ukuran: versi
    pertama membiarkan SEMUA region mengklaim, dan di jp_6 r2 (rapi, tanpa tanda
    hubung) mengambil 413 px dari r3 yang justru sedang tercekik — r3 turun 7->6
    dan tanda hubungnya TETAP ada, r2 turun 9->8. Reclaim yang membuat halaman
    lebih buruk daripada tidak dijalankan.

    Pikselnya DIPINDAH, bukan digandakan — dan itu bukan kerapian, itu syarat
    supaya langkah ini berguna sama sekali. render_page() menyusun forb_map dari
    interior region LAIN, jadi kalau piksel yang dikembalikan tetap tercatat
    sebagai interior tetangga, _clip_to_mask() menghapus tepat piksel itu dan
    hasilnya lebih buruk daripada tidak melakukan apa-apa: baris dinilai muat
    lalu tintanya dibuang. Jadi yang menerima menambah, yang melepas mengurangi,
    dalam satu operasi.

    Tiap klaim DIUJI dengan fit() lalu diterima atau DIBATALKAN, bukan dipercaya:
    luas yang bertambah ternyata bukan lebar yang bisa dipakai. Piksel rampasan
    disjoin berbentuk pita Voronoi yang bergerigi, jadi menambahkannya menaikkan
    luas tanpa menaikkan RUN bebas yang menyambung di band satu baris — terukur
    di r3, luas +365 px tapi run band-nya 24->20, 26->24, 26->25 (probe_reclaim3).
    Satu-satunya penilai yang jujur karena itu hasil fit() sesudahnya.

    Syarat terima: pengklaim kehilangan tanda hubung (atau, dengan jumlah tanda
    hubung sama, ukurannya naik), DAN tidak ada pelepas yang mendapat tanda
    hubung baru, luber, atau menyusut lebih dari _RECLAIM_LOSS. Pertukarannya
    searah: satu tanda hubung adalah cacat #4 yang disebut plan.txt, sedangkan
    menyusut sedikit adalah kebijakan yang sudah dipilih ("boleh panjang, font
    mengecil") — jadi tanda hubung boleh ditebus dengan ukuran, tidak pernah
    sebaliknya.

    Sumbernya fill_mask region itu sendiri (direkam build_fill_mask() SEBELUM
    pemangkasan), jadi langkah ini secara konstruksi tidak bisa memunculkan
    "teks keluar bubble": piksel yang dikembalikan selalu piksel yang dulu
    memang interior balon ini. Syarat "& interior region lain" perlu karena
    fill_mask dikikis lebih tipis daripada bubble_mask (fill_erode_stroke) —
    tanpa itu reclaim ikut memakan jarak aman ke garis balon.

    Dua fase, jadi fit() memang berjalan berkali-kali. Itu harganya, dan tidak
    bisa dihindari: kotak tinta tetangga baru diketahui SETELAH ditata, sementara
    lebar yang boleh diambil harus diketahui SEBELUM menata.

    Returns:
        Jumlah region yang interiornya berubah (melebar atau menyusut).
    """
    font_path = font_path or FONT_USED
    if not font_path or len(regions) < 2:
        return 0
    h, w = img.shape[:2]

    live = [r for r in regions if r.translation and not r.is_protected]
    if len(live) < 2:
        return 0
    by_idx = {r.idx: r for r in regions}

    maps = {r.idx: _paste_mask(*_region_box_mask(r), h, w) > 0 for r in regions}
    fills = {
        r.idx: (_paste_mask(r.fill_bbox, r.fill_mask, h, w) > 0
                if r.fill_mask is not None and r.fill_bbox is not None
                else np.zeros((h, w), bool))
        for r in regions
    }
    # Region terlindungi (SFX) tidak pernah jadi pelepas: interiornya ikut
    # menyusun forb_map, jadi merampasnya membuka jalan teks Inggris menimpa SFX
    # — dilarang plan.txt, dan dijaga assert_sfx_intact.
    keep_out = np.zeros((h, w), bool)
    for r in regions:
        if r not in live:
            keep_out |= maps[r.idx]

    def _lay(r: Region, mp: np.ndarray):
        """fit() pada peta halaman `mp`, bukan pada r.bubble_mask yang sekarang."""
        ys, xs = np.nonzero(mp)
        if ys.size == 0:
            return None
        box = (int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1)
        mask = np.where(mp[box[1]:box[3], box[0]:box[2]], 255, 0).astype(np.uint8)
        text = r.translation.upper() if SETTINGS.force_upper else r.translation
        size, lines, start_y, over = fit(text, mask, region_font_cap(mask), font_path)
        return box, mask, size, lines, start_y, over

    def _score(st) -> tuple[int, int, int]:
        """(tanda hubung, luber, ukuran); dua yang pertama makin kecil makin baik."""
        if st is None:
            return (99, 1, 0)
        _b, _m, size, lines, _sy, over = st
        return (sum(1 for x in lines if x.endswith("-")), int(bool(over)), size)

    # Fase 1: tata letak pada interior sekarang — hanya untuk tahu di mana
    # tintanya jatuh dan siapa yang tercekik. Tata letaknya sendiri dibuang;
    # render_region() menata ulang pada interior final.
    lay = {r.idx: _lay(r, maps[r.idx]) for r in live}

    def _ink_union() -> np.ndarray:
        """Kotak tinta semua region, dari tata letak terakhir yang sah."""
        m = np.zeros((h, w), bool)
        for q in live:
            st = lay[q.idx]
            if st is None:
                continue
            for y0, y1, x0, x1 in _line_bands(st[0], st[1], st[2], st[3],
                                              st[4], font_path):
                m[max(y0, 0):max(y1, 0), max(x0, 0):max(x1, 0)] = True
        return m

    # Fase 2: satu pengklaim per putaran, diuji lalu diterima atau dibatalkan.
    # Yang paling tercekik jalan lebih dulu (tanda hubung terbanyak, lalu luber,
    # lalu ukuran terkecil) supaya lebar yang terbatas jatuh ke yang paling
    # butuh; idx sebagai pemutus seri terakhir supaya hasilnya deterministik.
    changed: set[int] = set()
    for r in sorted(live, key=lambda q: (-_score(lay[q.idx])[0],
                                         -_score(lay[q.idx])[1],
                                         _score(lay[q.idx])[2], q.idx)):
        base = _score(lay[r.idx])
        if base[0] == 0 and base[1] == 0:
            continue                     # rapi: tidak berhak merampas tetangga
        ink = _ink_union()
        others = np.zeros((h, w), bool)
        for q in regions:
            if q.idx != r.idx:
                others |= maps[q.idx]
        cand = fills[r.idx] & ~maps[r.idx] & others & ~ink & ~keep_out
        if not cand.any():
            continue

        trial = {r.idx: maps[r.idx] | cand}
        losers = [q for q in live
                  if q.idx != r.idx and bool((maps[q.idx] & cand).any())]
        for q in losers:
            trial[q.idx] = maps[q.idx] & ~cand
        newlay = {i: _lay(by_idx[i], m) for i, m in trial.items()}
        if newlay[r.idx] is None or any(newlay[i] is None for i in trial):
            continue                     # jangan pernah mengosongkan balon

        gain = _score(newlay[r.idx])
        better = gain[0] < base[0] or (gain[0] == base[0] and gain[1] < base[1]) \
            or (gain[:2] == base[:2] and gain[2] > base[2])
        if not better:
            continue
        harmed = False
        for q in losers:
            was, now = _score(lay[q.idx]), _score(newlay[q.idx])
            floor = was[2] - max(1, int(round(was[2] * _RECLAIM_LOSS)))
            if now[0] > was[0] or now[1] > was[1] or now[2] < floor:
                harmed = True
                break
        if harmed:
            continue

        for i, m in trial.items():       # diterima: pasang, dan ingat ulang
            maps[i] = m
            lay[i] = newlay[i]
            changed.add(i)

    # Baru sekarang bubble_mask/bubble_bbox ditulis, sekali per region: selama
    # pengujian di atas semuanya masih di `maps` supaya klaim yang ditolak tidak
    # meninggalkan bekas apa pun di Region.
    for i in sorted(changed):
        r = by_idx[i]
        ys, xs = np.nonzero(maps[i])
        bx1, by1 = int(xs.min()), int(ys.min())
        bx2, by2 = int(xs.max()) + 1, int(ys.max()) + 1
        r.bubble_bbox = (bx1, by1, bx2, by2)
        r.bubble_mask = np.where(maps[i][by1:by2, bx1:bx2], 255, 0).astype(np.uint8)
    return len(changed)


def render_page(img: np.ndarray, regions: list[Region]) -> np.ndarray:
    out = img
    h, w = img.shape[:2]
    # Lantai ukuran font berskala lebar halaman — lihat min_font(). Di-set di sini
    # supaya pemanggil render_page() langsung (probe, notebook) ikut benar tanpa
    # perlu mengingat memanggilnya sendiri.
    set_page_width(w)
    n = len(regions)
    if n == 0:
        return img

    # Lebar yang disandera disjoin tanpa dipakai dikembalikan DULU, sebelum peta
    # dan plafon dihitung — semuanya turunan dari bubble_mask.
    reclaim_unused_interiors(img, regions)

    def _page_map(r: Region) -> np.ndarray:
        """Interior balon (atau bbox, untuk region tanpa balon) di halaman.

        Wajib memakai kotak+mask yang SAMA dengan render_region. Kalau keduanya
        berbeda, own_map bisa kosong tepat di tempat teks digambar dan
        _clip_to_mask menghapus seluruh barisnya.
        """
        return _paste_mask(*_region_box_mask(r), h, w)

    # Peta "area terlarang" tiap region = gabungan interior balon region LAIN,
    # dihitung lewat prefix/suffix max supaya biayanya O(N) bukan O(N^2).
    # Klip inilah yang menjamin teks panjang tidak pernah saling timpa, apa pun
    # bentuk balonnya (double bubble menyatu, balon saling tumpang tindih, dll).
    maps = [_page_map(r) for r in regions]
    pref = [np.zeros((h, w), np.uint8)] * (n + 1)
    for i in range(n):
        pref[i + 1] = np.maximum(pref[i], maps[i])
    suff = [np.zeros((h, w), np.uint8)] * (n + 1)
    for i in range(n - 1, -1, -1):
        suff[i] = np.maximum(suff[i + 1], maps[i])

    # Plafon dihitung per balon dari geometrinya (region_font_cap), bukan satu
    # angka halaman: referensi TERUKUR menskalakan teks ke besar balon, bukan
    # menyeragamkannya — lihat komentar di atas _REF_CAP_PER_MIN.
    for i, r in enumerate(regions):
        forb = np.maximum(pref[i], suff[i + 1])
        out = render_region(out, r, own_map=maps[i], forb_map=forb)
    return out



Writing /content/mangatl/typeset.py


In [13]:
%%writefile /content/mangatl/verify.py

"""Verifikasi residu + escalation ladder.

Jalankan ulang DETECTOR pada halaman bersih — jangan OCR. manga-ocr
terdokumentasi berhalusinasi pada input kosong dan akan terus memberi
false positive.
"""

from __future__ import annotations

import cv2
import numpy as np

from config import SETTINGS, Region
import detect
import erase


def _iou(a: tuple, b: tuple) -> float:
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    if inter == 0:
        return 0.0
    ua = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / max(ua, 1)


# Berapa px interior balon dilebarkan sebelum dipakai MENGURUNG lingkup
# pemeriksaan sisa — lihat _residue_scope(). Bukan nol, karena tinta Jepang yang
# menempel garis balon memang duduk sedikit di luar mask interior dan justru
# itulah tinta yang paling sering tertinggal; bukan besar, karena di luar sana
# ada ART yang ikut terbawa ink_mask dan akan dituduh sisa. Empat ambang diukur
# berdampingan di _erasescope.py pada halaman referensi:
#
#   N     alarm palsu   pengawasan rata2 / terburuk
#   lama  r7, r8        100.0% / 100.0%
#   0     -              97.6% /  81.3%
#   2     -              98.2% /  83.5%
#   3     -              98.3% /  84.4%   <- dipilih
#   4     r7, r8         98.5% /  85.4%
#   8     r7, r8         98.9% /  88.8%
#
# 3 adalah yang TERLEBAR yang masih nol alarm palsu, jadi butanya paling kecil
# di antara yang aman. Di kedua ujung sapuan, sisa BUATAN tetap tertangkap —
# baik di tengah interior (153 px) maupun tepat di tepi garis balon (28 px),
# yang terakhir adalah mekanisme cacat #1 yang sesungguhnya.
_SCOPE_NEAR = 3


def _residue_scope(clean: np.ndarray, region: Region) -> tuple[np.ndarray, np.ndarray] | None:
    """(dev, scope) untuk region ini: piksel menyimpang, dan di mana dicari.

    Lingkupnya `(bekas stroke | cincin 3 px di sekitarnya)` DIKURUNG ke interior
    balon yang dilebarkan `_SCOPE_NEAR` px.

    Cincin itu ada karena `textmask.protect_bubble_outline()` MENGHAPUS piksel
    dari ink_mask: tinta Jepang yang menempel garis balon memang tidak boleh
    dicat, kalau tidak garis balonnya ikut hilang. Konsekuensinya piksel itu
    (a) tidak dicat jalur ink_mask dan (b) tidak pernah diperiksa, karena
    lingkup pemeriksaan justru `ink_mask > 0` — dan piksel itu baru saja dibuang
    dari sana. Selama fill_mask ada, isian interior menutupinya; begitu
    build_fill_mask menyerah, ia tertinggal di halaman dan definisi lama
    melaporkan NOL. Terukur di halaman referensi (est_font_size TERISI seperti
    produksi): protect_bubble_outline melepas 215 px.

    Kurungan interiornya yang menahan ALARM PALSU, dan ini yang terukur paling
    mahal. Tanpa kurungan, r7 (36 px) dan r8 (103 px) ditandai — padahal
    _erasewho.py mengukur bahwa piksel itu (a) sama sekali TIDAK diubah erase
    (utuh = sisa, jadi bukan bekas cat yang gagal), (b) NOL px-nya di dalam
    bubble_mask, (c) nol di bawah guard, dan (d) gray minimumnya 113/4 dengan
    median 184/190 — itu ART di luar balon yang ikut terbawa ink_mask. Kedua
    jawaban atas tuduhan itu merusak halaman:
      * mengecatnya (gabungan fill|ink di erase_flat) menaruh bg_color putih di
        atas art gray 12-18 — 7 bercak TERLIHAT terukur di _eraseblotch.py,
        yaitu cacat #3 lewat pintu belakang, di region yang bahkan tidak punya
        sisa untuk diperbaiki;
      * membiarkannya membuat find_residue memanggil escalate(), dan mask
        eskalasinya memakan 250 px `bubble_outline_guard` (_erasegate.py) —
        persis yang protect_bubble_outline ada untuk mencegah.
    Jadi yang salah bukan cat maupun eskalasinya, melainkan LINGKUP tuduhannya.

    Kenapa cincin 3 px dan bukan seluruh interior: sudut kotak balon berisi
    ART, dan sejak `textmask._keep_ink_lobes()` art itu memang sengaja tidak
    dicat. Lingkup seluruh interior akan melaporkan art sebagai sisa lalu
    mengeskalasi inpaint ke atasnya. Terukur: r9 menyimpan 198 px art di sudut
    kanan-bawah interiornya (halaman y1280-1303 x608-625, gray 218-234);
    lingkup ini TIDAK melihatnya, karena 3 px dari bekas tinta tidak menjangkau
    sudut kotak.

    Cincin `near` tetap digabung, bukan diganti: lingkup cincin-saja kehilangan
    bekas stroke yang justru sedang diperiksa. `gabungan >= lama` terukur di
    seluruh 13 region — pada halaman referensi keduanya sama besar, jadi cincin
    ini bermotif struktural (ia menutup lubang pengawasan di atas) dan belum
    punya contoh positif di halaman ini.
    """
    if region.ink_mask is None:
        return None
    x1, y1, x2, y2 = region.bbox
    crop = clean[y1:y2, x1:x2]
    if crop.size == 0:
        return None
    gray = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)
    mh, mw = region.ink_mask.shape[:2]
    sub = region.ink_mask[: min(mh, gray.shape[0]), : min(mw, gray.shape[1])]
    area = gray[: sub.shape[0], : sub.shape[1]]
    inside = sub > 0
    if not inside.any():
        return None
    scope = inside
    if region.bubble_mask is not None and region.bubble_bbox is not None:
        big = np.zeros(clean.shape[:2], np.uint8)
        bx1, by1 = region.bubble_bbox[0], region.bubble_bbox[1]
        bh, bw = region.bubble_mask.shape[:2]
        yy = min(by1 + bh, clean.shape[0])
        xx = min(bx1 + bw, clean.shape[1])
        big[by1:yy, bx1:xx] = region.bubble_mask[: yy - by1, : xx - bx1]
        if big.any():
            itr = big[y1 : y1 + sub.shape[0], x1 : x1 + sub.shape[1]] > 0
            near = cv2.dilate(
                sub, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))) > 0
            wide = cv2.dilate(big, cv2.getStructuringElement(
                cv2.MORPH_ELLIPSE, (2 * _SCOPE_NEAR + 1,) * 2))
            keep = wide[y1 : y1 + sub.shape[0], x1 : x1 + sub.shape[1]] > 0
            scope = (inside | (near & itr[: sub.shape[0], : sub.shape[1]])) & keep
    bg = float(np.median(area[~inside])) if (~inside).any() else 255.0
    dev = np.abs(area.astype(np.int16) - bg) > SETTINGS.residue_deviation
    return dev, scope


def pixel_residue(clean: np.ndarray, region: Region) -> int:
    """Hitung piksel yang masih menyimpang dari median background lokal.

    Non-nol di region flat-fill berarti mask atau fill-nya salah.
    Lingkupnya dijelaskan di _residue_scope().
    """
    got = _residue_scope(clean, region)
    if got is None:
        return 0
    dev, scope = got
    return int((dev & scope).sum())


def residue_blob(clean: np.ndarray, region: Region) -> int:
    """Komponen tersambung TERBESAR dari sisa — bukan jumlahnya.

    Gerbang jumlah `max(30, 0.002*w*h)` berskala AREA balon, sementara satu
    titik kotor tidak. Di balon 121x199 halaman ini ambangnya 48 px, tapi di
    balon 400x500 ia menjadi 400 px — dan titik 60 px yang jelas terlihat lolos
    utuh tanpa satu ronde eskalasi. Yang dilihat mata adalah satu titik, jadi
    yang harus dijaga ukuran titik terbesarnya, bukan totalnya.
    """
    got = _residue_scope(clean, region)
    if got is None:
        return 0
    dev, scope = got
    hit = dev & scope
    if not hit.any():
        return 0
    n, _lab, stats, _c = cv2.connectedComponentsWithStats(hit.astype(np.uint8), 8)
    return 0 if n <= 1 else int(stats[1:, cv2.CC_STAT_AREA].max())


def find_residue(clean: np.ndarray, regions: list[Region]) -> list[Region]:
    """Region mana yang masih punya sisa teks setelah erase.

    Deteksi di luar region yang dibersihkan = SFX yang sengaja dijaga, abaikan.

    Region PROTECTED sengaja tetap dikecualikan: tintanya memang harus utuh, dan
    memeriksanya berarti mengeskalasi inpaint ke atas SFX — pelanggaran kontrak
    'SFX dan simbol tetap ada'. Yang diperiksa hanya route flat/lama.
    """
    erased = [r for r in regions if r.route in ("flat", "lama") and not r.is_protected]
    if not erased:
        return []

    try:
        new_regions, _ = detect.detect(clean, conf=max(SETTINGS.det_conf, 0.35))
    except (RuntimeError, FileNotFoundError):
        new_regions = []

    failed: list[Region] = []
    for r in erased:
        hit = any(_iou(r.bbox, nr.bbox) > 0.25 for nr in new_regions)
        # Dua gerbang, bukan satu: TOTAL berskala area balon (satu titik kotor
        # tidak), jadi ditambah gerbang komponen terbesar — lihat residue_blob().
        if (hit
                or pixel_residue(clean, r) > max(30, int(0.002 * r.width * r.height))
                or residue_blob(clean, r) > SETTINGS.residue_blob_max):
            failed.append(r)
    return failed


def assert_sfx_intact(erase_mask: np.ndarray, protected_mask: np.ndarray) -> bool:
    """Kontrak keras: mask hapus tidak boleh menyentuh satu piksel SFX pun."""
    if protected_mask.max() == 0:
        return True
    return not bool(cv2.bitwise_and(erase_mask, protected_mask).any())


def escalate(
    img: np.ndarray, clean: np.ndarray, failed: list[Region], device: str = "cuda"
) -> np.ndarray:
    """Perluas mask region yang gagal lalu inpaint ulang HANYA region itu.

    Ladder: kernel 3 -> 5 -> 7 mengikuti percobaan ke-berapa.

    Mask yang sudah dipekarkan DIKURUNG ke interior balon (dilebarkan
    `_SCOPE_NEAR` px, ambang yang sama dengan lingkup pemeriksaan). Tanpa
    kurungan itu, memekarkan ink_mask dengan kernel 5 memakan 250 px
    `bubble_outline_guard` di halaman referensi (terukur di _erasegate.py) —
    LaMa lalu melukis ulang garis balon yang `protect_bubble_outline` baru saja
    susah-susah selamatkan, dan hasilnya balon bergaris putus. Kurungannya tidak
    membuat eskalasi tumpul: sisa BUATAN di dalam interior tetap tertutup
    7727/7727 px dengan maupun tanpa kurungan. Region tanpa bubble_mask
    dibiarkan seperti dulu — tidak ada interior untuk dijadikan pagar.
    """
    if not failed:
        return clean
    out = clean
    for attempt in range(1, SETTINGS.max_escalation + 1):
        mask = np.zeros(img.shape[:2], np.uint8)
        k = 3 + 2 * attempt
        el = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
        for r in failed:
            if r.ink_mask is None:
                continue
            x1, y1, x2, y2 = r.bbox
            mh, mw = r.ink_mask.shape[:2]
            y2, x2 = min(y2, y1 + mh), min(x2, x1 + mw)
            grown = cv2.dilate(r.ink_mask[: y2 - y1, : x2 - x1], el, iterations=1)
            if r.bubble_mask is not None and r.bubble_bbox is not None:
                itr = np.zeros(img.shape[:2], np.uint8)
                bx1, by1 = r.bubble_bbox[0], r.bubble_bbox[1]
                bh, bw = r.bubble_mask.shape[:2]
                yy = min(by1 + bh, img.shape[0])
                xx = min(bx1 + bw, img.shape[1])
                itr[by1:yy, bx1:xx] = r.bubble_mask[: yy - by1, : xx - bx1]
                if itr.any():
                    wide = cv2.dilate(itr, cv2.getStructuringElement(
                        cv2.MORPH_ELLIPSE, (2 * _SCOPE_NEAR + 1,) * 2))
                    grown = cv2.bitwise_and(grown, wide[y1:y2, x1:x2])
            mask[y1:y2, x1:x2] = np.maximum(mask[y1:y2, x1:x2], grown)

        if not mask.any():
            break
        out = erase.erase_neural(out, mask, device)
        failed = find_residue(out, failed)
        if not failed:
            break
    return out


def report(regions: list[Region], failed: list[Region], font_used: str,
           notes: list[tuple[str, str, str]] | None = None) -> dict:
    """Ringkasan untuk report.json + tabel UI.

    `notes` = catatan config.note() yang muncul SELAMA halaman ini diproses
    (lihat config.notes_since). Ikut di sini, bukan cuma di batch, karena
    pertanyaan yang ditanyakan pembaca selalu per halaman: "halaman ini kenapa
    kosong?". Kalau catatan hanya ada di tingkat batch, halaman ke-3 mewarisi
    tuduhan halaman ke-1.
    """
    # untranslated_idx: region yang boleh diterjemahkan (bukan SFX) tapi tidak
    # dapat terjemahan. Ini yang menangkap cacat hasilnew/13.JPG di sidecar:
    # translated_count saja tidak cukup karena angkanya harus dibandingkan
    # dengan jumlah region yang MEMANG perlu diterjemahkan — dan pembacanya
    # tidak punya angka itu. Kalau daftar ini tidak kosong, ada balon yang
    # tercetak berbahasa Jepang.
    untranslated = [r.idx for r in regions
                    if not r.is_protected and r.src_text and not r.translation]
    notes = list(notes or [])
    return {
        "region_count": len(regions),
        "residue_count": len(failed),
        "residue_idx": [r.idx for r in failed],
        "overflow_count": sum(1 for r in regions if r.overflowed),
        "protected_count": sum(1 for r in regions if r.is_protected),
        "sfx_idx": [r.idx for r in regions if r.label == "SFX"],
        "translated_count": sum(1 for r in regions if r.translation),
        "untranslated_count": len(untranslated),
        "untranslated_idx": untranslated,
        # translatable_count: pembanding yang hilang selama ini. region_count
        # memuat SFX dan UNREADABLE yang memang TIDAK boleh diterjemahkan, jadi
        # "translated 0 dari 15 region" bisa berarti dua hal yang jauh berbeda:
        # 15 balon gagal, atau 15 region itu semuanya SFX. Tanpa angka ini
        # pembaca tabel tidak bisa membedakannya.
        "translatable_count": sum(1 for r in regions
                                  if not r.is_protected and r.src_text),
        "notes": [{"level": lv, "tag": tg, "msg": ms} for lv, tg, ms in notes],
        "error_count": sum(1 for lv, _t, _m in notes if lv == "error"),
        "warn_count": sum(1 for lv, _t, _m in notes if lv == "warn"),
        "route_flat": sum(1 for r in regions if r.route == "flat"),
        "route_lama": sum(1 for r in regions if r.route == "lama"),
        "font_used": font_used,
        "regions": [r.to_dict() for r in regions],
    }



Writing /content/mangatl/verify.py


In [14]:
%%writefile /content/mangatl/assets.py

"""Unduh weight model. Idempoten — file yang sudah ada dilewati."""

from __future__ import annotations

import urllib.error
import urllib.request
from pathlib import Path

from config import WEIGHT_URLS, WEIGHTS

_MIN_BYTES = 1_000_000  # weight terkecil ~94 MB; file kecil = halaman error HTML


def _fetch(url: str, dest: Path, chunk: int = 1 << 20) -> bool:
    """Unduh streaming supaya file 200 MB tidak menghabiskan RAM."""
    tmp = dest.with_suffix(dest.suffix + ".part")
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=120) as resp, open(tmp, "wb") as f:
            total = int(resp.headers.get("Content-Length") or 0)
            done = 0
            while True:
                buf = resp.read(chunk)
                if not buf:
                    break
                f.write(buf)
                done += len(buf)
                if total:
                    pct = done * 100 // total
                    print(f"\r  {dest.name}: {pct:3d}%  ({done >> 20} MB)", end="")
        print()
        if tmp.stat().st_size < _MIN_BYTES:
            tmp.unlink(missing_ok=True)
            return False
        tmp.replace(dest)
        return True
    except (urllib.error.URLError, urllib.error.HTTPError, OSError, TimeoutError) as exc:
        print(f"\n  {dest.name}: GAGAL ({exc})")
        tmp.unlink(missing_ok=True)
        return False


def download_weights(verbose: bool = True) -> dict[str, bool]:
    """Return {nama_file: tersedia}. Pipeline tetap jalan walau sebagian gagal.

    Tiap weight punya rantai mirror; mirror berikutnya dicoba hanya kalau yang
    sebelumnya gagal, jadi jalur normal tetap satu request per file.
    """
    WEIGHTS.mkdir(parents=True, exist_ok=True)
    status: dict[str, bool] = {}
    for name, urls in WEIGHT_URLS.items():
        dest = WEIGHTS / name
        if dest.exists() and dest.stat().st_size >= _MIN_BYTES:
            status[name] = True
            if verbose:
                print(f"  {name}: sudah ada ({dest.stat().st_size >> 20} MB)")
            continue
        status[name] = any(_fetch(u, dest) for u in urls)
    return status



Writing /content/mangatl/assets.py


In [15]:
%%writefile /content/mangatl/pipeline.py

"""Orkestrasi per halaman: detect -> mask -> OCR -> LLM -> erase -> verify -> typeset.

Urutan wajib: klasifikasi SFX dari LLM harus selesai SEBELUM compose mask,
karena exclusion SFX bergantung pada label. Menukar dua langkah ini membuat
pipeline menghapus persis apa yang diminta untuk dijaga.
"""

from __future__ import annotations

import gc
import json
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np

from config import DEBUG_DIR, OUTPUT, RUN_NOTES, SETTINGS, Region, note, notes_since
import detect
import erase
import imgio
import ocr
import textmask
import translate as tl
import typeset
import verify


@dataclass
class PageResult:
    stem: str
    original: np.ndarray
    cleaned: np.ndarray
    final: np.ndarray
    regions: list[Region]
    report: dict
    paths: dict[str, Path]


def _device() -> str:
    try:
        import torch

        return "cuda" if torch.cuda.is_available() else "cpu"
    except ImportError:
        return "cpu"


def _dump(stem: str, name: str, img: np.ndarray) -> None:
    d = DEBUG_DIR / stem
    d.mkdir(parents=True, exist_ok=True)
    arr = img if img.ndim == 3 else cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    cv2.imwrite(str(d / f"{name}.png"), cv2.cvtColor(arr, cv2.COLOR_RGB2BGR))


def _draw_boxes(img: np.ndarray, regions: list[Region]) -> np.ndarray:
    """Kotak berwarna per kelas untuk debug: merah = SFX (dijaga)."""
    out = img.copy()
    colors = {
        "SFX": (255, 0, 0), "UNREADABLE": (255, 128, 0), "DIALOGUE": (0, 200, 0),
        "THOUGHT": (0, 160, 255), "NARRATION": (200, 0, 200), "SIGN": (255, 200, 0),
    }
    for r in regions:
        x1, y1, x2, y2 = r.bbox
        c = colors.get(r.label, (128, 128, 128))
        cv2.rectangle(out, (x1, y1), (x2, y2), c, 2)
        cv2.putText(out, f"{r.idx}:{r.label[:4]}", (x1, max(y1 - 4, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, c, 1, cv2.LINE_AA)
    return out


def process_page(
    img: np.ndarray, stem: str, client=None, model: str = "",
    debug: bool | None = None, outdir: Path | None = None,
    progress=None, target_lang: str = "English",
    style: str = "Manga Natural", keep_honorifics: bool = True,
) -> PageResult:
    """Satu halaman penuh. client None = jalan tanpa terjemahan (halaman bersih)."""
    debug = SETTINGS.debug if debug is None else debug
    dev = _device()
    original = img.copy()
    # Batas awal catatan halaman ini. Diambil SEBELUM apa pun jalan supaya
    # notes_since() nanti hanya mengembalikan kegagalan halaman INI — kalau
    # tidak, halaman ke-3 dalam satu batch mewarisi error halaman ke-1 dan
    # banner UI menuduh halaman yang sebenarnya bersih. RUN_NOTES di-mutate
    # (bukan di-rebind) jadi len() atas nama yang diimpor tetap akurat.
    note_mark = len(RUN_NOTES)

    def step(frac: float, msg: str) -> None:
        if progress is not None:
            progress(frac, desc=msg)

    if debug:
        _dump(stem, "01_input", img)

    # Lantai ukuran font berskala lebar halaman (typeset.min_font()). Di-set di
    # sini, SEBELUM terjemahan, karena anggaran karakter yang dikirim ke model
    # (translate._page_budget) memakai lantai itu — kalau baru di-set saat render,
    # model diberi anggaran halaman kalibrasi dan menulis terlalu pendek.
    typeset.set_page_width(img.shape[1])

    step(0.10, "deteksi region")
    regions, bubbles = detect.detect(img)
    if not regions:
        # Nol region bukan sukses: halaman keluar identik dengan aslinya. Dicatat
        # sebagai warn supaya baris ini muncul di banner UI, karena tabel hanya
        # akan menampilkan angka 0 di semua kolom dan itu tidak menjelaskan apa pun.
        note("warn", "detect",
             f"{stem}: tidak ada region terdeteksi — halaman keluar TANPA perubahan")
        rep = verify.report([], [], typeset.FONT_USED, notes_since(note_mark))
        paths = imgio.save_outputs(original, stem, outdir)
        return PageResult(stem, original, original, original, [], rep, paths)

    step(0.25, "bangun mask teks")
    soft = textmask.ctd_soft_mask(img)
    for r in regions:
        textmask.build_region_mask(img, r, soft)
    # Setelah SEMUA ink_mask terisi: balon ganda dipartisi per lobus dari
    # interior gabungan. Butuh ink_mask semua region, jadi tidak bisa di dalam
    # loop di atas.
    textmask.partition_shared_interiors(img, regions)
    # ...lalu balon bertetangga yang interiornya beririsan dibuat saling lepas.
    # Tanpa ini _clip_to_mask memotong glyph di zona irisan (lihat docstring-nya).
    textmask.disjoin_overlapping_interiors(img, regions)
    # ...lalu garis balonnya dilepas dari mask hapus. Harus SESUDAH kedua
    # langkah di atas, karena penjaganya dihitung dari bubble_mask final.
    textmask.protect_bubble_outline(img, regions)
    del soft
    gc.collect()

    step(0.40, "OCR Jepang")
    ocr.read_all(img, regions)
    # Model OCR TETAP di memori antar halaman (dimuat sekali per batch).
    # Kalau dilepas tiap halaman, batch multi membayar ~5-8 dtk reload
    # manga-ocr per halaman dan kecepatan satuan jadi turun. release()
    # cukup di release_all() di akhir batch (lihat _run).
    gc.collect()

    step(0.55, "klasifikasi SFX + terjemah")
    if client is not None and model:
        try:
            tl.translate_page(client, model, regions, target_lang,
                              style, keep_honorifics)
        except Exception as exc:  # noqa: BLE001 - jaringan tidak boleh membunuh halaman
            note("error", "pipeline",
                 f"{stem}: LLM gagal ({exc}); pakai label heuristik — "
                 "halaman keluar TANPA terjemahan")
            tl._fallback_labels(regions)
    else:
        tl._fallback_labels(regions)

    if debug:
        _dump(stem, "03_boxes", _draw_boxes(img, regions))

    # SFX exclusion terjadi di sini, sebelum erase apa pun.
    erase_mask, protected_mask = textmask.compose_page_mask(img, regions)
    if debug:
        raw = np.zeros(img.shape[:2], np.uint8)
        for r in regions:
            if r.ink_mask is None:
                continue
            x1, y1, x2, y2 = r.bbox
            mh, mw = r.ink_mask.shape[:2]
            y2, x2 = min(y2, y1 + mh), min(x2, x1 + mw)
            raw[y1:y2, x1:x2] = np.maximum(raw[y1:y2, x1:x2], r.ink_mask[: y2 - y1, : x2 - x1])
        _dump(stem, "05_mask", raw)
        _dump(stem, "07_mask_after_sfx_exclusion", erase_mask)

    if not verify.assert_sfx_intact(erase_mask, protected_mask):
        raise AssertionError("mask hapus menyentuh SFX — kontrak pipeline dilanggar")

    step(0.70, "hapus teks asli")
    cleaned = erase.erase_page(img, regions, dev)

    step(0.80, "verifikasi residu")
    failed = verify.find_residue(cleaned, regions)
    if failed:
        cleaned = verify.escalate(img, cleaned, failed, dev)
        failed = verify.find_residue(cleaned, regions)
    if debug:
        _dump(stem, "09_cleaned", cleaned)

    step(0.90, "typeset Inggris")
    final = typeset.render_page(cleaned, regions)
    if debug:
        _dump(stem, "10_typeset", final)

    rep = verify.report(regions, failed, typeset.FONT_USED, notes_since(note_mark))
    rep["bubble_count"] = len(bubbles)
    if debug:
        (DEBUG_DIR / stem).mkdir(parents=True, exist_ok=True)
        (DEBUG_DIR / stem / "report.json").write_text(
            json.dumps(rep, ensure_ascii=False, indent=2), encoding="utf-8"
        )

    step(0.97, "simpan")
    paths = imgio.save_outputs(final, stem, outdir)
    # Sidecar: teks asli tersimpan walau terjemahan gagal — kerja tidak hilang.
    # Ambil path pertama yang ditulis (png/jpg sesuai format terpilih).
    next(iter(paths.values())).with_suffix(".json").write_text(
        json.dumps(rep, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return PageResult(stem, original, cleaned, final, regions, rep, paths)


def process_batch(
    files: list[str | Path], api_key: str | None = None, debug: bool = False,
    progress=None, target_lang: str = "English",
    style: str = "Manga Natural", keep_honorifics: bool = True,
    outdir: Path | None = None, reset_notes: bool = True,
) -> tuple[list[PageResult], dict]:
    """Beberapa halaman sekali jalan. Model dipilih sekali, dipakai ulang.

    `reset_notes=False` dipakai app.py: UI sudah mencatat kegagalannya sendiri
    (mis. "API key tidak terbaca") SEBELUM memanggil ini, dan pembersihan di sini
    akan menghapus justru catatan yang menjelaskan kenapa `api_key` None. Default
    True supaya pemanggil skrip (run_full.py, sel 25) tidak mewarisi catatan run
    sebelumnya di sesi yang sama.
    """
    SETTINGS.debug = debug
    # Dikosongkan in-place (bukan rebind) supaya `from config import RUN_NOTES`
    # di modul lain tetap menunjuk daftar yang sama.
    if reset_notes:
        RUN_NOTES.clear()
    client, model, probes = None, "", []
    if api_key:
        try:
            # Penyedia diambil dari SETTINGS.provider (diisi UI) — bukan parameter
            # baru — supaya process_page dan pemanggil lain tidak perlu ikut
            # meneruskannya. make_client yang memutuskan kelas client-nya.
            client = tl.make_client(api_key)
            model, probes = tl.pick_model(client, verbose=False)
        except (RuntimeError, ImportError) as exc:
            # error, bukan warn: tanpa client SELURUH batch keluar berbahasa
            # Jepang. Ini penyebab paling sering "diterjemah 0" di semua halaman
            # sekaligus (kunci salah/kosong), dan harus jadi baris pertama banner.
            note("error", "pipeline",
                 f"tidak ada model LLM: {exc} — SEMUA halaman keluar TANPA terjemahan")
            client = None
    # Anggaran balon memanggil typeset.layout(), jadi fontnya harus sudah ada
    # sebelum halaman pertama. Tanpa ini balon pertama membayar unduhan font di
    # tengah pengukuran, dan waktunya tampak seperti biaya anggaran.
    if client is not None and not typeset.FONT_USED:
        typeset.setup_fonts(verbose=False)

    results: list[PageResult] = []
    total = max(len(files), 1)
    for i, f in enumerate(files):
        stem = Path(f).stem
        img = imgio.load_any(f)

        def sub(frac: float, desc: str = "", _i: int = i) -> None:
            if progress is not None:
                progress((_i + frac) / total, desc=f"[{_i + 1}/{total}] {desc}")

        results.append(
            process_page(img, stem, client, model, debug=debug, progress=sub,
                         target_lang=target_lang, style=style,
                         keep_honorifics=keep_honorifics, outdir=outdir)
        )
        del img
        gc.collect()

    summary = {
        "pages": len(results),
        "model": model or "none",
        "provider": SETTINGS.provider,
        "target_lang": target_lang,
        "style": style,
        "keep_honorifics": keep_honorifics,
        "font_used": typeset.FONT_USED,
        "probes": [p.as_row() for p in probes],
        "residue_total": sum(r.report["residue_count"] for r in results),
        "overflow_total": sum(r.report["overflow_count"] for r in results),
        "sfx_total": sum(len(r.report["sfx_idx"]) for r in results),
        # Seluruh catatan run, termasuk yang terjadi SEBELUM halaman pertama
        # (mis. "tidak ada model LLM") yang tidak dimiliki report halaman mana
        # pun. app.py butuh keduanya: per halaman untuk kolom tabel, batch untuk
        # sebab yang berlaku menyeluruh.
        "notes": [{"level": lv, "tag": tg, "msg": ms} for lv, tg, ms in RUN_NOTES],
        "error_total": sum(1 for lv, _t, _m in RUN_NOTES if lv == "error"),
        "warn_total": sum(1 for lv, _t, _m in RUN_NOTES if lv == "warn"),
        "translated_total": sum(r.report["translated_count"] for r in results),
        "untranslated_total": sum(r.report["untranslated_count"] for r in results),
    }
    zip_path = imgio.make_zip(
        [p for r in results for p in r.paths.values()], OUTPUT / "manga_translated.zip"
    )
    summary["zip"] = str(zip_path)
    return results, summary


def release_all() -> None:
    """Bebaskan semua sesi model — RAM Colab cuma ~12.7 GB."""
    import inpaint

    detect.release()
    textmask.release()
    ocr.release()
    inpaint.release()
    gc.collect()



Writing /content/mangatl/pipeline.py


In [16]:
%%writefile /content/mangatl/selftest.py

"""Self-test tanpa input user: gambar halaman uji sendiri lalu assert pipeline.

Ini yang membuktikan pipeline hidup sebelum user upload apa pun.
"""

from __future__ import annotations

import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont

from config import Region, SETTINGS
import detect
import erase
import textmask
import typeset
import verify


def _jp_font(size: int) -> ImageFont.FreeTypeFont:
    """Cari font berglyph Jepang; kalau tidak ada, pakai default (tofu tetap OK)."""
    for cand in (
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/fonts-japanese-gothic.ttf",
        "/usr/share/fonts/opentype/ipafont-gothic/ipag.ttf",
        "C:/Windows/Fonts/msgothic.ttc",
    ):
        try:
            return ImageFont.truetype(cand, size)
        except OSError:
            continue
    return ImageFont.load_default()


def make_test_page() -> tuple[np.ndarray, list[tuple[int, int, int, int]]]:
    """Dua bubble + satu kotak narasi + satu SFX di luar bubble."""
    W, H = 900, 1300
    img = Image.new("RGB", (W, H), (245, 243, 240))
    d = ImageDraw.Draw(img)

    # Latar bergaris supaya ada region yang butuh inpaint, bukan flat-fill saja.
    for y in range(0, H, 9):
        d.line([(0, y), (W, y)], fill=(205, 205, 205), width=2)

    f = _jp_font(34)
    boxes: list[tuple[int, int, int, int]] = []

    d.ellipse([80, 90, 400, 340], fill=(255, 255, 255), outline=(0, 0, 0), width=4)
    d.text((240, 215), "こんにちは", font=f, fill=(0, 0, 0), anchor="mm")
    boxes.append((80, 90, 400, 340))

    d.ellipse([480, 430, 830, 700], fill=(255, 255, 255), outline=(0, 0, 0), width=4)
    d.text((655, 565), "セックスしよ", font=f, fill=(0, 0, 0), anchor="mm")
    boxes.append((480, 430, 830, 700))

    d.rectangle([100, 820, 780, 960], fill=(255, 255, 255), outline=(0, 0, 0), width=3)
    d.text((440, 890), "その夜、二人は。", font=f, fill=(0, 0, 0), anchor="mm")
    boxes.append((100, 820, 780, 960))

    # SFX di luar bubble, langsung di atas art — harus tetap utuh.
    d.text((250, 1120), "ドドド", font=_jp_font(76), fill=(0, 0, 0), anchor="mm")
    boxes.append((150, 1070, 360, 1170))

    return np.asarray(img, dtype=np.uint8), boxes


# Ketebalan garis balon halaman uji balon ganda.
_DB_STROKE = 4


def make_double_bubble_page() -> tuple[np.ndarray, np.ndarray, np.ndarray, list[Region]]:
    """Balon figura-8 SUNGGUHAN: dua elips menyatu, tanpa garis pemisah di leher.

    Inilah bentuk yang meloloskan cacat 'saling timpa'. Membelah KOTAK balon
    tidak memisahkan BENTUK lobusnya — tiap belahan persegi masih memuat
    sebagian lobus sebelah, jadi dua centroid jatuh berdekatan dan kedua
    terjemahan bertumpuk di leher. Test lama (dua persegi terpisah) tidak
    pernah menyentuh kasus ini.

    Returns:
        (clean, img, inner, regions)
        clean   halaman tanpa teks Jepang. Target render, jadi beda piksel
                terhadapnya = tinta Inggris saja, bukan sisa teks asli.
        img     clean + teks Jepang; input pembangun mask, seperti pipeline.
        inner   interior balon seukuran halaman. `inner == 0` mencakup garis
                balon DAN seluruh luar balon = kontrak 'tidak keluar bubble'.
        regions dua region teks, keduanya menunjuk kotak balon GABUNGAN —
                persis keluaran detector saat kedua lobus jadi satu kotak.
    """
    W, H = 900, 620
    page = Image.new("RGB", (W, H), (238, 236, 233))
    d = ImageDraw.Draw(page)
    for y in range(0, H, 9):
        d.line([(0, y), (W, y)], fill=(198, 198, 198), width=2)

    lobes = ((280, 300, 215, 165), (620, 300, 215, 165))
    fill = np.zeros((H, W), np.uint8)
    for cx, cy, ax, ay in lobes:
        cv2.ellipse(fill, (cx, cy), (ax, ay), 0, 0, 360, 255, -1)
    # Garis balon = pita tepi GABUNGAN, bukan dua outline elips. Menggambar
    # kedua outline meninggalkan garis pemisah di leher, dan itu bukan balon
    # ganda lagi — cuma dua balon yang bersinggungan, kasus yang jauh lebih mudah.
    k = 2 * _DB_STROKE + 1
    inner = cv2.erode(fill, np.ones((k, k), np.uint8))
    arr = np.asarray(page, np.uint8).copy()
    arr[fill > 0] = (255, 255, 255)
    arr[(fill > 0) & (inner == 0)] = (0, 0, 0)

    clean = arr.copy()
    page = Image.fromarray(arr)
    d = ImageDraw.Draw(page)
    f = _jp_font(34)
    for (cx, _, _, _), rows in zip(lobes, (("かいちょう", "さがした"), ("ミルク", "クラブ"))):
        for i, row in enumerate(rows):
            d.text((cx, 278 + i * 44), row, font=f, fill=(0, 0, 0), anchor="mm")

    ys, xs = np.nonzero(fill)
    shared = (int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1)
    regions = [
        Region(idx=0, bbox=(190, 255, 370, 345), det_class="text_bubble",
               bubble_bbox=shared),
        Region(idx=1, bbox=(560, 255, 680, 345), det_class="text_bubble",
               bubble_bbox=shared),
    ]
    return clean, np.asarray(page, dtype=np.uint8), inner, regions


def make_adjacent_bubbles_page() -> tuple[np.ndarray, np.ndarray, np.ndarray, list[Region]]:
    """Dua lobus MENYATU, tapi detector memberi tiap lobus kotaknya SENDIRI.

    Inilah konfigurasi yang benar-benar gagal di halaman nyata dan tidak
    tersentuh make_double_bubble_page(). Bedanya cuma satu hal, dan hal itu
    menentukan segalanya: di sana kedua region menunjuk SATU kotak balon
    (`shared_bubble_bbox` terisi) sehingga partition_shared_interiors() jalan;
    di sini kotaknya terpisah, jadi fungsi itu tidak pernah dipanggil.

    Bentuknya tetap menyatu — tidak ada garis pemisah di leher — jadi flood fill
    tiap region menelan SELURUH figura-8, dipotong hanya oleh kotaknya sendiri.
    Kotak yang saling tumpang tindih + isi yang sama = interior beririsan ribuan
    piksel. Terukur di jepang_002.webp: 6 pasang beririsan, terparah 5994 px
    (region 2-3), semuanya `shared_bubble_bbox = None`.

    Yang rusak karenanya bukan tumpang tindih melainkan GLYPH TERPOTONG:
    render_region menata teks di interiornya sendiri, lalu _clip_to_mask
    membuang piksel yang jatuh di interior region lain (forb_map), sehingga
    'OH, IS THIS THE SHIKO CLUB?' keluar sebagai 'IS THE :KO 4B?'.

    Kalau dua balon memang terpisah bergaris sendiri-sendiri, garis balon
    menghentikan flood fill masing-masing dan tidak ada irisan sama sekali —
    kasus itu sudah aman tanpa perbaikan apa pun.

    Returns:
        (clean, img, inner, regions) — sama seperti make_double_bubble_page().
    """
    W, H = 760, 460
    page = Image.new("RGB", (W, H), (240, 238, 235))
    d = ImageDraw.Draw(page)
    for y in range(0, H, 11):
        d.line([(0, y), (W, y)], fill=(200, 200, 200), width=2)

    lobes = ((250, 230, 170, 150), (505, 230, 170, 150))
    fill = np.zeros((H, W), np.uint8)
    for cx, cy, ax, ay in lobes:
        cv2.ellipse(fill, (cx, cy), (ax, ay), 0, 0, 360, 255, -1)
    # Pita tepi GABUNGAN: lehernya terbuka, jadi kedua lobus satu ruang putih.
    k = 2 * _DB_STROKE + 1
    inner = cv2.erode(fill, np.ones((k, k), np.uint8))
    arr = np.asarray(page, np.uint8).copy()
    arr[fill > 0] = (255, 255, 255)
    arr[(fill > 0) & (inner == 0)] = (0, 0, 0)

    clean = arr.copy()
    page = Image.fromarray(arr)
    d = ImageDraw.Draw(page)
    f = _jp_font(30)
    for (cx, _, _, _), rows in zip(lobes, (("あっシコ", "部の"), ("性徒会の", "記録"))):
        for i, row in enumerate(rows):
            d.text((cx, 212 + i * 40), row, font=f, fill=(0, 0, 0), anchor="mm")

    # Satu kotak per lobus — bukan satu kotak gabungan. Keduanya beririsan di x
    # karena lobusnya memang saling tumpuk.
    regions = []
    for i, (cx, cy, ax, ay) in enumerate(lobes):
        regions.append(Region(idx=i, bbox=(cx - 70, 190, cx + 70, 270),
                              det_class="text_bubble",
                              bubble_bbox=(cx - ax, cy - ay, cx + ax, cy + ay)))
    return clean, np.asarray(page, dtype=np.uint8), inner, regions


def make_grey_bubble_page() -> tuple[np.ndarray, np.ndarray, np.ndarray, list[Region]]:
    """Balon figura-8 ber-screentone KELABU di sebelah art gelap.

    Dua pembangun balon ganda di atas mengecat balon PUTIH
    (`arr[fill > 0] = (255,255,255)`), jadi tidak satu pun bisa menangkap cacat
    cacatbaru/jp_cacatnew1+2: aturan polaritas lama di `_interior_from_crop`
    memutuskan kelas Otsu mana yang interior lewat ambang ABSOLUT
    `median(kelas mayoritas) < 128`. Itu benar hanya untuk dua ujung (balon
    putih / balon hitam). Pada balon KELABU, median mayoritas ada DI ATAS 128
    sehingga polaritas tidak dibalik, dan yang diambil sebagai 'interior' adalah
    piksel TERANG di luar balon. Satu mask salah itu memunculkan dua cacat
    sekaligus: `build_fill_mask` tidak pernah menghapus tinta Jepangnya (
    `erase_flat` memakai `fill_mask`) dan `typeset._region_box_mask` menata
    terjemahan di sliver luar balon — tercetak mungil di atas art.

    Tiga sifat halaman ini yang membuatnya memancing cacat itu, dan tidak satu
    pun ada di dua pembangun sebelumnya:

    1. Interior balon KELABU (screentone), bukan putih, jadi median kelas
       mayoritas jatuh di sekitar 140 — di atas 128, tapi jauh dari putih.
    2. Ada piksel MENDEKATI PUTIH di dalam kotak balon (halaman di sudut kotak
       + kilau di dalam balon), jadi Otsu punya kelas terang untuk dipilih
       secara keliru sebagai interior.
    3. Art GELAP menempel di tepi balon. Setelah polaritas dibalik, garis balon
       masuk kelas yang sama dengan interior kelabu, jadi flood fill bisa
       menembus garis dan mengisi art — inilah yang dijaga dinding gelap
       (`_WALL_MAD`/`_WALL_MIN`) di `_interior_from_crop`.

    Returns:
        (clean, img, inner, regions) — sama seperti make_double_bubble_page().
    """
    W, H = 900, 620
    page = Image.new("RGB", (W, H), (250, 249, 247))
    d = ImageDraw.Draw(page)
    for y in range(0, H, 9):
        d.line([(0, y), (W, y)], fill=(214, 214, 214), width=1)
    arr = np.asarray(page, np.uint8).copy()

    # Art gelap yang MENEMPEL di tepi balon (rambut) — sasaran uji dinding.
    for x0 in (150, 190, 700, 745):
        cv2.line(arr, (x0, 40), (x0 + 60, H - 40), (18, 18, 20), 9)

    lobes = ((300, 300, 205, 158), (610, 300, 205, 158))
    fill = np.zeros((H, W), np.uint8)
    for cx, cy, ax, ay in lobes:
        cv2.ellipse(fill, (cx, cy), (ax, ay), 0, 0, 360, 255, -1)
    k = 2 * _DB_STROKE + 1
    inner = cv2.erode(fill, np.ones((k, k), np.uint8))

    # Screentone kelabu: dua nilai berpola, jadi MAD cincin > 0 dan dinding
    # gelap tidak boleh memotong balonnya sendiri.
    yy, xx = np.mgrid[0:H, 0:W]
    tone = np.where(((xx // 3) + (yy // 3)) % 2 == 0, 148, 132).astype(np.uint8)
    grey = np.dstack([tone] * 3)
    arr[fill > 0] = grey[fill > 0]
    # Kilau hampir-putih DI DALAM balon: memberi Otsu kelas terang yang menggoda.
    for cx, cy, _, _ in lobes:
        cv2.ellipse(arr, (cx - 96, cy - 96), (44, 26), 0, 0, 360, (246, 246, 246), -1)
    arr[(fill > 0) & (inner == 0)] = (0, 0, 0)

    clean = arr.copy()
    page = Image.fromarray(arr)
    d = ImageDraw.Draw(page)
    f = _jp_font(34)
    for (cx, _, _, _), rows in zip(lobes, (("それとも", "全てを"), ("生涯で", "最も"))):
        for i, row in enumerate(rows):
            d.text((cx, 278 + i * 44), row, font=f, fill=(0, 0, 0), anchor="mm")

    ys, xs = np.nonzero(fill)
    shared = (int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1)
    regions = [
        Region(idx=0, bbox=(212, 255, 390, 345), det_class="text_bubble",
               bubble_bbox=shared),
        Region(idx=1, bbox=(522, 255, 700, 345), det_class="text_bubble",
               bubble_bbox=shared),
    ]
    return clean, np.asarray(page, dtype=np.uint8), inner, regions


def _page_mask(r: Region, shape: tuple[int, int]) -> np.ndarray:
    """Interior balon region, ditempel ke kanvas seukuran halaman."""
    (bx1, by1, _, _), mask = typeset._region_box_mask(r)
    h, w = shape
    out = np.zeros((h, w), np.uint8)
    mh, mw = mask.shape[:2]
    by2, bx2 = min(by1 + mh, h), min(bx1 + mw, w)
    if by2 > by1 and bx2 > bx1:
        out[by1:by2, bx1:bx2] = mask[: by2 - by1, : bx2 - bx1]
    return out


def _ink_of(page: np.ndarray, base: np.ndarray) -> np.ndarray:
    """Piksel yang berubah terhadap halaman bersih = tinta terjemahan."""
    diff = np.abs(page.astype(np.int16) - base.astype(np.int16)).sum(axis=2)
    return diff > 120


def run(verbose: bool = True) -> bool:
    """Assert kontrak pipeline pada halaman sintetis. Tanpa GPU, tanpa API."""
    img, boxes = make_test_page()
    checks: list[tuple[str, bool, str]] = []

    regions = [
        Region(idx=i, bbox=b, det_class="text_bubble" if i < 3 else "text_free",
               bubble_bbox=b if i < 3 else None)
        for i, b in enumerate(boxes)
    ]
    for r in regions:
        textmask.build_region_mask(img, r, None)

    # Double bubble: satu kotak balon berisi dua region -> dibelah per region.
    _bb = (100, 300, 700, 620)
    _rA = Region(idx=10, bbox=(140, 360, 360, 420), det_class="text_bubble",
                 bubble_bbox=_bb)
    _rB = Region(idx=11, bbox=(440, 360, 660, 420), det_class="text_bubble",
                 bubble_bbox=_bb)
    detect._partition_shared_bubbles([_rA, _rB])
    checks.append((
        "double bubble dibelah per region",
        _rA.bubble_bbox is not None and _rB.bubble_bbox is not None
        and _rA.bubble_bbox != _rB.bubble_bbox
        and _rA.bubble_bbox[2] <= _rB.bubble_bbox[0],
        f"A={_rA.bubble_bbox} B={_rB.bubble_bbox}",
    ))

    # SATU blok teks terdeteksi DUA KALI: kotak kecil bersarang di kotak besar.
    # Bukan sintetis — bbox di bawah diambil apa adanya dari halaman
    # hitomi_3740721_015 (r0 di dalam r1, containment 0.974 tapi IoU cuma 0.280
    # jadi NMS kelompok teks melewatkannya). Kalau dibiarkan, KEDUANYA dapat
    # kotak balon yang sama, _partition_shared_bubbles menyangkanya balon ganda
    # dan MEMBELAH balonnya di x=956 — tiap belahan lalu mengukur fill_color dan
    # warna hurufnya sendiri, jadi satu balon keluar berjahitan dua warna dengan
    # kalimat yang sama tercetak dua kali. Lihat detect.drop_nested_duplicates().
    _dup = [Region(idx=0, bbox=(944, 130, 1024, 321), det_class="text_bubble"),
            Region(idx=1, bbox=(832, 135, 1027, 405), det_class="text_bubble")]
    _ndup = detect.drop_nested_duplicates(_dup)
    checks.append((
        "duplikat bersarang dibuang, yang bertahan dilebarkan ke gabungan",
        _ndup == 1 and len(_dup) == 1
        and _dup[0].bbox == (832, 130, 1027, 405),
        f"dibuang={_ndup} sisa={[r.bbox for r in _dup]}",
    ))
    # Sisi lain gerbangnya, dan yang lebih penting: kotak besar yang benar-benar
    # memuat DUA kotak kecil adalah balon ganda sungguhan, dan untuk kasus itu
    # _partition_shared_bubbles + partition_shared_interiors sudah benar. Kalau
    # penyingkiran di atas ikut menyalak di sini, jalur balon ganda yang bekerja
    # justru dirusak. Lobus BERJAJAR juga tidak boleh disentuh — containment-nya
    # rendah (tertinggi 0.33 pada halaman bersih yang diukur).
    _nd2 = [Region(idx=0, bbox=(100, 100, 300, 400), det_class="text_bubble"),
            Region(idx=1, bbox=(110, 110, 190, 390), det_class="text_bubble"),
            Region(idx=2, bbox=(210, 110, 290, 390), det_class="text_bubble")]
    _nd3 = [Region(idx=0, bbox=(100, 100, 205, 400), det_class="text_bubble"),
            Region(idx=1, bbox=(195, 100, 300, 400), det_class="text_bubble")]
    _k2, _k3 = (detect.drop_nested_duplicates(_nd2),
                detect.drop_nested_duplicates(_nd3))
    checks.append((
        "balon ganda & lobus berjajar TIDAK ikut dibuang",
        _k2 == 0 and _k3 == 0 and len(_nd2) == 3 and len(_nd3) == 2
        and _nd2[0].bbox == (100, 100, 300, 400),
        f"dua_lobus_bersarang={_k2} berjajar={_k3}",
    ))

    # Induk dipilih dari CAKUPAN dulu, area cuma pemutus seri. Angka di bawah
    # apa adanya dari hitomi_3740721_015: kotak lobus kiri lebih KECIL tapi hanya
    # memuat 0.677 teksnya, jadi aturan "terkecil yang memuat mayoritas" memilih
    # dia dan lobus kanan tidak pernah masuk interior — tinta Jepang di sana
    # tidak terhapus. Lihat detect._PARENT_SLACK.
    _pr = Region(idx=0, bbox=(832, 130, 1027, 405), det_class="text_bubble")
    detect.assign_bubbles([_pr], [(800, 117, 964, 440),      # lobus, cover 0.677
                                  (800, 96, 1046, 442)])     # balon penuh, 1.000
    checks.append((
        "induk = balon yang memuat teks utuh, bukan kotak lobus yang lebih kecil",
        _pr.bubble_bbox == (800, 96, 1046, 442),
        f"terpilih={_pr.bubble_bbox}",
    ))
    # Sisi lain pitanya: kalau teks cuma menonjol beberapa piksel keluar lobus,
    # lobus HARUS tetap menang — kalau tidak, tiap region balon ganda akan
    # memilih kotak gabungan dan kedua terjemahan kembali bertumpuk di tengah.
    _pr2 = Region(idx=0, bbox=(10, 10, 110, 110), det_class="text_bubble")
    detect.assign_bubbles([_pr2], [(10, 10, 107, 110),        # lobus, cover 0.97
                                   (0, 0, 200, 200)])         # gabungan, 1.000
    checks.append((
        "tonjolan beberapa piksel tidak memindahkan induk ke kotak gabungan",
        _pr2.bubble_bbox == (10, 10, 107, 110),
        f"terpilih={_pr2.bubble_bbox}",
    ))

    # Balon figura-8 sungguhan: belahan persegi di atas TIDAK cukup, interiornya
    # harus dipartisi mengikuti bentuk lobus. Lihat make_double_bubble_page().
    _dclean, _dimg, _dinner, _dbl = make_double_bubble_page()
    detect._partition_shared_bubbles(_dbl)
    for _r in _dbl:
        textmask.build_region_mask(_dimg, _r, None)
    _split = textmask.partition_shared_interiors(_dimg, _dbl)
    _shape = _dinner.shape
    _mA, _mB = _page_mask(_dbl[0], _shape), _page_mask(_dbl[1], _shape)
    _ov = int(((_mA > 0) & (_mB > 0)).sum())
    checks.append((
        "interior balon figura-8 dipartisi per lobus & disjoint",
        _split == 2 and _mA.any() and _mB.any() and _ov == 0,
        f"dipartisi={_split} overlap_px={_ov} A={int((_mA>0).sum())} "
        f"B={int((_mB>0).sum())}",
    ))
    _dbw = _dbl[0].shared_bubble_bbox[2] - _dbl[0].shared_bubble_bbox[0]
    _cA = _dbl[0].bubble_bbox[0] + typeset._centroid(_dbl[0].bubble_mask)[0]
    _cB = _dbl[1].bubble_bbox[0] + typeset._centroid(_dbl[1].bubble_mask)[0]
    checks.append((
        "centroid dua lobus terpisah >= 40% lebar balon",
        abs(_cB - _cA) >= _dbw * 0.40,
        f"dx={abs(_cB - _cA)} lebar_balon={_dbw}",
    ))

    # Balon BERTETANGGA (bukan figura-8): shared_bubble_bbox None, jadi
    # partition_shared_interiors() tidak jalan dan yang harus menyelamatkan
    # adalah disjoin_overlapping_interiors(). Ini kasus 5994 px di halaman nyata.
    _aclean, _aimg, _ainner, _adj = make_adjacent_bubbles_page()
    detect._partition_shared_bubbles(_adj)          # tidak boleh mengubah apa pun
    for _r in _adj:
        textmask.build_region_mask(_aimg, _r, None)
    _ashape = _ainner.shape
    _ov0 = int(((_page_mask(_adj[0], _ashape) > 0)
                & (_page_mask(_adj[1], _ashape) > 0)).sum())
    _afix = textmask.disjoin_overlapping_interiors(_aimg, _adj)
    _amA, _amB = _page_mask(_adj[0], _ashape), _page_mask(_adj[1], _ashape)
    _ov1 = int(((_amA > 0) & (_amB > 0)).sum())
    checks.append((
        "balon bertetangga: interior beririsan SEBELUM diperbaiki (test valid)",
        _ov0 > 0 and all(r.shared_bubble_bbox is None for r in _adj),
        f"overlap_awal={_ov0} shared={[r.shared_bubble_bbox for r in _adj]}",
    ))
    checks.append((
        "balon bertetangga: interior dibuat disjoint tanpa mengosongkan balon",
        _ov1 == 0 and _amA.any() and _amB.any() and _afix >= 1,
        f"overlap={_ov1} diperbaiki={_afix} A={int((_amA>0).sum())} "
        f"B={int((_amB>0).sum())}",
    ))
    # Menyusut hanya boleh MEMBUANG piksel di zona sengketa — kalau bisa
    # menambah, cacat 'teks keluar balon' bisa muncul lewat pintu ini.
    checks.append((
        "disjoin tidak pernah menambah piksel di luar interior balon",
        int(((_amA > 0) & (_ainner == 0)).sum()) == 0
        and int(((_amB > 0) & (_ainner == 0)).sum()) == 0,
        f"A_luar={int(((_amA>0)&(_ainner==0)).sum())} "
        f"B_luar={int(((_amB>0)&(_ainner==0)).sum())}",
    ))

    # Balon KELABU ber-screentone: kedua halaman balon ganda di atas memakai
    # balon PUTIH, jadi tidak satu pun menyentuh aturan polaritas. Di sini
    # median kelas mayoritas ada DI ATAS 128 (jadi aturan absolut lama TIDAK
    # membalik polaritas) padahal interiornya kelabu, bukan putih. Terukur di
    # _cngrey.py pada halaman ini: cakupan tinta aturan lama 0.000 dengan 0.828
    # piksel interior jatuh DI LUAR balon (itu halaman putih di sudut kotak),
    # aturan cincin 1.000 dan 0.000. Dua angka itulah dua cacat cacatbaru:
    # tinta Jepang tak terhapus (build_fill_mask memakai interior ini) dan
    # terjemahan ditata di sliver luar balon.
    _gclean, _gimg, _ginner, _grey = make_grey_bubble_page()
    for _r in _grey:
        textmask.build_region_mask(_gimg, _r, None)
    _gshape = _ginner.shape
    _gtinta, _gluar = [], []
    for _r in _grey:
        _gm = _page_mask(_r, _gshape)
        _gsel = _gm > 0
        _gluar.append(0.0 if not _gsel.any()
                      else float((_ginner[_gsel] == 0).mean()))
        _gx1, _gy1, _gx2, _gy2 = _r.bbox
        _gink = np.zeros(_gshape, np.uint8)
        _gink[_gy1:_gy2, _gx1:_gx2] = _r.ink_mask[: _gy2 - _gy1, : _gx2 - _gx1]
        _gtinta.append(-1.0 if not _gink.any()
                       else float((_gm[_gink > 0] > 0).mean()))
    checks.append((
        "balon kelabu: interior memuat tinta region-nya sendiri",
        all(t >= 0.90 for t in _gtinta),
        f"cakupan_tinta={[round(t, 3) for t in _gtinta]}",
    ))
    checks.append((
        "balon kelabu: interior tidak keluar balon (art gelap di tepi utuh)",
        all(l <= 0.02 for l in _gluar),
        f"fraksi_di_luar={[round(l, 3) for l in _gluar]}",
    ))
    # Alasan STRUKTURAL-nya diuji langsung, supaya kalau halaman uji berubah
    # bentuk pun aturannya tetap terjaga: polaritas ditentukan cincin latar
    # tinta, bukan kecerahan absolut. Kelas mayoritas di sini median > 128,
    # jadi aturan lama menjawab False dan aturan cincin harus menjawab True.
    _gr = _grey[0]
    _gb = _gr.bubble_bbox
    _gcrop = _gimg[_gb[1]:_gb[3], _gb[0]:_gb[2]]
    _ggray = cv2.cvtColor(_gcrop, cv2.COLOR_RGB2GRAY)
    _, _gbinv = cv2.threshold(_ggray, 0, 255,
                              cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _gvals = _ggray[_gbinv > 0]
    if _gvals.size < _gbinv.size - _gvals.size:
        _gvals = _ggray[_gbinv == 0]
    _gmay = float(np.median(_gvals))
    _gink_crop = textmask._ink_in_crop(_gr, _gb[0], _gb[1], _gcrop.shape[:2])
    _gbalik, _, _ = textmask._polarity_ring(_ggray, _gbinv, _gink_crop)
    checks.append((
        "polaritas dari cincin tinta, bukan ambang absolut",
        _gmay >= 128 and _gbalik,
        f"median_kelas_mayoritas={_gmay:.1f} (aturan_lama_balik="
        f"{_gmay < 128}) aturan_cincin_balik={_gbalik}",
    ))

    checks.append((
        "mask terbentuk di semua region",
        all(r.ink_mask is not None and r.ink_mask.any() for r in regions),
        "",
    ))
    checks.append((
        "estimasi font size masuk akal (8..120 px)",
        all(8 <= r.est_font_size <= 120 for r in regions),
        str([round(r.est_font_size, 1) for r in regions]),
    ))

    # Regresi 'SFX diterjemahkan' — heuristik baru di translate.py.
    # Kasus nyata yang bocor di heuristik lama (kana murni <=3/ABAB-4):
    #   "フー．．．" kana+simbol diterjemah, "ぴくぴくっ" ABAB+っ,
    #   "ドキドキドキ" ulangan 6 char, "ドキッ" SFX dalam balon.
    # Dan yang TIDAK boleh terkunci: それは (narasi), ちょっと, サッカー
    # (pinjaman ッ…ー), こんにちは (dialog).
    #
    # Arah kedua ditambahkan setelah cacat "balon pendek tidak diterjemah":
    # cabang lama di dalam balon `n == 3 and has_small` mengunci seruan yang
    # jelas ucapan jadi SFX, dan SFX = translation None + PROTECTED = balon
    # Jepang tercetak TANPA satu pun pesan error. _label3.py mengukur cabang
    # itu pada 72 kasus (45 dialog + 27 SFX): arah dialog 30 salah, arah SFX 4
    # salah. Yang di bawah wakilnya, dipilih supaya tiap bagian gerbang baru
    # punya penjaga:
    #   hiragana ber-っ (ええっ うんっ まてっ)      -> DIALOGUE
    #   ucapan ditulis katakana (ダメッ オイッ)     -> DIALOGUE lewat _kata2hira
    #   seluruhnya katakana, ッ di UJUNG (ハッ)     -> SFX (サッカー tidak: ッ
    #                                                  di tengah, ujungnya ー)
    #   hiragana yang batang-gandanya di kamus      -> SFX (どきっ びくっ)
    import translate as _tl
    _sfx_cases = [
        ("フー．．．", False, "SFX"), ("ピクッ", False, "SFX"),
        ("ーー", False, "SFX"),
        ("ドキドキドキ", False, "SFX"), ("ぴくぴくっ", False, "SFX"),
        ("ガッタンゴットン", False, "SFX"), ("ドキッ", True, "SFX"),
        ("はぁっ", True, "SFX"),
        ("それは．．．", False, "DIALOGUE"), ("ちょっと", False, "DIALOGUE"),
        ("サッカー", True, "DIALOGUE"), ("こんにちは", True, "DIALOGUE"),
        ("うう．．．", True, "DIALOGUE"), ("んっ", True, "DIALOGUE"),
        # seruan hiragana di dalam balon: WAJIB diterjemah
        ("ええっ", True, "DIALOGUE"), ("ええっ！？", True, "DIALOGUE"),
        ("うんっ", True, "DIALOGUE"), ("だめっ", True, "DIALOGUE"),
        ("いやっ", True, "DIALOGUE"), ("まてっ", True, "DIALOGUE"),
        ("うそっ", True, "DIALOGUE"), ("なにっ", True, "DIALOGUE"),
        ("ちょっ", True, "DIALOGUE"), ("そこっ", True, "DIALOGUE"),
        ("はいっ", True, "DIALOGUE"), ("ねえっ", True, "DIALOGUE"),
        ("もうっ", True, "DIALOGUE"), ("やめっ", True, "DIALOGUE"),
        ("うわっ", True, "DIALOGUE"), ("やだっ", True, "DIALOGUE"),
        ("あーっ", True, "DIALOGUE"), ("ふぇっ", True, "DIALOGUE"),
        # ucapan yang DITULIS KATAKANA — tanpa normalisasi kana semuanya SFX
        ("ダメッ", True, "DIALOGUE"), ("ハイッ", True, "DIALOGUE"),
        ("ウンッ", True, "DIALOGUE"), ("ムリッ", True, "DIALOGUE"),
        ("オイッ", True, "DIALOGUE"), ("マテッ", True, "DIALOGUE"),
        ("ナニッ", True, "DIALOGUE"), ("ウソッ", True, "DIALOGUE"),
        ("イヤッ", True, "DIALOGUE"), ("ヤダッ", True, "DIALOGUE"),
        ("ヤメッ", True, "DIALOGUE"),
        # arah sebaliknya tidak boleh ikut longgar
        ("ハッ", True, "SFX"), ("ズドンッ", True, "SFX"),
        ("ガシャッ", True, "SFX"), ("パチンッ", True, "SFX"),
        ("ゴクッ", True, "SFX"), ("どきっ", True, "SFX"),
        ("びくっ", True, "SFX"), ("ぴくっ", True, "SFX"),
        ("ごくっ", True, "SFX"), ("がくっ", True, "SFX"),
        ("ふわっ", True, "SFX"), ("どきどき", True, "SFX"),
        ("はぁはぁ", True, "SFX"), ("ぐちゅぐちゅ", True, "SFX"),
        # Arah ketiga, dari hasilnew5: satu balon dari 22 halaman keluar tetap
        # Jepang — 'ヒ．．．ッ！？' di balon hitam bergerigi. Aturan K
        # (<=6 kana, seluruhnya katakana, ッ di ujung) mengunci SFX, dan SFX =
        # PROTECTED = translate_page melewatinya tanpa pesan error.
        # Yang menyulitkan: ハッ di atas WAJIB tetap SFX dan strukturnya
        # IDENTIK (satu mora katakana + ッ, di dalam balon), jadi panjang
        # batang tidak bisa memisahkannya. Pemisahnya tanda BICARA di teks
        # ASLI — yang justru dibuang _sfx_core. _h5lbl.py mengukur 5 kandidat
        # atas 60 kasus di atas + 14 di bawah; hanya aturan V yang nol
        # kesalahan dua arah.
        ("ヒ．．．ッ！？", True, "DIALOGUE"),
        ("ヒッ！？", True, "DIALOGUE"), ("ヒ．．．ッ", True, "DIALOGUE"),
        ("ア．．．ッ", True, "DIALOGUE"), ("ウ．．．ッ！？", True, "DIALOGUE"),
        ("キャ．．．ッ", True, "DIALOGUE"),
        # ...dan yang TIDAK boleh ikut longgar karena V:
        ("ヒ．．．ッ！？", False, "SFX"),   # teks SAMA di LUAR balon = bunyi latar
        ("ヒッ", True, "SFX"),            # tanpa tanda bicara: tetap bunyi
        ("ドキッ！", True, "SFX"), ("ドキッ！？", True, "SFX"),
        ("ゴクッ．．．", True, "SFX"),      # jeda di UJUNG, bukan di ANTARA kana
        ("ガシャッ．．．", True, "SFX"),
        # ズドンッ！ / パチンッ！ menahan kandidat '？ atau ！': batang keduanya
        # TIDAK berkamus (ずどんずどん, ぱちんぱちん tidak ada), jadi aturan S
        # tidak menyelamatkannya dan mereka akan jatuh jadi DIALOGUE.
        ("ズドンッ！", True, "SFX"), ("パチンッ！", True, "SFX"),
    ]
    _sfx_bad = []
    for _t, _ib, _want in _sfx_cases:
        _tr = Region(idx=99, bbox=(0, 0, 10, 10),
                     bubble_bbox=(0, 0, 10, 10) if _ib else None)
        _tr.src_text = _t
        _tl._label_region(_tr)
        if _tr.label != _want:
            _sfx_bad.append(f"{_t!r}->{_tr.label} (ingin {_want})")
    checks.append((
        "heuristik SFX baru (フー.., ドキドキドキ, サッカー..)",
        not _sfx_bad, "; ".join(_sfx_bad),
    ))
    # Penjaga langsung untuk pemisah aturan V: jeda DI ANTARA kana vs di UJUNG.
    # Kalau _PAUSE suatu saat dilebarkan sampai memuat ー (U+30FC), 'フー．．．'
    # dan 'ゴクッ．．．' ikut terjaring dan bunyi memanjang berhenti jadi SFX.
    _brk = _tl._broken_kana
    _brk_want = [
        ("ヒ．．．ッ", True), ("キャ．．．ッ", True), ("ア．．．ッ", True),
        ("フー．．．", False), ("ゴクッ．．．", False), ("ドキッ", False),
        ("ヒッ", False), ("ー．．．", False), ("ハッ", False),
    ]
    _brk_bad = [f"{_t!r}->{_brk(_t)} (ingin {_w})"
                for _t, _w in _brk_want if _brk(_t) != _w]
    checks.append((
        "jeda DI ANTARA kana dibedakan dari jeda di UJUNG",
        not _brk_bad, "; ".join(_brk_bad),
    ))
    # Cacat hasilnew5 apa adanya: balon r8 harus keluar dari PROTECTED, karena
    # PROTECTED = translate_page melewatinya = balon Jepang tercetak diam-diam.
    _r8 = Region(idx=8, bbox=(485, 1140, 525, 1237),
                 bubble_bbox=(474, 1121, 536, 1254))
    _r8.src_text = "ヒ．．．ッ！？"
    _tl._label_region(_r8)
    checks.append((
        "balon hitam hasilnew5 'ヒ．．．ッ！？' tidak lagi PROTECTED",
        _r8.label == "DIALOGUE" and _r8.label not in _tl.PROTECTED_LABELS,
        f"label={_r8.label} protected={_r8.label in _tl.PROTECTED_LABELS}",
    ))
    _sym_ok = _tl._restore_symbols("大好き♥", "I love you") == "I love you♥"
    checks.append((
        "simbol emosi dipulihkan setelah terjemahan", _sym_ok, "",
    ))

    # Balon yang isinya HANYA simbol tidak punya kata untuk diterjemahkan. Model
    # membalas kosong, jalur perbaikan menuduhnya "BELUM DITERJEMAHKAN", dan
    # balonnya tercetak HAMPA (final_font_size 0) — simbol yang memang tertulis
    # di halaman aslinya ikut hilang. Terukur di hitomi_3740721_015 r12='．．．'.
    _so_bad: list[str] = []
    for _t in ("．．．", "。。。", "！？", "♥", "♡", "♪", "☆", "〜", "～", "…",
               "・・・", "ー．．．", "？", "．．．♥"):
        if not _tl._symbols_only(_t):
            _so_bad.append(f"{_t!r} dianggap berkata")
    # ー dan ・ duduk di blok katakana tapi tidak pernah jadi kata sendirian;
    # ぁ, ん, ッ satu huruf pun TETAP kata. Dua arah harus benar.
    for _t in ("ぁ", "ん", "ッ", "な．．．", "そん．．．", "でも、", "俺も",
               "ヒ．．．ッ！？", "A", "1", "", "   "):
        if _tl._symbols_only(_t):
            _so_bad.append(f"{_t!r} dianggap simbol")
    checks.append((
        "balon simbol-saja dikenali (dan huruf tunggal TIDAK ikut terjaring)",
        not _so_bad, "; ".join(_so_bad),
    ))
    # _clean_translation() membuang tanda baca di AWAL string, dan pada balon
    # simbol-saja SELURUH isinya ada di awal — jadi ia mengembalikan string
    # kosong. Itulah sebabnya ada helper terpisah; pemeriksaan kedua di bawah
    # membuktikan jebakannya masih ada supaya helper ini tidak dianggap mubazir.
    _sa_bad = [f"{_s!r}->{_tl._symbols_as_text(_s)!r} (ingin {_w!r})"
               for _s, _w in (("．．．", "..."), ("。。。", "..."), ("！？", "!?"),
                              ("♥", "♥"), ("〜", "~"), ("…", "..."),
                              ("？", "?"), ("．．．♥", "...♥"))
               if _tl._symbols_as_text(_s) != _w]
    checks.append((
        "simbol dipetakan ke ASCII tanpa dikosongkan lstrip",
        not _sa_bad and _tl._clean_translation("．．．") == "",
        "; ".join(_sa_bad) or
        f"_clean_translation('．．．')={_tl._clean_translation('．．．')!r}",
    ))
    # Bukti bahwa helper di atas benar-benar TERSAMBUNG: satu halaman yang isinya
    # cuma balon simbol harus selesai TANPA menyentuh penyedia. client=object()
    # bukan RouterClient dan modelnya karangan, jadi kalau region ini sampai
    # masuk `items` pemeriksaan ini akan meledak, bukan lolos diam-diam.
    _spr = [Region(idx=0, bbox=(0, 0, 40, 60), det_class="text_bubble")]
    _spr[0].src_text = "．．．"
    try:
        _tl.translate_page(object(), "tidak-ada-model", _spr)
        _sp_err = ""
    except Exception as _e:                       # noqa: BLE001 - pesannya dilaporkan
        _sp_err = f"{type(_e).__name__}: {_e}"
    checks.append((
        "balon simbol-saja selesai tanpa memanggil penyedia",
        not _sp_err and _spr[0].translation == "...",
        _sp_err or f"translation={_spr[0].translation!r}",
    ))

    # Kurung sudut Jepang lolos apa adanya dari DeepL dan Anime Ace merendernya
    # jadi kotak tofu di dalam balon; ♥ ♪ ☆ justru WAJIB bertahan (plan.txt).
    # 〜 juga bertahan, tapi dinormalkan ke '~': wave dash lebar tidak ada di
    # Anime Ace sementara tilde ASCII ada, jadi bentuknya tetap tampil dan
    # dirender font balon yang sama — lihat _PUNCT_MAP.
    _ct = _tl._clean_translation
    checks.append((
        "punctuation CJK dibuang, simbol emosi bertahan",
        not any(c in _ct("「会長っ」") for c in "「」")
        and _ct("大好き♥").endswith("♥")
        and _ct("ずっと〜").endswith("~")
        and _ct("ずっと～").endswith("~")
        and _ct("　. シ ズ ク…") == "シ ズ ク..."
        # ＼…／ tanda penekanan Jepang: tidak ada di Anime Ace -> tofu di depan
        # baris pertama ('＼MY APOLO-GIES' di halaman referensi).
        and _ct("＼My apologies") == "My apologies",
        f"{_ct('「会長っ」')!r} {_ct('大好き♥')!r} {_ct('ずっと〜')!r} "
        f"{_ct('　. シ ズ ク…')!r} {_ct('＼My apologies')!r}",
    ))

    # Simbol emosi yang TIDAK ada di Anime Ace harus dirutekan ke font yang
    # benar-benar punya glyph-nya, bukan ke penampung umum. Sebelum diperbaiki
    # ♡ ♪ ♫ ☆ ★ semuanya jatuh ke NotoSans — satu-satunya font di rantai yang
    # tidak punya satu pun dari simbol itu — dan dirender jadi kotak tofu.
    typeset.setup_fonts(verbose=False)
    _main = typeset._font(typeset.FONT_USED, 16)
    _mcmap = typeset._cmap(typeset.FONT_USED)
    _tofu = []
    for _ch in "♡♪♫☆★❤~":
        _f = typeset._char_font(_ch, _main, _mcmap, 16)
        if ord(_ch) not in typeset._cmap(_f.path):
            _tofu.append(f"{_ch} -> {_f.path.replace(chr(92), '/').rsplit('/', 1)[-1]}")
    checks.append((
        "simbol emosi dapat font yang punya glyph-nya (bukan tofu)",
        not _tofu, "; ".join(_tofu),
    ))

    # Cek kedua, dan ini yang menangkap cacat hasilnew/6.JPG: "punya glyph"
    # TIDAK SAMA dengan "punya glyph yang benar". anime_ace.ttf MEMETAKAN U+2665
    # ke glyph bernama `yat` (huruf Cyrillic Ѣ), jadi cek tofu di atas lolos
    # sementara yang tergambar di balon bukan hati. Rantai fallback lama hanya
    # menyala kalau font utama tidak punya codepoint-nya, jadi ia tidak pernah
    # menyala untuk ♥ — lihat _FORCE_SYMBOL di typeset.py.
    _wrong_shape = []
    for _ch in "♥♡♪☆★":
        _f = typeset._char_font(_ch, _main, _mcmap, 16)
        if _f.path == typeset.FONT_USED:
            _wrong_shape.append(f"{_ch} masih dari font utama")
    checks.append((
        "simbol emosi TIDAK diambil dari font utama (anime_ace ♥ = huruf `yat`)",
        not _wrong_shape, "; ".join(_wrong_shape),
    ))

    # Nama glyph di font yang dipilih harus benar-benar simbol yang dimaksud.
    # Assertion paling langsung yang bisa dibuat tanpa membandingkan piksel ke
    # gambar acuan: nama glyph U+2665 di font terpilih bukan nama huruf.
    _gname = ""
    try:
        from fontTools.ttLib import TTFont as _TTF
        _hf = typeset._char_font("♥", _main, _mcmap, 16)
        _gname = _TTF(_hf.path, fontNumber=0, lazy=True).getBestCmap()[0x2665]
    except Exception:  # noqa: BLE001 - fontTools opsional
        _gname = "heart"  # tanpa fontTools cek ini tidak berarti; jangan gagal
    checks.append((
        "glyph U+2665 di font terpilih bernama hati, bukan huruf",
        "heart" in _gname.lower() or _gname.startswith(("uni2665", "cid")),
        f"nama glyph = {_gname!r}",
    ))

    # Jalur UKUR dan jalur GAMBAR wajib memakai font yang sama per karakter.
    # Kalau _measure() memakai font utama sementara _draw_line() jatuh ke font
    # simbol, barisnya diukur salah: ☆ 14.0 px di anime_ace vs 21.0 px di
    # NotoSansSymbols2 pada size 20 — ukur-kurang 7 px yang lolos fit() lalu
    # melebar keluar balon saat digambar.
    _mismatch = []
    for _line in ("I LOVE YOU ♥", "WAIT ☆", "LA LA ♪", "PLAIN TEXT"):
        _a = typeset._measure(_line, _main)
        _b = typeset._line_width(_line, _main, _mcmap, 16)
        if abs(_a - _b) > 0.01:
            _mismatch.append(f"{_line!r} ukur={_a:.1f} gambar={_b:.1f}")
    checks.append((
        "lebar jalur ukur == lebar jalur gambar (simbol ikut dihitung)",
        not _mismatch, "; ".join(_mismatch),
    ))

    # ------------------------------------------------------------- condense
    #
    # Rapat horizontal (SETTINGS.condense) menutup selisih kerapatan TERUKUR
    # antara Anime Ace dan font typeset CONTOH/6.JPG (0.690; probe_reffont2.py),
    # dan angka 0.85 dipilih dari sapuan pada mask jp_6 sungguhan
    # (probe_cond.py: wording referensi 5 -> 1 tanda hubung, luber 1 -> 0).
    #
    # Yang diuji di sini bukan angkanya melainkan KESETIAANNYA: jalur ukur dan
    # jalur gambar harus memampatkan dengan faktor yang sama. Kalau tidak, fit()
    # menyangka baris muat lalu tintanya tergambar lebih lebar dan menembus garis
    # balon — cacat 'keluar bubble' yang dilarang plan.txt, dan cacat yang paling
    # sulit dilihat karena selisihnya cuma beberapa piksel per baris.
    _cnd = typeset._cond()
    checks.append((
        "faktor condense terbaca dan masuk rentang wajar",
        0.3 <= _cnd <= 1.0 and abs(_cnd - float(SETTINGS.condense)) < 1e-9,
        f"condense={_cnd}",
    ))
    # Tinta yang BENAR-BENAR tergambar diukur dari piksel, lewat transform yang
    # sama dengan render_region() — bukan dari rumus yang sama, kalau tidak yang
    # diuji cuma aritmetika terhadap dirinya sendiri.
    _cbad = []
    for _txt, _sz in (("WONDER", 12), ("EMBARASSING", 14), ("I'M PRAISING", 20),
                      ("LOVE YOU ♥", 16)):
        _f = typeset._font(typeset.FONT_USED, _sz)
        _w = typeset._line_width(_txt, _f, _mcmap, _sz)
        _lh = typeset._line_height(_f)
        _k = SETTINGS.oblique
        _pd = int(abs(_k) * _lh) + 4
        _tile = Image.new("RGBA", (int(_w / _cnd) + _pd * 2, _lh + _pd * 2),
                          (0, 0, 0, 0))
        typeset._draw_line(ImageDraw.Draw(_tile), (_pd, _pd), _txt, _f,
                           (0, 0, 0), _mcmap, _sz, 0)
        _th = _tile.height
        _tile = _tile.transform(
            _tile.size, Image.AFFINE,
            (1 / _cnd, _k / _cnd, _pd - (_pd + _k * _th / 2) / _cnd, 0, 1, 0),
            resample=Image.BICUBIC)
        _cols = np.where((np.asarray(_tile.getchannel("A")) > 24).any(axis=0))[0]
        _ink = int(_cols[-1] - _cols[0] + 1) if _cols.size else 0
        # Tinta selalu lebih SEMPIT dari advance (advance memuat side bearing
        # kanan), jadi yang dijaga: jangan pernah MELEBIHI lebar yang diukur, dan
        # jangan menyusut lebih dari 25% — menyusut jauh berarti transform-nya
        # memampatkan dua kali.
        if not (_w * 0.75 <= _ink <= _w + 2):
            _cbad.append(f"{_txt!r}@{_sz} ukur={_w:.1f} tinta={_ink}")
    checks.append((
        "condense: lebar tinta tergambar cocok dengan lebar yang diukur",
        not _cbad, "; ".join(_cbad),
    ))
    # Faktor harus benar-benar MERAPATKAN, bukan cuma dikalikan di satu sisi.
    # Perbandingannya terhadap getlength() mentah, satu-satunya angka di sini
    # yang tidak lewat _cond().
    _raw = typeset._font(typeset.FONT_USED, 20).getlength("EMBARASSING")
    _got = typeset._line_width("EMBARASSING",
                               typeset._font(typeset.FONT_USED, 20), _mcmap, 20)
    checks.append((
        "condense benar-benar mempersempit baris (bukan no-op)",
        abs(_got - _raw * _cnd) < 0.01 and (_cnd == 1.0 or _got < _raw),
        f"mentah={_raw:.1f} rapat={_got:.1f} faktor={_cnd}",
    ))

    # Label seolah dari LLM: region terakhir SFX. Panjang teks sengaja beda-beda
    # supaya jumlah baris tiap balon beda — yang diuji nanti bukan keseragaman
    # ukuran mentah (balonnya memang beda besar) melainkan keseragaman RASIO
    # ukuran terhadap balonnya, lihat assertion 'ukuran font proporsional'.
    _texts = [
        "HELLO THERE",
        "SO THIS IS WHERE YOU WERE ALL ALONG",
        "THAT NIGHT, THE TWO OF THEM WERE TOGETHER.",
    ]
    for r, t in zip(regions[:3], _texts):
        r.label, r.translation = "DIALOGUE", t
    regions[3].label = "SFX"

    erase_mask, protected_mask = textmask.compose_page_mask(img, regions)
    checks.append((
        "SFX tidak tersentuh mask hapus",
        verify.assert_sfx_intact(erase_mask, protected_mask),
        "",
    ))
    checks.append(("mask SFX tidak kosong", bool(protected_mask.any()), ""))

    cleaned = erase.erase_page(img, regions, "cpu")
    sfx_box = boxes[3]
    sx1, sy1, sx2, sy2 = sfx_box
    untouched = np.array_equal(img[sy1:sy2, sx1:sx2], cleaned[sy1:sy2, sx1:sx2])
    checks.append(("piksel SFX identik sebelum/sesudah erase", untouched, ""))

    resid = [verify.pixel_residue(cleaned, r) for r in regions[:3]]
    checks.append((
        "tidak ada residu piksel di region dialog",
        all(v < 60 for v in resid),
        str(resid),
    ))

    if typeset.FONT_USED:
        out = typeset.render_page(cleaned, regions)

        # Regresi 'saling timpa': teks panjang dikunci di dalam balon.
        _lr = Region(idx=12, bbox=(120, 60, 380, 220), det_class="text_bubble",
                     bubble_bbox=(120, 60, 380, 220))
        textmask.build_region_mask(img, _lr, None)
        _lr.translation = ("THIS IS A VERY LONG DIALOGUE THAT KEEPS GOING AND "
                           "GOING WITH NO END IN SIGHT AT ALL WHATSOEVER") * 3
        _lout = typeset.render_page(cleaned, [_lr])
        _ld = np.abs(_lout.astype(np.int16) - cleaned.astype(np.int16)).sum(axis=2)
        _bx1, _by1, _bx2, _by2 = _lr.bubble_bbox
        _mh, _mw = _lr.bubble_mask.shape[:2]
        _pmap = np.zeros_like(cleaned[..., 0], np.uint8)
        _pmap[_by1:_by1 + _mh, _bx1:_bx1 + _mw] = _lr.bubble_mask
        _leaked = int(((_ld > 120) & (_pmap == 0)).sum())
        checks.append((
            "teks panjang tidak bocor keluar balon",
            _leaked == 0, f"leaked_px={_leaked}",
        ))
        inside = all(
            r.final_font_size >= 8 and r.lines for r in regions[:3]
        )
        checks.append(("teks hasil fit dirender di semua bubble", inside, ""))
        checks.append((
            "tidak ada overflow",
            not any(r.overflowed for r in regions[:3]),
            "",
        ))
        _fs = [r.final_font_size for r in regions[:3] if r.final_font_size]
        # Kontrak ukuran font BUKAN 'satu angka untuk seluruh halaman'. Typeset
        # referensi CONTOH/2.webp diukur (probe_refnative.py, 13 balon) dan
        # ternyata MENSKALAKAN teks ke besar balon: cap_height/sisi-terpendek
        # interior konstan 0.117 sementara cap_height sendiri berkisar 13..27 px
        # (sebaran 2.08x). Tiga model diuji terhadap ukuran terukur itu
        # (probe_model.py): seragam-halaman galat 4.31 px, proporsional balon
        # 2.71 px, per panel 4.19 px. Jadi yang di-assert adalah RASIO-nya yang
        # seragam, bukan ukurannya — dan tiap region duduk di plafon
        # proporsionalnya sendiri, tidak lebih kecil.
        #
        # Tiga balon halaman uji ini sisi terpendeknya 250/270/140 px, jadi
        # ukuran mentahnya memang wajib beda ~1.9x. Assertion lama (max/min
        # <= 1.35) lolos hanya karena kebetulan: dulu ketiga region jatuh ke satu
        # ukuran seragam. Angkanya bukan bukti benar, cuma bukti seragam.
        _norm, _gap = [], []
        for _r in regions[:3]:
            if not _r.final_font_size:
                continue
            _m = typeset._region_box_mask(_r)[1]
            _norm.append(_r.final_font_size / max(min(_m.shape[:2]), 1))
            _gap.append(typeset.region_font_cap(_m) - _r.final_font_size)
        _nspread = max(_norm) / max(min(_norm), 1e-6) if _norm else 0.0
        checks.append((
            "ukuran font proporsional ke balon (rasio seragam, duduk di plafon)",
            len(_norm) == 3 and _nspread <= 1.15 and max(_gap) <= 1,
            f"sizes={_fs} rasio_spread={_nspread:.2f} selisih_plafon={_gap}",
        ))
        checks.append((
            "tidak ada tanda hubung buatan di hasil fit",
            not any(ln.endswith("-") for r in regions[:3] for ln in r.lines),
            str([ln for r in regions[:3] for ln in r.lines if ln.endswith("-")]),
        ))

        # ------------------------------------------------- reclaim lebar terpakai
        #
        # disjoin_overlapping_interiors() memutus irisan per PIKSEL secara
        # Voronoi, tanpa tahu di baris mana teks tetangga benar-benar jatuh.
        # Akibatnya lebar disandera di ketinggian yang tetangganya tidak sentuh:
        # di hasilnew/jp_6.JPG r3 kehilangan 20 px tetap di y=139..191 padahal
        # tinta r2 berhenti di y=168 (probe_row.py), dan 26 px sisanya membuat
        # 'WONDER' (32 px pada size 6) dipenggal jadi 'WON-/DER'.
        #
        # Wordingnya bukan pilihan bebas. 'MISUNDERSTANDING AGAIN, PREZ?' dicari
        # dengan probe_adjfind.py justru karena tanda hubungnya SEBAB LEBAR: di
        # interior hasil disjoin 2 tanda hubung, di interior + 9522 px sanderaan
        # tinggal 1, ukurannya tetap 40. Kata mustahil semacam
        # 'PNEUMONOULTRAMICROSCOPIC...' tidak bisa dipakai — tanda hubungnya tetap
        # ada berapa pun lebarnya, jadi test-nya lolos/gagal tanpa hubungan dengan
        # kode yang diuji. r1 dijaga pendek supaya perannya jelas: pelepas.
        #
        # Yang diuji EMPAT kontrak, karena reclaim yang salah merusak salah satunya:
        #   1. tanda hubung pengklaim berkurang — bukan cuma luas bertambah.
        #      Luas naik itu murah dan menipu: piksel rampasan disjoin bergerigi,
        #      jadi bisa naik tanpa menambah RUN bebas yang dipakai satu baris.
        #   2. pelepas tidak dirugikan (tanda hubung/luber tidak muncul)
        #   3. interior tetap saling lepas — kalau tidak, teks saling timpa
        #   4. tidak ada piksel di luar interior balon SENDIRI (fill_mask), yaitu
        #      kontrak 'tidak keluar bubble'
        _adj[0].translation = "MISUNDERSTANDING AGAIN, PREZ?"
        _adj[1].translation = "YES, THE RECORDS ARE HERE."
        _afill = [typeset._paste_mask(_r.fill_bbox, _r.fill_mask, *_ashape) > 0
                  if _r.fill_mask is not None else np.zeros(_ashape, bool)
                  for _r in _adj]

        def _rfit(_r):
            """(ukuran, jumlah tanda hubung, luber) di interior _r sekarang."""
            _m = typeset._region_box_mask(_r)[1]
            _s, _ls, _sy, _ov = typeset.fit(_r.translation.upper(), _m,
                                            typeset.region_font_cap(_m),
                                            typeset.FONT_USED)
            return _s, sum(1 for _x in _ls if _x.endswith("-")), int(bool(_ov))

        _rf0 = [_rfit(_r) for _r in _adj]
        _amoved = typeset.reclaim_unused_interiors(_aimg, _adj)
        _rf1 = [_rfit(_r) for _r in _adj]
        _rmA = _page_mask(_adj[0], _ashape) > 0
        _rmB = _page_mask(_adj[1], _ashape) > 0
        checks.append((
            "reclaim: tanda hubung pengklaim berkurang setelah lebar dikembalikan",
            _amoved >= 1 and _rf1[0][1] < _rf0[0][1] and _rf1[0][0] >= _rf0[0][0],
            f"berubah={_amoved} region A:{_rf0[0]}->{_rf1[0]}",
        ))
        # Yardstick pelepas HARUS memakai ambang yang sama dengan produksi
        # (typeset._RECLAIM_LOSS), bukan angka px tetap. Versi pertama test ini
        # memakai '-1 px' dan gagal pada pertukaran yang justru benar: pengklaim
        # 2 tanda hubung -> 1, pelepas 40 -> 37 di balon 335x291 yang masih
        # lapang. Yang dijaga di sini bahwa pelepas tidak dapat cacat BARU
        # (tanda hubung/luber) dan tidak digunduli — bukan bahwa ukurannya beku.
        _rloss = max(1, int(round(_rf0[1][0] * typeset._RECLAIM_LOSS)))
        checks.append((
            "reclaim: yang melepas tidak dapat tanda hubung/luber baru",
            _rf1[1][1] <= _rf0[1][1] and _rf1[1][2] <= _rf0[1][2]
            and _rf1[1][0] >= _rf0[1][0] - _rloss,
            f"B:{_rf0[1]}->{_rf1[1]} batas_susut={_rloss}",
        ))
        checks.append((
            "reclaim: interior tetap saling lepas (tidak ada piksel milik dua region)",
            int((_rmA & _rmB).sum()) == 0, f"overlap={int((_rmA & _rmB).sum())}",
        ))
        checks.append((
            "reclaim: tidak ada piksel di luar interior balon sendiri",
            int((_rmA & ~_afill[0]).sum()) == 0 and int((_rmB & ~_afill[1]).sum()) == 0,
            f"A_luar={int((_rmA & ~_afill[0]).sum())} "
            f"B_luar={int((_rmB & ~_afill[1]).sum())}",
        ))
        # Dan hasil akhirnya: dirender sungguhan, tinta kedua region tidak boleh
        # bersinggungan satu piksel pun maupun keluar dari garis balon gabungan.
        _rink = [_ink_of(typeset.render_page(_aclean, [_r]), _aclean) for _r in _adj]
        checks.append((
            "reclaim: tinta hasil render tidak saling timpa & tetap di dalam balon",
            int((_rink[0] & _rink[1]).sum()) == 0
            and int((_rink[0] & (_ainner == 0)).sum()) == 0
            and int((_rink[1] & (_ainner == 0)).sum()) == 0,
            f"timpa={int((_rink[0] & _rink[1]).sum())} "
            f"A_luar={int((_rink[0] & (_ainner == 0)).sum())} "
            f"B_luar={int((_rink[1] & (_ainner == 0)).sum())}",
        ))

        # Kontrak inti balon ganda. Tiap lobus dirender SENDIRI ke halaman
        # bersih supaya tintanya bisa dipisahkan: kalau forb_map yang menahan
        # tumpang tindih (bukan mask yang sudah disjoint), test ini yang gagal.
        _dbl[0].translation = "I'VE BEEN LOOKING ALL OVER FOR YOU, PREZ!"
        _dbl[1].translation = "IS THAT THE SUMMARY FOR THE MILKING CLUB?"
        _dink = [_ink_of(typeset.render_page(_dclean, [_r]), _dclean) for _r in _dbl]
        _clash = int((_dink[0] & _dink[1]).sum())
        checks.append((
            "tinta dua lobus tidak saling timpa satu piksel pun",
            _clash == 0, f"clash_px={_clash}",
        ))
        # Toleransi 3 px: batas Voronoi kedua lobus BERSINGGUNGAN dan tepi alpha
        # di _clip_to_mask di-feather, jadi 1-2 px terluar memang jatuh sebelah.
        _e3 = np.ones((7, 7), np.uint8)
        _inAB = int((_dink[0] & (cv2.erode(_mB, _e3) > 0)).sum())
        _inBA = int((_dink[1] & (cv2.erode(_mA, _e3) > 0)).sum())
        checks.append((
            "tinta lobus A tidak masuk interior lobus B (dan sebaliknya)",
            _inAB == 0 and _inBA == 0, f"A->B={_inAB} B->A={_inBA}",
        ))
        _outA = int((_dink[0] & (_dinner == 0)).sum())
        _outB = int((_dink[1] & (_dinner == 0)).sum())
        checks.append((
            "tinta tidak menyentuh garis balon dan tidak keluar balon",
            _outA == 0 and _outB == 0, f"A={_outA} B={_outB}",
        ))
        checks.append((
            "SFX tidak ditimpa teks Inggris",
            np.array_equal(cleaned[sy1:sy2, sx1:sx2], out[sy1:sy2, sx1:sx2]),
            "",
        ))

        # ---------------------------------------------------------- anggaran balon
        #
        # Yang diuji: apakah angka anggaran BENAR-BENAR menggambarkan balonnya.
        # Anggaran yang tidak berkorelasi dengan geometri sama tidak bergunanya
        # dengan tidak punya anggaran — model akan diberi angka yang salah dan
        # patuh pada angka yang salah. Jadi tiga sifat yang dijamin:
        #   hard >= soft   ukuran lebih kecil selalu memuat lebih banyak
        #   soft > 0       balon sebesar ini pasti memuat sesuatu
        #   soft berbeda   antara lobus dan balon sempit (bukan konstanta)
        _bud = [typeset.region_budget(_r, typeset.FONT_USED) for _r in _dbl]
        _bok = all(b["hard"] >= b["soft"] > 0 and b["word_hard"] >= b["word_soft"] > 0
                   for b in _bud)
        checks.append((
            "anggaran balon: hard >= soft > 0 untuk kedua lobus",
            _bok, "; ".join(f"soft={b['soft']} hard={b['hard']} "
                            f"kata={b['word_soft']}/{b['word_hard']}" for b in _bud),
        ))
        # Anggaran harus IKUT besar balon. regions[2] balon sempit (sisi 140 px),
        # _dbl[0] satu lobus balon ganda yang jauh lebih lapang; kalau keduanya
        # memberi angka yang sama, yang diukur bukan balonnya melainkan teks
        # pengisinya — tepat kegagalan yang bikin tujuh balon melaporkan soft=20.
        _bnarrow = typeset.region_budget(regions[2], typeset.FONT_USED)
        checks.append((
            "anggaran balon membedakan balon sempit dari balon lapang",
            _bnarrow["soft"] != _bud[0]["soft"],
            f"sempit={_bnarrow['soft']} lapang={_bud[0]['soft']}",
        ))
        # Lapis VALIDASI: teks yang mustahil harus tertangkap, teks yang muat
        # harus lolos. Tanpa kedua arah ini validator bisa lolos-semua (tidak
        # menjaga apa pun) atau tolak-semua (menuntut revisi tanpa akhir).
        #
        # Yang mustahil di sini SATU KATA panjang, bukan kalimat panjang, dan itu
        # bukan pilihan sembarang: kalimat panjang selalu bisa dipecah jadi banyak
        # baris, jadi di balon lapang 490 karakter pun masih muat (lobus ini
        # memuat ~790 pada ukuran minimum). Yang benar-benar tidak punya jalan
        # keluar adalah satu kata yang lebih lebar dari balonnya — layout() hanya
        # bisa memenggalnya atau gagal, dan itulah cacat yang divalidasi.
        #
        # Panjangnya dihitung dari GEOMETRI pada emergency_floor(), bukan
        # `word_hard + 40`. Angka tetap itu sudah pernah membuat test ini gagal
        # tanpa ada cacat: word_hard diukur di min_font() (9 px), sementara fit()
        # boleh turun ke emergency_floor() (7 px), dan begitu SETTINGS.condense
        # dipasang 0.85 lebar 'A' di 7 px menyusut 6.00 -> 5.10 px. 64 karakter
        # jadi cuma 326 px di mask 397 px — benar-benar muat, jadi _violations()
        # BENAR meloloskannya dan yang salah justru patokannya. Diambil dari
        # lebar mask dibagi lebar maju satu huruf di lantai terendah, plus marjin,
        # angkanya tetap mustahil berapa pun faktor condense-nya.
        _vfl = typeset.emergency_floor()
        _vadv = typeset._line_width("A", typeset._font(typeset.FONT_USED, _vfl),
                                    typeset._cmap(typeset.FONT_USED), _vfl)
        _vn = int(_dbl[0].bubble_mask.shape[1] / max(_vadv, 0.5) * 1.25) + 8
        _vbud = {_dbl[0].idx: _bud[0]}
        _vbad = _tl._violations({_dbl[0].idx: "A" * _vn}, _vbud, [_dbl[0]])
        _vok = _tl._violations({_dbl[0].idx: "SORRY."}, _vbud, [_dbl[0]])
        checks.append((
            "validasi anggaran: teks mustahil ditolak, teks pendek diloloskan",
            bool(_vbad) and not _vok,
            f"N={_vn} (lantai={_vfl} maju={_vadv:.2f}) "
            f"panjang={sorted(_vbad)} pendek={sorted(_vok)}",
        ))
        # '＼' dibuang _clean_translation sebelum typeset, jadi validasi harus
        # mengukur bentuk SETELAH pembersihan. Kalau tidak, '＼SORRY.' dilaporkan
        # tidak muat di balon yang memuat 39 karakter — dan model dipaksa merevisi
        # sesuatu yang sudah benar.
        checks.append((
            "validasi memakai bentuk setelah pembersihan (＼ tidak dihitung)",
            not _tl._violations({_dbl[0].idx: "＼SORRY."}, _vbud, [_dbl[0]]),
            "",
        ))
    else:
        checks.append(("font tersedia", False, "setup_fonts() belum dijalankan"))

    # ---------------------------------------------------------------- penyedia
    #
    # Tidak ada panggilan jaringan di sini: yang diuji PEMILIHAN jalur, bukan
    # jawaban API. Salah pilih penyedia adalah cacat yang paling mahal untuk
    # ditemukan lewat mata — hasilnya tetap keluar, cuma dari mesin yang salah.
    checks.append((
        "provider: nama UI -> kelas client yang benar",
        isinstance(_tl.make_client("x", "DeepL"), _tl.DeepLClient)
        and isinstance(_tl.make_client("x", "Router LLM (gorouter)"), _tl.RouterClient)
        and isinstance(_tl.make_client("x", "LLM (freetokenfaucet)"), _tl.FaucetClient)
        # Faucet TURUNAN RouterClient, jadi isinstance saja tidak membedakannya —
        # yang membedakan base URL-nya. Kalau urutan cabang di make_client()
        # terbalik, router yang mati justru dipakai walau UI memilih faucet.
        and not isinstance(_tl.make_client("x", "Router LLM (gorouter)"), _tl.FaucetClient)
        and "faucet" in _tl.make_client("x", "LLM (freetokenfaucet)").base
        and _tl._is_router("Router LLM (gorouter)")
        and _tl._is_router("LLM (freetokenfaucet)")
        and not _tl._is_router("DeepL"),
        "",
    ))
    # Faucet memakai model REASONING: tanpa thinking dimatikan, jatah keluaran
    # habis untuk berpikir dan content keluar STRING KOSONG tanpa error HTTP —
    # halaman keluar bersih tanpa terjemahan dan tidak ada yang mengeluh.
    _fc = _tl.make_client("x", "LLM (freetokenfaucet)")
    checks.append((
        "faucet: thinking dimatikan + batas waktu sendiri",
        _fc.extra.get("thinking", {}).get("type") == "disabled"
        and _fc.max_tokens > 0
        and _fc.timeout < _tl.ROUTER_TIMEOUT
        and _fc.deadline < _tl.ROUTER_DEADLINE,
        "",
    ))
    # Model faucet WAJIB salah satu dari tiga yang GRATIS. Terukur 17 Agu 2026:
    # 16 dari 19 model membalas HTTP 402 INSUFFICIENT_BALANCE, termasuk model
    # yang dulu jadi default di sini. Akibatnya tiga halaman Colab keluar TANPA
    # terjemahan. Check ini yang mencegahnya kembali diam-diam lewat edit sel
    # notebook: daftar putih, bukan daftar hitam — model berbayar baru pun
    # tertolak tanpa perlu menambah namanya di sini.
    _FREE = {"mimo-v2.5-pro", "mimo-v2.5", "gpt-5.6-terra"}
    checks.append((
        "faucet: model default GRATIS (bukan model 402)",
        _fc.model in _FREE and all(f in _FREE for f in _fc.fallback),
        f"model={_fc.model} fallback={_fc.fallback}",
    ))
    # Tanpa User-Agent, gorouter dibalas 403 "error code 1010" oleh Cloudflare —
    # dua bentuk auth sama-sama 403, dan key yang sama DENGAN header ini
    # membalas 200. Kalau atribut ini hilang, router mati total tanpa petunjuk
    # apa pun di pesan errornya. Faucet sebaliknya: terukur sehat tanpa UA.
    _rc = _tl.make_client("x", "Router LLM (gorouter)")
    checks.append((
        "router: header User-Agent wajib ada, faucet tanpa header",
        "User-Agent" in _rc.headers and _rc.headers["User-Agent"]
        and _fc.headers == {},
        f"router={sorted(_rc.headers)} faucet={sorted(_fc.headers)}",
    ))
    # Prompt anggaran cuma boleh menyebut max_chars kalau angkanya BENAR-BENAR
    # dikirim. Menyebut batas yang tidak ada di masukan membuat model menebak
    # batasnya sendiri, dan tebakannya tidak ada hubungannya dengan balon.
    _sp_bud = _tl._system_prompt("English", "Manga Natural", True, True)
    _sp_pln = _tl._system_prompt("English", "Manga Natural", True, False)
    checks.append((
        "system prompt menyebut max_chars HANYA saat anggaran dikirim",
        "max_chars" in _sp_bud and "max_chars" not in _sp_pln,
        "",
    ))
    checks.append((
        "system prompt membawa gaya terpilih + aturan honorifik",
        "Uncensored:" in _tl._system_prompt("English", "Uncensored", True, True)
        and "Keep honorifics" in _sp_bud
        and "Localise honorifics" in _tl._system_prompt(
            "English", "Manga Natural", False, True),
        "",
    ))
    # Router membalas text/event-stream walau non-stream: satu objek lalu
    # 'data: [DONE]' TANPA pemisah. json.loads gagal 'Extra data' padahal isinya
    # utuh — kalau ini regresi, SEMUA terjemahan router gagal sekaligus.
    checks.append((
        "body router text/event-stream ter-decode (data: + [DONE] tanpa pemisah)",
        _tl._decode_router('data: {"choices": [{"message": {"content": "x"}}]}'
                          'data: [DONE]')["choices"][0]["message"]["content"] == "x",
        "",
    ))

    # Kunci yang TIDAK dijawab model harus diminta ulang, bukan ditelan.
    # Inilah cacat hasilnew/13.JPG: balon 'えっ！？' terkirim ke model, model
    # memutuskan seruan sependek itu tidak perlu diterjemahkan, kuncinya hilang
    # dari JSON, dan loop lama (`if not t: continue`) membiarkan translation
    # tetap None — render_region() lalu keluar lebih awal dan balon itu tercetak
    # berbahasa Jepang tanpa satu pun peringatan. Diuji TANPA jaringan: yang
    # ditukar cuma _router_call_any, jadi yang diperiksa logika ronde ulangnya.
    _fake_calls: list[list[int]] = []

    def _fake_router(_client, _model, _system, user, *_a, **_kw):
        import json as _json
        ids = sorted(int(k) for k in _json.loads(
            user.split("Lines:\n", 1)[1].split("\n\nMISSING", 1)[0]))
        _fake_calls.append(ids)
        # Panggilan pertama sengaja MENGHILANGKAN idx 1 (meniru model sungguhan);
        # panggilan kedua — permintaan ulang — menjawabnya.
        return ({str(i): "HUH?!" for i in ids} if len(_fake_calls) > 1
                else {str(i): "REALLY?" for i in ids if i != 1}), "m"

    _rr = [Region(idx=i, bbox=(0, 0, 10, 10), det_class="text_bubble")
           for i in range(2)]
    _rr[0].src_text, _rr[1].src_text = "でも", "えっ！？"
    for _r in _rr:
        _r.label = "DIALOGUE"
    _orig_call, _orig_bud = _tl._router_call_any, SETTINGS.balloon_budget
    try:
        _tl._router_call_any = _fake_router
        SETTINGS.balloon_budget = False   # anggaran diuji terpisah; ini soal kelengkapan
        _tl._translate_router(_tl.make_client("x", "Router LLM (gorouter)"), "m",
                             _rr, _rr, "English", "Manga Natural", True)
    finally:
        _tl._router_call_any, SETTINGS.balloon_budget = _orig_call, _orig_bud
    checks.append((
        "kunci yang tidak dijawab model diminta ulang, bukan ditelan diam-diam",
        len(_fake_calls) == 2 and _fake_calls[1] == [1]
        and bool(_rr[0].translation) and bool(_rr[1].translation),
        f"panggilan={_fake_calls} hasil={[r.translation for r in _rr]}",
    ))
    # _missing_ids harus mengukur bentuk AKHIR, bukan apa yang model tulis:
    # jawaban '．．．' hilang seluruhnya di _clean_translation, jadi balonnya
    # sama kosongnya dengan kunci yang tidak ada — dan harus ikut diminta ulang.
    _mr = [Region(idx=i, bbox=(0, 0, 10, 10)) for i in range(4)]
    for _r in _mr:
        _r.src_text = "でも"
    _mi = _tl._missing_ids({0: "OK", 1: "   ", 2: "．．．"}, _mr)
    checks.append((
        "balon kosong terdeteksi: kunci hilang, spasi, dan yang habis dibersihkan",
        _mi == [1, 2, 3], f"missing={_mi}",
    ))

    # ------------------------------------------------------------- diagnostik
    #
    # Yang diuji: apakah kegagalan BENAR-BENAR terlihat. Sebelum lapisan ini ada,
    # satu run Colab keluar `diterjemah 0` di tiga halaman tanpa satu pun pesan,
    # dan tidak ada test yang bisa gagal karenanya — jadi empat kontrak di bawah
    # justru menjaga jalur yang paling mudah kembali jadi bisu.
    import config as _cfg
    import io as _io
    import contextlib as _ctx

    _n0 = len(_cfg.RUN_NOTES)
    _cap = _io.StringIO()
    with _ctx.redirect_stdout(_cap):
        _cfg.note("error", "uji", "pesan uji")
        _cfg.note("warn", "uji", "pesan warn")
    _cout = _cap.getvalue()
    checks.append((
        "note() mengisi RUN_NOTES DAN mencetak awalan per level",
        len(_cfg.RUN_NOTES) - _n0 == 2 and "[!!]" in _cout and "[!]" in _cout
        and _cfg.RUN_NOTES[-2][0] == "error"
        and len(_cfg.notes_since(_n0)) == 2,
        f"catatan={_cfg.RUN_NOTES[-2:]} keluaran={_cout.strip()!r}",
    ))

    # verify.report() harus MENERUSKAN catatan ke dict hasil — inilah pipa yang
    # membuat kolom `catatan` di tabel UI menunjuk halaman yang benar.
    _rep = verify.report([], [], "x", [("error", "t", "boom"), ("warn", "t", "hm")])
    checks.append((
        "verify.report(notes=...) meneruskan catatan + menghitung per level",
        len(_rep.get("notes", [])) == 2 and _rep.get("error_count") == 1
        and _rep.get("warn_count") == 1
        and _rep["notes"][0]["msg"] == "boom",
        f"notes={_rep.get('notes')} err={_rep.get('error_count')}",
    ))

    # _diagnose() diuji atas summary BIKINAN, tanpa jaringan dan tanpa Gradio:
    # itu sebabnya fungsinya dibuat murni. Kasusnya persis layar user — 15 region
    # bisa diterjemah, nol yang jadi.
    import app as _app

    class _FakeRes:
        def __init__(self, stem, rep):
            self.stem, self.report = stem, rep

    _dead = _app._diagnose(
        [_FakeRes("hitomi_006", {"region_count": 15, "translatable_count": 15,
                                 "translated_count": 0, "untranslated_count": 15,
                                 "residue_count": 0, "overflow_count": 0})],
        {"notes": [{"level": "error", "tag": "translate",
                    "msg": "faucet gagal (HTTP 429)"}],
         "residue_total": 0, "overflow_total": 0},
    )
    _dtxt = "\n".join(_dead)
    checks.append((
        "_diagnose(): 'diterjemah 0' -> banner merah + nama halaman + sebabnya",
        "TIDAK ADA TERJEMAHAN" in _dtxt and "hitomi_006" in _dtxt
        and "HTTP 429" in _dtxt and _app._RED in _dtxt,
        _dtxt[:160].replace("\n", " | "),
    ))

    # Arah sebaliknya, dan ini yang menjaga banner tetap berarti: halaman bersih
    # TIDAK boleh memerah. Banner yang selalu merah sama tidak bergunanya dengan
    # tidak ada banner — user berhenti membacanya.
    _clean = _app._diagnose(
        [_FakeRes("ok_001", {"region_count": 8, "translatable_count": 6,
                             "translated_count": 6, "untranslated_count": 0,
                             "residue_count": 0, "overflow_count": 0})],
        {"notes": [], "residue_total": 0, "overflow_total": 0,
         "translated_total": 6},
    )
    # Halaman yang isinya SFX semua juga tidak boleh dituduh gagal: 0 diterjemah
    # dari 0 yang bisa diterjemah itu BENAR, dan menuduhnya adalah cara tercepat
    # membuat user mengabaikan banner merah yang sungguhan.
    _sfx_only = _app._diagnose(
        [_FakeRes("sfx_001", {"region_count": 4, "translatable_count": 0,
                              "translated_count": 0, "untranslated_count": 0,
                              "residue_count": 0, "overflow_count": 0})],
        {"notes": [], "residue_total": 0, "overflow_total": 0},
    )
    checks.append((
        "_diagnose(): halaman bersih & halaman SFX-saja tidak memerah",
        _app._RED not in "\n".join(_clean) and _app._GREEN in "\n".join(_clean)
        and _app._RED not in "\n".join(_sfx_only),
        f"bersih={' '.join(_clean)[:80]!r} sfx={' '.join(_sfx_only)[:60]!r}",
    ))

    if verbose:
        for name, ok, extra in checks:
            print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + (f"  {extra}" if extra else ""))

    return all(ok for _, ok, _ in checks)



Writing /content/mangatl/selftest.py


In [17]:
%%writefile /content/mangatl/app.py

"""UI Gradio: upload -> TRANSLATE -> selesai.

Gradio 6: theme/css pindah ke launch(); di Colab share=True wajib dan
gr.Progress butuh .queue().
"""

from __future__ import annotations

import contextlib
import io
import json
import shutil
import subprocess
import traceback
from pathlib import Path

import gradio as gr

from config import (LANGUAGES, OUTPUT, PROVIDER_DEFAULT, PROVIDERS, RUN_NOTES,
                    SETTINGS, TRANSLATION_STYLES, note)
import pipeline
import translate as tl
import typeset

CSS = """
.gradio-container {max-width: 1280px !important;}
#go {font-size: 18px; font-weight: 700; height: 56px;}
footer {display: none !important;}
"""

# Nama file log di OUTPUT. Ditulis ke disk, bukan hanya ditaruh di gr.Code,
# karena gr.File butuh path nyata untuk diunduh — dan karena kalau sesi Colab
# mati mendadak, log-nya masih ada di /content/mangatl/output.
LOG_NAME = "run.log"


class _Tee:
    """Tulis ke stream asli SEKALIGUS tampung di buffer.

    Bukan sekadar redirect_stdout(StringIO): kalau keluaran dialihkan sepenuhnya,
    sel Colab yang memang sedang dilihat user jadi bisu total dan progress bar
    library pihak ketiga hilang. Tee menjaga keduanya — Colab tetap dapat aliran
    langsung, UI dapat salinan lengkap.

    Sengaja TIDAK mewarisi io.TextIOBase: sebagian library memeriksa atribut
    seperti `.encoding` atau memanggil `.fileno()`, dan pewarisan setengah jalan
    membuat pemeriksaan itu lolos lalu gagal belakangan. Di sini semua yang tidak
    dikenal diteruskan ke stream asli lewat __getattr__, jadi objeknya berperilaku
    persis seperti stdout aslinya.
    """

    def __init__(self, real, buf: io.StringIO):
        self._real, self._buf = real, buf

    def write(self, s: str) -> int:
        self._buf.write(s)
        try:
            return self._real.write(s)
        except Exception:  # noqa: BLE001 - stream asli boleh mati, buffer tidak
            return len(s)

    def flush(self) -> None:
        with contextlib.suppress(Exception):
            self._real.flush()

    def isatty(self) -> bool:
        return False

    def fileno(self) -> int:
        # tqdm dan sebagian library memanggil ini; diteruskan supaya mereka
        # mengambil keputusan yang sama seperti tanpa Tee.
        return self._real.fileno()

    def __getattr__(self, name):
        return getattr(self._real, name)


@contextlib.contextmanager
def _capture():
    """Jalankan blok dengan stdout+stderr di-tee ke satu buffer.

    yield buffer-nya, bukan teksnya: pemanggil perlu membaca isi buffer JUGA
    saat blok gagal di tengah jalan lewat except di luar sini — dan pada saat itu
    nilai balik context manager sudah tidak bisa diambil lagi.
    """
    buf = io.StringIO()
    import sys

    with contextlib.redirect_stdout(_Tee(sys.stdout, buf)), \
            contextlib.redirect_stderr(_Tee(sys.stderr, buf)):
        yield buf


def _write_log(text: str) -> str | None:
    """Simpan log mentah ke OUTPUT/run.log. None kalau tidak bisa ditulis."""
    try:
        OUTPUT.mkdir(parents=True, exist_ok=True)
        p = OUTPUT / LOG_NAME
        # unlink dulu: menimpa file yang sedang dipegang gr.File di klik
        # sebelumnya bisa menyisakan ekor log lama di unduhan.
        p.unlink(missing_ok=True)
        p.write_text(text or "(tidak ada keluaran)", encoding="utf-8")
        return str(p)
    except OSError:
        return None


def _archive(files: list[Path], dest: Path) -> tuple[str | None, str]:
    """Bungkus hasil jadi .rar. Kembalikan (path, catatan); path None = gagal.

    RAR itu format proprietary. Stdlib Python tidak punya penulis RAR dan
    `rarfile` cuma bisa MEMBACA, jadi biner `rar` resmi (dipasang di sel 3)
    satu-satunya cara membuat .rar asli. Kalau binernya tidak ada, pemanggil
    tetap punya ZIP yang sah; JANGAN pernah me-rename ZIP jadi .rar karena
    WinRAR menolaknya dan user baru sadar setelah unduhan selesai.
    """
    exe = shutil.which("rar")
    if exe is None:
        return None, "biner `rar` tidak terpasang"
    dest.unlink(missing_ok=True)
    # -ep1 buang path induk, -m5 kompresi maksimum, -idq senyap, -y ya ke semua.
    cmd = [exe, "a", "-ep1", "-m5", "-idq", "-y", str(dest), *(str(f) for f in files)]
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=900)
    except (OSError, subprocess.SubprocessError) as exc:
        return None, str(exc)
    if proc.returncode != 0 or not dest.exists():
        tail = (proc.stderr or proc.stdout).strip().splitlines()
        return None, tail[-1][:150] if tail else f"rar keluar kode {proc.returncode}"
    return str(dest), "ok"


# ---------------------------------------------------------------- diagnosa

# Penanda banner. Karakter, bukan warna CSS: Markdown Gradio tidak menjamin
# kelas CSS kustom lolos sanitizer-nya, sedangkan emoji selalu terlihat dan
# ikut terbaca kalau tabelnya di-copy ke tempat lain.
_RED, _YELLOW, _GREEN = "\U0001F534", "\U0001F7E1", "\U0001F7E2"


def _diagnose(results, summary: dict, notes: list | None = None) -> list[str]:
    """Baris banner untuk disisipkan di ATAS tabel. Murni, tanpa I/O.

    Tanpa I/O dan tanpa Gradio dengan sengaja: inilah satu-satunya bagian
    lapisan diagnostik yang punya logika keputusan, jadi ia harus bisa diuji di
    selftest offline dengan summary bikinan. Begitu fungsi ini menyentuh disk
    atau gr.*, kontraknya cuma bisa dibuktikan dengan menjalankan seluruh UI.

    `results` = daftar objek dengan `.stem` dan `.report`; `summary` = dict
    process_batch; `notes` = catatan tingkat batch (default: summary['notes']).
    Return daftar baris markdown, sudah urut dari yang paling parah.
    """
    notes = notes if notes is not None else (summary.get("notes") or [])
    notes = [n if isinstance(n, dict) else {"level": n[0], "tag": n[1], "msg": n[2]}
             for n in notes]
    out: list[str] = []

    def note_lines(level: str, limit: int = 6) -> list[str]:
        seen, picked = set(), []
        for n in notes:
            if n.get("level") != level:
                continue
            msg = str(n.get("msg", ""))
            # De-dup: satu kegagalan jaringan yang sama terulang di 3 halaman
            # menghasilkan 3 baris identik, dan banner jadi tembok teks yang
            # justru menyembunyikan sebab kedua yang berbeda.
            if msg in seen:
                continue
            seen.add(msg)
            picked.append(f">   `[{n.get('tag', '?')}]` {msg}")
            if len(picked) >= limit:
                picked.append(f">   ...dan {sum(1 for x in notes if x.get('level') == level) - limit} lagi, lihat **Log lengkap**")
                break
        return picked

    # 1. Paling parah: ada halaman yang butuh terjemahan tapi tidak dapat satu pun.
    # Syaratnya translatable_count, bukan region_count: halaman yang isinya SFX
    # semua memang SEHARUSNYA translated_count 0, dan menuduhnya gagal akan
    # melatih user mengabaikan banner ini.
    dead = [r for r in results
            if (r.report.get("translatable_count",
                             r.report.get("region_count", 0)) > 0
                and r.report.get("translated_count", 0) == 0)]
    if dead:
        names = ", ".join(f"`{r.stem}`" for r in dead[:8])
        more = f" (+{len(dead) - 8} lagi)" if len(dead) > 8 else ""
        out += [
            f"> ## {_RED} TIDAK ADA TERJEMAHAN",
            f"> {len(dead)} halaman keluar **masih berbahasa Jepang**: {names}{more}.",
            ">",
            "> Gambar dan ZIP tetap dibuat — teks aslinya tersimpan di sidecar "
            "JSON tiap halaman, jadi kerjanya tidak hilang. Sebab yang tercatat:",
        ]
        out += note_lines("error") or [">   _(tidak ada catatan error — lihat **Log lengkap** di bawah)_"]
        out.append("")
    else:
        # 2. Sebagian balon tertinggal. Bukan kegagalan total, tapi tetap merah:
        # halaman ini TERCETAK campur Jepang-Inggris dan tidak layak dipakai.
        partial = [r for r in results if r.report.get("untranslated_count", 0) > 0]
        if partial:
            det = "; ".join(
                f"`{r.stem}` balon {r.report.get('untranslated_idx', [])}"
                for r in partial[:6]
            )
            out += [
                f"> ## {_RED} ADA BALON YANG BELUM DITERJEMAH",
                f"> {sum(r.report.get('untranslated_count', 0) for r in partial)} balon "
                f"di {len(partial)} halaman masih berbahasa Jepang: {det}.",
                "",
            ]

    # 3. Error lain yang belum masuk banner di atas (OCR mati, klien gagal dibuat).
    if not dead:
        errs = note_lines("error")
        if errs:
            out += [f"> ## {_RED} ADA YANG GAGAL", *errs, ""]

    # 4. Kuning: hasil masih layak dipakai, cuma perlu dilihat.
    soft: list[str] = []
    res_tot = summary.get("residue_total", 0)
    ovf_tot = summary.get("overflow_total", 0)
    if res_tot:
        bad = [f"`{r.stem}` {r.report.get('residue_idx', [])}"
               for r in results if r.report.get("residue_count", 0)]
        soft.append(f"> {_YELLOW} **Sisa teks asli** di {res_tot} region: "
                    f"{'; '.join(bad[:6])} — teks Jepangnya masih terlihat di bawah hasil.")
    if ovf_tot:
        soft.append(f"> {_YELLOW} **{ovf_tot} balon overflow**: teks tercetak "
                    "melebihi ruang balon.")
    warns = note_lines("warn", limit=4)
    if warns:
        soft += [f"> {_YELLOW} **Peringatan:**", *warns]
    if soft:
        out += [*soft, ""]

    if not out:
        n_tr = summary.get("translated_total")
        extra = f" — {n_tr} balon diterjemah" if n_tr else ""
        out = [f"> {_GREEN} **Bersih:** tidak ada residu, overflow, "
               f"maupun balon yang gagal diterjemah{extra}.", ""]
    return out


def _probe_table(provider: str, api_key: str) -> str:
    """Cek key penyedia aktif: kuota DeepL, atau satu panggilan uji ke router.

    Dibungkus tangkap-keluaran + except: ini tombol PERTAMA yang diklik orang,
    jadi ia tidak boleh bisa gagal dalam diam. Sebelumnya hanya RuntimeError dan
    ImportError yang tertangkap — sebuah URLError atau TypeError dari dalam
    check_usage() membuat Gradio menampilkan toast merah tanpa isi, lalu user
    melihat UI yang diam persis seperti kasus `diterjemah 0`.
    """
    SETTINGS.provider = provider
    with _capture() as buf:
        try:
            key = tl.get_api_key(api_key, provider)
            client = tl.make_client(key, provider)
            if tl._is_router(provider):
                msg = (f"{provider}: **{tl.check_usage(client)}**. Anggaran balon "
                       f"{'AKTIF' if SETTINGS.balloon_budget else 'mati'} — teks dibuat "
                       "sependek balonnya sejak di sumber.")
            else:
                msg = (f"DeepL API: **{tl.check_usage(client)}** digunakan. "
                       "DeepL tidak menyensor konten apa pun.")
        except (RuntimeError, ImportError) as exc:
            msg = f"{_RED} **Gagal:** {exc}"
        except Exception:  # noqa: BLE001 - tombol probe tidak boleh diam
            msg = (f"{_RED} **Gagal tak terduga saat cek API.**\n\n"
                   f"```\n{traceback.format_exc()[-1500:]}\n```")
    log = buf.getvalue().strip()
    if log:
        msg += f"\n\n<details><summary>Log</summary>\n\n```\n{log[-2000:]}\n```\n\n</details>"
    return msg


def _run(files, provider: str, api_key: str, target_lang: str, style: str,
         balloon_budget: bool, debug: bool, font_file, out_format: str = "both",
         progress=gr.Progress()):
    """Handler tombol TRANSLATE — pembungkus yang menangkap log DAN exception.

    Isi sebenarnya ada di `_run_inner`. Dipisah supaya SATU tempat memegang dua
    jaminan yang berlaku untuk semua jalur keluar: (1) apa pun yang tercetak
    selama run masuk ke run.log dan ke accordion UI, (2) exception apa pun jadi
    banner traceback, bukan UI yang diam. Kalau logika ini ditempel di dalam
    _run_inner, tiap `return` awal harus mengulangnya sendiri.

    Selalu mengembalikan 7 nilai: gallery, rar, zip, tabel(+banner), json, log
    teks, log file.
    """
    with _capture() as buf:
        try:
            gallery, rar_path, zip_path, md, raw = _run_inner(
                files, provider, api_key, target_lang, style, balloon_budget,
                debug, font_file, out_format, progress,
            )
        except Exception:  # noqa: BLE001 - UI tidak boleh mati tanpa pesan
            tb = traceback.format_exc()
            print(tb)  # ikut ke buffer -> run.log, jadi tersimpan juga
            gallery, rar_path, zip_path, raw = None, None, None, None
            md = (
                f"> ## {_RED} PIPELINE BERHENTI KARENA ERROR\n"
                "> Ini bug, bukan kegagalan jaringan. Traceback lengkapnya:\n\n"
                f"```\n{tb[-3000:]}\n```\n\n"
                "Log lengkap ada di accordion di bawah dan bisa diunduh."
            )
    log_text = buf.getvalue()
    return gallery, rar_path, zip_path, md, raw, log_text, _write_log(log_text)


def _run_inner(files, provider: str, api_key: str, target_lang: str, style: str,
               balloon_budget: bool, debug: bool, font_file, out_format: str,
               progress):
    """Isi asli handler TRANSLATE. Return 5 nilai (tanpa bagian log)."""
    # RUN_NOTES hidup selama sesi Colab, jadi klik TRANSLATE kedua akan mewarisi
    # banner merah klik pertama kalau tidak dikosongkan. Dikosongkan DI SINI dan
    # bukan di process_batch, karena catatan "API key tidak terbaca" di bawah
    # terjadi sebelum process_batch dipanggil dan justru itu yang harus selamat.
    RUN_NOTES.clear()
    if not files:
        return None, None, None, "Belum ada gambar yang diunggah.", None

    SETTINGS.output_format = out_format
    SETTINGS.target_lang = target_lang
    SETTINGS.translation_style = style
    SETTINGS.provider = provider
    # Anggaran balon cuma berarti untuk router: DeepL tidak bisa diberi tahu
    # ukuran balon, jadi mengukurnya di sana hanya membakar 10 detik CPU.
    SETTINGS.balloon_budget = bool(balloon_budget) and tl._is_router(provider)
    # ALL CAPS hanya gaya lettering English; bahasa lain pakai huruf normal.
    SETTINGS.force_upper = target_lang == "English"

    if font_file:
        path = font_file if isinstance(font_file, str) else font_file.name
        typeset.set_user_font(path)

    if not typeset.FONT_USED:
        typeset.setup_fonts(verbose=False)

    paths = [f if isinstance(f, str) else f.name for f in files]
    try:
        key = tl.get_api_key(api_key, provider)
    except RuntimeError as exc:
        # Dulu ditelan tanpa jejak, dan inilah jalur yang menghasilkan "diterjemah
        # 0" paling sering: key tidak ada -> client None -> semua halaman keluar
        # berbahasa Jepang. Sekarang tercatat sebagai error supaya masuk banner.
        note("error", "app",
             f"API key tidak terbaca ({exc}) — jalan TANPA terjemahan, "
             "semua halaman keluar berbahasa Jepang")
        key = ""

    results, summary = pipeline.process_batch(paths, key or None, debug, progress,
                                              target_lang, style, reset_notes=False)
    pipeline.release_all()

    # Galeri ikut format terpilih; kalau "both", preview pakai PNG (lossless).
    gallery = [str(r.paths.get("png") or r.paths["jpg"]) for r in results]
    rows = [
        "| halaman | region | SFX dijaga | diterjemah | belum diterjemah | residu | overflow | catatan |",
        "|---|---|---|---|---|---|---|---|",
    ]
    for r in results:
        rp = r.report
        n_err, n_warn = rp.get("error_count", 0), rp.get("warn_count", 0)
        # Kolom catatan menunjuk ke accordion, bukan memuat pesannya: pesan
        # aslinya bisa 200 karakter dan akan merusak lebar tabel.
        cat = " ".join(x for x in (f"{_RED}{n_err}" if n_err else "",
                                   f"{_YELLOW}{n_warn}" if n_warn else "") if x) or "-"
        belum = rp.get("untranslated_count", 0)
        rows.append(
            f"| {r.stem} | {rp['region_count']} | {len(rp['sfx_idx'])} | "
            f"{rp['translated_count']} | {f'**{belum}**' if belum else 0} | "
            f"{rp['residue_count']} | {rp['overflow_count']} | {cat} |"
        )
    rar_path, rar_note = _archive(
        [p for r in results for p in r.paths.values()],
        Path(summary["zip"]).with_name("manga_translated.rar"),
    )
    summary["rar"] = rar_path or f"gagal: {rar_note}"
    # Catatan disisipkan ke `rows` (yang sudah di-join) supaya tabelnya tetap utuh.
    if rar_path is None:
        rows[:0] = [f"> RAR tidak dibuat ({rar_note}) - pakai unduhan ZIP.", ""]
    # Banner diagnosa paling atas, di atas catatan RAR: sebab kegagalan harus
    # jadi hal pertama yang terbaca tanpa perlu men-scroll.
    rows[:0] = _diagnose(results, summary)

    # Konfirmasi GPU di UI: user melihat langsung bahwa proses jalan di T4.
    try:
        import torch as _torch
        _gpu = _torch.cuda.get_device_name(0) if _torch.cuda.is_available() else "CPU (lambat)"
    except Exception:  # noqa: BLE001
        _gpu = "tidak terbaca"
    # font_used bisa None kalau setup_fonts() gagal total. Path(None) melempar
    # TypeError, dan dulu itu terjadi SETELAH semua halaman selesai diproses —
    # seluruh run hilang tanpa satu pesan pun. Jangan pernah kembalikan Path()
    # atas nilai yang boleh None.
    _font = summary.get("font_used")
    _font = Path(_font).name if _font else "GAGAL DIMUAT"
    md = (
        f"**GPU:** `{_gpu}` · **Penyedia:** `{summary.get('provider', '?')}` · "
        f"**Model:** `{summary['model']}` · **Font:** `{_font}`\n\n"
        + "\n".join(rows)
    )
    return (
        gallery, rar_path, summary["zip"], md,
        json.dumps(summary, ensure_ascii=False, indent=2),
    )


def build() -> gr.Blocks:
    with gr.Blocks(title="Manga Translator — Jepang ke semua bahasa") as demo:
        gr.Markdown(
            "# Manga Translator — Jepang ke semua bahasa\n"
            "Unggah halaman manga, pilih penyedia terjemahan dan bahasa tujuan, "
            "tekan **TRANSLATE**. SFX dibiarkan utuh, tidak ada penyensoran."
        )
        with gr.Row():
            with gr.Column(scale=1):
                files = gr.File(
                    label="Halaman manga - bisa banyak sekaligus",
                    file_count="multiple",
                    file_types=["image"],
                    height=200,
                )
                provider = gr.Dropdown(
                    label="Penyedia terjemahan",
                    choices=PROVIDERS,
                    value=PROVIDER_DEFAULT,
                    info="LLM (freetokenfaucet): paham konteks halaman DAN ukuran "
                         "balon, GRATIS, mimo-v2.5-pro terukur 5-8 dtk per "
                         "halaman — pakai ini. "
                         "DeepL: cepat tapi tidak bisa diberi tahu ukuran balon. "
                         "Router gorouter: mutu bahasa paling rapi tapi memakai "
                         "kredit berbayar.",
                )
                api_key = gr.Textbox(
                    label="API key (kosongkan kalau sudah di Colab Secrets)",
                    type="password",
                    placeholder="FAUCET_API_KEY / DEEPL_API_KEY / ROUTER_API_KEY",
                    info="Disimpan di Colab Secrets sebagai FAUCET_API_KEY, "
                         "DEEPL_API_KEY, atau ROUTER_API_KEY sesuai penyedia — "
                         "jangan ditulis di kode.",
                )
                lang = gr.Dropdown(
                    label="Bahasa terjemahan (Jepang → ...)",
                    choices=LANGUAGES,
                    value="English",
                )
                style = gr.Dropdown(
                    label="Gaya terjemahan",
                    choices=list(TRANSLATION_STYLES.keys()),
                    value="Manga Natural",
                )
                with gr.Accordion("Opsi", open=False):
                    balloon_budget = gr.Checkbox(
                        label="Anggaran balon (penyedia LLM saja)",
                        value=True,
                        info="Kirim batas karakter tiap balon ke model, ukur ulang "
                             "jawabannya, minta perbaikan yang melanggar. +~10 dtk "
                             "CPU per halaman; ini yang menjaga teks tidak keluar "
                             "balon. Tidak berlaku untuk DeepL.",
                    )
                    out_format = gr.Radio(
                        label="Format output",
                        choices=[
                            ("PNG + JPG", "both"),
                            ("PNG saja (lossless)", "png"),
                            ("JPG saja (ringan)", "jpg"),
                        ],
                        value="both",
                    )
                    debug = gr.Checkbox(label="Debug mode (dump tahapan)", value=False)
                    font_file = gr.File(
                        label="Font sendiri (.ttf/.otf) — opsional",
                        file_types=[".ttf", ".otf"],
                    )
                    probe_btn = gr.Button("Cek API", size="sm")
                go = gr.Button("TRANSLATE", variant="primary", elem_id="go")
            with gr.Column(scale=2):
                gallery = gr.Gallery(label="Hasil", columns=2, height=560)
                with gr.Row():
                    rar_out = gr.File(label="Unduh semua (RAR)")
                    zip_out = gr.File(label="Unduh semua (ZIP)")
        table = gr.Markdown()
        with gr.Accordion("Ringkasan JSON", open=False):
            raw = gr.Code(language="json")
        # Tertutup secara default: log mentah panjang dan tidak boleh mendorong
        # galeri keluar layar saat semuanya berjalan normal. Banner di atas tabel
        # yang memberi tahu kapan accordion ini perlu dibuka.
        with gr.Accordion("Log lengkap (buka kalau ada banner merah)", open=False):
            log_box = gr.Code(label="Keluaran mentah run terakhir")
            log_file = gr.File(label=f"Unduh {LOG_NAME}")

        go.click(
            _run,
            inputs=[files, provider, api_key, lang, style, balloon_budget,
                    debug, font_file, out_format],
            outputs=[gallery, rar_out, zip_out, table, raw, log_box, log_file],
        )
        probe_btn.click(_probe_table, inputs=[provider, api_key], outputs=[table])
    return demo


def launch(share: bool = True, debug: bool = False):
    """Di Colab share=True wajib: share=False + queueing -> ValueError."""
    OUTPUT.mkdir(parents=True, exist_ok=True)
    SETTINGS.debug = debug
    demo = build().queue()
    try:
        return demo.launch(share=share, css=CSS, debug=False, show_error=True)
    except (ValueError, RuntimeError) as exc:
        # frpc sering 403; jatuh ke mode lokal supaya notebook tidak mati.
        print(f"[app] share gagal ({exc}); coba tanpa share")
        return demo.launch(share=False, css=CSS, show_error=True)



Writing /content/mangatl/app.py


In [18]:
# Sel 20 — unduh weight + font. Idempoten, aman dijalankan ulang.
import importlib, assets, typeset

importlib.reload(assets); importlib.reload(typeset)

print("Weights:")
status = assets.download_weights()
print()
print("Fonts:")
font = typeset.setup_fonts()
print()
for name, ok in status.items():
    print(f"  {'OK  ' if ok else 'GAGAL'} {name}")
print(f"\nFont dipakai: {font}")
if not status.get("detector.onnx"):
    print("\n[!] detector.onnx gagal diunduh — deteksi region tidak akan jalan.")



Weights:
  detector.onnx: 100%  (160 MB)
  comictextdetector.pt.onnx: 100%  (90 MB)
  lama_large_512px.ckpt: 100%  (195 MB)

Fonts:
[font] pakai AnimeAce -> anime_ace.ttf

  OK   detector.onnx
  OK   comictextdetector.pt.onnx
  OK   lama_large_512px.ckpt

Font dipakai: /content/work/fonts/anime_ace.ttf


In [19]:
# Sel 21 — PROBE API. Berdiri sendiri dengan sengaja: ini satu-satunya bagian
# yang bergantung pada layanan pihak ketiga, dan hasilnya harus terlihat
# SEBELUM GPU dipakai. Kalau penyedianya menolak, kamu tahu di detik ini juga.
#
# Satu panggilan uji saja, dan sengaja SEKECIL mungkin (~6 token keluaran):
# token faucet terbatas dan jatahnya untuk menerjemahkan, bukan untuk probe.
# Harganya tetap sepadan — gagal di sini 3 detik, gagal setelah 20 halaman
# OCR+inpaint jauh lebih mahal.
import translate as tl
from config import PROVIDER_DEFAULT, SETTINGS

# Penyedia yang diprobe = yang jadi default UI, supaya yang diuji memang yang
# nanti dipakai. Ganti di UI kalau mau penyedia lain; sel ini tidak mengikat.
SETTINGS.provider = PROVIDER_DEFAULT
print('penyedia:', PROVIDER_DEFAULT)

MODEL, client = None, None
try:
    key = tl.get_api_key()
    client = tl.make_client(key, PROVIDER_DEFAULT)
    MODEL, _ = tl.pick_model(client)
    if tl._is_router(PROVIDER_DEFAULT):
        p = tl.probe_model(client, MODEL, True)
        print('uji  :', 'OK' if p.ok else 'GAGAL', f'{p.latency:.1f}s', p.reason)
        print('balas:', p.sample)
        if not p.ok:
            print()
            print('[!] Penyedia tidak menjawab. Pilih penyedia lain di dropdown UI.')
            print('    Pipeline tetap jalan: halaman keluar bersih, teks asli')
            print('    tersimpan di sidecar JSON, tidak ada yang hilang.')
    else:
        print('kuota:', tl.check_usage(client))
except RuntimeError as exc:
    print('[!]', exc)
    print('Pipeline tetap jalan — halaman keluar bersih, teks asli di sidecar JSON.')

[!] API key tidak ditemukan. Isi Colab Secrets 'DEEPL_API_KEY' atau tempel di field API Key pada UI.
Pipeline tetap jalan — halaman keluar bersih, teks asli di sidecar JSON.


In [20]:
# Sel 22 — warm-up + GERBANG FULL CUDA.
#
# Muat keempat model sekali supaya halaman pertama tidak lambat, lalu
# VERIFIKASI tiap stage benar-benar jalan di GPU. Fallback CPU itu diam-diam
# dan mahal (detector/CTD ~100 detik per halaman), jadi kalau GPU tersedia
# tapi ada stage yang jatuh ke CPU, sel ini BERHENTI dengan pesan yang bisa
# ditindaklanjuti — lebih baik gagal sekarang daripada user menunggu lama.
import gc, os, time

import onnxruntime as ort
import torch

import config, detect, inpaint, ocr, textmask

# T4 (Turing) tidak punya jalur TF32, jadi torch.set_float32_matmul_precision
# tidak berpengaruh. Yang berfaedah: cudnn.benchmark memilih kernel conv
# tercepat per ukuran input (LaMa menelusuri banyak tile berukuran beda).
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

t0 = time.time()
print("ort      :", ort.__version__, "| device:", ort.get_device())
print("provider :", ", ".join(ort.get_available_providers()))
print("torch    :", torch.__version__, "| cuda:", torch.version.cuda,
      "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu      :", torch.cuda.get_device_name(0))
print()

report: dict[str, str] = {}

try:
    detect.get_session()
    report["detector (ONNX)"] = config.ORT_REPORT.get("detector", "?")
except (FileNotFoundError, RuntimeError) as exc:
    report["detector (ONNX)"] = f"GAGAL: {exc}"

if textmask.get_ctd() is None:
    report["ctd (ONNX)"] = "TIDAK DIMUAT (mask pakai jalur Otsu)"
else:
    report["ctd (ONNX)"] = config.ORT_REPORT.get("ctd", "?")

dev = "cuda" if torch.cuda.is_available() else "cpu"
_lama = inpaint.get_model(dev)
if _lama is None:
    report["lama (torch)"] = "GAGAL DIMUAT (fallback cv2.inpaint)"
else:
    report["lama (torch)"] = str(next(_lama.parameters()).device)

_ocr = ocr.get_ocr()
if _ocr is None:
    report["manga-ocr (torch)"] = "GAGAL DIMUAT (semua region UNREADABLE)"
else:
    try:
        report["manga-ocr (torch)"] = str(next(_ocr.model.parameters()).device)
    except (AttributeError, StopIteration):
        report["manga-ocr (torch)"] = "dimuat (device tak terbaca)"

print("stage            device")
print("-" * 34)
for stage, devi in report.items():
    print(f"{stage:<17} {devi}")

print()
used = dict(config.ORT_REPORT)
for tag, prov in used.items():
    print(f"onnx[{tag}]".ljust(15), prov)
print("probe    :", os.environ.get("MANGATL_ORT_CUDA", "(sel 5 belum jalan)"))

gc.collect()
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM     : {free / 1e9:.2f} GB bebas / {total / 1e9:.1f} GB total")
print(f"Warm-up  : {time.time() - t0:.1f}s")

# ---- Gerbang FULL CUDA: stage neural di CPU padahal CUDA ada -> berhenti. ----
def _on_cpu(stage: str, devi: str) -> bool:
    if devi == "cpu":
        return True
    if "ExecutionProvider" in devi:
        return devi != "CUDAExecutionProvider"
    return False

cpu_stages = [s for s, d in report.items() if _on_cpu(s, d)]
if cpu_stages:
    raise RuntimeError(
        "TIDAK FULL CUDA — stage ini jalan di CPU: " + ", ".join(cpu_stages)
        + ". Perbaiki berurutan: (1) jalankan ulang sel 3; (2) Runtime -> "
        "Restart session; (3) mulai lagi dari sel 5 — sel itu menguji beberapa "
        "versi ORT di subproses secara otomatis dan mencetak alasan persisnya."
    )
if not used:
    print("[!] tidak ada sesi ONNX terbentuk — cek unduhan weight di sel 20.")



ort      : 1.22.0 | device: GPU
provider : TensorrtExecutionProvider, CUDAExecutionProvider, CPUExecutionProvider
torch    : 2.11.0+cu128 | cuda: 12.8 | available: True
gpu      : Tesla T4



2026-08-11 12:17:49.860 | INFO     | manga_ocr.ocr:__init__:16 - Loading OCR model from kha-white/manga-ocr-base


preprocessor_config.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/24.1k [00:00<?, ?B/s]

...

2026-08-11 12:17:57.582 | INFO     | manga_ocr.ocr:__init__:24 - Using CUDA


model.safetensors: reconstructing file:   0%|          |  0.00B /  444MB            

model.safetensors: downloading bytes:           |  0.00B            

2026-08-11 12:18:00.680 | INFO     | manga_ocr.ocr:__init__:37 - OCR ready


stage            device
----------------------------------
detector (ONNX)   CUDAExecutionProvider
ctd (ONNX)        CUDAExecutionProvider
lama (torch)      cuda:0
manga-ocr (torch) cuda:0

onnx[detector]  CUDAExecutionProvider
onnx[ctd]       CUDAExecutionProvider
probe    : ok
VRAM     : 14.41 GB bebas / 15.6 GB total
Warm-up  : 25.6s


In [21]:
# Sel 23 — self-test tanpa input user. Notebook menggambar halaman ujinya
# sendiri: dua balon, satu kotak narasi, satu SFX di luar balon.
import importlib, selftest

importlib.reload(selftest)
ok = selftest.run()
print("\n" + ("SEMUA CEK LULUS — pipeline siap." if ok
               else "ADA CEK GAGAL — periksa keluaran di atas."))



  [PASS] mask terbentuk di semua region
  [PASS] estimasi font size masuk akal (8..120 px)  [65.5, 70.5, 38.0, 11.0]
  [PASS] SFX tidak tersentuh mask hapus
  [PASS] mask SFX tidak kosong
  [PASS] piksel SFX identik sebelum/sesudah erase
  [PASS] tidak ada residu piksel di region dialog  [0, 0, 0]
  [PASS] teks hasil fit dirender di semua bubble
  [PASS] tidak ada overflow
  [PASS] SFX tidak ditimpa teks Inggris

SEMUA CEK LULUS — pipeline siap.


In [22]:
# Sel 24 — jalankan UI. Link share hidup 72 jam.
import app

app.launch(share=True)



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4046f8d3b16da5c42f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [23]:
# Sel 25 (opsional) — debug satu halaman, tampilkan tahapan berdampingan.
# Isi PAGE dengan path gambar, lalu jalankan.
PAGE = ""  # contoh: "/content/halaman.jpg"

if PAGE:
    from pathlib import Path
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import imgio, pipeline, typeset
    from config import DEBUG_DIR

    if not typeset.FONT_USED:
        typeset.setup_fonts(verbose=False)
    img = imgio.load_any(PAGE)
    res = pipeline.process_page(
        img, Path(PAGE).stem,
        globals().get("client"), globals().get("MODEL") or "",
        debug=True,
    )

    stages = ["01_input", "05_mask", "07_mask_after_sfx_exclusion",
              "09_cleaned", "10_typeset"]
    d = DEBUG_DIR / res.stem
    avail = [(s, d / f"{s}.png") for s in stages if (d / f"{s}.png").exists()]
    fig, axes = plt.subplots(1, len(avail), figsize=(5 * len(avail), 9))
    for ax, (name, p) in zip(np.atleast_1d(axes), avail):
        ax.imshow(plt.imread(p), cmap="gray")
        ax.set_title(name, fontsize=9)
        ax.axis("off")
    plt.tight_layout(); plt.show()

    df = pd.DataFrame(res.report["regions"])
    display(df[["idx", "label", "src_text", "translation", "route",
                "est_font_size", "final_font_size", "overflowed"]])
    print({k: v for k, v in res.report.items() if k != "regions"})
else:
    print("Isi PAGE dengan path gambar lebih dulu.")



Isi PAGE dengan path gambar lebih dulu.


In [ ]:
# Sel 26 (opsional) — AUDIT KEBERSIHAN BALON + status terjemah. NOL TOKEN:
# tidak memanggil LLM sama sekali (client=None), jadi aman dijalankan berkali-kali.
#
# Kebersihan diukur pada 09_cleaned (SEBELUM teks Inggris ditulis), karena
# sesudah typeset tinta baru tidak bisa dibedakan dari sisa tinta lama.
#
# Temuan dibatasi ke INTERIOR balon. Alasannya terukur (17 Agu 2026): ink_mask
# didilatasi 2 px supaya ekor antialias stroke ikut terperiksa, tapi pada balon
# kecil dilatasi itu menembus GARIS TEPI balon dan tinta panel sebelah — dan
# keduanya memang TIDAK BOLEH dihapus. Di jp_13 ketiga "komponen" yang dilaporkan
# semuanya berpiksel input == cleaned dan berada di luar mask hapus, yakni tepi
# balon. Sisa tinta Jepang selalu DI DALAM balon; tepi balon selalu di luar
# interiornya. Tanpa pembatasan ini audit menghukum garis yang wajib dijaga.
AUDIT_PAGE = ""  # kosong = pakai hasil Sel 25 (variabel `res`)

import cv2
import numpy as np

import imgio
import pipeline
import typeset


def _region_ink(r, shape, grow=2):
    """ink_mask region di kanvas halaman, didilatasi `grow` px."""
    h, w = shape
    out = np.zeros((h, w), np.uint8)
    if r.ink_mask is None:
        return out
    x1, y1, x2, y2 = r.bbox
    mh, mw = r.ink_mask.shape[:2]
    y2, x2 = min(y2, y1 + mh, h), min(x2, x1 + mw, w)
    if y2 <= y1 or x2 <= x1:
        return out
    out[y1:y2, x1:x2] = r.ink_mask[: y2 - y1, : x2 - x1]
    if grow:
        el = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (grow * 2 + 1,) * 2)
        out = cv2.dilate(out, el, iterations=1)
    return out


def _bubble_map(r, shape):
    """Interior balon region di kanvas halaman — mask yang sama dipakai typeset."""
    (bx1, by1, _, _), mask = typeset._region_box_mask(r)
    h, w = shape
    out = np.zeros((h, w), np.uint8)
    mh, mw = mask.shape[:2]
    by2, bx2 = min(by1 + mh, h), min(bx1 + mw, w)
    if by2 > by1 and bx2 > bx1:
        out[by1:by2, bx1:bx2] = mask[: by2 - by1, : bx2 - bx1]
    return out
def audit_clean(cleaned, regions, dev_thr=16):
    """Kebersihan area bekas teks pada halaman yang sudah dihapus.

    dev_thr 16 lebih ketat dari SETTINGS.residue_deviation (20) dengan sengaja:
    yang dituntut bukan "lolos gerbang pipeline" tapi "tidak ada titik sekecil
    apa pun", jadi ambangnya diturunkan sampai mendekati derau JPEG (~8-12 pada
    balon putih) tanpa menjadikan derau itu sendiri sebagai temuan.

    Komponen panjang-kurus (sisi panjang >= 12 px atau rasio >= 3) dilaporkan
    terpisah sebagai kandidat GARIS, karena coretan garis lebih mengganggu
    daripada titik dan penanganannya berbeda.
    """
    h, w = cleaned.shape[:2]
    gray = cv2.cvtColor(cleaned, cv2.COLOR_RGB2GRAY)
    per_region, dots, lines = [], [], []
    for r in regions:
        if r.is_protected or r.ink_mask is None:
            continue
        ink = _region_ink(r, (h, w)) > 0
        if not ink.any():
            continue
        bub = _bubble_map(r, (h, w)) > 0
        # Latar = interior balon DI LUAR bekas teks. Kalau balonnya tidak
        # dikenali, pakai bbox region; median tetap wakil yang jujur karena
        # sebagian besar piksel di sana memang latar.
        ref = bub & ~ink
        if ref.sum() < 50:
            x1, y1, x2, y2 = r.bbox
            box = np.zeros((h, w), bool)
            box[y1:y2, x1:x2] = True
            ref = box & ~ink
        bg = float(np.median(gray[ref])) if ref.any() else 255.0
        bad = (np.abs(gray.astype(np.int16) - bg) > dev_thr) & ink
        if bub.any():
            bad &= bub          # <-- pembatasan INTERIOR; lihat komentar di atas sel
        n, _, stats, _ = cv2.connectedComponentsWithStats(
            bad.astype(np.uint8), connectivity=8)
        comps = []
        for i in range(1, n):
            x, y, cw, ch, area = stats[i]
            long_side, short_side = max(cw, ch), max(min(cw, ch), 1)
            kind = ("garis" if (long_side >= 12 or long_side / short_side >= 3.0)
                    else "titik")
            comps.append({"kind": kind, "area": int(area),
                          "bbox": [int(x), int(y), int(x + cw), int(y + ch)]})
            (lines if kind == "garis" else dots).append({"idx": r.idx, **comps[-1]})
        per_region.append({
            "idx": r.idx, "bg": round(bg, 1), "bad_px": int(bad.sum()),
            "components": len(comps),
            "max_area": max((c["area"] for c in comps), default=0)})
    return {
        "dev_threshold": dev_thr,
        "dirty_px_total": sum(p["bad_px"] for p in per_region),
        "components_total": sum(p["components"] for p in per_region),
        "dots": sorted(dots, key=lambda d: -d["area"])[:20],
        "lines": sorted(lines, key=lambda d: -d["area"])[:20],
        "per_region": per_region,
    }
_res = globals().get("res")
_no_llm = False
if AUDIT_PAGE:
    from pathlib import Path as _P

    if not typeset.FONT_USED:
        typeset.setup_fonts(verbose=False)
    # client=None -> seluruh jalur terjemah dilewati process_page, jadi audit ini
    # tidak memakai satu token pun dan boleh diulang sesering perlu. Konsekuensinya
    # SEMUA balon tercatat "belum diterjemah" — itu bawaan mode ini, bukan cacat,
    # jadi bagian status terjemah di bawah sengaja tidak dicetak.
    _no_llm = True
    _res = pipeline.process_page(imgio.load_any(AUDIT_PAGE),
                                 "audit_" + _P(AUDIT_PAGE).stem, None, "",
                                 debug=True)

if _res is None:
    print("Jalankan Sel 25 lebih dulu, atau isi AUDIT_PAGE dengan path gambar.")
else:
    for _thr in (16, 20):
        a = audit_clean(_res.cleaned, _res.regions, _thr)
        print(f"=== {_res.stem} — kebersihan balon, ambang {_thr} ===")
        print(f"  piksel kotor total : {a['dirty_px_total']}")
        print(f"  komponen total     : {a['components_total']}")
        print(f"  kandidat GARIS     : {len(a['lines'])} {a['lines'][:6]}")
        print(f"  kandidat TITIK     : {len(a['dots'])} {a['dots'][:6]}")
        for p in a["per_region"]:
            print(f"    r{p['idx']:<3} bg={p['bg']:<6} kotor={p['bad_px']:<5} "
                  f"komponen={p['components']:<3} terbesar={p['max_area']}")
        if a["components_total"] == 0:
            print("  -> BERSIH: tidak ada titik maupun garis di dalam balon.")

    rep = _res.report or {}
    print(f"\n  region={rep.get('region_count')} bubble={rep.get('bubble_count')} "
          f"residu={rep.get('residue_count')} idx={rep.get('residue_idx')}")
    if _no_llm:
        print("  (status terjemah tidak diukur: mode audit tanpa LLM)")
    else:
        print(f"  diterjemah={rep.get('translated_count')}/"
              f"{rep.get('translatable_count')} "
              f"belum={rep.get('untranslated_count')} "
              f"{rep.get('untranslated_idx')}")
        # Balon yang MASIH berbahasa Jepang dicetak eksplisit: inilah yang dulu
        # lolos tanpa terlihat (mis. えっ！？ pada jp_13) karena laporan hanya
        # memberi angka, bukan teksnya.
        for r in _res.regions:
            if (r.label not in ("SFX", "UNREADABLE") and r.src_text
                    and not r.translation):
                print(f"    BELUM DITERJEMAH r{r.idx} {r.label} {r.src_text!r}")